In [ ]:
# ---------- inline code bundle (written verbatim; no dataset) ----------
import os, pathlib
ROOT = '/kaggle/working/code'
p = pathlib.Path(ROOT, 'prism/__init__.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""PRISM: Jacobian-based analysis of transformer models.\n\nPublic API. Import from ``prism`` rather than submodules when possible.\nBackward-compatible: ``from prism.jacobian import ...`` continues to work.\n"""\n\nfrom .bridge import jacobian_gap\nfrom .esg import (\n    compute_esg,\n    compute_esg_per_sample,\n    cross_positional_gramian,\n)\nfrom .jacobian import (\n    compute_jacobian_for_tokens,\n    compute_hidden_jacobian,\n    compute_hidden_jacobian_rows,\n    softmax_jacobian,\n    uniform_centered,\n    bridge_rows_backward,\n    h_norm_at_perturbed_h,\n)\nfrom .calibration import (\n    load_calibration_data,\n    extract_embeddings,\n    uniform_layer_mapping,\n    get_layer_types,\n    build_global_token_set,\n)\n# NOTE (audit docs/changed_review.md, 2026-07-18): prism.calibration imports\n# `datasets` at module level, so `import prism` REQUIRES `datasets` to be\n# installed (pre-v6.0 __init__ was docstring-only and had no such dependency).\n# Deliberately NOT wrapped in try/except and NOT made a lazy import:\n# `load_calibration_data` is a core entry point of the public API, and a\n# silent absence would surface as a confusing NameError deep in a\n# long-running experiment. If `datasets` is missing, `import prism` fails\n# with a clear ModuleNotFoundError at install/import time. To use the rest\n# of the package without it: `pip install datasets` (it is small).\n\nfrom .diagnostics import (\n    run_layer_diagnostics,\n    run_identity_test,\n    run_heatmap,\n)\nfrom .layer_selection import select_pairs, classify_pair\nfrom .cache_io import find_targets_dir\nfrom .cache import (\n    SVDCache,\n    MetricCache,\n    bootstrap_ci,\n    bootstrap_ci_paired,\n    deterministic_mode,\n    iter_position_pairs,\n)\nfrom .spectral import (\n    top_r_svd,\n    spectrum_from_gramian,\n    eigengap,\n    eigengap_min,\n    entropy_effective_rank,\n    stable_rank,\n    participation_ratio,\n    snr_top1_vs_rest,\n    snr_e14,\n    snr_e14_parts,\n    gauge_share,\n    gauge_share_top1,\n)\nfrom .holonomy import (\n    procrustes_align,\n    holonomy_accumulate,\n    holonomy_spectrum,\n    geodesic_curvature,\n    eigenframe_basis,\n    holonomy_along_path,\n    holonomy_along_2d_loop,\n)\nfrom .jstruct import (\n    build_M_K,\n    decompose_w_vs_uhat,\n    v_pred_from_JT,\n)\nfrom .interventions import (\n    EditSpec,\n    make_base,\n    make_readout_pull,\n    make_random_ortho,\n    make_random_matched,\n    make_tail_only,\n    apply_edit,\n    apply_edit_and_cache,\n)\nfrom .noise import (\n    calibrate_noise_floor,\n    validate_signal,\n)\nfrom .permutation import (\n    permutation_max_statistic,\n    generate_null,\n    cluster_based_permutation,\n)\nfrom .topology import (\n    zigzag_persistence,\n    project_to_subspace,\n    project_to_pca_subspace,\n)\nfrom .pipeline import (\n    ExperimentSpec,\n    OutputArtifacts,\n    run_spec,\n    run_many,\n    validate_spec,\n    load_artifacts,\n    default_output_subdir,\n)\n\n__all__ = [\n    "jacobian_gap",\n    "compute_esg",\n    "compute_esg_per_sample",\n    "cross_positional_gramian",\n    "compute_jacobian_for_tokens",\n    "compute_hidden_jacobian",\n    "compute_hidden_jacobian_rows",\n    "softmax_jacobian",\n    "uniform_centered",\n    "bridge_rows_backward",\n    "h_norm_at_perturbed_h",\n    "load_calibration_data",\n    "extract_embeddings",\n    "uniform_layer_mapping",\n    "get_layer_types",\n    "build_global_token_set",\n    "run_layer_diagnostics",\n    "run_identity_test",\n    "run_heatmap",\n    "select_pairs",\n    "classify_pair",\n    "find_targets_dir",\n    "SVDCache",\n    "MetricCache",\n    "bootstrap_ci",\n    "bootstrap_ci_paired",\n    "deterministic_mode",\n    "iter_position_pairs",\n    "top_r_svd",\n    "spectrum_from_gramian",\n    "eigengap",\n    "eigengap_min",\n    "entropy_effective_rank",\n    "stable_rank",\n    "participation_ratio",\n    "snr_top1_vs_rest",\n    "snr_e14",\n    "snr_e14_parts",\n    "gauge_share",\n    "gauge_share_top1",\n    "procrustes_align",\n    "holonomy_accumulate",\n    "holonomy_spectrum",\n    "geodesic_curvature",\n    "eigenframe_basis",\n    "holonomy_along_path",\n    "holonomy_along_2d_loop",\n    "build_M_K",\n    "decompose_w_vs_uhat",\n    "v_pred_from_JT",\n    "EditSpec",\n    "make_base",\n    "make_readout_pull",\n    "make_random_ortho",\n    "make_random_matched",\n    "make_tail_only",\n    "apply_edit",\n    "apply_edit_and_cache",\n    "calibrate_noise_floor",\n    "validate_signal",\n    "permutation_max_statistic",\n    "generate_null",\n    "cluster_based_permutation",\n    "zigzag_persistence",\n    "project_to_subspace",\n    "project_to_pca_subspace",\n    "ExperimentSpec",\n    "OutputArtifacts",\n    "run_spec",\n    "run_many",\n    "validate_spec",\n    "load_artifacts",\n    "default_output_subdir",\n]\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/behavior.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Behavioral endpoints — GATED module (ТЗ E40_math_core §II.10, M11).\n\nGate: tier-0 (``E40_*.py`` carriers, and the kernel modules) must NOT import\nthis module — behavior lives with E-42/E-43/E-44 (interventional/behavioral\ntiers). A pytest (``tests/test_nulls_tag_gate.py::\ntest_behavior_gate_no_import_in_kernel_and_tier0``) enforces it: CI is\npytest-only, so a CI import-check is a test that greps sources.\n\nBoundary conditions baked into the API (ТЗ §III.9/§III.10):\n- ``b_mag`` is ``‖Δz‖²`` — the SQUARE, ``‖·‖₂`` (first power) is not offered;\n- ``b_lin`` is the general ``ε²‖Jv‖²``; ``ε²σ₁²`` is the corollary at ``v = v₁``\n  (asserted in tests), never a separate definition;\n- ``KL_lin`` carries the Fisher factor ``F = diag(p) − ppᵀ = P(p)``, not ``I``.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nfrom scipy.stats import spearmanr\n\n__all__ = ["b_mag", "b_lin", "b_res", "lq_coupling", "b_qual", "KL_lin"]\n\n\ndef b_mag(delta_z: np.ndarray) -> float:\n    """Magnitude endpoint ``‖Δz‖²`` (logit-space energy of the perturbation).\n\n    NOTE (E44 §5.1.1): ``Δz`` is ``J_{t→t+k} v`` with ``v`` chosen by the\n    experimenter, so ``ρ(g_F, b_mag)`` correlates two operator-derived objects\n    and needs BOTH nulls (operator + content-matched) before interpretation.\n    """\n    dz = np.asarray(delta_z, dtype=np.float64)\n    return float((dz ** 2).sum())\n\n\ndef b_lin(J: np.ndarray, eps: float, v: np.ndarray) -> float:\n    """Linear response ``ε² ‖J v‖²`` for a unit-norm direction ``v``.\n\n    ``v`` is normalized here; passing ``v = v₁`` (top right singular vector)\n    yields ``ε² σ₁²`` — the test asserts exactly that identity.\n    """\n    J = np.asarray(J, dtype=np.float64)\n    v = np.asarray(v, dtype=np.float64)\n    nv = float(np.linalg.norm(v))\n    if nv <= 0:\n        raise ValueError("v must be nonzero")\n    Jv = J @ (v / nv)\n    return float(eps) ** 2 * float((Jv ** 2).sum())\n\n\ndef b_res(b_mag_value: float, b_lin_value: float) -> float:\n    """Residual response ``b_mag − b_lin`` (signed).\n\n    Order (патч v2 §1.1): ``‖Δz‖² = ε²‖Jv‖² + ε³⟨Jv, H[v,v]⟩ + O(ε⁴)``, so\n    ``b_res = O(ε³)`` — NOT ``O(ε⁴)``: the cubic term is the first nonzero\n    remainder unless ``⟨Jv, H[v,v]⟩ = 0`` (a checkable condition, not an\n    identity). Reduction control therefore tests ``b_res/(ε²‖Jv‖²) → 0``\n    (``o(ε²)``), never an ``O(ε⁴)`` rate — an honest ε³ term must not be read\n    as a reduction bug. The coefficient is :func:`lq_coupling`.\n    """\n    return float(b_mag_value) - float(b_lin_value)\n\n\ndef lq_coupling(Jv: np.ndarray, Hvv: np.ndarray) -> float:\n    """Linear-quadratic coupling ``⟨Jv, H[v,v]⟩`` (signed) — the ε³ coefficient\n    of ``b_res`` and a mandatory artifact field for E-44 (патч v2 §1.1.2).\n\n    ``Jv = J v`` and ``Hvv = H[v,v]`` (Hessian bilinear form applied to ``v``),\n    both already carrying the same ``v``; ``v`` normalization is the caller\'s\n    convention (unit norm assumed). This is a measurable object, not noise.\n    """\n    a = np.asarray(Jv, dtype=np.float64).ravel()\n    b = np.asarray(Hvv, dtype=np.float64).ravel()\n    if a.shape != b.shape:\n        raise ValueError(f"lq_coupling: shape {a.shape} vs {b.shape}")\n    return float(a @ b)\n\n\ndef b_qual(logits_base: np.ndarray, logits_perturbed: np.ndarray) -> float:\n    """Qualitative endpoint: rank correlation of logits (corr_f family,\n    Pearson-on-ranks = Spearman). Interventional channel only — forbidden in\n    the natural channel (paper §6 item 21)."""\n    a = np.asarray(logits_base, dtype=np.float64).ravel()\n    b = np.asarray(logits_perturbed, dtype=np.float64).ravel()\n    if a.shape != b.shape:\n        raise ValueError(f"logit shapes differ {a.shape} vs {b.shape}")\n    return float(spearmanr(a, b).statistic)\n\n\ndef KL_lin(J: np.ndarray, p: np.ndarray, eps: float, v: np.ndarray) -> float:\n    """Second-order KL of the softmax under ``h → h + ε v``:\n    ``½ ε² (Jv)ᵀ P(p) (Jv)`` with the Fisher–Rao metric ``P(p)`` (ТЗ §III.10:\n    the factor is ``diag(p) − ppᵀ``, not ``I``)."""\n    J = np.asarray(J, dtype=np.float64)\n    p = np.asarray(p, dtype=np.float64)\n    v = np.asarray(v, dtype=np.float64)\n    if p.ndim != 1 or J.shape[-2] != p.size:\n        raise ValueError(f"p [K] must match J token axis {J.shape}")\n    if abs(p.sum() - 1.0) > 1e-8:\n        raise ValueError("KL_lin: p must be normalized (Σp = 1)")\n    nv = float(np.linalg.norm(v))\n    if nv <= 0:\n        raise ValueError("KL_lin: v must be non-zero (same guard as b_lin; "\n                         "a zero v used to return NaN silently)")\n    Jv = J @ (v / nv)\n    # xᵀPx = Σ_i p_i x_i² − (Σ_i p_i x_i)² = Var_p(x) — closed form, P never materialized\n    return 0.5 * float(eps) ** 2 * (float(np.dot(p, Jv ** 2)) - float(np.dot(p, Jv)) ** 2)\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/bridge.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Jacobian Bridge: projection through vocabulary space + loss computation."""\n\nimport hashlib\nimport os\n\nimport torch\n\n_gramian_cache: dict[str, torch.Tensor] = {}\n_GRAMIAN_DIR = os.path.join("results", ".gramians")\n\n\ndef jacobian_gap(\n    J_L: torch.Tensor,\n    J_S: torch.Tensor,\n    E_L: torch.Tensor,\n    E_S: torch.Tensor,\n    probs_L: torch.Tensor,\n    threshold: float = 0.0,\n) -> float:\n    """Compute the Jacobian gap (alignment loss, LOWER = BETTER) between large and small models.\n\n    Terminology (v4.0): gap = weighted (1 - cos), lower=better; bridge = weighted\n    cos, higher=better; gap = 1 - bridge at equal weights. Callers reporting\n    "bridge" (E-16a/b) invert: bridge = 1 - jacobian_gap(...). The legacy\n    `jacobian_bridge` alias (which silently returned the GAP direction despite\n    its name) was removed after being identified as an inversion trap — the\n    inversion lives in the experiment kernels (kaggle_kernel/kernel-exp1*/...py),\n    not here.\n\n    The bridge chain: J_L @ E_L^T @ E_S @ (E_S^T @ E_S)^{-1}.\n    The inverse Gramian (E_S^T @ E_S)^{-1} (disk-cached, see _inverse_gramian)\n    corrects for non-orthogonal embeddings, so identity (J_L == J_S,\n    E_L == E_S) yields gap ~0.\n\n    threshold: row mask on BOTH norms, (n1 > threshold) & (n2 > threshold).\n    Default 0.0 = sentinel-only mode: only exactly-zero rows are masked\n    (no decay filtering). Historical default was 1e-6; the paper\'s\n    relative quantile tau_l = 0.01 * median_i ||J_S[i]|| (E-6 calibration,\n    ~1e-4 at L18) can be passed explicitly to enable decay masking.\n\n    Epsilon convention (docs/audit4.pdf M3): paper §2.4 writes the cosine\n    denominator as ``‖J‖·‖J_target‖ + ε``; the code uses ``F.cosine_similarity``\n    (divide by each norm with ``clamp_min=1e-12``). On live rows the two differ\n    by ``≲1e-9`` relative; on near-zero rows the additive form is WORSE — it\n    turns the cosine into an arbitrary small number, while clamping keeps the\n    direction. The formula-vs-code gap is therefore documented, not closed:\n    closing it would move published bridge numbers for no semantic gain.\n    """\n    G_inv = _inverse_gramian(E_S)\n    J_L_proj = _project_to_vocab(J_L, E_L)\n    J_S_target = _project_to_small(J_L_proj, E_S, G_inv)\n    mask = _valid_mask(J_S, J_S_target, threshold)\n\n    if mask.sum() == 0:\n        return 1.0\n\n    return _weighted_cosine_loss(J_S, J_S_target, probs_L, mask)\n\n\ndef _project_to_vocab(J_L: torch.Tensor, E_L: torch.Tensor) -> torch.Tensor:\n    """Project J_L [K, D_L] through E_L^T -> [K, V]."""\n    return J_L.float() @ E_L.float().T\n\n\ndef _project_to_small(\n    J_L_proj: torch.Tensor, E_S: torch.Tensor, G_inv: torch.Tensor\n) -> torch.Tensor:\n    """Project [K, V] through E_S @ G_inv -> [K, D_S].\n\n    G_inv = (E_S^T @ E_S)^{-1} corrects non-orthogonal embeddings.\n    """\n    return J_L_proj @ E_S.float() @ G_inv\n\n\ndef _inverse_gramian(E: torch.Tensor) -> torch.Tensor:\n    """Compute (E^T @ E)^{-1}, cached on disk per embedding hash."""\n    key = _gramian_key(E)\n    if key not in _gramian_cache:\n        path = os.path.join(_GRAMIAN_DIR, f"{key}.pt")\n        if os.path.exists(path):\n            _gramian_cache[key] = torch.load(path, weights_only=True, map_location="cpu")\n        else:\n            _gramian_cache[key] = _compute_inverse_gramian(E)\n            os.makedirs(_GRAMIAN_DIR, exist_ok=True)\n            torch.save(_gramian_cache[key], path)\n    return _gramian_cache[key].to(E.device)\n\n\ndef _gramian_key(E: torch.Tensor) -> str:\n    """Stable hash of an embedding matrix for disk caching.\n\n    Hashes shape/device header + the first row, the last row, and up to 9\n    deterministic mid-rows (uniformly spaced, seeded from row count so the\n    sample is stable across runs but covers the interior). First/last rows\n    alone would collide for matrices that share boundary rows (e.g. PAD/UNK)\n    but differ internally. Hashing the full 248k×1024 matrix would be\n    prohibitively expensive.\n\n    KNOWN SLOP, kept on purpose (audit4 Q4): rows 0 and V−1 are fed to the\n    digest twice (once explicitly, once through ``idx``), ``device`` is part of\n    the key so a CPU and a GPU run get separate files, and the torch version is\n    NOT in the key (a LAPACK change could alter the pseudoinverse in the last\n    bits). None of these can cause a false cache HIT — they cost bytes, not\n    correctness — and changing any of them invalidates the whole on-disk\n    ``results/.gramians`` cache (a full (EᵀE)⁻¹ recompute for a 248k×1024\n    embedding), so the key is frozen until something actually needs it.\n    """\n    V = E.shape[0]\n    idx = [0, V - 1]\n    if V > 4:\n        step = (V - 1) // 10\n        idx += sorted({(i * step) % (V - 1) + 1 for i in range(1, 10)})\n    header = f"{tuple(E.shape)}|{str(E.device)}".encode()\n    data = header + E[0].cpu().numpy().tobytes() + E[-1].cpu().numpy().tobytes()\n    for i in sorted(set(idx)):\n        data += E[i].cpu().numpy().tobytes()\n    h = hashlib.sha256(data).hexdigest()[:16]\n    return f"g_{h}"\n\n\ndef _compute_inverse_gramian(E: torch.Tensor) -> torch.Tensor:\n    """Compute (E^T @ E)^{-1} via pseudoinverse for numerical stability."""\n    G = E.float().T @ E.float()\n    return torch.linalg.pinv(G)\n\n\ndef _valid_mask(\n    J_S: torch.Tensor, J_target: torch.Tensor, threshold: float\n) -> torch.Tensor:\n    """Mask rows where either norm <= threshold (sentinel rows).\n\n    threshold=0.0 (default): only exactly-zero rows are masked.\n    threshold>0: decay masking (absolute threshold; the paper\'s relative\n    tau_l = 0.01*median is applied by the caller, E-6 calibration).\n    """\n    n1 = J_S.float().norm(dim=1)\n    n2 = J_target.float().norm(dim=1)\n    return (n1 > threshold) & (n2 > threshold)\n\n\ndef _weighted_cosine_loss(\n    J_S: torch.Tensor,\n    J_target: torch.Tensor,\n    probs: torch.Tensor,\n    mask: torch.Tensor,\n) -> float:\n    """The reported bridge metric (LOWER = WORSE alignment): `Σ_k w_k(1−cos_k)`,\n    `w = probs/Σprobs` over the valid rows. This is THE SAME functional that\n    `jacobian.bridge_objective` optimizes — one formula for objective and metric\n    (audit4 M1), so ""gap improved"" means the trained loss improved.\n    """\n    from .jacobian import bridge_objective      # local: keep bridge import-light\n\n    cos_sim = torch.nn.functional.cosine_similarity(\n        J_S.float()[mask], J_target[mask], dim=1\n    )\n    return bridge_objective(cos_sim, probs[mask] / probs[mask].sum()).item()\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/cache.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Caching infrastructure for the new experiment series (docs/new_code_1.md).\n\nTwo disk caches + bootstrap CI + determinism + position-pair iterators.\n\nDivision of labor with ``cache_io.py``: ``cache_io.find_targets_dir`` is\nread-only path resolution for EXISTING producer caches (Kaggle FUSE\nmounts); ``cache.py`` is read+WRITE with keys, atomic writes and\nversioning. ``find_targets_dir`` is reused (imported), not duplicated.\n\nConcurrency: ONE process per ``MetricCache`` (metric, edit, layer) triple\n— atomic ``os.replace`` prevents torn files, not last-writer-lost across\nthreads/processes. No parallel writes are supported.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport uuid\nimport warnings\nfrom collections import OrderedDict\nfrom contextlib import contextmanager\nfrom typing import Callable, Iterable, Sequence\n\nimport numpy as np\nimport torch\n\nfrom .cache_io import find_targets_dir\n\n__all__ = [\n    "SVDCache",\n    "MetricCache",\n    "bootstrap_ci",\n    "bootstrap_ci_paired",\n    "deterministic_mode",\n    "iter_position_pairs",\n]\n\n_LAYER_AGG_KEY = "__layer_aggregate__"\n\n\ndef _atomic_write(path: str, write_fn: Callable[[str], None]) -> None:\n    """Write via tmp file + os.replace (no torn files on crash)."""\n    tmp = f"{path}.tmp.{os.getpid()}.{uuid.uuid4().hex[:8]}"\n    try:\n        write_fn(tmp)\n        os.replace(tmp, path)\n    except BaseException:\n        try:\n            if os.path.exists(tmp):\n                os.remove(tmp)\n        except OSError:\n            pass\n        raise\n\n\ndef _ensure_root(root: str) -> None:\n    """Create the cache root if absent; reuse an existing producer cache.\n\n    ``find_targets_dir`` (cache_io, read-only listdir walk) is consulted\n    first so an existing Kaggle FUSE layout is picked up; if nothing is\n    found the root is created locally. A read-only FUSE root passes the\n    isdir check — writes there fail at ``put`` with a RuntimeError.\n    """\n    if os.path.isdir(root):\n        return\n    try:\n        find_targets_dir(root)  # raises if no *.pt under root / mounts\n    except FileNotFoundError:\n        os.makedirs(root, exist_ok=True)\n\n\ndef _is_writable(path: str) -> bool:\n    try:\n        probe = os.path.join(path, f".probe.{uuid.uuid4().hex[:8]}")\n        with open(probe, "w"):\n            pass\n        os.remove(probe)\n        return True\n    except OSError:\n        return False\n\n\n# ---------------------------------------------------------------------------\n# SVD cache\n# ---------------------------------------------------------------------------\n\n\nclass SVDCache:\n    """Disk cache of truncated SVD ``(U, S, V)`` keyed by\n    ``(edit, layer, doc, pos)`` (plus ``r``, ``dtype``, ``version``).\n\n    File format: ``torch.save({"U", "S", "V", "meta"})``, loaded with\n    ``weights_only=True``. Writes are atomic (tmp + ``os.replace``).\n    The cache is device-independent (CPU files); move results explicitly.\n    """\n\n    def __init__(\n        self,\n        root: str,\n        r: int,\n        dtype: torch.dtype = torch.float32,\n        version: str = "v1",\n        cache_size: int = 256,\n    ):\n        if not isinstance(r, int) or r < 1:\n            raise ValueError(f"SVDCache: r must be a positive int, got {r!r}")\n        self.root = root\n        self.r = r\n        self.dtype = dtype\n        self.version = version\n        self._lru: OrderedDict[str, tuple[torch.Tensor, torch.Tensor, torch.Tensor]] = (\n            OrderedDict()\n        )\n        self._cache_size = cache_size\n        _ensure_root(root)\n\n    # -- keys --------------------------------------------------------------\n\n    def key(self, edit: str, layer: int, doc: int, pos: int) -> str:\n        """SHA-256 of the request identifier (used for the in-memory LRU)."""\n        s = f"{edit}|L{layer}|d{doc}|p{pos}|r{self.r}|{self.dtype}|v{self.version}"\n        return hashlib.sha256(s.encode()).hexdigest()\n\n    def _path(self, edit: str, layer: int, doc: int, pos: int) -> str:\n        # dtype is part of the path: TZ §2.3 lists it in the key, so two\n        # caches with the same (r, version) but different dtypes must not\n        # share a file (review new_code_1_review.md P2.4 — TZ typo resolved\n        # in favor of the key).\n        return os.path.join(\n            self.root, edit, f"L{layer:02d}",\n            f"d{doc:04d}_p{pos:04d}_r{self.r}_{self.dtype}_{self.version}.pt",\n        )\n\n    # -- API ---------------------------------------------------------------\n\n    def has(self, edit: str, layer: int, doc: int, pos: int) -> bool:\n        return os.path.isfile(self._path(edit, layer, doc, pos))\n\n    def get(\n        self, edit: str, layer: int, doc: int, pos: int\n    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor] | None:\n        """Load (U, S, V) from memory LRU or disk.\n\n        Returns None if the file for THIS (r, dtype, version) does not\n        exist. A corrupted file raises RuntimeError (never silent None —\n        that would hide cache corruption). A torch-version mismatch in the\n        meta is a warning only (the format is compatible).\n        """\n        k = self.key(edit, layer, doc, pos)\n        if k in self._lru:\n            self._lru.move_to_end(k)\n            return self._lru[k]\n        path = self._path(edit, layer, doc, pos)\n        if not os.path.isfile(path):\n            return None\n        try:\n            payload = torch.load(path, weights_only=True)\n            U, S, V = payload["U"], payload["S"], payload["V"]\n            meta = payload["meta"]\n        except Exception as e:  # corrupted / unexpected file\n            raise RuntimeError(f"SVDCache: failed to load {path}: {e}") from e\n        if meta.get("r") != self.r or meta.get("version") != self.version:\n            return None  # key mismatch (different cache)\n        if meta.get("dtype") != str(self.dtype):\n            return None\n        if meta.get("torch") not in (None, torch.__version__):\n            warnings.warn(\n                f"SVDCache: {path} was written with torch {meta.get(\'torch\')}, "\n                f"current is {torch.__version__} (format-compatible, proceeding)",\n                RuntimeWarning,\n            )\n        U, S, V = U.to(self.dtype), S.to(self.dtype), V.to(self.dtype)\n        self._lru[k] = (U, S, V)\n        if len(self._lru) > self._cache_size:\n            self._lru.popitem(last=False)\n        return U, S, V\n\n    def put(\n        self,\n        edit: str,\n        layer: int,\n        doc: int,\n        pos: int,\n        U: torch.Tensor,\n        S: torch.Tensor,\n        V: torch.Tensor,\n    ) -> None:\n        """Store (U, S, V) with full validation; atomic on-disk write."""\n        if U.dtype != self.dtype or S.dtype != self.dtype or V.dtype != self.dtype:\n            raise ValueError(\n                f"SVDCache.put: dtype mismatch — expected {self.dtype}, got "\n                f"U={U.dtype}, S={S.dtype}, V={V.dtype}"\n            )\n        if U.dim() != 2:\n            raise ValueError(f"SVDCache.put: U must be 2-D, got {U.dim()}-D")\n        K, r = U.shape\n        if S.shape != (r,):\n            raise ValueError(f"SVDCache.put: S shape {tuple(S.shape)} != ({r},)")\n        if V.shape != (r, U.shape[1]):\n            raise ValueError(f"SVDCache.put: V shape {tuple(V.shape)} != ({r}, {U.shape[1]})")\n        if r > min(K, V.shape[1]):\n            raise ValueError(f"SVDCache.put: r={r} > min(K, D)={min(K, V.shape[1])}")\n        if r > 1 and not torch.all(S[:-1] >= S[1:]):\n            raise ValueError("SVDCache.put: S is not descending")\n        if not torch.allclose(U.T @ U, torch.eye(r, device=U.device, dtype=U.dtype), atol=1e-5):\n            raise ValueError("SVDCache.put: U is not orthonormal (‖UᵀU − I‖_F ≥ 1e-5)")\n        if not torch.allclose(V @ V.T, torch.eye(r, device=V.device, dtype=V.dtype), atol=1e-5):\n            raise ValueError("SVDCache.put: V is not orthonormal (‖VVᵀ − I‖_F ≥ 1e-5)")\n\n        path = self._path(edit, layer, doc, pos)\n        if not _is_writable(self.root):\n            raise RuntimeError(\n                f"SVDCache.put: root is not writable (read-only Kaggle FUSE "\n                f"mount?): {self.root}"\n            )\n        meta = {\n            "edit": edit, "layer": layer, "doc": doc, "pos": pos,\n            "r": r, "dtype": str(self.dtype), "version": self.version,\n            "torch": torch.__version__,\n        }\n        payload = {"U": U.cpu(), "S": S.cpu(), "V": V.cpu(), "meta": meta}\n\n        def _save(tmp: str) -> None:\n            os.makedirs(os.path.dirname(tmp), exist_ok=True)\n            torch.save(payload, tmp)\n\n        _atomic_write(path, _save)\n        k = self.key(edit, layer, doc, pos)\n        self._lru[k] = (U, S, V)\n        self._lru.move_to_end(k)\n        if len(self._lru) > self._cache_size:\n            self._lru.popitem(last=False)\n\n\n# ---------------------------------------------------------------------------\n# Scalar metric cache\n# ---------------------------------------------------------------------------\n\n\ndef _json_dump_value(v: float) -> object:\n    if math.isnan(v):\n        return "nan"\n    if math.isinf(v):\n        return "inf" if v > 0 else "-inf"\n    return v\n\n\ndef _json_load_value(v: object) -> float:\n    if isinstance(v, str):\n        if v == "nan":\n            return float("nan")\n        if v == "inf":\n            return float("inf")\n        if v == "-inf":\n            return float("-inf")\n    return float(v)\n\n\nclass MetricCache:\n    """Disk cache of scalar metrics keyed by (metric, edit, layer, doc).\n\n    JSON files ``{root}/{metric}/{version}/{edit}/L{layer:02d}.json``\n    (human-readable, git-mergeable; scalars are small — .pt would be\n    overengineering). ``nan``/``inf`` are stored as the strings\n    "nan"/"inf"/"-inf". Document keys are zero-padded (``doc_{N:04d}``) so\n    that git diffs stay stable. ``doc=None`` addresses the per-layer\n    aggregate (key ``__layer_aggregate__``), which bulk_get/bulk_put do\n    NOT include. No in-memory cache (files are small; a cache would\n    complicate consistency under parallel writes).\n\n    Version semantics: a change of ``version`` produces a NEW file\n    (``{version}/`` in the path); old files remain untouched. A given\n    root is therefore monoversion by construction.\n    """\n\n    def __init__(self, root: str, version: str = "v1"):\n        self.root = root\n        self.version = version\n        _ensure_root(root)\n\n    def _path(self, metric: str, edit: str, layer: int) -> str:\n        # version is part of the path: TZ §2.3 says "version in the key;\n        # version change -> new file" (review new_code_1_review.md P2.4 —\n        # TZ typo resolved in favor of the key: without this, a version\n        # change would mix v1/v2 documents under a single meta). A root\n        # must therefore be used by exactly one version; see class docstring.\n        return os.path.join(self.root, metric, self.version, edit, f"L{layer:02d}.json")\n\n    def _load(self, metric: str, edit: str, layer: int) -> dict:\n        path = self._path(metric, edit, layer)\n        if not os.path.isfile(path):\n            return {}\n        try:\n            with open(path) as f:\n                data = json.load(f)\n        except (json.JSONDecodeError, OSError) as e:\n            raise RuntimeError(f"MetricCache: failed to load {path}: {e}") from e\n        return data\n\n    def _save(self, metric: str, edit: str, layer: int, data: dict) -> None:\n        path = self._path(metric, edit, layer)\n        if not _is_writable(self.root):\n            raise RuntimeError(\n                f"MetricCache.put: root is not writable (read-only Kaggle FUSE "\n                f"mount?): {self.root}"\n            )\n        data = {**data, "meta": {"version": self.version}}\n\n        def _w(tmp: str) -> None:\n            os.makedirs(os.path.dirname(tmp), exist_ok=True)\n            with open(tmp, "w") as f:\n                json.dump(data, f, indent=1, sort_keys=True)\n\n        _atomic_write(path, _w)\n\n    @staticmethod\n    def _check_value(value: object) -> float:\n        # Python float only on purpose: np.float64 passes (it subclasses\n        # float in CPython) but np.float32 does NOT — a silent-precision-loss\n        # guard against JSON serialization. Call .item() explicitly.\n        if not isinstance(value, float):\n            raise ValueError(\n                f"MetricCache.put: value must be a Python float "\n                f"(got {type(value).__name__}); call .item() explicitly"\n            )\n        return value\n\n    def has(self, metric: str, edit: str, layer: int, doc: int | None = None) -> bool:\n        data = self._load(metric, edit, layer)\n        key = "__layer_aggregate__" if doc is None else f"doc_{int(doc):04d}"\n        return key in data\n\n    def get(self, metric: str, edit: str, layer: int, doc: int | None = None) -> float | None:\n        data = self._load(metric, edit, layer)\n        key = "__layer_aggregate__" if doc is None else f"doc_{int(doc):04d}"\n        if key not in data:\n            return None\n        return _json_load_value(data[key])\n\n    def put(\n        self,\n        metric: str,\n        edit: str,\n        layer: int,\n        doc: int | None,\n        value: float,\n    ) -> None:\n        value = self._check_value(value)\n        key = "__layer_aggregate__" if doc is None else f"doc_{int(doc):04d}"\n        data = self._load(metric, edit, layer)\n        data[key] = _json_dump_value(value)\n        self._save(metric, edit, layer, data)\n\n    def bulk_get(self, metric: str, edit: str, layer: int) -> dict[int, float]:\n        """{doc: value} for all present documents (aggregate excluded)."""\n        data = self._load(metric, edit, layer)\n        return {\n            int(k[4:]): _json_load_value(v)\n            for k, v in data.items()\n            if k.startswith("doc_")\n        }\n\n    def bulk_put(\n        self,\n        metric: str,\n        edit: str,\n        layer: int,\n        values: dict[int, float],\n    ) -> None:\n        data = self._load(metric, edit, layer)\n        for doc, value in values.items():\n            data[f"doc_{int(doc):04d}"] = _json_dump_value(self._check_value(value))\n        self._save(metric, edit, layer, data)\n\n\n# ---------------------------------------------------------------------------\n# Bootstrap CI\n# ---------------------------------------------------------------------------\n\n\ndef bootstrap_ci(\n    values: Sequence[float],\n    B: int = 1000,\n    alpha: float = 0.05,\n    seed: int = 0,\n    statistic: Callable[[np.ndarray], float] = np.median,\n) -> tuple[float, float]:\n    """Percentile bootstrap CI of a statistic (default: median) over values.\n\n    Resampling WITH replacement, ``len(values)`` draws per iteration,\n    ``np.random.default_rng(seed)`` (deterministic). Paper §13: B=1000,\n    percentile CI. NaN in values → ValueError (silently dropping them\n    would hide bugs). For heavy-tailed metrics median (default) is the\n    robust choice; ``mean`` is available via the ``statistic`` argument.\n\n    I.i.d. assumption: resampling treats ``values`` as independent draws.\n    Do NOT pass a flattened (doc x position) array without checking the\n    within-doc correlation (positions are correlated, E-31: C_F(1) up to\n    0.996) — the CI would be ~sqrt(n_positions) times too narrow. Use\n    per-doc statistics + ``bootstrap_ci_paired`` for such comparisons.\n    """\n    arr = np.asarray(values, dtype=np.float64)\n    if arr.ndim != 1:\n        raise ValueError(f"bootstrap_ci: expected 1-D values, got {arr.ndim}-D")\n    n = arr.size\n    if n < 2:\n        raise ValueError(f"bootstrap_ci: need >= 2 values, got {n}")\n    if not np.all(np.isfinite(arr)):\n        raise ValueError("bootstrap_ci: NaN/Inf in values")\n    if B < 100:\n        raise ValueError(f"bootstrap_ci: B must be >= 100, got {B}")\n    if not (0 < alpha < 1):\n        raise ValueError(f"bootstrap_ci: alpha must be in (0, 1), got {alpha}")\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, n, size=(B, n))\n    samples = arr[idx]\n    stats = np.array([statistic(s) for s in samples])\n    lo, hi = np.percentile(stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])\n    return float(lo), float(hi)\n\n\ndef bootstrap_ci_paired(\n    values_a: Sequence[float],\n    values_b: Sequence[float],\n    B: int = 1000,\n    alpha: float = 0.05,\n    seed: int = 0,\n) -> tuple[float, float]:\n    """Paired bootstrap CI for ``median(a) − median(b)`` on the SAME docs.\n\n    Pair indices are resampled together, so the CI of the difference is\n    (correctly) narrower than the one from two independent bootstraps —\n    use it for base-vs-edit comparisons on identical document sets\n    (ε, T2\').\n    """\n    a = np.asarray(values_a, dtype=np.float64)\n    b = np.asarray(values_b, dtype=np.float64)\n    if a.shape != b.shape:\n        raise ValueError(\n            f"bootstrap_ci_paired: length mismatch {a.size} != {b.size}"\n        )\n    n = a.size\n    if n < 2:\n        raise ValueError(f"bootstrap_ci_paired: need >= 2 pairs, got {n}")\n    if not (np.all(np.isfinite(a)) and np.all(np.isfinite(b))):\n        raise ValueError("bootstrap_ci_paired: NaN/Inf in values")\n    if B < 100:\n        raise ValueError(f"bootstrap_ci_paired: B must be >= 100, got {B}")\n    if not (0 < alpha < 1):\n        raise ValueError(f"bootstrap_ci_paired: alpha must be in (0, 1), got {alpha}")\n    rng = np.random.default_rng(seed)\n    idx = rng.integers(0, n, size=(B, n))\n    diffs = np.median(a[idx], axis=1) - np.median(b[idx], axis=1)\n    lo, hi = np.percentile(diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])\n    return float(lo), float(hi)\n\n\n# ---------------------------------------------------------------------------\n# Determinism\n# ---------------------------------------------------------------------------\n\n\n@contextmanager\ndef deterministic_mode(seed: int = 0, strict: bool = False):\n    """Seed torch/numpy/random; restore ALL global state on exit.\n\n    Nesting is supported: each level restores the state captured at ITS\n    entry. ``strict=True`` additionally enables\n    ``torch.use_deterministic_algorithms(True)`` + cudnn.deterministic —\n    caveat: some GDN kernels have no deterministic implementation and will\n    raise RuntimeError in strict mode (use strict only for non-GDN paths).\n    """\n    torch_state = torch.random.get_rng_state()\n    cuda_states = (\n        torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None\n    )\n    np_state = np.random.get_state()\n    random_state = random.getstate()\n    torch_det = torch.are_deterministic_algorithms_enabled()\n    cudnn_det = torch.backends.cudnn.deterministic\n    if strict and torch_det:\n        warnings.warn("deterministic_mode: strict=True but deterministic algorithms already enabled (redundant)", RuntimeWarning)\n\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    random.seed(seed)\n    if strict:\n        torch.use_deterministic_algorithms(True)\n        torch.backends.cudnn.deterministic = True\n    try:\n        yield\n    finally:\n        if strict:\n            torch.use_deterministic_algorithms(torch_det)\n            torch.backends.cudnn.deterministic = cudnn_det\n        random.setstate(random_state)  # seed(tuple) is invalid — setstate restores\n        np.random.set_state(np_state)\n        torch.random.set_rng_state(torch_state)\n        if cuda_states is not None:\n            torch.cuda.set_rng_state_all(cuda_states)\n\n\n# ---------------------------------------------------------------------------\n# Position pairs\n# ---------------------------------------------------------------------------\n\n\ndef iter_position_pairs(\n    S: int, mode: str = "adjacent", stride: int = 1\n) -> Iterable[tuple[int, int]]:\n    """Pairs (t, t\') with 0 ≤ t < t\' < S, in a fixed order.\n\n    mode:\n      "adjacent": (t, t+stride) for t in 0, stride, 2·stride, … (< S−stride)\n      "strided":  (t, t+stride) for t = 0, 1, …, S−stride−1\n      "all":      every (t, t\') with t < t\' — O(S²); for S=128 that is\n                  8128 pairs, expensive; prefer adjacent/strided.\n\n    Used by ε (adjacent), T2\' (adjacent baseline), T1\' (strided).\n    """\n    if not isinstance(S, int) or S < 2:\n        raise ValueError(f"iter_position_pairs: need S >= 2, got {S}")\n    if not isinstance(stride, int) or stride < 1:\n        raise ValueError(f"iter_position_pairs: stride must be >= 1, got {stride}")\n    if mode == "adjacent":\n        return ((t, t + stride) for t in range(0, S - stride, stride))\n    if mode == "strided":\n        return ((t, t + stride) for t in range(0, S - stride))\n    if mode == "all":\n        return ((t, tp) for t in range(S) for tp in range(t + 1, S))\n    raise ValueError(f"iter_position_pairs: unknown mode {mode!r} (adjacent/strided/all)")\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/cache_io.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Shared target-cache resolution (Kaggle kernel_sources mount).\n\nKaggle FUSE mount: os.path.isdir() lies on /kaggle/input (returns False for\nexisting dirs) but os.listdir() works. So every check here is listdir-based,\nand the walk is layout-independent: it finds a dir containing *.pt at any\ndepth under any mounted slug — no hardcoded producer layout assumed.\n"""\n\nimport os\n\n# Max depth under input_root to search (producer puts cache at depth 2:\n# results/targets; 4 covers any reasonable layout).\n_MAX_DEPTH = 4\n\n\ndef _has_pt(d: str) -> bool:\n    try:\n        return any(f.endswith(".pt") for f in os.listdir(d))\n    except OSError:\n        return False\n\n\ndef _walk(root: str, depth: int):\n    """Yield dirs under root (listdir-based), depth-limited."""\n    try:\n        entries = sorted(os.listdir(root))\n    except OSError:\n        return\n    for e in entries:\n        p = os.path.join(root, e)\n        if _has_pt(p):\n            yield p\n        if depth > 1:\n            yield from _walk(p, depth - 1)\n\n\ndef find_targets_dir(preferred: str, input_root: str = "/kaggle/input") -> str:\n    """Return the dir holding the target-cache *.pt files.\n\n    Order: `preferred` (local path / runner-injected), then a listdir walk\n    of every mount under `input_root`. Raises FileNotFoundError with the\n    mount inventory when nothing is found (no silent fallback, no listdir crash).\n    """\n    if _has_pt(preferred):\n        return preferred\n    for hit in _walk(input_root, _MAX_DEPTH):\n        return hit\n    mounts = []\n    try:\n        for d in sorted(os.listdir(input_root)):\n            try:\n                mounts.append(f"{d}/:{os.listdir(os.path.join(input_root, d))[:4]}")\n            except OSError:\n                mounts.append(f"{d}/:<unreadable>")\n    except OSError:\n        mounts.append("<no input mount>")\n    raise FileNotFoundError(\n        f"No target cache .pt in {preferred} nor under {input_root}/ "\n        f"(mounts: {mounts}). Run Stage 0 first — it must publish results/targets/*.pt."\n    )\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/calibration.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Calibration: data loading, tokenization, global token set, layer mapping."""\n\nimport hashlib\nimport json\nimport os\nimport re\nimport warnings\nfrom collections import Counter\nfrom pathlib import Path\n\nimport torch\nfrom datasets import load_dataset\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\n# --- закрепление пула (PLAN «Запрос 11»: все результаты воспроизводимы) -------------------\n# Порядок streaming-потока на Hub ничем не закреплён, поэтому «первые подходящие строки»\n# меняются от прогона к прогону: E-0.1 при идентичных params дал другой top-token набор\n# (pk_hash 5c12c174… -> 30d82ade…) и сдвинул C_b(sigma_d) до x31. Пин = сырые тексты принятых\n# строк + отпечаток; при наличии пина поток не читается вообще, значит прогон воспроизводим\n# и офлайн. Тексты (не тензоры): input_ids по-прежнему производит токенизатор.\nPIN_DIR_ENV = "PRISM_CALIB_PIN_DIR"\nPIN_WRITE_ENV = "PRISM_CALIB_PIN_WRITE"      # =1: разрешить записать новый пин\nPIN_REQUIRE_ENV = "PRISM_CALIB_REQUIRE_PIN"  # =1: отсутствие пина = громкий отказ\nLAST_POOL_FPS: dict[str, str] = {}           # domain -> отпечаток последнего загруженного пула\n\n\ndef pool_fingerprint(texts: list[str]) -> str:\n    """Отпечаток пула по сырым текстам (от токенизатора не зависит)."""\n    h = hashlib.sha256()\n    for t in texts:\n        h.update(t.encode("utf-8"))\n        h.update(b"\\x00")\n    return h.hexdigest()[:16]\n\n\ndef _default_pin_dir() -> Path | None:\n    """Каталог пинов по умолчанию, или None если искать негде.\n\n    В flattening-Kaggle-ядре модулей нет на диске и `__file__` не определён (упавший exp3:\n    `NameError: name \'__file__\' is not defined` прямо в `_pin_path`) — значит путь репозитория\n    строится только когда модуль реально лежит в репозитории; иначе пин ищется по `PIN_DIR_ENV`,\n    а без него — поток (или громкий отказ при `PIN_REQUIRE_ENV`).\n    """\n    try:\n        here = Path(__file__).resolve()\n    except NameError:\n        return None\n    return here.parent.parent / "data" / "calib_pins"\n\n\ndef _pin_path(domain, model_name, n_samples, min_length, max_length,\n              save_offset_mapping) -> Path | None:\n    root = os.environ.get(PIN_DIR_ENV)\n    base = Path(root) if root else _default_pin_dir()\n    if base is None:\n        return None\n    slug = re.sub(r"[^0-9A-Za-z.]+", "_", model_name)\n    mode = "offset" if save_offset_mapping else "plain"\n    return base / f"{domain}__{slug}__n{n_samples}__min{min_length}__max{max_length}__{mode}.json"\n\n\ndef uniform_layer_mapping(n_layers_large: int, n_layers_small: int) -> dict[int, int]:\n    """Uniform layer mapping from large to small model layers."""\n    mapping = {}\n    for i in range(n_layers_large):\n        j = round(i * (n_layers_small - 1) / (n_layers_large - 1))\n        mapping[i] = j\n    return mapping\n\n\ndef get_layer_types(model: AutoModelForCausalLM) -> list[str]:\n    """Get layer types from model config (full_attention / linear_attention)."""\n    cfg = model.config\n    return list(\n        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types\n    )\n\n\n_DOMAIN_DATASETS = {\n    "wiki": ("allenai/c4", "en", "train", "text"),\n    "math": ("nvidia/OpenMathInstruct-2", None, "train", "problem"),\n    "code": ("neyralabs/apex-agents", None, "train", "prompt"),\n    "chat": ("HuggingFaceH4/ultrachat_200k", None, "train_sft", "messages"),\n}\n\n\ndef _load_dataset_for_domain(\n    domain: str,\n) -> tuple[str, str | None, str, str]:\n    """Return (dataset_id, config, split, text_field) for a domain.\n\n    Falls back to wiki (C4) if the target dataset is unavailable, with a\n    RuntimeWarning: a fallback run silently poisons cross-domain comparisons\n    (E-13/E-27 T_domain), so the substitution must be visible in the log.\n    """\n    ds_id, cfg, split, field = _DOMAIN_DATASETS[domain]\n    try:\n        kw: dict[str, object] = {"split": split, "streaming": True}\n        if cfg is not None:\n            kw["name"] = cfg\n        ds = load_dataset(ds_id, **kw)  # type: ignore[call-overload]\n        next(iter(ds))  # force the first fetch: streaming is lazy, a broken/\n        # gated dataset would otherwise pass the check and fail inside\n        # load_calibration_data (loud crash instead of the intended fallback).\n        return ds_id, cfg, split, field\n    except Exception:\n        warnings.warn(\n            f"domain {domain} ({ds_id}) unavailable — falling back to wiki/C4. "\n            f"Domain labels in this run are INVALID for cross-domain "\n            f"comparisons (E-13/E-27).",\n            RuntimeWarning,\n            stacklevel=2,\n        )\n        return _DOMAIN_DATASETS["wiki"]\n\n\ndef _extract_text_from_sample(\n    sample: dict,\n    field: str,\n) -> str:\n    """Extract text from a dataset sample, handling different formats."""\n    if field == "messages":\n        # Chat format: list of {role, content}\n        texts = [m.get("content", "") for m in sample.get("messages", [])]\n        return " ".join(texts)\n    val = sample.get(field, "")\n    if isinstance(val, list):\n        # e.g. code_contests solutions: list of strings\n        return "\\n".join(str(v) for v in val)\n    if isinstance(val, dict):\n        return str(val)\n    return val\n\n\ndef load_calibration_data(\n    model_name: str,\n    max_length: int = 64,\n    min_length: int = 64,\n    n_samples: int = 4,\n    domain: str = "wiki",\n    save_offset_mapping: bool = False,\n) -> list[dict[str, torch.Tensor]]:\n    """Load and tokenize calibration data.\n\n    Defaults match the tested protocol (paper §12/C.6: S=64, filter >= 64,\n    N=4); experiments pass explicit lengths. domain: \'wiki\' (C4), \'math\'\n    (OpenMathInstruct-2), \'code\' (apex-agents), \'chat\' (ultrachat). Falls back\n    to wiki if the target dataset is unavailable. save_offset_mapping: adds\n    `offset_mapping` to each sample (E-31 sentence boundaries).\n    """\n    tokenizer = AutoTokenizer.from_pretrained(model_name)\n    pin = _pin_path(domain, model_name, n_samples, min_length, max_length, save_offset_mapping)\n    texts: list[str] | None = None\n    ds_id = cfg = split = field = None\n    if pin is not None and pin.exists():\n        meta = json.loads(pin.read_text(encoding="utf-8"))\n        texts = list(meta["texts"])\n        print(f"[calib] {domain}: пин {pin.name}, строк={len(texts)}, "\n              f"pool_fp={meta[\'fp\']}, поток не читается", flush=True)\n    elif os.environ.get(PIN_REQUIRE_ENV) == "1":\n        raise RuntimeError(\n            f"нет пина пула {pin or f\'каталог пинов недоступен — задайте {PIN_DIR_ENV}\'} при "\n            f"{PIN_REQUIRE_ENV}=1: прогон на потоке с Hub недопустим, "\n            f"иначе числа не воспроизводимы. Сгенерируйте пин с {PIN_WRITE_ENV}=1 один раз и "\n            f"закоммитьте его."\n        )\n    else:\n        ds_id, cfg, split, field = _load_dataset_for_domain(domain)\n        kw: dict[str, object] = {"split": split, "streaming": True}\n        if cfg is not None:\n            kw["name"] = cfg\n        dataset = load_dataset(ds_id, **kw)  # type: ignore[call-overload]\n    samples = []\n    # NOTE (audit4 Q9): the offset-mapping path tokenizes with\n    # add_special_tokens=False, the plain path with the tokenizer default (True).\n    # Deliberate, not a bug: offset_mapping must stay aligned 1:1 with the text\n    # span (special tokens carry no span), and the two paths feed different\n    # consumers. Consequence to keep in mind: a token set / calibration built\n    # through the offset path is NOT the same sequence as the mainline E-16\n    # calibration of the same text — do not mix their token sets.\n    accepted: list[str] = []\n    # Из пина идут уже извлечённые тексты (извлечение = _extract_text_from_sample на момент\n    # записи пина); с потока идут сырые строки.\n    rows = ((t, None) for t in texts) if texts is not None else ((None, raw) for raw in dataset)\n    for pinned_text, raw in rows:\n        text = pinned_text if pinned_text is not None else _extract_text_from_sample(raw, field)\n        if not text:\n            continue\n        if save_offset_mapping:\n            # Токенизатор вызывается на ОДНОЙ строке, поэтому input_ids — плоский список id, а\n            # offset_mapping — список пар; индексация `[0]` здесь была багом (len(int) падал на\n            # первом же вызове, найденном E-45b).\n            toks = tokenizer(\n                text, max_length=max_length, truncation=True,\n                add_special_tokens=False, return_offsets_mapping=True,\n            )\n            if len(toks["input_ids"]) >= min_length:\n                accepted.append(text)\n                samples.append({\n                    "input_ids": toks["input_ids"],\n                    "offset_mapping": torch.tensor(\n                        toks["offset_mapping"], dtype=torch.int32\n                    ),\n                })\n        else:\n            tokens = tokenizer(\n                text, max_length=max_length, truncation=True,\n                return_tensors="pt",\n            )\n            if tokens["input_ids"].shape[1] >= min_length:\n                accepted.append(text)\n                samples.append(tokens)\n        if len(samples) >= n_samples:\n            break\n\n    if texts is None:\n        texts = accepted\n        fp = pool_fingerprint(texts)\n        print(f"[calib] {domain}: поток с Hub, строк={len(texts)}, pool_fp={fp}", flush=True)\n        if os.environ.get(PIN_WRITE_ENV) == "1":\n            if pin is None:\n                raise RuntimeError(\n                    f"{PIN_WRITE_ENV}=1, но каталога пинов нет: {PIN_DIR_ENV} не задан, а "\n                    "`__file__` недоступен (flattening-ядро). Задайте PRISM_CALIB_PIN_DIR "\n                    "на путь, который попадает в вывод ядра."\n                )\n            pin.parent.mkdir(parents=True, exist_ok=True)\n            pin.write_text(\n                json.dumps(\n                    {\n                        "protocol": {"domain": domain, "model": model_name, "n_samples": n_samples,\n                                     "min_length": min_length, "max_length": max_length,\n                                     "offset_mapping": bool(save_offset_mapping)},\n                        "source": {"dataset": ds_id, "config": cfg, "split": split, "field": field},\n                        "fp": fp,\n                        "texts": texts,\n                    },\n                    ensure_ascii=False,\n                ),\n                encoding="utf-8",\n            )\n            print(f"[calib] {domain}: пин записан -> {pin}", flush=True)\n    else:\n        fp = pool_fingerprint(texts)\n        if len(samples) < n_samples:\n            raise RuntimeError(\n                f"пин {pin.name} дал {len(samples)} строк вместо {n_samples}: токенизатор или "\n                f"фильтр длины изменились относительно момента записи пина (pool_fp={fp}). "\n                f"Молча читать другой пул нельзя — числа стали бы невоспроизводимыми."\n            )\n    LAST_POOL_FPS[domain] = fp\n    return samples\n\n\ndef build_global_token_set(\n    model: AutoModelForCausalLM,\n    calibration_samples: list[dict[str, torch.Tensor]],\n    K: int = 64,\n) -> list[int]:\n    """Frequency token set: the K tokens that most often appear in the per-sample\n    top-K of the LAST position (`logits[:, -1]`) across the calibration pool.\n\n    One basis for every position, so a stored p-row is SUB-stochastic (~half of the\n    exp27b positions carry < 0.5 of the softmax mass) — that basis is what all\n    published Jacobian numbers sit on. The positional convention (each position gets\n    its own top-K, so p is a real truncation) is produced by\n    `experiments/exp27b_dense.py: column_mode="positional"` (cache exp27c), not here;\n    "union top-K at fixed K" is not definable, since the union of per-position top-K\n    exceeds K (audit4 Q10 verdict round). Never mix numbers from the two bases:\n    the basis belongs in the ChannelTag slice.\n    """\n    token_counter: Counter[int] = Counter()\n    model.eval()  # type: ignore[attr-defined]\n\n    device = next(model.parameters()).device  # type: ignore[attr-defined]\n    with torch.no_grad():\n        for sample in calibration_samples:\n            inputs = {\n                "input_ids": sample["input_ids"].to(device),\n                "attention_mask": sample["attention_mask"].to(device),\n            }\n            outputs = model(**inputs)  # type: ignore[operator]\n            logits = outputs.logits[:, -1, :]\n            top_idx = logits.topk(K, dim=-1).indices[0]\n            token_counter.update(top_idx.tolist())\n\n    U = [tid for tid, _ in token_counter.most_common(K)]\n    return sorted(U)\n\n\ndef load_models(\n    large_name: str, small_name: str, dtype: torch.dtype = torch.float32\n) -> tuple[torch.nn.Module, torch.nn.Module]:\n    """Load large and small models with eager attention.\n\n    dtype defaults to float32: the tested protocols run on CPU, where fp16\n    is ~10x slower (paper §12/C.9). Pass torch.float16 explicitly for GPU.\n    Thread count (torch.set_num_threads(4) in §12/C.9) is set by the caller\'s\n    setup, not here.\n    """\n    kwargs = {\n        "torch_dtype": dtype,\n        "attn_implementation": "eager",\n    }\n    large = AutoModelForCausalLM.from_pretrained(large_name, **kwargs)\n    large.eval()  # type: ignore[attr-defined]\n    small = AutoModelForCausalLM.from_pretrained(small_name, **kwargs)\n    small.eval()  # type: ignore[attr-defined]\n    return large, small\n\n\ndef extract_embeddings(\n    model: AutoModelForCausalLM,\n) -> torch.Tensor:\n    """Extract embedding matrix from model.\n\n    The bridge projects through the AN-embedding matrix W (§3/§5). For the\n    tied-weight checkpoints in scope, W = E (embed_tokens). If a checkpoint\n    is loaded WITHOUT tied weights, W = lm_head.weight ≠ E and the projection\n    would silently use the wrong matrix. We therefore warn ONCE per model if\n    lm_head.weight is not the same storage as embed_tokens.weight — a loud,\n    non-fatal signal that the W = E assumption does not hold for this model.\n    """\n    emb = model.model.embed_tokens.weight.detach()  # type: ignore[attr-defined]\n    _warn_if_not_tied(model, emb)\n    return emb\n\n\ndef _warn_if_not_tied(model: AutoModelForCausalLM, emb: torch.Tensor) -> None:\n    lm_head = getattr(getattr(model, "lm_head", None), "weight", None)\n    if lm_head is None:\n        return\n    if not lm_head.data_ptr() == emb.data_ptr():\n        # data_ptr identity covers tied weights; fall back to value equality\n        # for the (rare) case of a tied-but-copied view.\n        if not torch.equal(lm_head.detach(), emb):\n            warnings.warn(\n                f"extract_embeddings: lm_head.weight is NOT tied to "\n                f"embed_tokens.weight (model={type(model).__name__}). The "\n                f"bridge projects through E assuming W = E (§3); with untied "\n                f"weights the an-embedding matrix is lm_head.weight and the "\n                f"gap/bridge numbers are silently wrong. Extract lm_head "\n                f"explicitly if this is intended.",\n                UserWarning,\n                stacklevel=2,\n            )\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/cheap_invariants.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Cheap triple invariants on ``P(p)`` and ``J`` (ТЗ E40_math_core §II.8, M10).\n\nRe-exports the triple from :mod:`prism.operator_checks` (single code path —\ndo NOT reimplement, E6 was exactly a two-code-path problem) and adds the\nisotropic predictor of ``c``:\n\n- ``c_pred(p) = tr P / sqrt(K · tr P²)`` — the alignment coefficient of the\n  operator\'s own spectrum (1 iff P ∝ I); compares a measured ``c`` against\n  what the OPERATOR alone suggests, before reading anything off the data.\n- ``A = c / c_pred`` — data-vs-operator alignment ratio.\n\n``frac_K``: the ТЗ staged it ``blocked_on_E6``; E6 was closed the same day\n(the published table mixed slices; the JSON triple was self-consistent:\n``c/s = 2.4418e-3/0.018772 = 0.13008 = cos_J_PJ`` to 5 digits), so it is a\nnormal core member with one rule: numerator, denominator and the identity\nmust be quoted on ONE aggregation and ONE position slice (paper §6 item 23).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .operator_checks import c, cos_J_PJ, frac_K, s  # re-exports, one path\nfrom .p_algebra import trace_P, trace_P2\n\n__all__ = ["c", "s", "cos_J_PJ", "frac_K", "c_pred", "A_alignment"]\n\n\ndef c_pred(p: np.ndarray) -> float:\n    """Isotropic predictor ``tr P / sqrt(K · tr P²)`` ∈ (0, 1]."""\n    p = np.asarray(p, dtype=np.float64)\n    K = p.size\n    return trace_P(p) / float(np.sqrt(K * trace_P2(p)))\n\n\ndef A_alignment(J: np.ndarray, p: np.ndarray) -> float:\n    """``A = c / c_pred`` — measured cross-term over its operator-level value."""\n    return float(c(J, p) / c_pred(p))\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/diagnostics.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Diagnostics: per-layer bridge loss heatmaps and baselines."""\n\nimport json\nimport math\nimport os\nfrom typing import Any\n\nimport matplotlib\nimport torch\n\nfrom prism.bridge import jacobian_gap\nfrom prism.calibration import (\n    extract_embeddings,\n    uniform_layer_mapping,\n)\nfrom prism.jacobian import compute_jacobian_for_tokens\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\n\n\ndef run_layer_diagnostics(\n    model_large,\n    model_small,\n    token_set,\n    layer_mapping,\n    samples,\n    results_dir="results",\n    exp_name="exp2_heatmap",\n) -> dict[str, float]:\n    """Compute bridge loss for each layer pair and save results."""\n    E_L = extract_embeddings(model_large)\n    E_S = extract_embeddings(model_small)\n    results: dict[str, float] = {}\n\n    for l_L, l_S in layer_mapping.items():\n        losses = []\n        for sample in samples:\n            ids = sample["input_ids"].to(model_large.device)\n            mask = sample["attention_mask"].to(model_large.device)\n\n            J_L = compute_jacobian_for_tokens(model_large, l_L, ids, mask, token_set)\n            J_S = compute_jacobian_for_tokens(model_small, l_S, ids, mask, token_set)\n\n            with torch.no_grad():\n                logits_L = model_large(input_ids=ids).logits[:, -1, :]\n                # softmax over the token_set SUBSET (renormalized), not the full-\n                # vocab probs at those columns. Harmless numerically: weights enter\n                # _weighted_cosine_loss normalized (w/w.sum), so the subset\n                # renormalization cancels — but do not read `probs` as p(token).\n                probs = torch.softmax(logits_L[0, token_set], dim=-1)\n\n            # jacobian_gap returns the GAP (lower = better), NOT bridge = 1 − gap\n            # (bridge.py docstring v4.0; E-16a/b report bridge by inverting).\n            loss = jacobian_gap(J_L, J_S, E_L, E_S, probs)\n            if not math.isfinite(loss):\n                raise FloatingPointError(\n                    f"non-finite gap for layer pair {l_L}->{l_S} (sample "\n                    f"{len(losses)}) — refusing to average NaN into the JSON")\n            losses.append(loss)\n\n        avg_loss = sum(losses) / len(losses)\n        results[f"{l_L}->{l_S}"] = avg_loss\n\n    _save_results(results, exp_name, results_dir)\n    return results\n\n\ndef run_identity_test(\n    model,\n    token_set,\n    samples,\n    n_layers: int,\n    results_dir="results",\n) -> dict[str, float]:\n    """Identity test: same model as large and small."""\n    mapping = uniform_layer_mapping(n_layers, n_layers)\n    results = run_layer_diagnostics(\n        model, model, token_set, mapping, samples, results_dir, "exp1_identity"\n    )\n    return results\n\n\ndef run_heatmap(\n    model_large,\n    model_small,\n    token_set,\n    layer_mapping,\n    samples,\n    results_dir="results",\n) -> dict[str, float]:\n    """Run cross-model heatmap (large -> small)."""\n    results = run_layer_diagnostics(\n        model_large,\n        model_small,\n        token_set,\n        layer_mapping,\n        samples,\n        results_dir,\n        "exp2_heatmap",\n    )\n    _plot_heatmap(results, layer_mapping, results_dir)\n    return results\n\n\ndef _save_results(results: dict[str, float], exp_name: str, results_dir: str) -> None:\n    """Save results to JSON."""\n    # ponytail: fail-fast is intentional for experiments\n    os.makedirs(results_dir, exist_ok=True)\n    path = os.path.join(results_dir, f"{exp_name}.json")\n    with open(path, "w") as f:\n        json.dump(results, f, indent=2)\n\n\ndef _plot_heatmap(\n    results: dict[str, Any],\n    layer_mapping: dict[int, int],\n    results_dir: str,\n) -> None:\n    """Save heatmap as PNG."""\n    l_L_vals = sorted(layer_mapping.keys())\n    l_S_vals = [layer_mapping[li] for li in l_L_vals]\n    losses = [\n        results.get(f"{li}->{layer_mapping[li]}", float("nan")) for li in l_L_vals\n    ]\n\n    fig, ax = plt.subplots(figsize=(10, 6))\n    im = ax.imshow([losses], aspect="auto", cmap="YlOrRd")\n    ax.set_xticks(range(len(l_L_vals)))\n    ax.set_xticklabels(\n        [f"{li}->{si}" for li, si in zip(l_L_vals, l_S_vals, strict=True)]\n    )\n    ax.set_yticks([0])\n    ax.set_yticklabels(["gap (lower=better)"])\n    fig.colorbar(im, ax=ax)\n    ax.set_title("Jacobian gap per layer pair")\n    fig.tight_layout()\n    fig.savefig(os.path.join(results_dir, "exp2_heatmap.png"))\n    plt.close(fig)\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/esg.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Empirical Sensitivity Gramian (ESG) computation.\n\nPooling semantics (paper v4.0, §6 vs §15 E-0.1):\n- `compute_esg` pools Jacobians of ONE trajectory (positions of a single\n  document). Intra-sample coherence C_a is 8-41x the noise floor F (E-27.6),\n  so pooling positions of one sample is valid.\n- Cross-SAMPLE pooling of Jacobians is INVALID (E-0.1: C_b ≈ 0.003-0.05,\n  near-orthogonal): different samples almost-orthogonal J\'s average to noise.\n  Aggregate across samples ONLY at the scalar-metric level (lambda, PR, SNR,\n  r_eff), never at the matrix level. Use `compute_esg_per_sample`.\n\n**Cross-positional Gramian (`cross_positional_gramian`).** `Δ_U(t, t\') = J_t J_{t\'}^T`\n— token correlations between positions. For `t = t\'` it coincides with the\nper-position ESG. Used in `E-ε` (spectral Goodhart) and as the baseline for\n`E-T2\'` (holonomy). Do not confuse with `compute_esg`, which pools along a\ntrajectory.\n"""\n\nimport torch\n\n\ndef _validate(jacobians: list[torch.Tensor]) -> None:\n    if not jacobians:\n        raise ValueError("jacobians: empty list")\n    ref = jacobians[0]\n    for J in jacobians[1:]:\n        if J.shape != ref.shape:\n            raise ValueError(f"jacobians: shape mismatch {tuple(J.shape)} vs {tuple(ref.shape)}")\n        if J.device != ref.device:\n            raise ValueError(f"jacobians: device mismatch {J.device} vs {ref.device}")\n\n\ndef compute_esg(\n    jacobians: list[torch.Tensor],\n    n_components: int = 32,\n) -> tuple[torch.Tensor, torch.Tensor]:\n    """Compute ESG from a list of Jacobian matrices.\n\n    Pooling is INTRA-TRAJECTORY: all J_i must be positions of ONE sample\n    (document). Cross-sample pooling is invalid — see module docstring /\n    paper §15 (E-0.1); aggregate across samples at the scalar-metric level.\n\n    Args:\n        jacobians: list of N Jacobian matrices (one trajectory), each [K, D].\n        n_components: number of top eigenvalues/vectors to return.\n\n    Returns:\n        eigenvalues: [min(n_components, K)] sorted descending (K < n_components\n        possible in degenerate cases; in practice K = |𝒮| ≥ 32).\n        eigenvectors: [min(n_components, K), K] corresponding eigenvectors.\n    """\n    _validate(jacobians)\n    G = _build_gramian(jacobians)\n    eigenvalues, eigenvectors = _spectral_decomposition(G, n_components)\n    return eigenvalues, eigenvectors\n\n\ndef compute_esg_per_sample(\n    sample_jacobians: list[list[torch.Tensor]],\n    n_components: int = 32,\n) -> tuple[list[torch.Tensor], list[torch.Tensor]]:\n    """Sample-local ESG: one Gramian per sample, NO cross-sample pooling.\n\n    Args:\n        sample_jacobians: list over samples; each entry is a list of the\n            sample\'s per-position Jacobians [K, D].\n        n_components: number of top eigenvalues/vectors to return.\n\n    Returns:\n        eigenvalues: list [n_samples] of [n_components].\n        eigenvectors: list [n_samples] of [n_components, K].\n\n    Cross-sample aggregation happens on scalars (lambda, PR, SNR, r_eff),\n    never on the matrices (E-0.1: cross-sample J\'s are near-orthogonal).\n    """\n    if not sample_jacobians:\n        raise ValueError("sample_jacobians: empty list")\n    eigenvalues = []\n    eigenvectors = []\n    for Js in sample_jacobians:\n        ev, evec = compute_esg(Js, n_components=n_components)\n        eigenvalues.append(ev)\n        eigenvectors.append(evec)\n    return eigenvalues, eigenvectors\n\n\ndef cross_positional_gramian(\n    J_t: torch.Tensor,\n    J_tp: torch.Tensor,\n) -> torch.Tensor:\n    """Cross-positional Gramian Δ_U(t, t\') = J_t @ J_tp.T.\n\n    Args:\n        J_t:  [K, D] gauge-free Jacobian at position t.\n        J_tp: [K, D] gauge-free Jacobian at position t\'.\n\n    Returns:\n        [K, K] Gramian (symmetric iff t == t\').\n\n    Note:\n        - NOT a pooling operation: each call uses exactly two positions.\n          See module docstring for intra-trajectory pooling (compute_esg)\n          and cross-sample pooling validity (E-0.1).\n        - When t == t\', result equals the inner-product part of\n          compute_esg([J_t]) for N=1: compute_esg([J_t])[0][0] ==\n          eigenvalue_0(Δ_U(t, t)). Checked in selfcheck.\n        - Caller must ensure J_t, J_tp come from the same model / edit /\n          token_set; the function does not check semantic compatibility,\n          only shape / device / dtype (via _validate).\n    """\n    _validate([J_t, J_tp])\n    return J_t.float() @ J_tp.float().T\n\n\ndef _build_gramian(jacobians: list[torch.Tensor]) -> torch.Tensor:\n    """G = (1/N) * sum_i (J_i @ J_i^T)  (intra-trajectory pool)."""\n    U = jacobians[0].shape[0]\n    device = jacobians[0].device\n    G = torch.zeros(U, U, device=device, dtype=torch.float)\n\n    for J in jacobians:\n        G += J.float() @ J.float().T\n    G /= len(jacobians)\n    return G\n\n\ndef _spectral_decomposition(\n    G: torch.Tensor, n_components: int\n) -> tuple[torch.Tensor, torch.Tensor]:\n    """Eigen decomposition, sorted descending."""\n    eigenvalues, eigenvectors = torch.linalg.eigh(G)\n\n    # Sort descending\n    idx = eigenvalues.argsort(descending=True)\n    eigenvalues = eigenvalues[idx][:n_components]\n    eigenvectors = eigenvectors[:, idx][:, :n_components].T\n\n    return eigenvalues, eigenvectors\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/exp38_hi_common.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Shared math for E-38h/i: L23 readout-Jacobian factorization.\n\nJacobian of the FINAL LAYER readout at a response token t, input = L23\npre-final-norm hidden h_t (V15 contract), output = next-token logit scores:\n\n    J(t) = (sqrt(D)/r_t) * M_K (I - u u^T)          (E-35 lemma [M-1])\n\nwith r_t = ||h_t||_2,  u = h_t/r_t,  M_K = W_lm[U128] * gamma (gamma = 1+w,\nRMSNorm weight).  This is an EXACT identity for the last-layer readout\nJacobian for ANY W_lm:\n\n    J_norm(t) = (sqrt(D)/r_t) * diag(gamma) (I - u u^T),   J(t) = W_lm @ J_norm(t)\n\nE-38i uses this: a READOUT-only edit changes W (and hence J(s), M_K(s)) but\nleaves h_t, r_t, u, J_norm, sqrt(D)/r_t invariant  =>  the identity holds\nfor every edited W; the residual is structurally zero and the test measures\nthe numeric floor + the leaf drift (J rows, M_K rows).\n\nE-38h uses the two-channel (left = 256-token space / right = D=1024 hidden\nspace) SVD split of Delta_W across edit steps.\n\nData (local, deterministic — no model forward):\n  - snapshots: results/exp16_deltas/exp16_dW_snapshots.pt\n               {\'W_orig\' [256,1024] fp16, \'snapshots\' {step: [256,1024]},\n                \'token_set\' [256]}  (100 steps, 10..1000)\n  - U128 + H/JR: kaggle_kernel/results/exp27b_jcache_L23.pt\n  - model:    Qwen3.5-0.8B safetensors — tied embed_tokens rows (== W_lm)\n              and model.language_model.norm.weight (final RMSNorm gamma=1+w)\n"""\nfrom __future__ import annotations\n\nimport os\nimport torch\n\n# ---------- constants / paths ----------\nU128_PATH = "kaggle_kernel/results/exp27b_jcache_L23.pt"\nSNAPSHOTS = "results/exp16_deltas/exp16_dW_snapshots.pt"\nEPS = 1e-6  # Qwen3.5-0.8B config rms_norm_eps (verified)\nD = 1024\n\n# Pre-registered subsample of the 100-step grid (10..1000), bracketing the\n# E-16 knee (knee_step=84, knee_dose≈100, results/exp16_lm_head_knee_info.pt).\nK_STEPS = [10, 20, 40, 80, 90, 100, 200, 400, 800, 1000]\n\nREPO = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))\n\n\ndef _p(rel):\n    return os.path.join(REPO, rel)\n\n\n# ---------- data loading ----------\ndef load_u128():\n    """(U128 long tensor, L23 dense cache dict with H[(dom,si,t)] etc.)."""\n    ck = torch.load(_p(U128_PATH), map_location="cpu", weights_only=False)\n    U = torch.tensor(ck["U128"], dtype=torch.long)\n    return U, ck\n\n\ndef load_snapshots():\n    """snapshots[k] IS the ΔW = (W_curr − W_orig)[token_set] (NOT W_curr) —\n    consumer must NOT subtract raw["W_orig"] again (E38_fixes P0)."""\n    d = torch.load(_p(SNAPSHOTS), map_location="cpu", weights_only=True)\n    snaps = {int(k): v.to(torch.float64) for k, v in d["snapshots"].items()}\n    # sanity: catch a W_orig bookkeeping bug (E38_fixes P0 class). The OLD\n    # guard ‖s @ Wo.T‖_F/(‖s‖_F‖Wo‖_F) is a matrix-product norm, NOT a\n    # cosine: ≈ 1/√D ≈ 0.03 for correct ΔW AND ≈ 1/√256 ≈ 0.06 for the bug\n    # case — both << 0.5, it never fired (docs/E38_fixes.md §38h P1 / §38i\n    # P1). The SIGNED Frobenius cosine |⟨s, Wo⟩_F|/(‖s‖_F‖Wo‖_F) separates\n    # them: correct ΔW (measured 0.02–0.26 across the 100-step grid, max at\n    # high E-16 dose) vs s = +W_curr (cos ≈ +1, W_orig not subtracted) or\n    # s = 2·W_curr − W_orig ≈ +Wo (cos ≈ +1, double subtraction).\n    Wo = d["W_orig"].to(torch.float64)\n    wo_norm = float(Wo.norm())\n    for k, s in snaps.items():\n        inner = float((s * Wo).sum())\n        cos = abs(inner) / (float(s.norm()) * wo_norm)\n        assert cos < 0.5, (\n            f"snapshot step {k}: cos(snap, W_orig)={cos:.3f} — "\n            "snapshots[k] is ΔW (measured cos 0.02–0.26 across the grid); "\n            "a cos ≈ ±1 means the snapshot is W_curr (W_orig not "\n            "subtracted) or a double subtraction, not ΔW")\n    return snaps, d\n\n\ndef load_model_tensors():\n    """(embed, gamma): tied embed_tokens weight and final RMSNorm weight\n    (1+w). Verified: embed[U128] reproduces cached J23 via the identity."""\n    from safetensors import safe_open\n    import glob\n\n    root = os.path.expanduser(\n        "~/.cache/huggingface/hub/models--Qwen--Qwen3.5-0.8B/snapshots"\n    )\n    f = sorted(glob.glob(root + "/*/model*.safetensors"))[0]\n    with safe_open(f, framework="pt") as sf:\n        emb = sf.get_tensor("model.language_model.embed_tokens.weight")\n        gamma = sf.get_tensor("model.language_model.norm.weight")\n    return emb.to(torch.float64), gamma.to(torch.float64)\n\n\n# ---------- factorization ----------\ndef j_raw(W, h, gamma):\n    """Exact final-layer readout Jacobian for ONE position:\n    h [D] -> J [n_tokens, D].  J(t) = (sqrt(D)/r) * M_K (I - u u^T),\n    M_K = W*gamma, u = h/||h||.  VERIFIED: cos vs cached exp27b JR (fp32)\n    = 1.0000 (E-35)."""\n    r = h.norm()\n    u = h / r\n    M = W * gamma[None, :]\n    c = float(D) ** 0.5\n    return (c / r) * (M - torch.outer(M @ u, u))\n\n\ndef factor_residual(W, h, gamma):\n    """Independent check: dense  W @ J_norm(t)  vs closed form, ONE position.\n    J_norm(t) = (sqrt(D)/r) * diag(gamma)(I - u u^T).\n    Residual is structural zero for ANY W; measures numeric floor only."""\n    r = h.norm()\n    u = h / r\n    c = float(D) ** 0.5\n    I = torch.eye(W.shape[1], dtype=W.dtype)\n    P = I - torch.outer(u, u)\n    # gamma scales OUTPUT directions (rows of J_norm): J_norm = (sqrt(D)/r) diag(gamma)(I-uu^T)\n    Jnorm = (c / r) * (P * gamma[:, None])\n    Jd = W @ Jnorm\n    M = W * gamma[None, :]\n    Jc = (c / r) * (M - torch.outer(M @ u, u))\n    return float((Jd - Jc).norm() / Jc.norm())\n\n\n# ---------- SVD / subspace helpers (float64) ----------\ndef top_r_subspace(A, r):\n    """Column space of A\'s top-r right singular vectors (D x r, orthonormal)."""\n    _, _, Vt = torch.linalg.svd(A, full_matrices=False)\n    return Vt[:r].t().contiguous()\n\n\ndef top_r_subspaces(A, r):\n    """E38_paper D.5: returns the RIGHT singular vectors (V, row-space of A) —\n    NOT the left (U). Callers compare subspace overlaps in the V space."""\n    _, s, Vt = torch.linalg.svd(A, full_matrices=False)\n    return Vt[:r].t().contiguous(), s[:r]\n\n\ndef overlap(Q1, Q2, r):\n    """cos^2 of principal angles (mean) between two r-dim subspaces.\n\n    ``r`` is unused (the frames are already sliced to r columns by the caller);\n    the parameter is kept because the E-38h/i notebooks call it positionally\n    (audit4 Q17 — dead argument, documented rather than silently re-cut)."""\n    s = (Q1.t() @ Q2).abs().clamp(max=1.0)\n    return float(s.pow(2).mean())\n\n\ndef procrustes_gap(Q1, Q2):\n    """||Q1 - Q2 O||_F / r MINIMIZED over orthogonal O (frame alignment).\n\n    The minimizer is O* = polar(Q2ᵀ Q1). For Q1ᵀ Q2 = U Σ Vᵀ that is O* = V Uᵀ,\n    so the SVD must be taken of the REVERSED product (Q2ᵀ Q1 = V Σ Uᵀ) to let\n    ``O = U @ Vt`` be the optimum. SVD-ing Q1ᵀQ2 and taking U Vᵀ gives the\n    TRANSPOSE of the optimum, which is not orthogonal-equivalent to it: two\n    frames of the SAME subspace (Q2 = Q1 R, R orthogonal) have a true gap of 0\n    and a transposed gap > 0 (audit docs/audit4.pdf B5: 0.499 vs 0.405 on random\n    frames). No committed E-38h number used this function (no consumers)."""\n    U, _, Vt = torch.linalg.svd(Q2.t() @ Q1, full_matrices=False)\n    O = U @ Vt\n    return float((Q1 - Q2 @ O).norm() / Q1.shape[1])\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/holonomy.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Module 3: holonomy — pure geometric primitives (TZ part 2, docs/new_functions_2.md).\n\nProcrustes-based holonomy (κ = ‖R_total − I‖_F) in **subspace space**\n(Grassmannian Gr(r, K) for left U_r, Gr(r, D_S) for right V_r).\n\nNo forward passes, no model dependency: only tensor math over already-\ncomputed subspaces. Concrete paths (W₀→W₁₀₀→W₀) and loops (V₁,V₂ vs\nPC₁,PC₂) live in experiments/exp_T2.py — they use the primitives here.\n\nGauge invariance: replacing U_i with U_i @ Q_i (any orthogonal Q_i, a\nbasis change inside the subspace) leaves every κ unchanged. This is a\nfundamental property; violating it is a bug (selfcheck 3.6.15).\n\nRelated: paper.md §8.6, §8.7.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport warnings\nfrom typing import Callable, Sequence\n\nimport numpy as np\nimport torch\n\n__all__ = [\n    "procrustes_align",\n    "holonomy_accumulate",\n    "holonomy_spectrum",\n    "geodesic_curvature",\n    "eigenframe_basis",\n    "holonomy_along_path",\n    "holonomy_along_2d_loop",\n]\n\n\ndef _validate_orthonormal(U: torch.Tensor, tol: float) -> None:\n    """‖U^T U − I‖_F < tol, else ValueError.\n\n    Silent work on non-orthonormal input gives a wrong R (the SVD polar\n    factor solves the orthonormal problem); fail loudly.\n    """\n    dev = U.device\n    gram = U.T @ U\n    err = torch.norm(gram - torch.eye(gram.shape[0], device=dev, dtype=gram.dtype),\n                     p="fro")\n    if not torch.isfinite(err) or err.item() >= tol:\n        raise ValueError(\n            f"input not orthonormal: ‖U^T U − I‖_F = {err.item():.3e} (tol {tol:.0e})"\n        )\n\n\ndef _check_loop_basis(b1: torch.Tensor, b2: torch.Tensor, tol: float = 1e-4) -> None:\n    """‖b1‖ ≈ 1, ‖b2‖ ≈ 1, |b1·b2| < tol — else ValueError.\n\n    A non-orthogonal loop pair makes κ uninterpretable as holonomy\n    (the loop is not a coordinate chart on the sphere).\n    """\n    n1, n2 = torch.norm(b1).item(), torch.norm(b2).item()\n    dot = torch.dot(b1, b2).abs().item()\n    if abs(n1 - 1.0) > tol or abs(n2 - 1.0) > tol:\n        raise ValueError(f"loop basis not unit: ‖b1‖={n1:.6f}, ‖b2‖={n2:.6f}")\n    if dot >= tol:\n        raise ValueError(f"loop basis not orthogonal: |b1·b2| = {dot:.3e} (tol {tol:.0e})")\n\n\ndef procrustes_align(\n    U1: torch.Tensor, U2: torch.Tensor,\n) -> tuple[torch.Tensor, float]:\n    """Procrustes alignment of two orthonormal subspaces (TZ §3.3).\n\n    Args:\n        U1, U2: [K, r] with U^T U ≈ I_r, same shape.\n\n    Returns:\n        (R, residual_sq): R is [r, r] orthogonal, minimizing\n        ‖U1 @ R − U2‖_F; residual_sq = ‖U1 R − U2‖_F² as a Python float.\n\n    Semantics (subspaces, not bases): R is **invariant** to a joint sign\n    flip of columns of U1 and U2, and residual_sq is invariant to any\n    column sign changes — the distance is between the subspaces span(U1)\n    and span(U2). If U1 and U2 span the same subspace in different\n    bases, residual_sq ≈ 0 even though R ≠ I (R is that basis change).\n\n    Algorithm: M = U2^T U1 [r, r]; SVD M = P Σ Q^T; R = Q P^T (polar\n    factor, O(r) — no det-fix; with sign-gauged bases the SO(r)\n    projection flips a column and breaks H = I for identical subspaces).\n    SVD computed in float64 for stability; R cast back to U1.dtype.\n\n    Degenerate SVD (s_i = s_j, i ≠ j): R is not unique; this function\n    returns one valid minimizer. residual_sq is always well-defined:\n    ‖U1 R − U2‖_F² = 2r − 2·tr(M^T R) = 2r − 2·Σs exactly (the clamp to\n    0 guards only against ~1e-16 floating noise).\n\n    Ill-conditioned (subspaces nearly orthogonal, all s_i ≈ 0): R is\n    numerically unstable; residual_sq → 2r (the maximum) and remains\n    meaningful. Consumers should not rely on R in that regime.\n\n    Errors:\n        ValueError: shape mismatch, U1.dim() != 2, non-orthonormal input\n            (tol 1e-4 for float32, 1e-6 for float64).\n    """\n    if U1.dim() != 2:\n        raise ValueError(f"U1 must be 2-D [K, r], got {U1.dim()}-D")\n    if U1.shape != U2.shape:\n        raise ValueError(f"shape mismatch: {tuple(U1.shape)} vs {tuple(U2.shape)}")\n    if U1.dtype not in (torch.float32, torch.float64):\n        raise ValueError(f"U1 dtype must be float32/float64, got {U1.dtype}")\n\n    tol = 1e-6 if U1.dtype == torch.float64 else 1e-4\n    _validate_orthonormal(U1, tol)\n    _validate_orthonormal(U2, tol)\n\n    # SVD in float64 for stability (TZ §3.4); result cast back to U1.dtype.\n    M = (U2.T @ U1).double()\n    P, s, Qt = torch.linalg.svd(M, full_matrices=False)\n    R64 = Qt.T @ P.T\n    R = R64.to(U1.dtype)\n\n    r = U1.shape[1]\n    residual = (2.0 * r - 2.0 * s.sum()).clamp(min=0.0).item()\n    return R, residual\n\n\ndef holonomy_accumulate(\n    U_list: Sequence[torch.Tensor],\n    closed_loop: bool = True,\n    return_total: bool = False,\n) -> tuple[float, list[float]] | tuple[float, list[float], torch.Tensor]:\n    """Accumulate Procrustes rotation along a (closed) cycle (TZ §3.3).\n\n    Args:\n        U_list: list of [K, r] orthonormal, len ≥ 2.\n        closed_loop: True — include the final R_{N-1,0} step (a real\n            holonomy); False — open path: R_total = R_{0,1} ... R_{N-2,N-1}\n            (measures accumulated parallel transport, not holonomy in the\n            strict sense).\n        return_total: False (default, backward-compatible) — return\n            (kappa, per_step_residuals). True — additionally return\n            R_total [r, r] float64 (the accumulated product, already\n            computed internally) for spectral analysis (see\n            :func:`holonomy_spectrum`), i.e. (kappa, per_step, R_total).\n\n    Returns:\n        (kappa, per_step_residuals[, R_total]): kappa = ‖R_total − I_r‖_F\n        as a Python float; per_step_residuals — the residual_sq of each\n        step in cycle order (len = N−1 open, N closed). Diagnostic: large\n        kappa with all small per-step residuals → composition problem;\n        small kappa with one large per-step → one bad step. R_total is\n        the float64 [r, r] product (only when return_total=True).\n\n    Semantics: R_{i,i+1} satisfies U_{i+1} ≈ U_i @ R_{i,i+1} (left\n    multiplication in cycle order: R_total = R_{0,1} @ R_{1,2} @ ...).\n    R_total ≈ I means the cycle is trivial (flat bundle).\n\n    len == 2 + closed_loop: the pair is traversed forward and backward;\n    the inverse Procrustes is the exact inverse, so κ ≈ 0 always.\n\n    dtype: R_total accumulated in float64; kappa returned as float.\n\n    Gauge invariant: U_i → U_i @ Q_i (orthogonal Q_i) leaves κ unchanged.\n\n    Group-orbit property: if the path is a pure orbit U_i = U₀ @ g_i\n    (e.g. U @ Q^i, or U @ exp(θᵢA)), the CLOSING step is the exact\n    inverse of the accumulated product, so with closed_loop=True\n    **κ = 0 always**. Non-zero κ requires the subspace path to fail to\n    return to its start (genuine curvature in Grassmannian space) —\n    the open-path κ for such an orbit is ‖g_end − I‖ (e.g. ‖Q^{N−1}−I‖\n    or ‖exp(2πA)−I‖). This is the correct semantics: a loop in\n    parameter space that the subspace bundle maps to a closed loop in\n    subspace space has trivial holonomy.\n\n    Errors:\n        ValueError: len(U_list) < 2.\n    """\n    n = len(U_list)\n    if n < 2:\n        raise ValueError(f"need ≥ 2 subspaces, got {n}")\n\n    r = U_list[0].shape[1]\n    R_total = torch.eye(r, dtype=torch.float64)\n    per_step: list[float] = []\n\n    steps = range(n) if closed_loop else range(n - 1)\n    for i in steps:\n        a, b = U_list[i], U_list[(i + 1) % n]\n        R_step, res = procrustes_align(a, b)\n        R_total = R_total @ R_step.double()\n        per_step.append(res)\n\n    kappa = torch.norm(R_total - torch.eye(r, dtype=torch.float64), p="fro").item()\n    if return_total:\n        return kappa, per_step, R_total\n    return kappa, per_step\n\n\ndef holonomy_spectrum(\n    U_list: Sequence[torch.Tensor],\n    closed_loop: bool = True,\n) -> tuple[np.ndarray, torch.Tensor]:\n    """Spectral angles of R_total; convenience wrapper over\n    ``holonomy_accumulate(..., return_total=True)``.\n\n    NOT a substitute for the scalar κ (``holonomy_accumulate()[0]``):\n    κ collapses R_total to one number; this returns the full angle set\n    used in paper [M-2] / §6.\n\n    Args:\n        U_list: list of [K, r] orthonormal, len ≥ 2 (see\n            :func:`holonomy_accumulate`).\n        closed_loop: see :func:`holonomy_accumulate`.\n\n    Returns:\n        (theta, R_total): theta — np.ndarray [r//2] of rotation angles\n        in (0, π], sorted ascending (one per conjugate pair\n        {e^{iθ}, e^{−iθ}} of R_total; +1-axes dropped; odd r leaves a\n        trivial +1 axis); R_total — [r, r] float64.\n\n        NOT a reconstruction recipe for κ: when ``det R_total = −1`` (a\n        Procrustes step may be a reflection) one real eigenvalue −1 exists, is\n        reported as a π angle, and contributes 4 to ``‖H − I‖_F²`` while the\n        ``4·Σ(1 − cos θ)`` form implies 8 — so that identity closes only when\n        summed over ALL r eigenvalues, ``‖H − I‖_F² = Σ_j 2(1 − cos ang_j)``\n        (audit docs/audit3_math.md п. А; the convention is a warning at runtime,\n        and no consumer in the repo derives κ from these angles — κ comes from\n        ``holonomy_accumulate``).\n\n    Uses ``torch.linalg.eigvals`` (complex) — ``spectrum_from_gramian`` /\n    ``eigvalsh`` CANNOT be used: R_total is orthogonal, not symmetric.\n\n    Warnings:\n        RuntimeWarning: odd r (trivial +1 axis); eigenvalues off the\n        unit circle (|λ| < 1 − 1e-6); duplicate angles (degenerate\n        spectrum, basis ambiguity).\n\n    Errors:\n        RuntimeError: R_total not orthogonal (‖R^T R − I‖_F > 1e-4).\n    """\n    _, _, R_total = holonomy_accumulate(U_list, closed_loop=closed_loop,\n                                        return_total=True)\n    r = R_total.shape[0]\n    orth_err = torch.norm(R_total.T @ R_total - torch.eye(r, dtype=torch.float64),\n                          p="fro").item()\n    if orth_err > 1e-4:\n        raise RuntimeError(\n            f"holonomy_spectrum: R_total not orthogonal: "\n            f"‖R^T R − I‖_F = {orth_err:.3e} (tol 1e-4)"\n        )\n\n    eigvals = torch.linalg.eigvals(R_total)\n    if torch.any(eigvals.abs() < 1 - 1e-6):\n        off = float((1.0 - eigvals.abs()).max().item())\n        warnings.warn(\n            f"holonomy_spectrum: eigenvalue off the unit circle by {off:.2e} "\n            "(numerical instability)",\n            RuntimeWarning,\n        )\n\n    # One angle per conjugate pair {e^{iθ}, e^{-iθ}}: a 2D rotation\n    # block at θ contributes angles +θ and −θ (equal in magnitude);\n    # at θ = π both eigenvalues are −1 and the angles are ±π — the SIGN\n    # of the atan2 result depends on fp noise in the imaginary part\n    # (LAPACK), so the old `angles > 1e-6` filter silently dropped −π\n    # and could lose an entire π block (all−π pair → "R ≈ I").\n    # docs/E38_fixes.md §2 HIGH. Work with ABS values: every 2D block\n    # then contributes exactly 2 equal angles, so mult = counts // 2\n    # UNIFORMLY (no special π halving needed; conjugate pairs from real\n    # Schur blocks are bit-exact in |imag|, so abs is exact here).\n    # A LONE real -1 eigenvalue (reflection axis; det R_total = -1,\n    # possible because a Procrustes step may be a reflection) also lands\n    # in the π bin (counts = 1 → mult = 0) and is reported as a π angle\n    # — the standard convention. It is NOT a 2D rotation block: it\n    # contributes 4 to ||R-I||_F^2, while the reported π angle implies\n    # 8 in the 4*sum(1-cos theta) identity. Exact identities must sum\n    # over ALL r eigenvalues:\n    #   ||R-I||_F^2 = sum_j 2 (1 - cos ang_j).\n    angles = torch.angle(eigvals).abs()  # [0, π]\n    positive = angles[angles > 1e-6]\n    rounded = positive.round(decimals=8)\n    uniques, counts = torch.unique(rounded, return_counts=True)\n    mult = counts // 2\n    # mult 0: a lone −1 (reflection axis) — merge into one π angle.\n    if (mult == 0).any():\n        warnings.warn(\n            "holonomy_spectrum: lone -1 eigenvalue (reflection axis) — "\n            "reported as a π angle (standard convention)",\n            RuntimeWarning,\n        )\n        mult = mult.clamp(min=1)\n    # mult > 1: the same angle appears in several pairs — the spectrum is\n    # degenerate and the basis inside that block is ambiguous, but the\n    # angle set (with multiplicities) is well-defined and REQUIRED for\n    # tr(R) = 2 Σ_j cos θ_j — keep the multiplicity\n    if (mult > 1).any():\n        warnings.warn(\n            "holonomy_spectrum: degenerate spectrum (repeated angles) — "\n            "basis is ambiguous inside the degenerate block; multiplicities "\n            "kept",\n            RuntimeWarning,\n        )\n    # repeat_interleave is unimplemented for float64 on CPU — expand in Python\n    expanded_vals = [float(u) for u, m in zip(uniques.tolist(), mult.tolist())\n                     for _ in range(int(m))]\n    expanded = torch.tensor(sorted(expanded_vals), dtype=torch.float64)\n    expected = r // 2\n    if r % 2 == 1:\n        warnings.warn(\n            f"holonomy_spectrum: odd r = {r}, trivial +1 axis present",\n            RuntimeWarning,\n        )\n    elif expanded.numel() != expected:\n        if positive.numel() == 0:\n            # R_total ≈ I: all angles are +1 — report r//2 zeros\n            expanded = torch.zeros(expected, dtype=torch.float64)\n        else:\n            warnings.warn(\n                f"holonomy_spectrum: expected {expected} angle pairs, "\n                f"got {expanded.numel()}",\n                RuntimeWarning,\n            )\n    return expanded.numpy().astype(np.float64), R_total\n\n\ndef geodesic_curvature(\n    u1: torch.Tensor, u2: torch.Tensor, u3: torch.Tensor,\n    eps: float = 1e-8,\n) -> float:\n    """Discrete geodesic curvature κ_g ∈ R≥0 at u2, from three unit vectors.\n\n    Local Abelian invariant of the û_t curve on S^{D-1} (lemma [M-3],\n    corrected formulation) — for comparison with κ (holonomy). NOT the\n    triangle-inequality defect.\n\n    Args:\n        u1, u2, u3: [D] unit vectors (‖·‖ = 1, checked).\n        eps: collinearity tolerance for the tangent norms.\n\n    Returns:\n        turn / arc, where turn = π − arccos(<t1, t3>/(‖t1‖‖t3‖)) is the\n        exterior angle between the tangents at u2 (toward u1 / u3) and\n        arc = (‖u1 − u2‖ + ‖u3 − u2‖) / 2 (chord lengths ≈ geodesic\n        lengths for small steps). Collinear (degenerate) triple → 0.0.\n\n    Errors:\n        ValueError: non-unit input (index and value reported), D < 2.\n        RuntimeError: all three points coincide (arc < eps).\n    """\n    for i, u in enumerate((u1, u2, u3)):\n        u = u.reshape(-1)\n        n = torch.norm(u).item()\n        if abs(n - 1.0) > 1e-4:\n            raise ValueError(\n                f"geodesic_curvature: u{i + 1} not unit: ‖u‖ = {n:.6f}"\n            )\n        if u.shape[0] < 2:\n            raise ValueError(\n                f"geodesic_curvature: D < 2 (got D = {u.shape[0]}) — "\n                "cannot build tangents"\n            )\n    u1, u2, u3 = u1.reshape(-1), u2.reshape(-1), u3.reshape(-1)\n\n    arc = (torch.norm(u1 - u2).item() + torch.norm(u3 - u2).item()) / 2\n    if arc < eps:\n        raise RuntimeError(\n            "geodesic_curvature: all three points coincide (arc < eps) — "\n            "not informative"\n        )\n\n    t1 = u1 - torch.dot(u1, u2) * u2   # tangent at u2, toward u1\n    t3 = u3 - torch.dot(u3, u2) * u2   # tangent at u2, toward u3\n    n1, n3 = float(torch.norm(t1)), float(torch.norm(t3))\n    if n1 < eps or n3 < eps:\n        return 0.0                      # u1 or u3 collinear with u2\n    cos_turn = float(torch.clip(torch.dot(t1, t3) / (n1 * n3), -1.0, 1.0))\n    turn = math.pi - math.acos(cos_turn)\n    return turn / arc\n\n\ndef eigenframe_basis(\n    M_K: torch.Tensor,   # [K, D]\n    uhat: torch.Tensor,  # [D] unit\n) -> torch.Tensor:\n    """Canonical basis B [D, D-1] orthonormal, B^T uhat = 0, ordered by\n    decreasing eigenvalue of P_⊥ M_K^T M_K P_⊥.\n\n    This is a CONVENTION per [M-5]; the basis is data-determined up to\n    permutation and sign, but the decomposition of H_{012} using it is\n    fixed once chosen.\n\n    NOT a substitute for spectral.top_r_svd: the operator is the symmetric\n    P_⊥ M_K^T M_K P_⊥ (D×D, in uhat^⊥), NOT J (general K×D). top_r_svd\n    returns right singular vectors of J; this returns eigenvectors of the\n    gauge-projected M_K Gramian. Consumers needing right singular vectors\n    of J must call top_r_svd directly.\n\n    Warnings:\n        RuntimeWarning: the trivial û-axis is not isolated (|λ_min| >\n        1e-8, M_K^T M_K non-degenerate on û) — the smallest-λ vector is\n        dropped anyway; degenerate eigenvalues (basis ambiguity inside\n        a degenerate block).\n\n    Errors:\n        ValueError: uhat not unit, M_K.shape[1] != D, D < 2.\n    """\n    if M_K.dim() != 2:\n        raise ValueError(f"eigenframe_basis: M_K must be 2-D, got {M_K.dim()}-D")\n    uhat = uhat.reshape(-1)\n    K, D = M_K.shape\n    if uhat.shape[0] != D:\n        raise ValueError(\n            f"eigenframe_basis: M_K.shape[1] = {D} != uhat dim = {uhat.shape[0]}"\n        )\n    if D < 2:\n        raise ValueError(f"eigenframe_basis: D < 2 (got D = {D})")\n    n = torch.norm(uhat).item()\n    if abs(n - 1.0) > 1e-4:\n        raise ValueError(f"eigenframe_basis: uhat not unit: ‖uhat‖ = {n:.6f}")\n\n    A = M_K.T @ M_K                              # [D, D] symmetric PSD\n    Au = A @ uhat\n    A_proj = A - torch.outer(uhat, Au) - torch.outer(Au, uhat) \\\n        + torch.dot(uhat, Au) * torch.outer(uhat, uhat)   # P_⊥ A P_⊥\n\n    eigvals, eigvecs = torch.linalg.eigh(A_proj)   # ascending\n    mask = eigvals.abs() > 1e-8\n    if mask.sum() == D:\n        # no ~0 eigenvalue: trivial û-axis not isolated — drop the smallest\n        warnings.warn(\n            "eigenframe_basis: trivial û-axis not isolated "\n            "(|λ_min| > 1e-8, M_K^T M_K non-degenerate on û) — "\n            "dropping the smallest-λ vector",\n            RuntimeWarning,\n        )\n        eigvals_pos, eigvecs_pos = eigvals[1:], eigvecs[:, 1:]\n    else:\n        eigvals_pos, eigvecs_pos = eigvals[mask], eigvecs[:, mask]\n    # degenerate spectrum check (consecutive gaps below 1e-8 of the scale)\n    if eigvals_pos.numel() > 1:\n        scale = eigvals_pos.max().abs()\n        if scale > 0 and ((eigvals_pos[1:] - eigvals_pos[:-1]).abs() < 1e-8 * scale).any():\n            warnings.warn(\n                "eigenframe_basis: degenerate eigenvalues — basis is "\n                "ambiguous inside the degenerate block",\n                RuntimeWarning,\n            )\n    order = torch.argsort(eigvals_pos, descending=True)\n    if eigvals_pos.numel() == 0:\n        # fully degenerate (e.g. M_K = 0): no non-trivial axis — return [D, 0]\n        warnings.warn(\n            "eigenframe_basis: spectrum fully degenerate — returning an "\n            "empty basis [D, 0]",\n            RuntimeWarning,\n        )\n        return eigvecs[:, :0].contiguous()\n    B = eigvecs_pos[:, order]\n    B = B / B.norm(dim=0, keepdim=True)\n    return B.contiguous()\n\n\ndef holonomy_along_path(\n    path_fn: Callable[[float], tuple[torch.Tensor, torch.Tensor]],\n    thetas: Sequence[float],\n    closed_loop: bool = True,\n) -> dict:\n    """Holonomy of a family of subspaces along a 1-D parameter (TZ §3.3).\n\n    Args:\n        path_fn: θ → (U_r, V_r), U_r: [K, r], V_r: [r, D_S] (the\n            ``top_r_svd`` convention: rows of V_r are the right singular\n            vectors). ALL model logic (weight setup, forward, SVD, top-r\n            selection) lives in path_fn — this function does not know about\n            models and does not cache. V_r is transposed internally to the\n            [D_S, r] orthonormal-column basis required by procrustes_align;\n            callers pass V_r exactly as top_r_svd returns it.\n        thetas: sequence of θ, len ≥ 2. With closed_loop=True the last\n            θ must NOT duplicate the first (holonomy_accumulate adds the\n            closing step R_{N-1,0} itself; a duplicate would just add a\n            trivial Procrustes I and double-count a measurement).\n        closed_loop: see holonomy_accumulate.\n\n    Returns:\n        dict with keys: kappa_U, kappa_V, per_step_U, per_step_V, thetas\n        (thetas as a list for JSON-friendliness).\n\n    Use case: parameter-space holonomy, cycle W₀→W₁₀₀→W₀ —\n    path_fn(θ) = compute top-r U, V at interpolated lm_head weights.\n    Note the group-orbit property (holonomy_accumulate): if the\n    subspaces form a pure orbit U(θ) = U₀·g(θ), a closed loop has\n    κ = 0; non-zero κ means the subspaces do NOT return to their start\n    at the end of the parameter cycle — that is the signal the\n    experiment is looking for.\n\n    Fully deterministic given path_fn and thetas.\n\n    Errors:\n        ValueError: len(thetas) < 2.\n        TypeError: path_fn does not return a (U, V) pair of tensors.\n    """\n    thetas = list(thetas)\n    if len(thetas) < 2:\n        raise ValueError(f"need ≥ 2 thetas, got {len(thetas)}")\n\n    U_list: list[torch.Tensor] = []\n    V_list: list[torch.Tensor] = []\n    for th in thetas:\n        out = path_fn(th)\n        if not (isinstance(out, tuple) and len(out) == 2\n                and isinstance(out[0], torch.Tensor) and isinstance(out[1], torch.Tensor)):\n            raise TypeError(\n                f"path_fn must return (U_r [K, r], V_r [r, D_S]) tensors, "\n                f"got {type(out).__name__}"\n            )\n        U_list.append(out[0])\n        V_list.append(out[1].T)  # [r, D_S] -> [D_S, r] orthonormal columns\n\n    kappa_U, per_U = holonomy_accumulate(U_list, closed_loop=closed_loop)\n    kappa_V, per_V = holonomy_accumulate(V_list, closed_loop=closed_loop)\n    return {\n        "kappa_U": kappa_U,\n        "kappa_V": kappa_V,\n        "per_step_U": per_U,\n        "per_step_V": per_V,\n        "thetas": thetas,\n    }\n\n\ndef holonomy_along_2d_loop(\n    base: torch.Tensor,\n    b1: torch.Tensor,\n    b2: torch.Tensor,\n    epsilon: float,\n    n_theta: int,\n    measure_fn: Callable[[torch.Tensor], tuple[torch.Tensor, torch.Tensor]],\n) -> dict:\n    """Holonomy around a small 2-D loop (ε·(cos θ·b1 + sin θ·b2)) (TZ §3.3).\n\n    The loop h(θ) = base + ε(cos θ·b1 + sin θ·b2), θ = 0, 2π/n, ...,\n    2π(n−1)/n (no 2π endpoint — holonomy_accumulate closes it), probes\n    the local curvature of the subspace bundle around `base`.\n\n    Args:\n        base: [D] / [1, D] / [1, 1, D] base point (normalized to [D]).\n        b1, b2: [D] orthonormal directions (‖·‖ = 1, |b1·b2| < 1e-4 —\n            checked; a non-orthogonal pair makes κ uninterpretable).\n            For \'sens\': top-2 right singular vectors of J(base); for\n            \'canon\': top-2 PCs of h_batch (wrappers in exp_T2.py).\n        epsilon: loop radius, must be > 0 and ≲ 0.2·‖base‖ (a larger\n            loop wanders far from the operating point and measure_fn may\n            return degenerate subspaces).\n        n_theta: loop discretization, ≥ 4 (a 3-point \'loop\' is a\n            triangle — crude; 16 recommended).\n        measure_fn: h → (U_r, V_r), V_r: [r, D] (top_r_svd convention;\n            transposed internally to [D, r] for procrustes_align). All model\n            logic lives there.\n\n    Returns:\n        dict: kappa_U, kappa_V, per_step_U, per_step_V, h_perturbed\n        (list of [D] loop points, for diagnostics).\n\n    Errors:\n        ValueError: epsilon ≤ 0; n_theta < 4; b1/b2 not orthonormal;\n            base not squeezable to [D].\n    """\n    if epsilon <= 0:\n        raise ValueError(f"epsilon must be > 0, got {epsilon}")\n    if n_theta < 4:\n        raise ValueError(f"n_theta must be ≥ 4, got {n_theta}")\n\n    base = base.reshape(-1)\n    b1 = b1.reshape(-1)\n    b2 = b2.reshape(-1)\n    if b1.shape != b2.shape or b1.shape != base.shape:\n        raise ValueError(\n            f"shape mismatch: base {tuple(base.shape)}, b1 {tuple(b1.shape)}, "\n            f"b2 {tuple(b2.shape)}"\n        )\n    _check_loop_basis(b1, b2)\n\n    U_list: list[torch.Tensor] = []\n    V_list: list[torch.Tensor] = []\n    h_perturbed: list[torch.Tensor] = []\n    for i in range(n_theta):\n        th = 2.0 * math.pi * i / n_theta\n        h = base + epsilon * (math.cos(th) * b1 + math.sin(th) * b2)\n        out = measure_fn(h)\n        if not (isinstance(out, tuple) and len(out) == 2\n                and isinstance(out[0], torch.Tensor) and isinstance(out[1], torch.Tensor)):\n            raise TypeError(\n                f"measure_fn must return (U_r, V_r) tensors, got {type(out).__name__}"\n            )\n        U_list.append(out[0])\n        V_list.append(out[1].T)  # [r, D] -> [D, r] orthonormal columns\n        h_perturbed.append(h)\n\n    kappa_U, per_U = holonomy_accumulate(U_list, closed_loop=True)\n    kappa_V, per_V = holonomy_accumulate(V_list, closed_loop=True)\n    return {\n        "kappa_U": kappa_U,\n        "kappa_V": kappa_V,\n        "per_step_U": per_U,\n        "per_step_V": per_V,\n        "h_perturbed": h_perturbed,\n    }\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/interventions.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Module 4: interventions — EditSpec and its application to the model (TZ part 2).\n\nFive T0 edit types: base, readout_pull, random_ortho, random_matched,\ntail_only. Plus `apply_edit` (context manager) and `apply_edit_and_cache`\n(→ h_t, p_t on disk).\n\nKey decisions (docs/new_functions_2.md §4.1):\n1. **delta_W is [K, D_S], applied only to `lm_head.weight[token_set]`**\n   — not [V, D_S]. ~1200× memory savings (storage of the DELTA only);\n   semantics: J depends only on the token_set rows. Note: the Adam state\n   in `edit_lm_head` is allocated on the FULL [V, D_S] weight (~2 GB\n   extra for 0.8B) — the savings apply to the persisted delta, not to\n   optimizer state (docs/E38_fixes.md §6).\n2. **EditSpec knows its token_set** — never recomputed at apply time.\n3. **apply_edit restores the model in `finally`** — even on exception.\n4. **apply_edit_and_cache caches only h_t and p_t, not J** — J is\n   reconstructed on the fly via compute_jacobian_for_tokens(h_override=\n   h_t[l][t]) + SVDCache (in the experiment code, NOT here): ~30× less\n   disk and any r without recompute.\n\nNot used here (deliberate): bridge.jacobian_gap (needs ready J; the loss\nneeds create_graph rows), compute_jacobian_for_tokens (downstream only),\ncache_io.find_targets_dir (read-only discovery; interventions write new\ncaches), _clone_gdn_states_on_entry around apply_edit_and_cache\n(no_grad, no backward — in-place mutation is safe).\n\nRelated: paper.md §E-16a (readout_pull protocol), §19.A (EditSpec).\n"""\n\nfrom __future__ import annotations\n\nimport contextlib\nimport json\nimport os\nimport uuid\nfrom dataclasses import dataclass, field, replace\nfrom typing import Any, Iterator, Sequence\n\nimport torch\nimport torch.nn.functional as F\n\nfrom .jacobian import compute_hidden_jacobian_rows\n\n__all__ = [\n    "EditSpec",\n    "make_base",\n    "make_readout_pull",\n    "make_random_ortho",\n    "make_random_matched",\n    "make_tail_only",\n    "apply_edit",\n    "apply_edit_and_cache",\n]\n\n# Module-level nesting guard for apply_edit (one edit at a time; TZ §4.3).\n_APPLY_EDIT_ACTIVE = False\n\n\n@dataclass(frozen=True)\nclass EditSpec:\n    """Immutable edit specification (TZ §4.3).\n\n    delta_W is [K, D_S] and refers to `lm_head.weight[token_set]` ONLY —\n    applying it to the full [V, D_S] weight is a caller bug (apply_edit\n    applies it to the token_set rows and checks shapes).\n\n    `projection_onto_top_r` may be filled in after the fact via\n    `dataclasses.replace(edit, projection_onto_top_r=...)` — frozen\n    dataclasses support replace (TZ §4.3, make_readout_pull).\n\n    `meta` is a dict; treated as immutable after creation (callers copy\n    it at creation; no field-level enforcement).\n    """\n\n    name: str\n    delta_W: torch.Tensor | None                    # [K, D_S], None for base\n    token_set: tuple[int, ...]                      # frozen tuple, hashable\n    target_layer: str                               # "lm_head" (only supported)\n    norm_frobenius: float\n    projection_onto_top_r: float\n    seed: int | None\n    meta: dict = field(default_factory=dict)\n\n\ndef _resolve_lm_head_weight(model) -> torch.Tensor:\n    """model.lm_head.weight, with a get_output_embeddings() fallback."""\n    lm = getattr(model, "lm_head", None)\n    if lm is not None and getattr(lm, "weight", None) is not None:\n        return lm.weight\n    out = model.get_output_embeddings()\n    if out is None or getattr(out, "weight", None) is None:\n        raise AttributeError("model has no lm_head.weight / output embeddings")\n    return out.weight\n\n\ndef _validate_orthonormal_columns(basis: torch.Tensor) -> None:\n    K, r = basis.shape\n    gram = basis.T @ basis\n    err = torch.norm(gram - torch.eye(r, device=gram.device, dtype=gram.dtype), p="fro").item()\n    if err > 1e-4:\n        raise ValueError(f"top_r_basis not orthonormal: ‖U^T U − I‖_F = {err:.3e}")\n\n\ndef _projection_fraction(delta_W: torch.Tensor, top_r_basis: torch.Tensor) -> float:\n    """‖P_topr ΔW‖_F / ‖ΔW‖_F, P_topr = U U^T ([K,K] projector on the\n    token-space span of top_r_basis; each of the D_S columns is a [K]\n    vector)."""\n    P = top_r_basis @ top_r_basis.T\n    nrm = delta_W.norm()\n    if nrm.item() < 1e-12:\n        return 0.0\n    return (P @ delta_W).norm().item() / nrm.item()\n\n\ndef _scale_to_norm(delta_W: torch.Tensor, target_norm: float) -> torch.Tensor:\n    nrm = delta_W.norm()\n    if nrm.item() < 1e-12:\n        raise RuntimeError("_scale_to_norm: zero-norm input")\n    return delta_W * (target_norm / nrm)\n\n\ndef make_base(token_set: Sequence[int]) -> EditSpec:\n    """No-op edit: uniform signature with the other edit types\n    (apply_edit(model, base) enters and leaves the model untouched)."""\n    return EditSpec(\n        name="base",\n        delta_W=None,\n        token_set=tuple(token_set),\n        target_layer="lm_head",\n        norm_frobenius=0.0,\n        projection_onto_top_r=0.0,\n        seed=None,\n        meta={},\n    )\n\n\ndef make_readout_pull(\n    model,\n    input_ids: torch.Tensor,\n    attention_mask: torch.Tensor,\n    layer_idx: int,\n    token_set: list[int],\n    target_J_p: torch.Tensor,\n    n_steps: int = 100,\n    lr: float = 1e-3,\n    kl_leash_weight: float = 0.1,\n    seed: int = 0,\n    verbose: bool = False,\n    top_r_basis: torch.Tensor | None = None,\n) -> EditSpec:\n    """E-16a protocol: Adam on lm_head.weight[token_set] rows only (TZ §4.3).\n\n    Loss per step: (1 − cos(J_S_p, target_J_p)).mean() +\n    kl_leash_weight · KL(p_base ‖ p_edit), where\n      - J_S_p [K, D_S] via compute_hidden_jacobian_rows(create_graph=True)\n        (reused from jacobian.py — the only create_graph site; its inner\n        _clone_gdn_states_on_entry handles the GDN state),\n      - cos — per-row F.cosine_similarity,\n      - KL — FULL-vocabulary KL(p_edit ‖ p_base) at the last position (same\n        operand order as the Kaggle kernel: input = log p_base, target =\n        log p_edit), with the\n        base computed ONCE up front (ref-math «leash»: a K-subset KL renormalized\n        to 1 is blind to mass leaking out of the token set). Weight semantics are\n        literal: KL is a sum, not a mean, so kl_leash_weight=0.1 means 0.1\n        (audit B3 — the previous 1-D batchmean divided it by K),\n      - the J-term keeps the E-16 gauge: p = softmax over the token_set, and\n        p_edit is the model\'s own last-position distribution.\n\n    PROTOCOL DIFFERENCE from kaggle_kernel/exp16_tpu/exp16_readout_pull.py\n    (which produced the published E-16a numbers): that kernel does NOT mask\n    gradients, so Adam moves all V rows of lm_head and α* = ‖ΔW‖/‖W‖ is measured\n    on the full W; here the TZ masking restricts Adam to the K token_set rows.\n    Same loss and same KL direction as that kernel; the only difference is the\n    optimizer support — not interchangeable, and this is\n    why E-16a numbers must not be reproduced through this function expecting a\n    match (audit B3, third defect).\n\n    Determinism: torch.manual_seed(seed); torch.use_deterministic_\n    algorithms is NOT enabled (Adam on GDN models may raise).\n\n    Memory note: ``params = [weight]`` — the Adam optimizer allocates full\n    [V, D_S] state (m + v), not just the [K, D_S] delta: gradient masking\n    keeps non-token_set rows frozen, but the state is still ~2 GB for\n    0.8B. The "~1200× savings" claim covers the persisted delta only\n    (docs/E38_fixes.md §6).\n\n    Restoration: W0 = lm_head.weight[token_set].detach().clone() up\n    front; copy_(W0) in finally; requires_grad saved/restored too.\n\n    Returns:\n        EditSpec(name="readout_pull", delta_W = W_final[token_set] − W0).\n        projection_onto_top_r: 0.0 if top_r_basis is None, else computed\n        (TZ §4.3: pass top_r_basis to fill it in-place; the experiment\n        can also use dataclasses.replace later).\n\n    The model is left exactly as found (weights, requires_grad, training\n    mode).\n\n    Errors:\n        ValueError: target_J_p.shape != (K, D_S); n_steps < 1; lr ≤ 0.\n    """\n    K = len(token_set)\n    if n_steps < 1:\n        raise ValueError(f"n_steps must be ≥ 1, got {n_steps}")\n    if lr <= 0:\n        raise ValueError(f"lr must be > 0, got {lr}")\n\n    weight = _resolve_lm_head_weight(model)\n    D_S = weight.shape[1]\n    ts = torch.tensor(token_set, device=weight.device)\n    if target_J_p.shape != (K, D_S):\n        raise ValueError(f"target_J_p must be ({K}, {D_S}), got {tuple(target_J_p.shape)}")\n    target_J_p = target_J_p.detach().to(dtype=weight.dtype)\n\n    global _APPLY_EDIT_ACTIVE\n    if _APPLY_EDIT_ACTIVE:\n        raise RuntimeError("make_readout_pull: nested model edit in progress")\n\n    torch.manual_seed(seed)\n    was_training = model.training\n    model.eval()\n\n    W0 = weight[ts].detach().clone()\n    rg_was = weight.requires_grad\n    weight.requires_grad_(True)\n    _APPLY_EDIT_ACTIVE = True  # AFTER the setup that can raise below\n    delta_W: torch.Tensor | None = None\n\n    try:\n        params = [weight]  # gradient masked below: only token_set rows update\n        ts_mask = torch.zeros(weight.shape[0], dtype=torch.bool, device=weight.device)\n        ts_mask[ts] = True\n\n        # p_base: once, under no_grad, over the K-subset (E-16 gauge) — this p\n        # is the gauge of the J-term. The KL leash is NOT computed on it: see\n        # the leash block (full vocabulary, audit B3).\n        with torch.no_grad():\n            out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)\n            p_base = F.softmax(out.logits[0, -1, ts].float(), dim=-1)\n            # Full-vocab log-probs of the base distribution, [1, V] — the leash\n            # must see mass leaking OUT of the token set (ref-math "leash"/\n            # E-16a), which a renormalized K-subset KL is structurally blind to.\n            log_p_base_full = torch.log_softmax(out.logits[0, -1].float(), dim=-1)[None]\n\n        opt = torch.optim.Adam(params, lr=lr)\n        for step in range(n_steps):\n            opt.zero_grad(set_to_none=True)\n\n            # (1) Jacobian rows through the (trainable) lm_head rows.\n            W_rows = weight[ts]\n            J_S_p = compute_hidden_jacobian_rows(\n                model, layer_idx, input_ids, attention_mask, token_set,\n                p=p_base, W_rows=W_rows, create_graph=True,\n            )\n            cos = F.cosine_similarity(J_S_p, target_J_p, dim=1, eps=1e-12)\n            j_loss = (1.0 - cos).mean()\n\n            # (2) KL leash on the token_set distribution at the last pos.\n            # h_t is recomputed every step so the leash tracks the edit.\n            out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)\n            log_p_edit_full = torch.log_softmax(out.logits[0, -1].float(), dim=-1)[None]\n            # KL(base || edit) over the FULL vocabulary, [1, V] with\n            # batchmean → divided by batch size 1 → the true KL. The previous\n            # one-liner had two defects (audit docs/audit4.pdf B3): it fed a 1-D\n            # [K] tensor to batchmean, which divides by K=128 (leash 128×\n            # weaker than the nominal weight — 0.1 became 7.8e-4), and it\n            # measured KL on the renormalized K-subset, blind to mass leaving\n            # the token set. Matches kaggle_kernel/exp16_tpu/exp16_readout_pull\n            # (batchmean on [1, V], log_target=True); the remaining protocol\n            # difference is deliberate and stated in the docstring: Adam here is\n            # restricted to the token_set rows, the Kaggle kernel moves all V.\n            # Operand order matches the kernel: input = log p_base (no grad),\n            # target = log p_edit (grad) → Σ p_edit(log p_edit − log p_base)\n            # = KL(edit ‖ base), the direction the published E-16a used.\n            kl = F.kl_div(log_p_base_full, log_p_edit_full,\n                          reduction="batchmean", log_target=True)\n\n            loss = j_loss + kl_leash_weight * kl\n            loss.backward()\n            # TZ: "Adam state: only lm_head.weight[token_set]" — the KL term\n            # (logits = h @ W^T) gradients the FULL [V, D_S] weight; mask\n            # everything outside the token_set slice so Adam moves only the\n            # K rows (zero grad → zero Adam update for the other rows).\n            if weight.grad is not None:\n                # ASSIGNMENT, not ``weight.grad.data[~ts_mask].zero_()``: boolean\n                # indexing returns a COPY, so ``t[mask].zero_()`` mutates the copy\n                # and leaves the gradient untouched (it went unnoticed while the\n                # K-subset KL produced exactly zero gradient outside the token\n                # set; the full-vocab leash makes it nonzero, and a no-op mask\n                # then moves all V rows).\n                weight.grad.data[~ts_mask] = 0.0\n            opt.step()\n\n            if verbose:\n                print(f"    readout_pull step {step + 1}/{n_steps}: "\n                      f"loss={loss.item():.6f} (cos={(cos.mean()).item():.4f})")\n            # Capture the delta BEFORE finally restores the weights.\n            delta_W = weight[ts].detach().clone() - W0\n    finally:\n        # index_copy_ on .data — in-place; a leaf with requires_grad=True\n        # rejects in-place ops even under no_grad (graph corruption), and\n        # weight[ts].copy_ would be a no-op (advanced indexing → copy).\n        weight.data.index_copy_(0, ts, W0)\n        weight.requires_grad_(rg_was)\n        model.train(was_training)\n        _APPLY_EDIT_ACTIVE = False\n\n    assert delta_W is not None  # n_steps ≥ 1 guarantees one capture\n    return EditSpec(\n        name="readout_pull",\n        delta_W=delta_W,\n        token_set=tuple(token_set),\n        target_layer="lm_head",\n        norm_frobenius=delta_W.norm().item(),\n        projection_onto_top_r=(\n            _projection_fraction(delta_W, top_r_basis) if top_r_basis is not None else 0.0\n        ),\n        seed=seed,\n        meta={\n            "target_J_p_hash": _tensor_fingerprint(target_J_p),\n            "lr": lr,\n            "n_steps": n_steps,\n            "kl_leash_weight": kl_leash_weight,\n            "layer_idx": layer_idx,\n        },\n    )\n\n\ndef _tensor_fingerprint(t: torch.Tensor) -> str:\n    """Deterministic 16-hex fingerprint (sum + abs-max) for meta."""\n    t = t.detach().double()\n    s = int(t.sum().item() * 1e9) & 0xFFFFFFFF\n    m = int(t.abs().max().item() * 1e9) & 0xFFFFFFFF\n    return f"{s:08x}{m:08x}"\n\n\ndef make_random_ortho(\n    delta_W_ref: torch.Tensor,\n    top_r_basis: torch.Tensor,\n    token_set: Sequence[int],\n    seed: int,\n) -> EditSpec:\n    """Random edit with ‖·‖_F = ‖ref‖_F and P_topr ΔW ≈ 0 (TZ §4.3).\n\n    Algorithm: G ~ randn(K, D_S); G_proj = G − P G (orthogonal\n    complement in token-space, P = U U^T [K, K]); scale to ref norm.\n    G is the only source of randomness (seeded → deterministic).\n\n    Post-checks (RuntimeError with diagnostics on failure):\n        ‖delta_W‖_F ≈ ‖ref‖_F (1e-6 rel); ‖P ΔW‖_F / ‖ΔW‖_F < 1e-3.\n\n    Errors:\n        ValueError: delta_W_ref not 2-D / K mismatch vs token_set or\n            basis; top_r_basis not orthonormal.\n        RuntimeError: ref norm ≈ 0; G nearly entirely in top-r\n            (‖G_proj‖ < 1e-6·‖G‖ — caller must reduce r or change basis).\n    """\n    _check_ref_and_basis(delta_W_ref, top_r_basis, token_set)\n    K, D_S = delta_W_ref.shape\n    torch.manual_seed(seed)\n    G = torch.randn(K, D_S, dtype=delta_W_ref.dtype)\n    P = top_r_basis @ top_r_basis.T\n    G_proj = G - P @ G\n    if G_proj.norm().item() < 1e-6 * G.norm().item():\n        raise RuntimeError(\n            "random_ortho: G is nearly entirely in the top-r span; "\n            "reduce r or choose a different basis"\n        )\n    delta_W = _scale_to_norm(G_proj, delta_W_ref.norm().item())\n\n    proj = _projection_fraction(delta_W, top_r_basis)\n    if proj > 1e-3:\n        raise RuntimeError(\n            f"random_ortho post-check failed: top-r projection fraction "\n            f"{proj:.3e} > 1e-3"\n        )\n    nrm = delta_W.norm().item()\n    if abs(nrm - delta_W_ref.norm().item()) > 1e-6 * delta_W_ref.norm().item():\n        raise RuntimeError(\n            f"random_ortho post-check failed: norm {nrm:.9f} vs ref "\n            f"{delta_W_ref.norm().item():.9f}"\n        )\n    return EditSpec(\n        name="random_ortho",\n        delta_W=delta_W,\n        token_set=tuple(token_set),\n        target_layer="lm_head",\n        norm_frobenius=nrm,\n        projection_onto_top_r=proj,\n        seed=seed,\n        meta={"ref_norm": delta_W_ref.norm().item(), "r": top_r_basis.shape[1]},\n    )\n\n\ndef make_random_matched(\n    delta_W_ref: torch.Tensor,\n    top_r_basis: torch.Tensor,\n    token_set: Sequence[int],\n    seed: int,\n) -> EditSpec:\n    """Random edit preserving the top-r projection of ref (TZ §4.3).\n\n    delta_W = ref_proj + G_residual · (‖ref_residual‖ / ‖G_residual‖):\n    the top-r component is EXACTLY ref\'s; only the orthogonal complement\n    is randomized. Same total norm as ref.\n\n    Contrast with random_ortho (zero top-r projection): the two controls\n    separate "effect of the norm" from "effect of landing in the signal".\n\n    Errors: ValueError as in random_ortho; RuntimeError if ref is\n    entirely in top-r (nothing to randomize).\n    """\n    _check_ref_and_basis(delta_W_ref, top_r_basis, token_set)\n    K, D_S = delta_W_ref.shape\n    torch.manual_seed(seed)\n    P = top_r_basis @ top_r_basis.T\n    ref_proj = top_r_basis @ (top_r_basis.T @ delta_W_ref)\n    ref_residual = delta_W_ref - ref_proj\n    if ref_residual.norm().item() < 1e-6 * delta_W_ref.norm().item():\n        raise RuntimeError(\n            "random_matched: reference is entirely in the top-r span; "\n            "nothing to randomize (reduce r or change reference)"\n        )\n    G = torch.randn(K, D_S, dtype=delta_W_ref.dtype)\n    G_residual = G - P @ G\n    if G_residual.norm().item() < 1e-9:\n        raise RuntimeError("random_matched: degenerate random residual")\n    delta_W = ref_proj + _scale_to_norm(G_residual, ref_residual.norm().item())\n\n    diff_proj = (P @ (delta_W - delta_W_ref)).norm().item()\n    if diff_proj > 1e-4 * delta_W_ref.norm().item():\n        raise RuntimeError(\n            f"random_matched post-check failed: top-r projection changed by "\n            f"{diff_proj:.3e} (rel to ref norm)"\n        )\n    nrm = delta_W.norm().item()\n    if abs(nrm - delta_W_ref.norm().item()) > 1e-5 * delta_W_ref.norm().item():\n        raise RuntimeError(\n            f"random_matched post-check failed: norm {nrm:.9f} vs ref "\n            f"{delta_W_ref.norm().item():.9f}"\n        )\n    return EditSpec(\n        name="random_matched",\n        delta_W=delta_W,\n        token_set=tuple(token_set),\n        target_layer="lm_head",\n        norm_frobenius=nrm,\n        projection_onto_top_r=_projection_fraction(delta_W, top_r_basis),\n        seed=seed,\n        meta={"ref_norm": delta_W_ref.norm().item(), "r": top_r_basis.shape[1]},\n    )\n\n\ndef make_tail_only(\n    delta_W_ref: torch.Tensor,\n    top_r_basis: torch.Tensor,\n    token_set: Sequence[int],\n) -> EditSpec:\n    """Reference minus its top-r projection — the tail ESG modes only\n    (TZ §4.3).\n\n    NOT renormalized: tail-only is exactly "what was left", not\n    "as if the full norm". ‖delta_W‖_F = ‖ref_residual‖_F ≤\n    ‖ref‖_F (strict if ref has a top-r component).\n\n    Control semantics: if the Goodhart effect appears in both\n    readout_pull and tail_only (but not random_ortho), the reference\'s\n    STRUCTURE matters, not just tail landing; if only in readout_pull —\n    the top-r component matters.\n\n    Errors: ValueError as in random_ortho; RuntimeError if ref is\n    entirely in top-r (nothing to keep).\n    """\n    _check_ref_and_basis(delta_W_ref, top_r_basis, token_set)\n    ref_proj = top_r_basis @ (top_r_basis.T @ delta_W_ref)\n    delta_W = delta_W_ref - ref_proj\n    if delta_W.norm().item() < 1e-6 * delta_W_ref.norm().item():\n        raise RuntimeError(\n            "tail_only: reference is entirely in the top-r span; "\n            "nothing to keep (reduce r or change reference)"\n        )\n    return EditSpec(\n        name="tail_only",\n        delta_W=delta_W,\n        token_set=tuple(token_set),\n        target_layer="lm_head",\n        norm_frobenius=delta_W.norm().item(),\n        projection_onto_top_r=_projection_fraction(delta_W, top_r_basis),\n        seed=None,\n        meta={"ref_norm": delta_W_ref.norm().item(), "r": top_r_basis.shape[1]},\n    )\n\n\ndef _check_ref_and_basis(delta_W_ref: torch.Tensor, top_r_basis: torch.Tensor,\n                         token_set: Sequence[int]) -> None:\n    K = len(token_set)\n    if delta_W_ref.dim() != 2 or delta_W_ref.shape[0] != K:\n        raise ValueError(\n            f"delta_W_ref must be 2-D with rows = len(token_set) = {K}, "\n            f"got {tuple(delta_W_ref.shape)}"\n        )\n    if top_r_basis.shape[0] != K:\n        raise ValueError(\n            f"top_r_basis rows ({top_r_basis.shape[0]}) != len(token_set) ({K})"\n        )\n    _validate_orthonormal_columns(top_r_basis)\n\n\ndef _validate_edit_spec(edit: EditSpec, D_S: int) -> None:\n    """TZ §4.4: check target_layer and delta_W.shape against the model."""\n    if edit.target_layer != "lm_head":\n        raise NotImplementedError(\n            f"only target_layer=\'lm_head\' supported, got {edit.target_layer!r}"\n        )\n    if edit.delta_W is not None and \\\n            edit.delta_W.shape != (len(edit.token_set), D_S):\n        raise ValueError(\n            f"delta_W shape {tuple(edit.delta_W.shape)} != "\n            f"({len(edit.token_set)}, {D_S})"\n        )\n\n\n@contextlib.contextmanager\ndef apply_edit(model, edit: EditSpec) -> Iterator[None]:\n    """Apply edit in-place, restore in finally (TZ §4.3).\n\n    lm_head.weight[token_set] += delta_W under torch.no_grad(); after\n    yield — copy_ the saved slice back. requires_grad of the full weight\n    is saved/restored. `base` (delta_W=None) is a no-op (uniform API).\n\n    NOT re-entrant: nested apply_edit → RuntimeError (one edit at a\n    time). target_layer other than "lm_head" → NotImplementedError.\n\n    Errors:\n        ValueError: delta_W shape != (len(token_set), D_S).\n    """\n    global _APPLY_EDIT_ACTIVE\n    if _APPLY_EDIT_ACTIVE:\n        raise RuntimeError(\n            "apply_edit is not re-entrant: one edit at a time "\n            "(nested apply_edit detected)"\n        )\n\n    weight = _resolve_lm_head_weight(model)\n    _validate_edit_spec(edit, weight.shape[1])\n    ts = torch.tensor(edit.token_set, device=weight.device)\n    delta = edit.delta_W.to(weight) if edit.delta_W is not None else None\n\n    W_orig = weight[ts].detach().clone()\n    rg_was = weight.requires_grad\n    _APPLY_EDIT_ACTIVE = True\n    try:\n        if delta is not None:\n            with torch.no_grad():\n                # .data.index_add_: in-place. (weight[ts].add_(delta) is a\n                # no-op — advanced indexing returns a copy; plain weight.\n                # index_add_ fails on a grad-requiring leaf even under\n                # no_grad.)\n                weight.data.index_add_(0, ts, delta)\n        yield\n    finally:\n        with torch.no_grad():\n            weight.data.index_copy_(0, ts, W_orig)\n        weight.requires_grad_(rg_was)\n        _APPLY_EDIT_ACTIVE = False\n\n\ndef _atomic_torch_save(obj: Any, final_path: str) -> None:\n    d = os.path.dirname(final_path) or "."\n    os.makedirs(d, exist_ok=True)\n    tmp = f"{final_path}.tmp.{os.getpid()}.{uuid.uuid4().hex[:8]}"\n    torch.save(obj, tmp)\n    os.replace(tmp, final_path)\n\n\ndef apply_edit_and_cache(\n    model,\n    edit: EditSpec,\n    samples: list[dict],\n    layers: list[int],\n    cache_dir: str,\n    overwrite: bool = False,\n) -> dict:\n    """Forward under the edit; cache h_t [S, D_S] and p_t [S, K] (TZ §4.3).\n\n    Layout: cache_dir/{edit.name}/L{layer:02d}/doc{doc:04d}_{h,p}.pt\n    (+ meta.json per (edit, layer, doc)). h stored in float32 (downstream\n    convert to model dtype; fp32 on disk is precise and uniform), p in\n    float32 (softmax precision for gauge-free P(p)).\n\n    J is NOT cached (decision #4): reconstruct via\n    compute_jacobian_for_tokens(h_override=h_t[l][t]) + SVDCache in the\n    experiment — ~30× less disk, any r without recompute. ``h_t[l][t]`` is the\n    OUTPUT of layer l (hidden_states[l+1]) at position t, which is exactly what\n    ``compute_jacobian_for_tokens(layer_idx=l)`` takes; it is only consistent\n    with ``input_ids`` at position ``t = S-1`` — for other positions pass a\n    prefix (see experiments/_exp_common.py, audit B2).\n\n    No _clone_gdn_states_on_entry here: no_grad, no backward — in-place\n    lm_head mutation is safe for the GDN recurrent state.\n\n    overwrite=False: existing files are kept, but their meta is checked\n    for consistency (edit/layer/doc/S/D_S/K) — mismatch → RuntimeError.\n    Partial cache (some of h/p/meta missing): existing files are kept,\n    missing ones are written, meta is checked if present.\n    overwrite=True: atomic (tmp + os.replace) rewrite.\n\n    Returns:\n        {"root", "edit", "n_docs", "layers", "S", "D_S", "K", "files"}.\n\n    Errors:\n        ValueError: empty samples; layer index out of [0, n_layers);\n            samples with differing S.\n        RuntimeError: read-only cache_dir with overwrite=True (Kaggle\n            FUSE) — explicit, not silent.\n    """\n    if not samples:\n        raise ValueError("samples is empty")\n    S = samples[0]["input_ids"].shape[1]\n    for sample in samples:\n        if sample["input_ids"].shape[1] != S:\n            raise ValueError(\n                f"all samples must have the same S, expected {S}, got "\n                f"{sample[\'input_ids\'].shape[1]}"\n            )\n    n_layers = _n_layers(model)\n    for L in layers:\n        if not (isinstance(L, int) and 0 <= L < n_layers):\n            raise ValueError(f"layer {L} out of range [0, {n_layers})")\n\n    was_training = model.training\n    model.eval()\n    weight = _resolve_lm_head_weight(model)\n    K = len(edit.token_set)\n    D_S = weight.shape[1]\n    ts = torch.tensor(edit.token_set, device=weight.device)\n    edit_root = os.path.join(cache_dir, edit.name)\n\n    files: list[str] = []\n    try:\n        _cache_edit(model, edit, samples, layers, ts, S, D_S, K, edit_root,\n                    overwrite, files, cache_dir)\n    finally:\n        model.train(was_training)\n\n    return {\n        "root": cache_dir,\n        "edit": edit.name,\n        "n_docs": len(samples),\n        "layers": list(layers),\n        "S": S,\n        "D_S": D_S,\n        "K": K,\n        "files": files,\n    }\n\n\ndef _cache_edit(model, edit, samples, layers, ts, S, D_S, K, edit_root,\n                overwrite, files, cache_dir) -> None:\n    with apply_edit(model, edit):\n        for doc_idx, sample in enumerate(samples):\n            with torch.no_grad():\n                out = model(**sample, output_hidden_states=True, use_cache=False)\n            h_states = out.hidden_states\n            # p_t = softmax over the token_set (re-normalized on K) — the\n            # p_S distribution of paper.md §4; required for gauge-free\n            # P(p) = diag(p) − pp^T (P(p)·1 = 0 iff Σp = 1).\n            p_t = F.softmax(out.logits[0][:, ts].float(), dim=-1)  # [S, K]\n            for L in layers:\n                # hidden_states[0] = embeddings, hidden_states[l+1] = OUTPUT of\n                # layer l. The consumer contract (docstring: reconstruct via\n                # compute_jacobian_for_tokens(layer_idx=L), which reads\n                # hidden_states[L+1]) needs the OUTPUT of L. Writing h_states[L]\n                # cached the INPUT of L = output of L-1, i.e. the linearization\n                # point of the neighbouring layer, while the tail replay skipped\n                # L entirely (audit docs/audit4.pdf B1: J off by 10.7 %).\n                h_t = h_states[L + 1][0].float()            # [S, D_S] float32\n                layer_dir = os.path.join(edit_root, f"L{L:02d}")\n                meta = {\n                    "edit": edit.name,\n                    "layer": L,\n                    "doc": doc_idx,\n                    "S": S,\n                    "D_S": D_S,\n                    "K": K,\n                    "model_dtype": str(model.dtype),\n                    "torch": torch.__version__,\n                    "transformers": _transformers_version(),\n                    "seed": edit.seed,\n                    # Cache-layout marker: caches written before the B1 fix\n                    # carry h = hidden_states[L] (input of L). They have no\n                    # "h_source" key, so the meta check below rejects them\n                    # instead of silently feeding layer-L-1 activations.\n                    "h_source": "hidden_states[L+1]",\n                }\n                h_path = os.path.join(layer_dir, f"doc{doc_idx:04d}_h.pt")\n                p_path = os.path.join(layer_dir, f"doc{doc_idx:04d}_p.pt")\n                meta_path = os.path.join(layer_dir, f"doc{doc_idx:04d}_meta.json")\n\n                if not overwrite:\n                    if os.path.exists(meta_path):\n                        _check_meta_consistency(meta_path, meta)\n                    if os.path.exists(h_path) and os.path.exists(p_path) \\\n                            and os.path.exists(meta_path):\n                        files.extend([h_path, p_path, meta_path])\n                        continue\n                try:\n                    # overwrite=False: keep existing files, write missing ones.\n                    if overwrite or not os.path.exists(h_path):\n                        _atomic_torch_save({"h": h_t, "meta": meta}, h_path)\n                    if overwrite or not os.path.exists(p_path):\n                        _atomic_torch_save({"p": p_t, "meta": meta}, p_path)\n                    if overwrite or not os.path.exists(meta_path):\n                        _atomic_torch_save_text(\n                            json.dumps(meta, indent=2, sort_keys=True), meta_path\n                        )\n                except OSError as e:\n                    raise RuntimeError(\n                        f"cannot write cache to {cache_dir!r} (read-only / Kaggle "\n                        f"FUSE?): {e}. The existing cache is incomplete "\n                        f"(overwrite={overwrite}); use a writable dir."\n                    ) from e\n                files.extend([h_path, p_path, meta_path])\n\n\ndef _atomic_torch_save_text(text: str, final_path: str) -> None:\n    d = os.path.dirname(final_path) or "."\n    os.makedirs(d, exist_ok=True)\n    tmp = f"{final_path}.tmp.{os.getpid()}.{uuid.uuid4().hex[:8]}"\n    with open(tmp, "w") as f:\n        f.write(text)\n    os.replace(tmp, final_path)\n\n\ndef _check_meta_consistency(meta_path: str, expected: dict) -> None:\n    with open(meta_path) as f:\n        existing = json.load(f)\n    # dtype/torch/transformers are checked too (docs/E38_fixes.md §2): a\n    # silent dtype switch between runs (e.g. fp16 -> fp32) would otherwise\n    # mix stale and fresh cache files without any error, since\n    # overwrite=False only writes missing files.\n    for key in ("edit", "layer", "doc", "S", "D_S", "K",\n                "model_dtype", "torch", "transformers", "h_source"):\n        if existing.get(key) != expected[key]:\n            raise RuntimeError(\n                f"cache meta mismatch for {meta_path}: {key} = "\n                f"{existing.get(key)!r} != expected {expected[key]!r} "\n                f"(pass overwrite=True to rewrite)"\n            )\n\n\ndef _n_layers(model) -> int:\n    cfg = getattr(model, "config", None)\n    tc = getattr(cfg, "text_config", cfg)\n    return int(tc.num_hidden_layers)\n\n\ndef _transformers_version() -> str:\n    import transformers\n    return transformers.__version__\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/invariants.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""M0 assert-hub: registry of cross-module identities (ТЗ E40_math_core §II.2).\n\nRun the whole registry after ANY kernel change; a status change is an event\nfor the artifact manifest. Every entry must be FALSIFIABLE — e.g.\n``channels_sum`` only counts when ``E_resid``/``E_off_manifold`` are computed\nindependently (norms of matrix differences), never as conservation leftovers\n(ТЗ §III.8, E40_ans §2/§7).\n\nStatuses: ``passing`` | ``failing`` | ``blocked``. The ``triple_identity``\nentry was ``blocked_on_E6`` in the ТЗ; E6 was closed 2026-11-03 (the published\ntable mixed slices; data were self-consistent), so it is ``passing`` here —\nthe check enforces the LESSON: the identity is asserted on ONE aggregation,\nnot across median/mean mixes (paper §6 item 23).\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Callable\n\nimport numpy as np\n\nfrom .kinetic import kinetic_decomposition\nfrom .operator_checks import O1, apply_P, c, cos_J_PJ, frac_K, s\nfrom .tag import make_tag\nfrom .p_algebra import (\n    P_matrix, g_uni, gauge_share, isotropic_reference, secular_residual,\n    secular_roots, trace_P, trace_P2,\n)\nfrom .transport import holonomy\n\n__all__ = ["INVARIANTS", "Invariant", "run_invariants"]\n\n\n@dataclass\nclass Invariant:\n    identity: str\n    modules: tuple[str, ...]\n    check: Callable[[np.random.Generator], None]     # raises AssertionError\n    status: str = "passing"\n    note: str = ""\n\n\ndef _fixtures(rng):\n    K, D, n = 32, 16, 8\n    x = rng.standard_normal((K,)) * 2.0\n    p = np.exp(x) / np.exp(x).sum()\n    J = rng.standard_normal((n, K, D))\n    return K, D, p, J\n\n\ndef _chk_O2_eq_O3(rng):\n    K, D, p, J = _fixtures(rng)\n    g = g_uni(J)\n    G = sum(Jt @ Jt.T for Jt in J)\n    gs = gauge_share(G)\n    assert abs(g - gs) <= 1e-12 + 1e-10 * abs(gs), f"O2≡O3: {g!r} vs {gs!r}"\n\n\ndef _chk_g_uni_gf_zero(rng):\n    K, D, p, J = _fixtures(rng)\n    g = g_uni(np.stack([apply_P(Jt, p) for Jt in J]))\n    assert g < 1e-12, f"g_uni(J^gf) = {g:g} ≠ 0"\n\n\ndef _chk_P_properties(rng):\n    K, D, p, J = _fixtures(rng)\n    P = P_matrix(p)\n    assert np.abs(P @ np.ones(K)).max() < 1e-12, "P·1 ≠ 0"\n    assert np.linalg.eigvalsh(P).min() >= -1e-12, "P not PSD"\n    # non-projector, falsifiable at interior p: uniform p gives P² = (1/K)P.\n    # (Vertex p = e_i ⇒ P = 0 IS idempotent — the boundary excluded here.)\n    pu = np.full(K, 1.0 / K)\n    Pu = P_matrix(pu)\n    assert np.abs(Pu @ Pu - Pu).max() > 1e-6, "P looks like a projector at uniform p"\n    assert np.abs(Pu @ Pu - Pu / K).max() < 1e-12, "P² ≠ (1/K)P at uniform p"\n\n\ndef _chk_iso_inline_eq_closed(rng):\n    K, D, p, J = _fixtures(rng)\n    inline = float((P_matrix(p) ** 2).sum()) / K\n    closed = isotropic_reference(p)\n    assert abs(inline - closed) <= 1e-9 * closed, f"iso {inline} vs {closed}"\n\n\ndef _chk_p_tilde_sum(rng):\n    K, D, p, J = _fixtures(rng)\n    mass = float(rng.uniform(0.05, 0.95))              # U128 coverage of a position\n    raw = p * mass                                     # sub-stochastic row (P128_RAW)\n    tilde = raw / raw.sum()                            # renormalized (P128)\n    assert abs(raw.sum() - mass) < 1e-12\n    assert abs(tilde.sum() - 1.0) < 1e-12, "Σp̃ ∉ {{1, 1 − mass}}"\n\n\ndef _chk_secular(rng):\n    K, D, p, J = _fixtures(rng)\n    p = p + rng.uniform(1e-3, 1e-2, size=K)            # force distinct p_i\n    p = p / p.sum()\n    lam = secular_roots(p)\n    ev = np.sort(np.linalg.eigvalsh(P_matrix(p)))[::-1]\n    for x in lam:\n        # f′ ~ Σp²/(p−λ)² explodes near close poles; 1e-4 is the honest\n        # tolerance for f AT the bisection root — the real identity check is\n        # the eigvalsh match below.\n        assert abs(secular_residual(p, x)) < 1e-4, f"residual {x}: {secular_residual(p, x)}"\n    assert np.abs(lam - ev[: K - 1]).max() < 1e-8, "secular roots ≠ eigvalsh"\n\n\ndef _chk_trace_identities(rng):\n    K, D, p, J = _fixtures(rng)\n    P = P_matrix(p)\n    assert abs(float(np.trace(P)) - trace_P(p)) < 1e-12\n    assert abs(float(np.trace(P @ P)) - trace_P2(p)) < 1e-9\n\n\nfrom scipy.linalg import expm\n\n\ndef _rot(rng, r, angle):\n    M = rng.standard_normal((r, r))\n    A = M - M.T\n    mu = np.abs(np.linalg.eigvalsh(1j * A)).max()      # skew eigenvalue scale\n    return expm(angle / mu * A)                        # max rotation angle = angle\n\n\ndef _frames(rng, r=4, d=24, angle=0.15):\n    A = rng.standard_normal((d, r))\n    Q, _ = np.linalg.qr(A)\n    return Q, Q @ _rot(rng, r, angle)\n\n\ndef _chk_H_orthogonal(rng):\n    V0, V1 = _frames(rng)\n    V2 = V1 @ _rot(rng, 4, 0.2)\n    H = holonomy(V0, V1, V2)\n    assert np.abs(H @ H.T - np.eye(H.shape[0])).max() < 1e-10, "holonomy not orthogonal"\n\n\ndef _blkrot(theta: float) -> np.ndarray:\n    """2×2 rotation by θ — a paired-spectrum building block with det = +1."""\n    return np.array([[np.cos(theta), -np.sin(theta)],\n                     [np.sin(theta), np.cos(theta)]])\n\n\ndef _chk_kappa_curv_algebra(rng):\n    """κ_curv² = 2r − 2 tr H — UNCONDITIONAL for any H ∈ O(r) (патч v2 §1.4).\n    Small-angle holonomy H and large-angle H both pass — no spectral guard."""\n    V0, V1 = _frames(rng)\n    V2 = V1 @ _rot(rng, V1.shape[1], 2.1)\n    for H in (holonomy(V0, V1, V2), np.eye(2)):        # large-angle + H = I (κ=0)\n        r = H.shape[0]\n        lhs = float(np.linalg.norm(H - np.eye(r), "fro") ** 2)\n        rhs = 2.0 * r - 2.0 * float(np.trace(H))\n        assert abs(lhs - rhs) < 1e-10 * max(1.0, rhs), f"κ_curv algebra: {lhs} vs {rhs}"\n\n\ndef _chk_kappa_curv_spectral(rng):\n    """κ_curv² = 4Σ(1−cosθⱼ) — CONDITIONAL: needs paired spectrum\n    (r even, det H = +1, no eigenvalues ±1). Guarded precondition (патч v2 §1.4)."""\n    thetas = rng.uniform(0.05, 2.5, size=3)            # strictly inside (0, π)\n    H = np.zeros((6, 6))\n    for j, th in enumerate(thetas):\n        H[2 * j:2 * j + 2, 2 * j:2 * j + 2] = _blkrot(th)\n    r = H.shape[0]\n    assert r % 2 == 0 and abs(np.linalg.det(H) - 1.0) < 1e-10\n    ev = np.linalg.eigvals(H)\n    assert np.min(np.abs(np.abs(ev) - 1.0)) < 1e-9      # all on unit circle\n    assert np.min(np.abs(ev - 1.0)) > 1e-6 and np.min(np.abs(ev + 1.0)) > 1e-6\n    lhs = float(np.linalg.norm(H - np.eye(r), "fro") ** 2)\n    rhs = 4.0 * float(np.sum(1.0 - np.cos(thetas)))\n    assert abs(lhs - rhs) < 1e-10 * max(1.0, rhs)\n    # negative control: the identity FAILS for H with a −1 eigenvalue\n    Hbad = np.diag([-1.0, 1.0, 1.0, 1.0, 1.0, 1.0]) @ H\n    bad = float(np.linalg.norm(Hbad - np.eye(6), "fro") ** 2) - 4.0 * float(\n        np.sum(1.0 - np.cos(np.concatenate([thetas, [0.0]]))))\n    assert abs(bad) > 1e-6, "spectral identity must NOT close through θ=0/π"\n\n\ndef _chk_kappa_cond_invariance(rng):\n    """κ_cond(OᵀGO) = κ_cond(G) for orthogonal O (патч v2 §1.4). Tautological\n    for exact arithmetic (σ(OᵀGO)=σ(G)); guards our cond usage, not the math."""\n    A = rng.standard_normal((8, 8))\n    G = A @ A.T + 1e-3 * np.eye(8)\n    O, _ = np.linalg.qr(rng.standard_normal((8, 8)))\n    assert abs(np.linalg.cond(O.T @ G @ O) - np.linalg.cond(G)) < 1e-8\n\n\ndef _kin_pair(rng, K=12, D=24, r=4, angle=0.12):\n    U0, S0, W0t = np.linalg.svd(rng.standard_normal((K, D)), full_matrices=False)\n    U0r, V0, S0r = U0[:, :r], W0t[:r].T, S0[:r]      # J = U0r diag(S0r) V0ᵀ\n    S1 = S0r * np.exp(rng.normal(0, 0.002, r))       # radial part ≪ angular, so\n    #                                     eps3/‖ΔJ‖² ~ O(θ³) is inside the 1e-2\n    #                                     calibration band of E40_ans §4\n    Jt = U0r @ np.diag(S0r) @ V0.T\n    # deterministic rank-1 drift OUT of span(U0)⊗span(V0) (next null directions),\n    # so E_off_manifold is nonzero but closes exactly (off ⊥ frame channels):\n    up, vp, sp = U0[:, r], W0t[r], float(S0r[-1]) / 3.0   # below the r-th σ\n    J1 = ((U0r @ _rot(rng, r, angle)) @ np.diag(S1) @ (V0 @ _rot(rng, r, angle)).T\n          + sp * np.outer(up, vp))\n    return Jt, J1\n\n\ndef _chk_channels_sum(rng):\n    Jt, J1 = _kin_pair(rng)\n    dJ2 = float(((J1 - Jt) ** 2).sum())\n    out = kinetic_decomposition(Jt, J1, r=4)\n    six = (out["E_radial"] + out["E_right"] + out["E_left"] + out["E_cross"]\n           + out["E_resid"] + out["E_off_manifold"])\n    assert out["flags"]["linearization_valid"], "fixture must be in the valid regime"\n    assert abs(dJ2 - six) / dJ2 < 1e-2, f"six-channel closure: {out[\'closure_rel\']:.2e}"\n    # non-vacuity: E_resid is an independent matrix difference, and the Gram\n    # accounting sums to ‖ΔJ‖² exactly (channel_energy_matrix does that).\n    assert out["E_resid"] > 0, "E_resid == 0 exactly? suspect conservation form"\n\n\ndef _chk_r1_cross_zero(rng):\n    Jt, J1 = _kin_pair(rng, r=4)\n    out = kinetic_decomposition(Jt, J1, r=1)\n    assert out["E_cross"] is not None and abs(out["E_cross"]) < 1e-12, \\\n        f"r=1 cross = {out[\'E_cross\']}"\n    assert out["E_right"] < 1e-24 and out["E_left"] < 1e-24, "r=1 angular energies ≠ 0"\n\n\ndef _chk_O1_eq_s2(rng):\n    K, D, p, J = _fixtures(rng)\n    assert abs(O1(J, p) - s(J, p) ** 2) < 1e-12, "O1 ≠ s²"\n\n\ndef _chk_b_lin_r1(rng):\n    # lazy import: the behavioral module is GATED out of tier-0 kernel imports\n    # (tests/test_nulls_tag_gate.py::test_behavior_gate_no_import_in_kernel_and_tier0);\n    # the hub may execute it without importing\n    # it into the kernel namespace at load time.\n    from .behavior import b_lin\n    K, D, p, J = _fixtures(rng)\n    Jm = J[0]\n    _, S, Vt = np.linalg.svd(Jm, full_matrices=False)\n    v1 = Vt[0]\n    eps = 0.037\n    got = b_lin(Jm, eps, v1)\n    want = eps ** 2 * S[0] ** 2\n    assert abs(got - want) <= 1e-9 * want, f"b_lin(r=1): {got} vs {want}"\n\n\ndef _chk_triple_identity(rng):\n    K, D, p, J = _fixtures(rng)\n    got, want = frac_K(J, p), 1.0 - 2.0 * c(J, p) + s(J, p) ** 2\n    assert abs(got - want) < 1e-12, f"frac_K identity: {got!r} vs {want!r}"\n    # the cos is the SAME aggregation: cos = c/s on aggregates\n    assert abs(cos_J_PJ(J, p) - c(J, p) / s(J, p)) < 1e-12\n\n\ndef _chk_same_slice_aggregation(rng):\n    """ТЗ §II.2 (ревизия 2): slice+aggregation identity-relevant in the tag; the\n    triple identity must NOT close across mixed aggregations (test IV.6)."""\n    t = make_tag(protocol="natural", slice="raw", p_mode="truncated_U128",\n                 p_slice="p_t", norm_type="energy")\n    assert t.aggregation == "ratio_of_means", "default must be the identity-closing one"\n    assert t.slice == "raw"\n    try:\n        make_tag(protocol="natural", slice="raw", p_mode="truncated_U128",\n                 norm_type="energy", aggregation=None)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("aggregation=None must be rejected (identity-relevant)")\n    # патч v2 §1.6: p_mode has no "other"; triple reporting needs p_slice.\n    try:\n        make_tag(protocol="natural", slice="raw", p_mode="other",\n                 norm_type="energy")\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("p_mode=\'other\' must be a stop condition, not a tag")\n    from .tag import require_triple_tag\n    try:\n        require_triple_tag(make_tag(protocol="natural", slice="raw",\n                                    p_mode="truncated_U128", norm_type="energy"))\n    except ValueError:\n        pass\n    else:\n        raise AssertionError("triple number without p_slice must be rejected")\n    require_triple_tag(t)\n    # negative half (IV.6): mixing median c/s with aggregated frac_K breaks the identity\n    K, D, p, J = _fixtures(rng)\n    mixed = 1.0 - 2.0 * c(J, p, "median") + s(J, p, "median") ** 2\n    assert abs(frac_K(J, p) - mixed) > 1e-9, "mixed aggregations must NOT close the identity"\n\n\ndef _chk_r_can_definition(rng):\n    """R_can must be the λ-share of the CANONICAL slice E_K = (PJ)(PJ)ᵀ\n    (ТЗ §8.1, paper §2.11.1) — not of (I−P)J, which is ≈ raw J (audit M1)."""\n    from .profiles import R_can\n    K, D, p, J = _fixtures(rng)\n    Jm = J[0]\n    PJ = apply_P(Jm, p)\n    G = PJ @ PJ.T\n    want = float(np.linalg.eigvalsh(G)[-1] / np.trace(G))\n    got = R_can(Jm, p)\n    assert abs(got - want) < 1e-12, f"R_can: {got} vs {want}"\n    # negative half: the (I−P)J slice is a DIFFERENT object and must not agree\n    Kc = Jm - PJ\n    Gc = Kc @ Kc.T\n    other = float(np.linalg.eigvalsh(Gc)[-1] / np.trace(Gc))\n    assert abs(got - other) > 1e-6, "R_can collapsed onto the (I−P)J slice again"\n\n\ndef _chk_agg_closure_matrix(rng):\n    """Documented closure matrix (audit M3): the triple identity closes for\n    ratio_of_means and per-position, NOT for mean (Jensen on s) — the kernel\n    must behave as its docstrings claim. The two closures are written with\n    DIFFERENT members on purpose: ``per_position`` deliberately ignores ``root``\n    and returns RAW energy ratios (``s(..., "per_position") == O1(...)``), so\n    positionally the identity is ``frac_t = 1 − 2c_t + O1_t``, while pooled it is\n    ``frac = 1 − 2c + s²`` with s the root of the pooled ratio."""\n    K, D, p, J = _fixtures(rng)\n    got = frac_K(J, p, "ratio_of_means")\n    want = 1.0 - 2.0 * c(J, p, "ratio_of_means") + s(J, p, "ratio_of_means") ** 2\n    err = float(np.abs(got - want).max())\n    assert err < 1e-12, f"ratio_of_means must close, err={err}"\n    got = frac_K(J, p, "per_position")\n    want = (1.0 - 2.0 * c(J, p, "per_position")\n            + O1(J, p, "per_position"))\n    err = float(np.abs(got - want).max())\n    assert err < 1e-12, f"per_position must close, err={err}"\n    assert float(np.abs(s(J, p, "per_position")\n                       - O1(J, p, "per_position")).max()) < 1e-15, \\\n        "per_position must stay root-free (raw energy ratios)"\n    # mean: s(mean) is the mean of the per-position ROOT — Jensen gap is real\n    assert s(J, p, "mean") ** 2 <= O1(J, p, "mean") + 1e-15, "Jensen direction flipped"\n    assert abs(frac_K(J, p, "mean") - (1.0 - 2.0 * c(J, p, "mean")\n                                       + s(J, p, "mean") ** 2)) > 1e-12, \\\n        "mean closure must NOT hold silently (it would mask the Jensen gap)"\n\n\ndef _chk_mp_edge_labelled(rng):\n    """MP edges under the XᵀX/n convention, pinned with n (ТЗ §III.11; the\n    retracted 1.4425/1.45 pair must not creep back — audit M8)."""\n    from .nulls_and_stats import mp_upper_edge\n    assert abs(mp_upper_edge(6, 32) - 2.053525) < 1e-6\n    assert abs(mp_upper_edge(8, 32) - 2.250000) < 1e-6\n    assert abs(mp_upper_edge(6, 192) - 1.384803) < 1e-6\n    assert abs(mp_upper_edge(8, 192) - 1.449915) < 1e-6\n\n\ndef _chk_svd_bundle_contract(rng):\n    """`svd_bundle` величин фазы (E-40.2/0.4 их отчитывают), а их в реестре не\n    было — пробел M13. Пиним: бандл реконструирует J, `energy` = доля топ-r, нулевой\n    спектр имеет ранг 0 (Q1), `principal_angles` требует ортонормированный фрейм\n    (M10), и `stable` — σ-форма, а НЕ σ²-форма `kinetic.rotation_unique` (M9)."""\n    from .svd_bundle import effective_rank, principal_angles, svd_bundle\n    K, D, p, J = _fixtures(rng)\n    Jm = J[0]\n    b = svd_bundle(Jm, rank=4)\n    full = svd_bundle(Jm, rank=min(K, D))            # ранг = полная размерность ⇒ J точно\n    rec = (full["U"] * full["Sigma"]) @ full["V"].T\n    assert np.linalg.norm(Jm - rec) / np.linalg.norm(Jm) < 1e-10, "bundle must rebuild J"\n    assert b["energy"] < 1.0 - 1e-9, "top-r energy share of a full-rank J must be < 1"\n    s_full = np.linalg.svd(Jm, compute_uv=False)\n    want = float((b["Sigma"] ** 2).sum() / (s_full ** 2).sum())\n    assert abs(b["energy"] - want) < 1e-12, "energy must be the top-r σ² share"\n    assert effective_rank(np.zeros(5)) == 0, "zero spectrum must report rank 0"\n    V = full["V"]\n    ang = principal_angles(V, V)\n    assert np.all(ang > 1 - 1e-9), "principal_angles(V, V) must be ~1"\n    try:\n        principal_angles(V * 2.0, V)                 # non-orthonormal frame\n        raise AssertionError("principal_angles accepted a non-orthonormal frame")\n    except ValueError:\n        pass\n    # M9: на [1.0, 0.991] конвенции σ и σ² расходятся по построению — это пинит,\n    # что `stable` остаётся σ-формой (E-38c gap_min), а не тихой σ²-формой.\n    Jn = np.zeros((8, 8))\n    Jn[0, 0], Jn[1, 1] = 1.0, 0.991\n    st = svd_bundle(Jn, rank=2)["stable"]\n    sigma_form = abs(1.0 - 0.991) > 1e-2                        # False\n    sigma2_form = abs(1.0 - 0.991 ** 2) / 1.0 > 1e-2            # True\n    assert st == sigma_form and sigma_form != sigma2_form, \\\n        "svd_bundle.stable must stay the σ-form (M9 conventions must not merge)"\n\n\nINVARIANTS: dict[str, Invariant] = {\n    "O2_eq_O3": Invariant("g_uni(J) == gauge_share(G=ΣJJᵀ)", ("p_algebra",),\n                          _chk_O2_eq_O3),\n    "g_uni_gf_zero": Invariant("g_uni(P(p)J) ≡ 0", ("p_algebra", "operator_checks"),\n                               _chk_g_uni_gf_zero),\n    "P_properties": Invariant("P·1=0, P⪰0, P²≠P (interior p; vertex P=0 excluded)",\n                              ("p_algebra",), _chk_P_properties),\n    "iso_inline_eq_closed": Invariant("iso inline == iso closed form", ("p_algebra",),\n                                      _chk_iso_inline_eq_closed),\n    "p_tilde_sum": Invariant("Σp̃ ∈ {1, 1 − mass}", ("p_algebra", "tag"), _chk_p_tilde_sum),\n    "secular_equation": Invariant("Σ pᵢ²/(pᵢ−λ) = 1 at λ ∈ eig(P)∖{0}", ("p_algebra",),\n                                  _chk_secular),\n    "trace_identities": Invariant("tr P = 1−Σp²; tr P² = Σp²−2Σp³+(Σp²)²", ("p_algebra",),\n                                  _chk_trace_identities),\n    "H_orthogonal": Invariant("H Hᵀ = I", ("transport",), _chk_H_orthogonal),\n    "kappa_curv_algebra": Invariant("κ_curv² = 2r − 2trH (любая H ∈ O(r))",\n                                    ("transport",), _chk_kappa_curv_algebra),\n    "kappa_curv_spectral": Invariant("κ_curv² = 4Σ(1−cosθ) (парный спектр: r чётн., det +1, ±1 нет)",\n                                     ("transport",), _chk_kappa_curv_spectral,\n                                     note="патч v2 §1.4: условное; precondition guard + "\n                                          "negative control θ=0/π"),\n    "kappa_cond_invariance": Invariant("κ_cond(OᵀGO) = κ_cond(G), O орт.",\n                                       ("p_algebra", "spectral"), _chk_kappa_cond_invariance,\n                                       note="vacuous-by-invariance check kept per патч v2; "\n                                            "κ(OᵀHO)=κ(H) запись удалена по его же требованию"),\n    "channels_sum": Invariant("Σ6 channels + eps3 = ‖ΔJ‖², E_resid/E_off independent",\n                              ("kinetic",), _chk_channels_sum,\n                              note="6 channels per docs/E40_ans.md, not the ТЗ\'s 5"),\n    "r1_cross_zero": Invariant("r=1 ⇒ E_cross = 0", ("kinetic",), _chk_r1_cross_zero),\n    "O1_eq_s2": Invariant("O1 ≡ s² on one aggregation", ("operator_checks",), _chk_O1_eq_s2),\n    "b_lin_r1": Invariant("b_lin(v=v₁) = ε²σ₁²", ("behavior",), _chk_b_lin_r1),\n    "triple_identity": Invariant("frac_K = 1 − 2c + s² (one aggregation, one slice)",\n                                 ("cheap_invariants", "operator_checks"),\n                                 _chk_triple_identity,\n                                 note="was blocked_on_E6; E6 closed 2026-11-03"),\n    "same_slice_aggregation": Invariant("tag carries slice+aggregation; mixed aggregations "\n                                        "must not close the triple identity (IV.6)",\n                                        ("tag", "cheap_invariants"),\n                                        _chk_same_slice_aggregation,\n                                        note="NEW ТЗ §II.2 ревизия 2"),\n    "R_can_definition": Invariant("R_can = λshare((PJ)(PJ)ᵀ), ≠ λshare(((I−P)J)…) ",\n                                  ("profiles", "operator_checks"), _chk_r_can_definition,\n                                  note="NEW audit_core_math_v1 M1 (critical): wrong slice "\n                                       "made R_can ≈ R_raw and E-40.6 blind"),\n    "agg_closure_matrix": Invariant("triple closes for rom/per_position, Jensen gap for "\n                                    "mean is real and documented", ("operator_checks",),\n                                    _chk_agg_closure_matrix,\n                                    note="NEW audit M3"),\n    "mp_edge_labelled": Invariant("MP edges pinned with explicit n (2.0535/2.25/1.3848/1.4499)",\n                                  ("nulls_and_stats",), _chk_mp_edge_labelled,\n                                  note="NEW audit M8"),\n    "svd_bundle_contract": Invariant("svd_bundle rebuilds J; energy = top-r σ² share; "\n                                     "rank(0)=0; principal_angles needs an orthonormal "\n                                     "frame; stable is the σ-form",\n                                     ("svd_bundle",), _chk_svd_bundle_contract,\n                                     note="NEW audit_core_math_v1 M13 (closes the last "\n                                          "phase-magnitude module without registry cover)"),\n}\n\n\ndef run_invariants(seed: int = 0) -> dict[str, dict]:\n    """Run every registered identity on synthetic fixtures; a declared\n    ``passing`` entry that raises is a FAILURE of the hub (returned ok=False,\n    the pytest driver asserts all-ok)."""\n    out = {}\n    for name, inv in INVARIANTS.items():\n        rng = np.random.default_rng(seed + sum(map(ord, name)))\n        try:\n            inv.check(rng)\n            out[name] = {"ok": True, "detail": "ok", "declared": inv.status}\n        except AssertionError as e:\n            out[name] = {"ok": False, "detail": str(e), "declared": inv.status}\n    return out\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/jacobian.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Jacobian computation for decoder layers."""\n\nimport contextlib\nimport logging\n\nimport torch\nfrom transformers import DynamicCache\nfrom transformers.cache_utils import (  # type: ignore[import-not-found]\n    DynamicLayer,\n    LinearAttentionLayer,\n)\n\n\ndef compute_jacobian_for_tokens(\n    model,\n    layer_idx: int,\n    input_ids: torch.Tensor,\n    attention_mask: torch.Tensor,\n    token_set: list[int],\n    prefix_cache=None,\n    h_override: torch.Tensor | None = None,\n) -> torch.Tensor:\n    """Compute Jacobian of logits w.r.t. hidden state at layer_idx.\n\n    Args:\n        model: Qwen3_5ForCausalLM instance.\n        layer_idx: Layer index (0-based) to compute Jacobian at.\n        input_ids: [1, S] token IDs.\n        attention_mask: [1, S] attention mask.\n        token_set: list of token IDs to compute gradients for.\n        prefix_cache: optional pre-computed prefix cache (K/V for positions 0..S-2).\n        h_override: if given, used as the hidden state of the last position at\n            layer_idx INSTEAD of extracting it from a full forward. The prefix\n            cache is still built from input_ids[:, :-1] (context unchanged); the\n            caller is responsible for h_override being consistent with\n            input_ids (e.g. h_override from the same sequence). Shapes: [1, 1, D],\n            [1, D] or [D]. Device must match input_ids.device (no auto-move).\n            dtype is cast to the model dtype. The tensor is cloned and detached\n            (fresh leaf; the caller\'s tensor is never mutated).\n\n    Returns:\n        J: [K, D] Jacobian matrix (K = len(token_set), unnormalized).\n\n    Interaction with compute_hidden_jacobian:\n        compute_jacobian_for_tokens takes `h_override`; compute_hidden_jacobian\n        takes `h`. Both mean "use this h instead of computing it from the\n        forward", but the names differ for historical reasons (compute_hidden_\n        jacobian already had `h`). With h_override=None the behavior is\n        bit-identical to pre-v6.0.\n    """\n    model.eval()\n    S = input_ids.shape[1]\n    # Constraint: single sequence, no padding. The replay phase builds causal\n    # masks from the cache length with attention_mask=None, so any padding\n    # token in the prefix would be silently treated as valid context.\n    if attention_mask is not None and not bool(attention_mask.all()):\n        raise ValueError(\n            "compute_jacobian_for_tokens: only unpadded single sequences are "\n            "supported (the replay phase cannot honor a padding mask)."\n        )\n    if prefix_cache is None:\n        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)\n    if h_override is None:\n        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)\n        h = h.to(next(model.parameters()).dtype).detach().requires_grad_(True)\n    else:\n        h = _normalize_h_override(h_override, model, input_ids)\n    # Re-enable grad: the decoder forward must build a graph over the leaf\n    # h AND the row loop below must run autograd.grad with grad mode ON,\n    # even when the caller is inside torch.no_grad() (e.g. eval loops in\n    # exp18/19). A no_grad caller otherwise gets "element 0 ... does not\n    # have a grad_fn" — the rows loop is part of the same enable_grad block.\n    with torch.enable_grad():\n        logits = _run_decoder_pass(model, prefix_cache, h, S, layer_idx)\n        J = _compute_jacobian_rows(logits, h, token_set)\n    return J\n\n\ndef _normalize_h_override(\n    h_override: torch.Tensor, model, input_ids: torch.Tensor\n) -> torch.Tensor:\n    """Validate + normalize h_override to a detached [1, 1, D] leaf.\n\n    Checks (in order, before any compute): shape reducible to [1, 1, D],\n    D == model hidden_size, device == input_ids.device. dtype is cast to the\n    model dtype. The tensor is cloned+detached (fresh leaf; caller\'s tensor\n    never mutated, no parasitic CopyBackwards nodes).\n    """\n    if h_override.dim() not in (1, 2, 3) or h_override.numel() == 0:\n        raise ValueError(\n            f"h_override: shape {tuple(h_override.shape)} is not reducible "\n            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"\n        )\n    if any(s != 1 for s in h_override.shape[:-1]):\n        raise ValueError(\n            f"h_override: shape {tuple(h_override.shape)} is not reducible "\n            "to [1, 1, D] (accepted: [D], [1, D], [1, 1, D])"\n        )\n    h = h_override.reshape(1, 1, -1)\n    # D lookup: flat CausalLM has config.hidden_size; the multimodal wrapper\n    # (Qwen3_5ForConditionalGeneration) exposes it on config.text_config only\n    # (the outer config is a heterogeneity spec without hidden_size).\n    D_expected = None\n    cfg = model.config\n    for cand in (getattr(cfg, "text_config", None), cfg):\n        if cand is not None and hasattr(cand, "hidden_size"):\n            D_expected = cand.hidden_size\n            break\n    if D_expected is None:\n        raise ValueError(\n            "h_override: cannot resolve model hidden_size "\n            "(neither config nor config.text_config has it)"\n        )\n    D = h.shape[-1]\n    if D != D_expected:\n        raise ValueError(\n            f"h_override: D={D} does not match model hidden_size={D_expected}"\n        )\n    if h.device != input_ids.device:\n        raise ValueError(\n            f"h_override: device {h.device} does not match "\n            f"input_ids.device {input_ids.device} (no auto-move)"\n        )\n    model_dtype = next(model.parameters()).dtype\n    # clone first, then cast: detach/clone act on the caller\'s values before\n    # any cast, and `to(same dtype)` is a no-op, so the typical fp32 case\n    # allocates exactly once (audit docs/changed_review.md, item 3).\n    return h.detach().clone().to(model_dtype).requires_grad_(True)\n\n\ndef _run_prefix_pass(\n    model, input_ids: torch.Tensor, attention_mask: torch.Tensor\n) -> DynamicCache:\n    """Prefix pass: all tokens except last -> cache."""\n    with torch.no_grad():\n        outputs = model(\n            input_ids=input_ids[:, :-1],\n            attention_mask=attention_mask[:, :-1],\n            use_cache=True,\n            output_hidden_states=False,\n        )\n    return outputs.past_key_values\n\n\ndef last_position_logits(\n    model,\n    h: torch.Tensor,\n    S: int,\n    layer_idx: int,\n    prefix_cache: DynamicCache,\n) -> torch.Tensor:\n    """Decode h (residual at position S-1, output of layer layer_idx) through the\n    remaining layers + final norm + lm_head against a prefix cache over positions\n    0..S-2 and return logits [1, V].\n\n    This is EXACTLY the path compute_jacobian_for_tokens differentiates through,\n    so with prefix_cache = _run_prefix_pass(m, ids, attn) and h =\n    hidden_states[layer_idx+1][:, -1:, :] it reproduces the full-forward\n    logits[:, -1, :] (KL ~ 1e-7, verified). Passing the cache of a FULL forward\n    (all S positions) instead double-counts the last token — the bug this helper\n    exists to prevent (audit 2026-11-05, E-11a/12a/12b)."""\n    return _run_decoder_pass(model, prefix_cache, h, S, layer_idx)\n\n\ndef _extract_hidden_state(model, input_ids, attention_mask, layer_idx) -> torch.Tensor:\n    """Full pass: extract hidden state at layer_idx for last token."""\n    with torch.no_grad():\n        outputs = model(\n            input_ids=input_ids,\n            attention_mask=attention_mask,\n            output_hidden_states=True,\n            use_cache=False,\n        )\n    # hidden_states[0] = embeddings, [l+1] = output of layer l\n    h = outputs.hidden_states[layer_idx + 1][:, -1:, :]\n    return h  # [1, 1, D]\n\n\ndef _deep_clone_cache(cache):\n    """Deep clone cache, ensuring all tensors are independent copies.\n\n    Must copy every per-layer state field, not just the obvious tensors. Both\n    layer types swallow unknown constructor kwargs, so passing\n    ``keys=`` / ``conv_states=`` / ``recurrent_states=`` silently discards the\n    state and yields an empty cache. We therefore assign attributes directly:\n\n    * DynamicLayer (full attention): keys / values / is_initialized / dtype /\n      device. Without this, ``update()`` lazy-initialises an empty cache and\n      each layer attends only to the current token.\n    * LinearAttentionLayer (GDN): conv_states / recurrent_states plus the flag\n      dicts (``has_previous_state`` etc.). Without these, GDN layers treat the\n      new token as the first in the sequence and drop all history.\n\n    All cloned state tensors are .detach()-ed: the prefix cache is pure data\n    (built under no_grad). Detaching drops pointless CopyBackwards graph nodes\n    in the enable_grad decode. The transformers 5.15 in-place cache updates\n    (update_recurrent_state\'s copy_, causal_conv1d_update\'s state update —\n    cudagraph static-address optimisation) are NOT fixed by the detach —\n    the in-place copy_ lands on the kernel\'s saved input regardless. Those\n    are handled by _clone_gdn_states_on_entry (clone at kernel entry so the\n    read object and the in-place-written object are different objects).\n\n    Extra unknown attributes are deep-copied as a forward-compat safety net.\n\n    Cache OBJECT-level attributes (e.g. ``offloading`` on 5.15.1; 4.x carried\n    ``_seen_tokens``) are copied from ``vars(cache)`` after the layer loop:\n    a fresh ``DynamicCache()`` would reset them. On 5.15.1 ``get_seq_length()``\n    is derived from ``keys.shape`` (not an object field), so the post-clone\n    assert below passes on both paths — on a version that tracks sequence\n    length in cache state it turns a silent Jacobian corruption into a loud\n    error.\n\n    ``copy.deepcopy(prefix_cache)`` DOES produce independent copies on 5.15.1\n    (tensor values are not shared). _deep_clone_cache exists for (a) detached\n    tensors (no spurious CopyBackwards in the enable_grad decode) and\n    (b) robustness to per-layer attribute renames — not because deepcopy\n    shares storage. Neither clone fixes the in-place write into a graph-saved\n    node; that is _clone_gdn_states_on_entry\'s job (see its docstring).\n    """\n    import copy as _copy\n\n    def _cl(t):\n        return t.clone().detach() if t is not None else None\n\n    new_cache = DynamicCache()\n    for idx in range(len(cache.layers)):\n        layer = cache.layers[idx]\n        if isinstance(layer, DynamicLayer):\n            # Deliberately NOT a vars()-walk: full-attention state fields are\n            # stable across versions, and a rename would fail LOUD\n            # (AttributeError), not silently corrupt the decode. The GDN\n            # branch below IS generic, because GDN state fields have\n            # demonstrably drifted between versions/backends.\n            nl = DynamicLayer()\n            nl.keys = _cl(layer.keys)\n            nl.values = _cl(layer.values)\n            nl.is_initialized = layer.is_initialized\n            nl.dtype = getattr(layer, "dtype", None)\n            nl.device = getattr(layer, "device", None)\n            for k, v in vars(layer).items():\n                if k not in vars(nl):\n                    setattr(nl, k, _copy.deepcopy(v))\n            new_cache.layers.append(nl)\n        else:\n            # GDN (LinearAttentionLayer). Clone every state field dynamically\n            # rather than hardcoding transformers-5.15.1 attribute names\n            # (number_of_states, conv_states, ...): other transformers versions\n            # on other backends (e.g. the Kaggle TPU torch 2.8 image) name some\n            # differently, and a hardcoded name crashes with AttributeError.\n            # Tensors (or dicts of tensors) are clone+detached; scalars and\n            # flag-dicts are deep-copied. This is the generic form of the old\n            # field-by-field copy and behaves identically on 5.15.1.\n            nl = LinearAttentionLayer()\n            for k, v in vars(layer).items():\n                if isinstance(v, torch.Tensor):\n                    setattr(nl, k, _cl(v))\n                elif isinstance(v, dict) and v and all(\n                    isinstance(x, torch.Tensor) for x in v.values()\n                ):\n                    setattr(nl, k, {kk: _cl(xv) for kk, xv in v.items()})\n                else:\n                    setattr(nl, k, _copy.deepcopy(v))\n            new_cache.layers.append(nl)\n    # Cache object-level state (offloading flag on 5.15.1; 4.x: _seen_tokens):\n    # a fresh DynamicCache() would reset it. ``layers`` is rebuilt above;\n    # list-valued attrs are skipped to avoid aliasing the original.\n    for k, v in vars(cache).items():\n        if k == "layers" or isinstance(v, list):\n            continue\n        setattr(new_cache, k, _copy.deepcopy(v))\n    # Guard: the clone must report the same sequence length (loud, not\n    # silent, on a version that tracks it in cache state). A bare `assert`\n    # disappears under `python -O`, turning a fail-loud check into a silent\n    # one — the class of bug this guard exists to catch (audit4 Q12).\n    seq_new, seq_old = new_cache.get_seq_length(), cache.get_seq_length()\n    if seq_new != seq_old:\n        raise RuntimeError(\n            f"cache clone lost the sequence length: {seq_new} != {seq_old} "\n            "(transformers cache layout changed? the clone is not a clone)")\n    return new_cache\n\n\n@contextlib.contextmanager\ndef _clone_gdn_states_on_entry():\n    """Clone GDN kernel state inputs at entry: read object != written object.\n\n    transformers 5.15 Qwen3_5GatedDeltaNet.forward resolves the kernels as\n    free variables of its own module (modeling_qwen3_5), so patching the\n    module attributes works — verified on 5.15.1 (calls at modeling L457/\n    L506). On\n    a version where GDN forward binds the kernels by a different route\n    (an alias imported into another module), the patch silently no-ops and\n    the in-place autograd error below resurfaces LOUD — that is the\n    detection signal; there is no silent degradation.\n\n    The recurrent loop saves the read object (version 0) for backward.\n    Afterwards update_recurrent_state does a stored.copy_(new) IN PLACE on\n    the very same object (5.15 cudagraph static-address optimisation) →\n    version 0→1 → "modified by an inplace operation" in any backward through\n    the decode (create_graph not required; the clone is also a no-op safety\n    net for create_graph=True graph retention). Cloning at kernel entry makes\n    the loop\'s saved input the fresh clone; the in-place write lands on the\n    cache\'s original (a different object) → no in-place on a saved node.\n\n    Re-entrancy: nesting is a NO-OP (counter below) — the outer context\'s\n    wrappers are already active and cloning is idempotent (clone-of-clone\n    changes nothing). Overwriting mq.<kernel> from OUTSIDE during the patch\n    window is user error and is NOT protected (the outer exit restores its\n    captured orig, discarding the overwrite).\n\n    If the Qwen3.5 module cannot be imported, a WARNING is logged ONCE per\n    interpreter (not per decode; non-Qwen3.5 builds are fine — for Qwen3.5\n    with enable_grad=True the in-place autograd error is NOT protected).\n    """\n    global _GDN_PATCH_DEPTH\n    try:\n        from transformers.models.qwen3_5 import modeling_qwen3_5 as mq\n    except Exception:\n        if not getattr(_clone_gdn_states_on_entry, "_warned", False):\n            logging.getLogger(__name__).warning(\n                "GDN monkey-patch skipped (transformers.models.qwen3_5 not "\n                "importable). If the model is Qwen3.5 with enable_grad=True, "\n                "in-place autograd errors may occur.")\n            _clone_gdn_states_on_entry._warned = True\n        yield  # non-Qwen3.5 build: nothing to patch\n        return\n\n    _GDN_PATCH_DEPTH += 1\n    if _GDN_PATCH_DEPTH > 1:\n        # Nested: outer context already patched; wrappers chain, and the\n        # clone is idempotent, so nothing to do here.\n        yield\n        _GDN_PATCH_DEPTH -= 1\n        return\n\n    orig_rec = mq.torch_recurrent_gated_delta_rule\n    orig_conv = mq.causal_conv1d_update\n\n    def rec(q, k, v, g, beta, *args, initial_state=None, output_final_state=True,\n            use_qk_l2norm_in_kernel=False, **kw):\n        # initial_state/output_final_state arrive as KEYWORDS in 5.15.1\n        # (and may be positional on other versions, hence *args); clone the\n        # bound state, forward everything else untouched. A kernel whose\n        # signature drops these names falls through to **kw and fails loud.\n        if initial_state is not None:\n            initial_state = initial_state.clone()\n        return orig_rec(q, k, v, g, beta, *args, initial_state=initial_state,\n                        output_final_state=output_final_state,\n                        use_qk_l2norm_in_kernel=use_qk_l2norm_in_kernel, **kw)\n\n    def conv(h, cs, *args, **kw):\n        # (cs, weight, [bias, activation], ...) — positional tail is\n        # version-dependent; forward everything we don\'t own.\n        if cs is not None:\n            cs = cs.clone()\n        return orig_conv(h, cs, *args, **kw)\n\n    mq.torch_recurrent_gated_delta_rule = rec\n    mq.causal_conv1d_update = conv\n    try:\n        yield\n    finally:\n        mq.torch_recurrent_gated_delta_rule = orig_rec\n        mq.causal_conv1d_update = orig_conv\n        _GDN_PATCH_DEPTH -= 1\n\n\n_GDN_PATCH_DEPTH = 0\n\n\ndef _run_decoder_pass(\n    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int\n) -> torch.Tensor:\n    """Decode h through remaining layers using prefix cache."""\n    # Clone cache tensors independently to avoid in-place mutation\n    cache_clone = _deep_clone_cache(prefix_cache)\n    # Wrapper-safe: flat CausalLM -> model.model; mm wrapper -> .language_model\n    text_model = _lm_module(model)\n\n    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)\n    text_position_ids = pos[0]\n    rope_position_ids = pos[1:]\n    position_embeddings = text_model.rotary_emb(h, rope_position_ids)\n\n    x = h\n    layer_types = _layer_types(model)\n    n_layers = _n_layers(model)\n    start = layer_idx + 1\n\n    causal_mask_mapping = _build_causal_masks(\n        text_model, text_position_ids, cache_clone, x\n    )\n\n    with _clone_gdn_states_on_entry():\n        for i in range(start, n_layers):\n            layer_type = layer_types[i]\n            attn_mask = causal_mask_mapping[layer_type]\n            x = text_model.layers[i](\n                x,\n                position_embeddings=position_embeddings,\n                attention_mask=attn_mask,\n                position_ids=text_position_ids,\n                past_key_values=cache_clone,\n            )\n\n    x = text_model.norm(x)\n    logits = model.lm_head(x[:, -1, :])\n    return logits\n\n\ndef _build_causal_masks(text_model, text_pos, cache, x):\n    """Build causal mask mapping for a single token with cache."""\n    from transformers.masking_utils import (\n        create_causal_mask,\n        create_recurrent_attention_mask,\n    )\n\n    dummy_embeds = x.new_zeros(1, 1, text_model.config.hidden_size)\n    mask_kwargs = {\n        "config": text_model.config,\n        "inputs_embeds": dummy_embeds,\n        "attention_mask": None,\n        "past_key_values": cache,\n        "position_ids": text_pos,\n    }\n    return {\n        "full_attention": create_causal_mask(**mask_kwargs),\n        "linear_attention": create_recurrent_attention_mask(**mask_kwargs),\n    }\n\n\ndef _compute_jacobian_rows(\n    logits: torch.Tensor, h: torch.Tensor, token_set: list[int]\n) -> torch.Tensor:\n    """Compute Jacobian rows via ONE batched backward (is_grads_batched).\n\n    Replaces the per-row autograd.grad loop (K backwards through the\n    decoder tail) with a single vmap-style pass: grad_outputs=eye(K)\n    gives all K rows at once, 3-15x faster on T4 (review point 5).\n    Verified identical to the loop on synthetic linear + nonlinear graphs;\n    on the real GDN decoder tail this is an EMPIRICAL property of the\n    transformers 5.15 kernels (torch_recurrent_gated_delta_rule / causal\n    conv1d update under vmap-style batched backward) — there is no\n    per-row fallback, so a transformers upgrade that breaks this shows up\n    as a runtime error, not silent degradation.\n    """\n    K = len(token_set)\n    return torch.autograd.grad(\n        outputs=logits[0, token_set],\n        inputs=h,\n        grad_outputs=torch.eye(K, device=h.device, dtype=h.dtype),\n        is_grads_batched=True,\n        retain_graph=False,\n    )[0][:, 0, 0, :]\n\n\ndef softmax_jacobian(J: torch.Tensor, p: torch.Tensor) -> torch.Tensor:\n    """Gauge-invariant Jacobian: J^(p) = (diag(p) - pp^T) @ J.\n\n    J^(p)_i = p_i * (J_i - J_bar_p),  J_bar_p = sum_k p_k * J_k\n    Removes the common-mode shift (gauge) from raw Jacobian.\n    """\n    J_bar = (p[:, None] * J).sum(dim=0, keepdim=True)  # [1, D]\n    return p[:, None] * (J - J_bar)  # [K, D]\n\n\ndef uniform_centered(J: torch.Tensor) -> torch.Tensor:\n    """Uniform centering: J - mean(J). Removes average shift."""\n    return J - J.mean(dim=0, keepdim=True)\n\n\ndef _last_layer_jacobian(h: torch.Tensor, norm: torch.nn.Module) -> torch.Tensor:\n    """Analytical J = d h_norm / d h at the final norm (no backward passes).\n\n    Both Qwen3_5RMSNorm and nn.LayerNorm are handled. J is evaluated at the\n    norm INPUT (h), never at h_norm — evaluating the formula at the norm\n    output gives a matrix with ~0.87 relative error (no longer LayerNorm-like\n    statistics). Qwen3.5\'s final norm is Qwen3_5RMSNorm:\n      h_norm = (h * rsqrt(mean(h^2) + eps) * (1 + w))\n      J = r · diag(1+w) · (I - xhat xhat^T / D),  r = rsqrt(mean(h^2)+eps), xhat = r*h\n    LayerNorm: J = (I - 1/D - xhat xhat^T / D) / std, rows scaled by w.\n    """\n    # h may arrive as [1, S, D] (full-sequence hidden_states) or [1, D]/[D].\n    # The last-layer READOUT Jacobian is the FINAL-position map (it feeds the\n    # final-token logits), so collapse a [1, S, D] to its last position first.\n    # BUG this fixes: [1, S, D].reshape(1,-1) -> [1, S*D] -> [S*D, S*D] identity\n    # (e.g. [65536, 65536] = 16 GB) -> OOM on GPU / XLA compile-abort on TPU\n    # (previously misattributed to an "XLA phantom dot" shape bug).\n    if h.dim() == 3:\n        h = h[:, -1, :]\n    x = h.reshape(1, -1).float()  # [1, D], the norm INPUT (row vector)\n    D = x.shape[-1]\n    eps = float(getattr(norm, "eps", None) or 1e-5)\n    w = getattr(norm, "weight", None)\n    dev = x.device\n    if getattr(norm, "bias", None) is not None:  # LayerNorm family\n        mu = x.mean(dim=-1, keepdim=True)\n        std = (x.var(dim=-1, unbiased=False, keepdim=True) + eps).sqrt()\n        xhat = (x - mu) / std  # [1, D]\n        J = (\n            torch.eye(D, device=dev)\n            - torch.ones(D, D, device=dev) / D\n            - xhat.T @ xhat / D\n        ) / std.squeeze(0)\n        if w is not None:\n            J = J * w.float().squeeze(0).unsqueeze(1)  # J = diag(w) @ J\n        return J\n    # RMSNorm family (Qwen3_5RMSNorm, Llama-style).\n    r = torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)\n    xhat = x * r  # [D]\n    # y_j = (1+w_j) r x_j; J = r * diag(1+w) @ (I - xhat xhat^T / D)\n    # (derived: J_ij = g_j (r d_ij - r^3 x_i x_j / D) = g_j r (d_ij - xhat_i xhat_j / D))\n    g = 1.0 + w.float().squeeze(0) if w is not None else None\n    M = torch.eye(D, device=dev) - xhat.T @ xhat / D\n    if g is None:\n        return r.squeeze(0) * M\n    return r.squeeze(0) * torch.diag(g) @ M\n\n\ndef compute_hidden_jacobian(\n    model,\n    layer_idx: int,\n    input_ids: torch.Tensor,\n    attention_mask: torch.Tensor,\n    create_graph: bool = False,\n    verbose: bool = False,\n    h: torch.Tensor | None = None,\n) -> torch.Tensor:\n    """Compute Jacobian of normalized last hidden state w.r.t. hidden state at layer_idx.\n\n    J_norm = d(h_norm) / d(h_{layer_idx}), shape [D, D].\n    Independent of lm_head — computed once, reused across W updates.\n    Then J_S = lm_head.weight[token_set] @ J_norm is differentiable through W.\n\n    For last_layer: J_norm is LayerNorm Jacobian (analytical, no backward passes).\n\n    Args:\n        create_graph: if True, J_norm has grad_fn.\n            - If h is None: h is extracted from a full forward and stays\n              connected to model params (full gradient flow, as before).\n            - If h is given: h is used AS-IS instead of the forward (the\n              forward is skipped). A ready leaf h (requires_grad=True,\n              grad_fn=None) is used IN PLACE — its .grad receives the\n              backpropagated derivative d J_norm/d h. J_norm still depends\n              on the decoder WEIGHTS between layer_idx and the final norm\n              (the decode pass), so gradients also reach those params; only\n              the params of layers <= layer_idx / embeddings are excluded.\n                      Costs ~D backward passes per sample.\n        verbose: print progress during J_norm computation.\n        h: pre-computed hidden state at layer_idx (reuse from a prior forward,\n            e.g. KL-forward in exp17). Respected in BOTH create_graph paths\n            (v6.0 bug-fix: the create_graph=True path previously ignored it\n            and unconditionally re-extracted h from a full forward). Not\n            renamed to h_override for backward compatibility; the asymmetry\n            with compute_jacobian_for_tokens(h_override=...) is intentional.\n    """\n    model.eval()\n    n_layers = _n_layers(model)\n\n    if create_graph:\n        # Keep gradient graph alive: h is connected to model params (+ any hooks)\n        S = input_ids.shape[1]\n        # Build prefix cache FIRST — before any forward involving hooks.\n        prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)\n\n        if h is None:\n            # Full forward to get h at layer_idx — connected to all model params + hooks\n            with torch.enable_grad():\n                outputs = model(\n                    input_ids=input_ids,\n                    attention_mask=attention_mask,\n                    output_hidden_states=True,\n                    use_cache=False,\n                )\n            assert outputs.hidden_states is not None  # requested\n            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D], connected!\n        # Normalize [1, D] / [D] -> [1, 1, D]; keep any existing grad graph intact.\n        h = h.reshape(1, 1, -1) if h.dim() in (1, 2) else h\n        model_dtype = next(model.parameters()).dtype\n        assert h is not None  # narrowed: param was Tensor|None\n        h = h.to(model_dtype)  # preserve grad graph\n        if h.grad_fn is None:\n            if h.requires_grad:\n                # Caller passed a ready leaf that already requires grad\n                # (e.g. a representation-holonomy h_leaf). Use it IN PLACE:\n                # detach() would create a NEW leaf and the caller\'s tensor\n                # would never receive .grad. It is already a leaf (grad_fn\n                # None), so no graph surgery is needed.\n                pass\n            else:\n                # Fully frozen graph: no trainable path reaches this layer\n                # (e.g. linear_attention pair with no LoRA patch, or a\n                # no_grad caller), OR a disconnected h was passed in. Make h\n                # a leaf so the decoder pass below still yields a\n                # differentiable h_norm; J_norm values are identical to the\n                # connected case.\n                h = h.detach().requires_grad_(True)\n\n        if layer_idx == n_layers - 1:\n            with torch.enable_grad():\n                h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]\n        else:\n            with torch.enable_grad():\n                h_norm = _run_decoder_to_norm_compat(\n                    model, prefix_cache, h, S, layer_idx\n                )\n\n        # [1, 1, D] — _compute_jacobian_rows_full indexes grad[0, 0, :].\n        # enable_grad: autograd.grad needs grad mode ON even when the\n        # caller is inside torch.no_grad().\n        with torch.enable_grad():\n            J_norm = _compute_jacobian_rows_full(\n                h_norm, h, create_graph=True, verbose=verbose\n            )\n        return J_norm\n\n    # Last layer: analytical norm Jacobian, no backward passes.\n    if layer_idx == n_layers - 1:\n        with torch.no_grad():\n            if h is None:\n                outputs = model(\n                    input_ids=input_ids,\n                    attention_mask=attention_mask,\n                    output_hidden_states=True,\n                    use_cache=False,\n                )\n                assert outputs.hidden_states is not None  # requested\n                h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]\n            assert h is not None\n            h = h.to(next(model.parameters()).dtype)\n        return _last_layer_jacobian(h, _lm_module(model).norm)\n\n    # General case: decode through remaining layers + norm\n    S = input_ids.shape[1]\n    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)\n    if h is None:\n        h = _extract_hidden_state(model, input_ids, attention_mask, layer_idx)\n    model_dtype = next(model.parameters()).dtype\n    h = h.to(model_dtype).detach().requires_grad_(True)\n    # Re-enable grad: h_norm must carry a grad_fn over h even if the caller\n    # is inside torch.no_grad() (e.g. eval loops in exp16r/exp17). The row\n    # loop below (autograd.grad) ALSO needs grad mode ON — a no_grad caller\n    # otherwise gets "element 0 ... does not have a grad_fn".\n    with torch.enable_grad():\n        h_norm = _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)\n        J_norm = _compute_jacobian_rows_full(\n            h_norm, h, create_graph=create_graph, verbose=verbose\n        )\n    return J_norm\n\n\ndef _run_decoder_to_norm(\n    text_model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int\n) -> torch.Tensor:\n    """Decode h through remaining layers, return norm(x) without lm_head."""\n    cache_clone = _deep_clone_cache(prefix_cache)\n    pos = torch.full((4, 1, 1), S - 1, dtype=torch.long, device=h.device)\n    text_position_ids = pos[0]\n    rope_position_ids = pos[1:]\n    position_embeddings = text_model.rotary_emb(h, rope_position_ids)\n\n    x = h\n    n_layers = text_model.config.num_hidden_layers\n    start = layer_idx + 1\n    causal_mask_mapping = _build_causal_masks(\n        text_model, text_position_ids, cache_clone, x\n    )\n    with _clone_gdn_states_on_entry():\n        for i in range(start, n_layers):\n            layer_type = text_model.config.layer_types[i]\n            attn_mask = causal_mask_mapping[layer_type]\n            x = text_model.layers[i](\n                x,\n                position_embeddings=position_embeddings,\n                attention_mask=attn_mask,\n                position_ids=text_position_ids,\n                past_key_values=cache_clone,\n            )\n    return text_model.norm(x[:, -1, :])\n\n\ndef _compute_jacobian_rows_full(\n    output: torch.Tensor,\n    h: torch.Tensor,\n    create_graph: bool = False,\n    verbose: bool = False,\n) -> torch.Tensor:\n    """Full [D, D] Jacobian via autograd.grad."""\n    import time as _time\n\n    D = h.shape[-1]\n    out_flat = output.flatten()\n    grads = []\n    # create_graph=True: every row must retain the graph — the caller\n    # backpropagates a final loss through J_norm afterwards (exp17 bridge\n    # loss -> o_proj HVP). retain=(i<D-1) would free the (second-order)\n    # graph after the last row, so the caller\'s first backward would fail\n    # with "backward through the graph a second time". Memory is bounded\n    # by the refresh schedule (one graph per si, freed at the next refresh\n    # / pair end).\n    start = _time.time()\n    for i in range(D):\n        grad = torch.autograd.grad(\n            outputs=out_flat[i],\n            inputs=h,\n            retain_graph=(True if create_graph else i < D - 1),\n            create_graph=create_graph,\n        )[0]\n        grads.append(grad[0, 0, :])\n        if verbose and i % 128 == 0:\n            elapsed = _time.time() - start\n            rate = i / elapsed if elapsed > 0 else 999\n            eta = elapsed * (D / i) - elapsed if i > 0 else 0.0\n            print(f"        J_norm: {i}/{D} ({rate:.0f}/s, eta={eta:.0f}s)")\n    J = torch.stack(grads)\n    elapsed_total = _time.time() - start\n    if verbose:\n        print(f"        J_norm done: {D}/{D} in {elapsed_total:.1f}s")\n    return J\n\n\ndef _lm_module(model):\n    """The text model with .layers/.norm/.rotary_emb.\n\n    Flat CausalLM: model.model IS the text model. Multimodal wrapper\n    (Qwen3_5ForConditionalGeneration): model.model.language_model.\n    """\n    mm = model.model\n    if hasattr(mm, "layers"):\n        return mm\n    return mm.language_model\n\n\ndef _n_layers(model) -> int:\n    """num_hidden_layers for flat or wrapper configs."""\n    cfg = model.config\n    return int(\n        cfg.num_hidden_layers\n        if hasattr(cfg, "num_hidden_layers")\n        else cfg.text_config.num_hidden_layers\n    )\n\n\ndef _layer_types(model) -> list:\n    """layer_types for flat or wrapper configs."""\n    cfg = model.config\n    return list(\n        cfg.layer_types if hasattr(cfg, "layer_types") else cfg.text_config.layer_types\n    )\n\n\ndef _run_decoder_to_norm_compat(\n    model, prefix_cache, h: torch.Tensor, S: int, layer_idx: int\n) -> torch.Tensor:\n    """_run_decoder_to_norm over _lm_module (flat or nested model)."""\n    return _run_decoder_to_norm(_lm_module(model), prefix_cache, h, S, layer_idx)\n\n\ndef h_norm_at_perturbed_h(\n    model,\n    layer_idx: int,\n    input_ids: torch.Tensor,\n    attention_mask: torch.Tensor,\n    h: torch.Tensor,\n) -> torch.Tensor:\n    """h_norm after the prefix, with the l_S hidden state pinned to `h`.\n\n    Finite-diff support (B2 spot check): call twice, h and h + eps*e_j,\n    and difference the two h_norms. h shape [1, 1, D] (any device/dtype);\n    the pinning is done on a copy — the input is not mutated.\n    """\n    S = input_ids.shape[1]\n    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)\n    with torch.no_grad():\n        return _run_decoder_to_norm_compat(model, prefix_cache, h, S, layer_idx)\n\n\ndef _forward_h_norm(\n    model,\n    layer_idx,\n    input_ids,\n    attention_mask,\n    h: torch.Tensor | None = None,\n):\n    """Forward once -> (h, h_norm), h graph-connected at layer l_S.\n\n    Shared by compute_hidden_jacobian_rows and bridge_rows_backward (A4):\n    the prefix pass runs before the model forward (hook-safe), h is the\n    l_S hidden state (kept in graph so a caller that patched o_proj / lm_head\n    gets HVP/bridge grads from the downstream backward), h_norm = RMSNorm(h)\n    (last layer) or decoder-to-norm (otherwise).\n\n    h (optional): reuse a pre-computed l_S hidden state instead of doing a\n    fresh full forward (v23.1: exp17\'s KL forward already produced it, so\n    the refresh reuses it and only runs the cheap decoder-tail-to-norm).\n    """\n    model.eval()\n    n_layers = _n_layers(model)\n    S = input_ids.shape[1]\n    prefix_cache = _run_prefix_pass(model, input_ids, attention_mask)\n    with torch.enable_grad():\n        if h is None:\n            # standalone: full forward to obtain h (original A4 behavior)\n            outputs = model(\n                input_ids=input_ids,\n                attention_mask=attention_mask,\n                output_hidden_states=True,\n                use_cache=False,\n            )\n            assert outputs.hidden_states is not None  # requested\n            h = outputs.hidden_states[layer_idx + 1][:, -1:, :]  # [1, 1, D]\n        assert h is not None\n        h = h.to(next(model.parameters()).dtype)\n        if h.grad_fn is None:\n            h = h.detach().requires_grad_(True)\n        if layer_idx == n_layers - 1:\n            h_norm = _lm_module(model).norm(h[:, -1, :])  # [1, D]\n        else:\n            h_norm = _run_decoder_to_norm_compat(\n                model, prefix_cache, h, S, layer_idx\n            )\n    return h, h_norm\n\n\ndef bridge_objective(cos: torch.Tensor, w: torch.Tensor) -> torch.Tensor:\n    """The bridge objective `Σ_k w_k (1 − cos_k)` — ONE formula for the thing that is\n    OPTIMIZED and the thing that is REPORTED (audit4 M1).\n\n    Before this, `bridge_rows_backward` minimized the unweighted mean `(1−cos).mean()`\n    while `bridge._weighted_cosine_loss` reported `Σ w(1−cos)` with `w = p_L/Σp_L`, so\n    ""gap improved"" was not a statement about the trained objective. Both sides call\n    this function now.\n\n    `w` is used AS GIVEN: it must be a probability vector over the rows you pass in, and\n    the CALLER normalizes. Deliberate — the chunked path normalizes `p` once and then\n    slices, so per-chunk losses add up to the global objective. Normalizing inside here\n    would renormalize every chunk by its own weight mass: chunk-size dependent, and it\n    quietly changes the objective (this is exactly what the chunk-invariance test caught).\n\n    The objective stays a convex combination like the old mean — same scale, no LR\n    retuning — but concentrated: on sharp `p` the top token carries most of the gradient,\n    by design. Rows dropped by a validity mask must enter with `w = 0` (the metric masks\n    them at its caller; on unmasked rows the two sides are identical).\n    """\n    if cos.shape != w.shape:\n        raise ValueError(f"bridge_objective: cos {tuple(cos.shape)} vs w {tuple(w.shape)}")\n    return (w.reshape(-1).to(cos.dtype) * (1.0 - cos)).sum()\n\n\ndef bridge_rows_backward(\n    model,\n    layer_idx: int,\n    input_ids: torch.Tensor,\n    attention_mask: torch.Tensor,\n    token_set: list[int],\n    p: torch.Tensor,\n    W_rows: torch.Tensor,\n    J_tgt_p: torch.Tensor,\n    extra_loss: torch.Tensor | None = None,\n    chunk_size: int = 8,\n    verbose: bool = False,\n    h: torch.Tensor | None = None,\n) -> tuple[torch.Tensor, float]:\n    """v23 A4: chunked bridge backward (the v22 OOM fix).\n\n    Forward once + J_bar once, then build each chunk of K rows and backprop\n    its `bridge_objective` loss immediately (extra_loss — the KL — is\n    backproped with the first chunk). Backproping per chunk frees that\n    chunk\'s 2nd-order row-graphs at once, so peak = chunk_size + 1 (the\n    shared J_bar) second-order graphs, not K. Per-chunk gradients are\n    identical to a single backward on the total weighted loss\n    `Σ_k (p_k/Σp)(1 − cos_k)` — the SAME functional `bridge.jacobian_gap`\n    reports (audit4 M1: the objective must be the metric; it used to be the\n    unweighted mean).\n\n    Gradients reach W_rows (live lm_head + B_lh @ A_lh) and o_proj (HVP\n    through h). Returns (J_S_p [K, D] detached, bridge_loss scalar).\n\n    h (optional): reuse a pre-computed l_S hidden state (v23.1) so this\n    does NOT redo the full forward — only the cheap decoder-tail-to-norm\n    runs. exp17\'s KL forward already produced the same h.\n    """\n    import time as _time\n\n    K = len(token_set)\n    w_obj = p.reshape(-1) / p.reshape(-1).sum()          # metric weights, Σw = 1\n    h, h_norm = _forward_h_norm(\n        model, layer_idx, input_ids, attention_mask, h=h\n    )\n\n    def _row_grad(w: torch.Tensor) -> torch.Tensor:\n        return torch.autograd.grad(\n            outputs=h_norm @ w,\n            inputs=h,\n            retain_graph=True,\n            create_graph=True,\n        )[0][0, 0, :]\n\n    start = _time.time()\n    J_bar = _row_grad(p @ W_rows)  # shared; freed by the last chunk\'s bwd\n    # extra_loss (the KL) gets a SEPARATE backward before the chunk loop.\n    # retain_graph=True keeps the shared h-graph alive for the chunks while\n    # the KL\'s OWN forward graph (a separate forward in exp17) frees here.\n    # We deliberately do NOT add it to chunk 0: in a shared-forward test/\n    # wiring that would free the h-graph the chunks still need.\n    if extra_loss is not None:\n        extra_loss.backward(retain_graph=True)\n    chunks: list[torch.Tensor] = []\n    total_bridge = 0.0\n    for c in range(0, K, chunk_size):\n        hi = min(c + chunk_size, K)\n        is_last = (c + chunk_size) >= K\n        rows_c = torch.stack(\n            [p[i] * (_row_grad(W_rows[i]) - J_bar) for i in range(c, hi)]\n        )\n        tgt_c = J_tgt_p[c:hi].float()\n        cos_c = torch.nn.functional.cosine_similarity(rows_c.float(), tgt_c, dim=1)\n        chunk_loss = bridge_objective(cos_c, w_obj[c:hi])\n        # retain_graph on all-but-last keeps the ONE shared h forward graph\n        # alive so the next chunk can still build its rows; the chunk\'s\n        # 2nd-order row-graph is released by refcounting once we del it\n        # (forward nodes stay, referenced by h / h_norm / J_bar).\n        chunk_loss.backward(retain_graph=not is_last)\n        total_bridge += chunk_loss.item()\n        chunks.append(rows_c.detach())\n        del rows_c, cos_c, chunk_loss, tgt_c\n        if verbose:\n            print(\n                f"        bridge_rows: {hi}/{K} "\n                f"({hi / max(_time.time() - start, 1e-6):.1f} chunks/s)"\n            )\n    return torch.cat(chunks), total_bridge\n\n\ndef compute_hidden_jacobian_rows(\n    model,\n    layer_idx: int,\n    input_ids: torch.Tensor,\n    attention_mask: torch.Tensor,\n    token_set: list[int],\n    p: torch.Tensor,\n    W_rows: torch.Tensor,\n    create_graph: bool = False,\n    chunk_size: int = 8,\n    verbose: bool = False,\n) -> torch.Tensor:\n    """v23 A1: per-row J_S_p = p_i*(J_i - J_bar), no dense [D,D] J_norm.\n\n    K-subset softmax gauge — the SAME gauge as the exp16 target (p_s =\n    softmax(full_logits[token_set])) and the exp17 dense bridge\n    (softmax_jacobian(W[token_set]@J_norm, p_L)):\n      p = re-normalized dist over the K-subset [K], W_rows = W[token_set]\n      [K, D]; J_bar = d(h_norm @ (p @ W_rows))/dh; row_i =\n      p[i] * (d(h_norm @ W_rows[i])/dh - J_bar).\n    h stays graph-connected (via _run_decoder_to_norm_compat, which\n    routes through _lm_module so both flat CausalLM and the multimodal\n    wrapper work), so a caller that patched the target layer\'s o_proj\n    and/or lm_head gets its HVP/bridge gradients from the downstream\n    loss backward. chunk_size (A4) bounds the 2nd-order graphs alive\n    at once; create_graph=True retains them for a later loss backward,\n    create_graph=False frees each chunk after the gradient (for cheap\n    no-grad refreshes / evals).\n    """\n    model.eval()\n    n_layers = _n_layers(model)\n    S = input_ids.shape[1]\n\n    h, h_norm = _forward_h_norm(model, layer_idx, input_ids, attention_mask)\n\n    return _compute_jacobian_rows_matrixfree(\n        h_norm,\n        h,\n        W_rows,\n        p,\n        token_set,\n        create_graph=create_graph,\n        chunk_size=chunk_size,\n        verbose=verbose,\n    )\n\n\ndef _compute_jacobian_rows_matrixfree(\n    h_norm: torch.Tensor,\n    h: torch.Tensor,\n    W_rows: torch.Tensor,\n    p: torch.Tensor,\n    token_set: list[int],\n    create_graph: bool = True,\n    chunk_size: int = 8,\n    verbose: bool = False,\n) -> torch.Tensor:\n    """J_S_p [K, D] rows, K-subset softmax gauge, no dense [D,D] J_norm.\n\n    p [K] = re-normalized softmax over the K-subset (exp16/exp17 gauge),\n    W_rows [K, D] = the selected rows. J_bar = d(h_norm @ (p @ W_rows))/dh;\n    row_i = p[i]*(d(h_norm @ W_rows[i])/dh - J_bar). Identical to\n    softmax_jacobian(W_rows @ J_norm, p) but with K backward passes\n    instead of a retained [D,D] 2nd-order matrix (v22 OOM).\n    Chunked (A4): only chunk_size row-graphs alive at once.\n    """\n    import time as _time\n\n    K = len(token_set)\n\n    def _row_grad(w: torch.Tensor) -> torch.Tensor:\n        return torch.autograd.grad(\n            outputs=h_norm @ w,\n            inputs=h,\n            retain_graph=True,\n            create_graph=create_graph,\n        )[0][0, 0, :]\n\n    start = _time.time()\n    J_bar = _row_grad(p @ W_rows)  # [D]; p@W_rows = effective selected row\n    rows = []\n    for c in range(0, K, chunk_size):\n        for local, i in enumerate(range(c, min(c + chunk_size, K))):\n            row = p[i] * (_row_grad(W_rows[i]) - J_bar)\n            rows.append(row)\n        if verbose:\n            done = min(c + chunk_size, K)\n            el = _time.time() - start\n            print(f"        J_S_p: {done}/{K} ({done / max(el, 1e-6):.1f}/s)")\n    out = torch.stack(rows)\n    elapsed = _time.time() - start\n    if verbose:\n        print(f"        J_S_p done: {K}/{K} in {elapsed:.1f}s")\n    return out\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/json_utils.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Strict RFC-8259 JSON serialization for experiment outputs.\n\n``json.dump``\'s ``default=`` hook is NOT called for Python floats:\n``float(\'nan\')`` / ``float(\'inf\')`` are serialized natively as the\nnon-standard ``NaN`` / ``Infinity`` literals (and ``np.float64`` is a\n``float`` subclass, so it has the same property; only ``np.float32``\nreaches ``default=``). A ``default=`` function that stringifies\nnon-finite floats is therefore a dead guard for the common case.\n\nFix = recursive pre-sanitization + ``allow_nan=False`` (defense in\ndepth: any NaN that slips through raises ``ValueError`` loudly) +\n``default=_raise`` (a caller-passed ``default=str`` cannot swallow an\nunserializable object into a useless string).\n\nConvention (same as ``prism.cache._json_dump_value``): non-finite\nfloats become the strings ``"nan"`` / ``"inf"`` / ``"-inf"`` — valid\nJSON. Writer-only: the matching reader is external\n(``prism.cache._json_load_value`` handles these markers for\n``MetricCache`` fields, non-recursively); a recursive loader is NOT\nprovided here.\n\nFail-loud boundary: ``np.ndarray`` / ``torch.Tensor`` fall through\n``sanitize`` and are rejected by ``json.dump`` (default=_raise) — call\n``.tolist()`` / ``.detach().cpu().tolist()`` at the caller if intended.\n\nCaller pattern (atomic write — so a sanitize/default raise does not\nleave a 0-byte OUT that downstream may read as "the experiment ran"):\n\n    tmp = OUT + ".tmp"\n    with open(tmp, "w") as f:\n        json_dump(out, f, indent=1)\n    os.replace(tmp, OUT)\n"""\nfrom __future__ import annotations\n\nimport json\nimport math\n\nimport numpy as np\n\n__all__ = ["sanitize", "dump"]\n\n\ndef sanitize(o):\n    """Recursively convert a JSON tree to strict-RFC-8259 form.\n\n    - ``np.bool_`` -> TypeError (unwrapped numpy bool; fail loud)\n    - ``np.integer`` -> ``int``\n    - ``np.floating`` -> ``float`` (then non-finite check)\n    - ``float`` NaN/Inf -> ``"nan"`` / ``"inf"`` / ``"-inf"``\n    - dict keys AND values -> elementwise (a non-sanitizable key, e.g.\n      ``np.bool_`` / ``np.int64``, raises here with the friendly message\n      instead of a raw TypeError from ``json.dump``)\n    - list/tuple -> elementwise (tuple -> list)\n    - ``np.ndarray`` / ``torch.Tensor`` -> pass through; ``json.dump``\n      rejects them (fail loud; see module docstring)\n    """\n    if isinstance(o, np.bool_):\n        raise TypeError(f"unwrapped numpy bool_ in JSON: {o!r}")\n    if isinstance(o, np.integer):\n        return int(o)\n    if isinstance(o, np.floating):\n        o = float(o)\n    if isinstance(o, float):\n        if math.isnan(o):\n            return "nan"\n        if math.isinf(o):\n            return "inf" if o > 0 else "-inf"\n        return o\n    if isinstance(o, dict):\n        return {sanitize(k): sanitize(v) for k, v in o.items()}\n    if isinstance(o, (list, tuple)):\n        return [sanitize(v) for v in o]\n    return o\n\n\ndef _raise(o):\n    raise TypeError(\n        f"unserializable object in JSON: {type(o).__name__!r} "\n        f"(convert at the caller: .tolist() / .item() / float() / int())")\n\n\ndef dump(o, f, **kwargs):\n    """``json.dump(sanitize(o), f, allow_nan=False, default=_raise, **kwargs)``.\n\n    ``allow_nan`` and ``default`` are forced — a caller cannot bypass\n    the strict contract.\n    """\n    kwargs["allow_nan"] = False\n    kwargs["default"] = _raise\n    json.dump(sanitize(o), f, **kwargs)\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/jstruct.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Module: structural primitives for the L23 gauge story (new_fun_core.md §3).\n\nThin tensor helpers formalizing existing quantities (M_K, û_t^⊥,\nw = α û + β v_⊥) — no model, no forward passes, no caching (W_lm changes\nunder ``apply_edit``; callers own any cache invalidation).\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom typing import Sequence\n\nimport torch\n\n__all__ = [\n    "build_M_K",\n    "decompose_w_vs_uhat",\n    "v_pred_from_JT",\n]\n\n\ndef build_M_K(\n    W_lm: torch.Tensor,     # [V, D]\n    token_set: Sequence[int],\n    g: torch.Tensor,        # [D] — RMSNorm weight (1 + w, NOT raw w)\n) -> torch.Tensor:         # [K, D]\n    """M_K = W_lm[S] · diag(g) ∈ R^{K×D} (lemma [M-1], fundamental for L23).\n\n    Args:\n        W_lm: [V, D] lm_head weight rows.\n        token_set: token row indices (duplicates allowed — the caller is\n            responsible for uniqueness).\n        g: [D] RMSNorm weight, i.e. (1 + w), broadcast over the K rows.\n\n    Returns:\n        M_K [K, D].\n\n    NOT cached inside: W_lm mutates under ``apply_edit`` — a ``data_ptr``-keyed\n    cache would be unsafe. If an experiment needs caching, the caller owns\n    invalidation.\n\n    Errors:\n        ValueError: empty token_set, index out of [0, V) (first offender\n            reported), W_lm.shape[1] != g.shape[0], NaN/Inf in g.\n    """\n    if len(token_set) == 0:\n        raise ValueError("build_M_K: token_set is empty")\n    if W_lm.dim() != 2:\n        raise ValueError(f"build_M_K: W_lm must be 2-D, got {W_lm.dim()}-D")\n    if g.dim() != 1 or W_lm.shape[1] != g.shape[0]:\n        raise ValueError(\n            f"build_M_K: W_lm.shape[1] = {W_lm.shape[1]} != g dim = {g.shape[0] if g.dim() else 0}"\n        )\n    V = W_lm.shape[0]\n    for i, idx in enumerate(token_set):\n        if idx < 0 or idx >= V:\n            raise ValueError(\n                f"build_M_K: token_set[{i}] = {idx} out of [0, V) = [0, {V})"\n            )\n    if not torch.isfinite(g).all():\n        bad = int(torch.nonzero(~torch.isfinite(g), as_tuple=False).flatten()[0])\n        raise ValueError(f"build_M_K: non-finite g at index {bad}")\n\n    ts = torch.tensor(token_set, device=W_lm.device, dtype=torch.long)\n    # Cast g to W_lm\'s dtype: an fp64 g would silently promote the result\n    # to fp64 (torch type promotion), surprising the caller (docs/E38_fixes.md §5).\n    return W_lm[ts] * g.to(W_lm.dtype).unsqueeze(0)\n\n\ndef decompose_w_vs_uhat(\n    w: torch.Tensor,     # [D]\n    uhat: torch.Tensor,  # [D] unit\n) -> tuple[float, float, torch.Tensor]:\n    """Rank-1 split w = α·uhat + β·v_perp (paper [M-1], step 1).\n\n    Returns (α: float, β: float ≥ 0, v_perp: [D] unit if β > 0 else zeros).\n\n    NOT a generic projection (cf. topology.project_to_subspace): this\n    returns a rank-1 coefficient + residual norm + unit residual direction,\n    working on a single vector [D]. project_to_subspace returns\n    h @ basis.T for [N, D] inputs and does not compute a residual.\n\n    Errors:\n        ValueError: uhat not unit (‖·‖ reported), w.shape != uhat.shape,\n            NaN/Inf in w.\n    """\n    w = w.reshape(-1)\n    uhat = uhat.reshape(-1)\n    if w.shape != uhat.shape:\n        raise ValueError(\n            f"decompose_w_vs_uhat: shape mismatch {tuple(w.shape)} vs {tuple(uhat.shape)}"\n        )\n    n = torch.norm(uhat).item()\n    if abs(n - 1.0) > 1e-4:\n        raise ValueError(f"decompose_w_vs_uhat: uhat not unit: ‖uhat‖ = {n:.6f}")\n    if not torch.isfinite(w).all():\n        bad = int(torch.nonzero(~torch.isfinite(w), as_tuple=False).flatten()[0])\n        raise ValueError(f"decompose_w_vs_uhat: non-finite w at index {bad}")\n\n    alpha = float(torch.dot(w, uhat))\n    r = w - alpha * uhat\n    beta = float(r.norm())\n    if beta > 1e-12:\n        v_perp = r / beta\n    else:\n        v_perp = torch.zeros_like(w)\n    return alpha, beta, v_perp\n\n\ndef v_pred_from_JT(\n    J: torch.Tensor,                  # [K, D]\n    uniform: torch.Tensor | None = None,  # [K], default 1/√K\n    eps: float = 1e-10,\n) -> torch.Tensor:                    # [D] unit\n    """Normalized transpose projection v_pred = J^T·u / ‖J^T·u‖, u = 1/√K\n    (paper [M-1], step 2). No J^+ — a pseudo-inverse would emphasize the\n    tail, not the gauge.\n\n    NOT a generic projection: transpose-side of J, returns a UNIT vector,\n    raises on ‖J^T u‖ ≈ 0 (lemma [M-1] degeneracy). Cf.\n    topology.project_to_subspace (h @ basis.T, [N, D] → [N, r]).\n\n    A non-unit ``uniform`` is normalized silently (documented behavior).\n\n    Errors:\n        ValueError: J.dim() != 2, K == 0, uniform.shape != (K,),\n            ‖uniform‖ == 0.\n        RuntimeError: ‖J^T u‖ < eps ([M-1] degeneracy, β ≈ 0).\n    """\n    if J.dim() != 2:\n        raise ValueError(f"v_pred_from_JT: J must be 2-D, got {J.dim()}-D")\n    K, D = J.shape\n    if K == 0:\n        raise ValueError("v_pred_from_JT: K == 0")\n    if uniform is None:\n        uniform = torch.full((K,), 1.0 / math.sqrt(K), device=J.device, dtype=J.dtype)\n    else:\n        if uniform.shape != (K,):\n            raise ValueError(\n                f"v_pred_from_JT: uniform shape {tuple(uniform.shape)} != (K,) = ({K},)"\n            )\n        n = uniform.norm()\n        if not (n > 0) or float(n) < 1e-30:\n            raise ValueError("v_pred_from_JT: uniform has (near-)zero norm")\n        uniform = uniform / n\n    x = J.T @ uniform\n    nx = float(x.norm())\n    if nx < eps:\n        raise RuntimeError(\n            f"v_pred_from_JT: ‖J^T u‖ = {nx:.2e} < {eps:.0e}; "\n            "this is the [M-1] degeneracy (β≈0); see lemma"\n        )\n    return x / nx\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/kinetic.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Kinetic SVD decomposition with SIX channels (ТЗ E40_math_core §I.7/§II.6,\nM6; full specification in docs/E40_ans.md — that file supersedes the ТЗ\nwherever they differ).\n\nLinearization recipe (E40_ans §1): with aligned frames,\n\n    ΔJ_lin = ΔΣ + M Σ̄ − Σ̄ N,   M = log polar(U_tᵀ U_{t+1}),  N likewise,\n    Σ̄ = (Σ_t + Σ_{t+1})/2.\n\nChannels (ENERGIES, never shares):\n    E_radial = ‖ΔΣ‖², E_right = ‖Σ̄N‖², E_left = ‖MΣ̄‖²,\n    E_cross  = 2 tr(Σ̄ M Σ̄ N)  (SIGNED; report signed and abs),\n    E_resid  = ‖U_tᵀ ΔJ V_t − ΔJ_lin‖²_F            (independent, matrix diff),\n    E_off_manifold = ‖ΔJ − U_t U_tᵀ ΔJ V_t V_tᵀ‖²_F (independent; mandatory at\n                                                    r ≪ D, E40_ans §3),\n    eps3_correction = ‖ΔJ‖² − Σ6                    (diagnostic, NOT a channel).\n\nValidity flags (E40_ans §5): ``linearization_valid`` = max principal angle\n< π/2 − eps_angle (else log is off the principal branch); ``rotation_unique``\n= min |σᵢ²−σⱼ²| > eps_gap·σ₁² (else left/right/cross are basis-ambiguous\ninside near-degenerate blocks). On the real cache E-40.5 PRE-REGISTERS:\nL6–L22 invalid (Δ_SVD ≈ 1.4–1.55), L23 valid-but-ambiguous (0.266, near\ndegeneracy) — only E_radial and E_off_manifold are interpretation-safe\nanywhere. Getting ``linearization_valid = false`` on L6–L22 is the expected\nresult, not a failure.\n"""\n\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\n\nfrom .svd_bundle import principal_angles\nfrom .transport import align_signs_and_columns, polar_factor, transport_log\n\n__all__ = ["kinetic_decomposition", "channel_energy_matrix"]\n\n_EPS = 1e-12\n\n\ndef _fsq(A: np.ndarray) -> float:\n    return float((A ** 2).sum())\n\n\ndef _check_svd(triple, J: np.ndarray, name: str):\n    """Validate a caller-supplied ``np.linalg.svd`` triple against ``J`` — a\n    stale or swapped ``svds`` must fail loud, not silently renumber channels.\n    Shape check alone cannot see a swap of same-shaped inputs, so the triple is\n    checked by reconstruction (one matmul, ~10× cheaper than the SVD)."""\n    U, S, W = triple\n    U = np.asarray(U, dtype=np.float64)\n    S = np.asarray(S, dtype=np.float64)\n    W = np.asarray(W, dtype=np.float64)\n    K, D = J.shape\n    if U.shape != (K, min(K, D)) or S.shape != (min(K, D),) or W.shape != (min(K, D), D):\n        raise ValueError(f"{name}: SVD shapes {U.shape}/{S.shape}/{W.shape} "\n                         f"do not match [K,D]=[{K},{D}] (full_matrices=False)")\n    rec = float(np.linalg.norm((U * S) @ W - J)\n                / max(np.linalg.norm(J), 1e-300))\n    if rec > 1e-8:\n        raise ValueError(f"{name}: U diag(S) Vᵀ ≠ J (rel {rec:.2e}) — stale or "\n                         "swapped svds (swapping two same-shaped triples is "\n                         "otherwise undetectable)")\n    return U, S, W\n\n\ndef _min_gap_ratio(S: np.ndarray) -> float:\n    if S.size < 2:\n        return 1.0\n    s2 = S ** 2\n    d = s2[:-1] - s2[1:]\n    return float(d.min() / s2[0]) if d.size else 1.0\n\n\ndef kinetic_decomposition(J_t: np.ndarray, J_t1: np.ndarray, r: int = 16,\n                          eps_angle: float = 0.3, eps_gap: float = 1e-2,\n                          svds: tuple | None = None) -> dict:\n    """Six-channel kinetic accounting between consecutive Jacobians.\n\n    ``J_t``, ``J_t1``: ``[K, D]`` in a common transport frame (the E-40.5\n    pipeline transports before calling; nothing here is model-aware).\n    Returns every field listed in E40_ans §7/§8; when a flag invalidates a\n    channel the number is still computed (if real) for diagnostics, but the\n    flags tell the report what may be interpreted.\n\n    ``svds=((U0,S0,W0),(U1,S1,W1))``: precomputed ``np.linalg.svd(...,\n    full_matrices=False)`` of the two fp64 inputs, for callers that sweep a\n    chain of positions where each ``J`` appears in two consecutive pairs — a\n    full [K,D] SVD is the whole cost of this function. Bit-identical to the\n    internal call when the cached triple came from the same fp64 array.\n    """\n    J_t = np.asarray(J_t, dtype=np.float64)\n    J_t1 = np.asarray(J_t1, dtype=np.float64)\n    if J_t.shape != J_t1.shape or J_t.ndim != 2:\n        raise ValueError(f"J_t, J_t1 must be matching [K, D], got {J_t.shape}, {J_t.shape}")\n    K, D = J_t.shape\n    if not (0 < r <= min(K, D)):\n        raise ValueError(f"r must be in (0, {min(K, D)}], got {r}")\n\n    if svds is None:\n        U0, S0, W0 = np.linalg.svd(J_t, full_matrices=False)\n        U1, S1, W1 = np.linalg.svd(J_t1, full_matrices=False)\n    else:\n        (U0, S0, W0) = _check_svd(svds[0], J_t, "svds[0]")\n        (U1, S1, W1) = _check_svd(svds[1], J_t1, "svds[1]")\n    V0, V1 = W0[:r].T, W1[:r].T                       # [D, r]\n    U0a = U0[:, :r]\n    U1a = U1[:, :r]\n\n    U0a, U1a, fU = align_signs_and_columns(U0a, U1a)\n    V0, V1, fV = align_signs_and_columns(V0, V1)\n    order_cross = bool(fU["order_cross"] or fV["order_cross"])\n    # M2 fix (audit_core_math_v1): σ must travel WITH the matched columns —\n    # column j of the aligned t+1 frame is S1[perm[j]], not S1[j]. σ follows\n    # the U-side perm; a U/V perm disagreement means the singular triplets\n    # themselves split (near-degenerate block) and is flagged.\n    permU, permV = fU["perm"], fV["perm"]\n    sigma_perm_mismatch = bool(permU != permV)\n\n    dJ = J_t1 - J_t\n    Sig_t = np.diag(S0[:r])\n    Sig_t1 = np.diag(np.asarray(S1[:r])[permU])\n    dSig = Sig_t1 - Sig_t\n    Sb = 0.5 * (Sig_t + Sig_t1)\n\n    # validity A: rotation size (E40_ans §5.1 — MAX principal angle threshold)\n    cos_pa = principal_angles(U0a, U1a)\n    max_angle = float(np.arccos(np.clip(cos_pa.min(), -1.0, 1.0)))\n    linearization_valid = bool(max_angle < math.pi / 2 - eps_angle)\n    # validity B: σ-separation inside the top-r block (both t\'s must separate)\n    gap_ratio = min(_min_gap_ratio(S0[:r]), _min_gap_ratio(S1[:r]))\n    rotation_unique = bool(gap_ratio > eps_gap)\n\n    M = N = None\n    unstable_transport = False\n    if linearization_valid:\n        try:\n            M = transport_log(polar_factor(U0a.T @ U1a))\n            N = transport_log(polar_factor(V0.T @ V1))\n        except ValueError:\n            # Either side may fail (U-transport is computed first): the linearization is\n            # rejected as a whole, so BOTH must go to None — leaving `M` set while `N` is\n            # None makes the `M is not None` branch below compute `Sb @ None` (TypeError on\n            # a 0-dim object array) instead of reporting the rejected cell.\n            M = N = None\n            unstable_transport = True\n    # r = 1: after alignment Q = {+1}, log = 0 — exact, no try needed\n    if r == 1:\n        M = np.zeros((1, 1))\n        N = np.zeros((1, 1))\n        linearization_valid = True\n\n    B = U0a.T @ dJ @ V0                                # coords of ΔJ in the t-frame\n    out: dict = {"r": r, "max_principal_angle": max_angle,\n                 "min_sigma_gap_ratio": gap_ratio,\n                 "flags": {"linearization_valid": linearization_valid,\n                           "rotation_unique": rotation_unique,\n                           "order_cross_U": fU["order_cross"],\n                           "order_cross_V": fV["order_cross"],\n                           "unstable_transport": unstable_transport,\n                           "sign_flips_U": fU["sign_flips"],\n                           "sign_flips_V": fV["sign_flips"],\n                           "perm_U": permU,\n                           "perm_V": permV,\n                           "sigma_perm_mismatch": sigma_perm_mismatch}}\n    dJ_proj = U0a @ (U0a.T @ dJ @ V0) @ V0.T\n    E_off = _fsq(dJ - dJ_proj)\n    E_radial = _fsq(dSig)\n\n    if M is not None:\n        dJ_lin = dSig + M @ Sb - Sb @ N\n        E_left = _fsq(M @ Sb)\n        E_right = _fsq(Sb @ N)\n        E_cross = float(2.0 * np.trace(Sb @ M @ Sb @ N))   # signed (E40_ans §1)\n        E_resid = _fsq(B - dJ_lin)\n    else:\n        dJ_lin = E_left = E_right = E_cross = E_resid = None\n\n    six_sum = (E_radial + E_off + E_resid + E_left + E_right + E_cross\n               if dJ_lin is not None else None)\n    out.update({\n        "E_radial": E_radial,\n        "E_right": E_right,\n        "E_left": E_left,\n        "E_cross": E_cross,\n        "E_cross_abs": abs(E_cross) if E_cross is not None else None,\n        "E_resid": E_resid,\n        "E_off_manifold": E_off,\n        "norm_dJ_sq": _fsq(dJ),\n        "eps3_correction": (_fsq(dJ) - six_sum) if six_sum is not None else None,\n        "diag": {"normM_fro": float(np.linalg.norm(M, "fro")) if M is not None else None,\n                 "normN_fro": float(np.linalg.norm(N, "fro")) if N is not None else None},\n        # The transport objects themselves. Consumers that build a NULL over the\n        # left channel must perturb the ACTUAL M at the ACTUAL Σ̄ — re-deriving\n        # them in the script duplicates the pipeline and silently loses the\n        # σ↔column permutation (the M2 lesson), so they are exposed here.\n        "M_transport": None if M is None else np.array(M),\n        "sigma_bar": np.array(Sb),\n    })\n    # reportable set (E40_ans §5.3/§8)\n    if not linearization_valid:\n        out["channels_reportable"] = ["E_radial", "E_off_manifold"]\n    elif not rotation_unique:\n        out["channels_reportable"] = ["E_radial", "E_off_manifold", "E_resid"]\n        out["ambiguous"] = ["E_right", "E_left", "E_cross"]\n    else:\n        out["channels_reportable"] = ["E_radial", "E_right", "E_left", "E_cross",\n                                      "E_resid", "E_off_manifold"]\n    if six_sum is not None:\n        # non-vacuous closure: eps3 is a DIAGNOSTIC residual, not a channel —\n        # the assertion bounds it, it never redefines E_resid (E40_ans §4).\n        out["closure_rel"] = abs(_fsq(dJ) - six_sum) / max(_fsq(dJ), _EPS)\n    if sigma_perm_mismatch and "ambiguous" not in out:\n        out["ambiguous"] = []\n    if sigma_perm_mismatch:\n        out["ambiguous"].append("E_radial")\n        if "E_radial" in out["channels_reportable"]:\n            out["channels_reportable"].remove("E_radial")\n    return out\n\n\ndef channel_energy_matrix(J_t: np.ndarray, J_t1: np.ndarray, r: int = 16,\n                          svds: tuple | None = None) -> np.ndarray:\n    """5×5 Gram of the vector components ``A1..A5`` with\n    ``ΔJ = A1 + A2 + A3 + A4 + A5``:\n    ``radial`` = ΔΣ embedded, ``left`` = U M Σ̄ Vᵀ, ``right`` = −U Σ̄ N Vᵀ,\n    ``resid`` = U (B − ΔJ_lin) Vᵀ, ``off`` = ΔJ − ΔJ_proj.\n    ``sum(matrix)`` == ``‖ΔJ‖²`` EXACTLY (accounting identity, all cross terms\n    visible off-diagonal); the diagonal holds channel energies,\n    ``2·[left, right]`` is ``E_cross``. Energies, not shares (shares are\n    compositional, rank ≤ 4 by construction — ТЗ §I.7).\n\n    Slice caveat (патч v2 §3.3): this 5×5 includes ``off``, so it closes\n    ``‖ΔJ‖²``. Restricting the Gram to the four in-frame vectors\n    {radial, left, right, resid} closes only ``‖ΔJ_proj‖²`` — and on real\n    exp27b data (r = 16 ≪ D = 1024, Δ_SVD ≈ π/2 on L6–L22)\n    ``E_off ≈ ‖ΔJ‖²/2``, i.e. "4×4 closes ‖ΔJ‖²" is FALSE there and TRUE on\n    synthetic rank(ΔJ) ≤ r. Full closure always needs Σ6 + eps3.\n    """\n    J_t = np.asarray(J_t, dtype=np.float64)\n    J_t1 = np.asarray(J_t1, dtype=np.float64)\n    K, D = J_t.shape\n    if svds is None:\n        U0, S0, W0 = np.linalg.svd(J_t, full_matrices=False)\n        U1, S1, W1 = np.linalg.svd(J_t1, full_matrices=False)\n    else:\n        (U0, S0, W0) = _check_svd(svds[0], J_t, "svds[0]")\n        (U1, S1, W1) = _check_svd(svds[1], J_t1, "svds[1]")\n    U0a, S0r, V0 = U0[:, :r], S0[:r], W0[:r].T\n    U1a, _, fU = align_signs_and_columns(U0a, U1[:, :r])\n    V1 = align_signs_and_columns(V0, W1[:r].T)[1]\n    S1r = S1[:r][fU["perm"]]                      # σ follows the frame (M2)\n    dJ = J_t1 - J_t\n    M = transport_log(polar_factor(U0a.T @ U1a))\n    N = transport_log(polar_factor(V0.T @ V1))\n    dSig = np.diag(S1r - S0r)\n    Sb = 0.5 * np.diag(S0r + S1r)\n    B = U0a.T @ dJ @ V0\n    dJ_lin = dSig + M @ Sb - Sb @ N\n    embed = lambda X: U0a @ X @ V0.T                    # noqa: E731\n    A1 = embed(dSig)\n    A2 = embed(M @ Sb)\n    A3 = embed(-Sb @ N)\n    A4 = embed(B - dJ_lin)\n    A5 = dJ - embed(B)\n    comps = [A1, A2, A3, A4, A5]\n    G = np.array([[float((Ai * Aj).sum()) for Aj in comps] for Ai in comps])\n    return G\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/layer_selection.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Layer re-selection rule (docs/exp17_change.md 0.1/0.2, user 2026-08-25).\n\nl_S = full_attention small-model layers + l_S=0 (0_0) + last layer.\nl_L = nearest preimage under uniform layer mapping (tie -> smaller l_L).\nIterate per selected l_S (mapping is many-to-one; per-l_L iteration dupes).\nClassify per l_S layer type: full_attention -> "bridge" (o_proj exists),\nelse "kl_only" (GDN: no o_proj; edge pairs train lm_head LoRA only).\n\nSTANDALONE ON PURPOSE: no `prism.*` imports. The Kaggle kernel build inlines\neach prism module as a notebook cell and strips prism imports only from the\nEXPERIMENT cell, not from inlined modules — a bare `from prism.X import ...`\nhere would survive into the notebook and crash on Kaggle (no prism package)\nwith ModuleNotFoundError. The two small helpers are inlined here instead.\n"""\n\nfrom __future__ import annotations\n\n\ndef _uniform_layer_mapping(n_L: int, n_S: int) -> dict[int, int]:\n    """Uniform large->small layer mapping: l_L -> round(l_L*(n_S-1)/(n_L-1)).\n\n    Inlined copy of prism.calibration.uniform_layer_mapping (identical formula)\n    so this module stays standalone for kernel inlining.\n    """\n    return {i: round(i * (n_S - 1) / (n_L - 1)) for i in range(n_L)}\n\n\ndef get_layer_types(model) -> list[str]:\n    """Per-layer attention type from model.config.layer_types.\n\n    Qwen3.5 style hybrid: entries \'full_attention\' or \'linear_attention\'.\n    `model.config` (post-AutoModel load) exposes `layer_types` at top level\n    (Qwen3_5TextConfig). Returns lowercase list.\n    """\n    types = getattr(model.config, "layer_types", None)\n    if types is None:\n        # Fallback: nested text_config (defensive; Qwen3_5 exposes top-level).\n        tc = getattr(model.config, "text_config", None)\n        types = getattr(tc, "layer_types", None)\n    if types is None:\n        raise RuntimeError(\n            "model.config has no layer_types (and no text_config.layer_types)"\n        )\n    return [str(t) for t in types]\n\n\ndef select_pairs(n_L: int, n_S: int, layer_types: list[str]) -> list[tuple[int, int]]:\n    """Return (l_L, l_S) pairs: full-attn l_S + 0_0 + last, sorted by l_S."""\n    if n_L < 1 or n_S < 1 or len(layer_types) != n_S:\n        raise ValueError(\n            f"invalid: n_L={n_L}, n_S={n_S}, layer_types={len(layer_types)}"\n        )\n    mapping = _uniform_layer_mapping(n_L, n_S)\n    full_S = {i for i, t in enumerate(layer_types) if t == "full_attention"}\n    selected = {0, n_S - 1} | full_S\n    pairs = [_nearest_preimage(mapping, j) for j in sorted(selected)]\n    if (n_L - 1, n_S - 1) not in pairs:\n        pairs.append((n_L - 1, n_S - 1))\n    return pairs\n\n\ndef _nearest_preimage(mapping: dict[int, int], j: int) -> tuple[int, int]:\n    """l_L = argmin |mapping[l] - j|, tie -> smaller l (the selected l_S is j)."""\n    best_l, best_d = 0, float("inf")\n    for li in range(len(mapping)):\n        d = abs(mapping[li] - j)\n        if d < best_d:\n            best_l, best_d = li, d\n    return (best_l, j)\n\n\ndef classify_pair(l_S: int, layer_types: list[str]) -> str:\n    """\'bridge\' if small-model layer l_S has o_proj (full_attention), else \'kl_only\'."""\n    return (\n        "bridge"\n        if l_S < len(layer_types) and layer_types[l_S] == "full_attention"\n        else "kl_only"\n    )\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/noise.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Noise-floor calibration and signal validation (TZ part 3, §5).\n\nGeneric: no model, no architecture specifics, no experiment logic. A\n"noise floor" is the natural variance of REPEATED COMPUTATION of a\nmetric callable — it reflects forward/backward non-determinism\n(recurrent-ordering effects, CUDA atomic adds, thread scheduling) when\nthe callable includes model passes, and is ≈ 0 when the callable reads\nfrom a deterministic cache (correct: the cache IS deterministic).\n\npaper.md §8.8 — noise floor and signal validation.\n\nNOT covered here (TZ §5.8): architecture-specific policy\n(experiments/exp_T0.py), ``deterministic_mode`` (cache.py — orthogonal:\nit ELIMINATES variance), cross-version checks (exp_T0.py), paired\ncomparison of two floors (caller applies validate_signal twice), floor\ncaching (MetricCache).\n\nDo NOT call ``calibrate_noise_floor`` inside\n``deterministic_mode(strict=True)`` — the floor would be 0 (correct, but\nnot the "natural" floor).\n"""\n\nfrom __future__ import annotations\n\nimport gc\nimport math\nfrom collections.abc import Callable\nfrom typing import Any\n\nimport torch\n\n__all__ = ["calibrate_noise_floor", "validate_signal"]\n\n\ndef _reset_caches() -> None:\n    """Drop dynamo/CUDA caches and collect GC. Never raises.\n\n    Increases the natural variance of repeated metric_fn calls (warm\n    caches give std ≈ 0 even when real non-determinism exists).\n    """\n    try:\n        torch._dynamo.reset()\n    except Exception:  # dynamo not imported / not available\n        pass\n    try:\n        torch.cuda.empty_cache()\n    except Exception:  # no CUDA\n        pass\n    gc.collect()\n\n\ndef _require_scalar(value: Any, i: int) -> float:\n    """Accept Python scalars and 0-d/1-element tensors; reject the rest."""\n    if isinstance(value, (bool,)):\n        raise TypeError(f"metric_fn() returned bool at repeat {i}; need a float")\n    if isinstance(value, float):\n        out = value\n    elif isinstance(value, int):\n        out = float(value)\n    elif isinstance(value, torch.Tensor):\n        if value.numel() != 1:\n            raise TypeError(\n                f"metric_fn() returned a {value.numel()}-element tensor at "\n                f"repeat {i}; need a scalar (use .item())"\n            )\n        out = value.item()\n    else:\n        raise TypeError(\n            f"metric_fn() returned {type(value).__name__} at repeat {i}; "\n            f"need a scalar float (use .item())"\n        )\n    if math.isnan(out) or math.isinf(out):\n        raise ValueError(\n            f"metric_fn() returned {\'nan\' if math.isnan(out) else \'inf\'} at "\n            f"repeat {i}; a zero-division or invalid computation is hiding "\n            f"in the metric"\n        )\n    return out\n\n\ndef calibrate_noise_floor(\n    metric_fn: Callable[[], float],\n    n_repeats: int = 10,\n    reset_between: bool = True,\n    verbose: bool = False,\n) -> dict:\n    """Measure the natural variance of repeated metric_fn() calls.\n\n    Seeds are NOT set (that would force the floor to 0). ``metric_fn``\n    must be pure (no persistent mutation) — the caller\'s responsibility.\n\n    Args:\n        metric_fn: Callable[[], float]; must not install seeds.\n        n_repeats: ≥ 2 (else std is undefined); 10 recommended.\n        reset_between: True — reset dynamo/CUDA caches + gc between\n            calls (realistic natural floor); False — back-to-back calls\n            (warm cache may give std ≈ 0).\n        verbose: print ``[i/N] value`` per repeat.\n\n    Returns:\n        dict with ``mean``, ``std`` (ddof=0), ``values`` (list of\n        floats, len = n_repeats), ``is_deterministic`` (std < 1e-10),\n        ``n_repeats``, ``reset_between``.\n\n    ``is_deterministic=True`` means the computation is deterministic in\n    this configuration: any non-zero signal is a real effect (not\n    "no signal" — see validate_signal).\n\n    Errors:\n        ValueError: n_repeats < 2; NaN/Inf in a value (with repeat index).\n        TypeError: metric_fn() returns a non-scalar (with repeat index).\n    """\n    if n_repeats < 2:\n        raise ValueError(f"n_repeats must be >= 2, got {n_repeats}")\n\n    values: list[float] = []\n    for i in range(n_repeats):\n        if reset_between and i > 0:\n            _reset_caches()\n        v = _require_scalar(metric_fn(), i)\n        values.append(v)\n        if verbose:\n            print(f"[{i + 1}/{n_repeats}] {v}")\n\n    mean = float(sum(values) / len(values))\n    var = sum((x - mean) ** 2 for x in values) / len(values)  # ddof=0\n    std = float(math.sqrt(var))\n    return {\n        "mean": mean,\n        "std": std,\n        "values": values,\n        "is_deterministic": std < 1e-10,\n        "n_repeats": n_repeats,\n        "reset_between": reset_between,\n    }\n\n\ndef _validate_floor(floor: dict) -> None:\n    missing = [k for k in ("mean", "std", "is_deterministic") if k not in floor]\n    if missing:\n        raise KeyError(f"floor dict is missing key(s): {missing}")\n\n\ndef validate_signal(\n    signal: float,\n    floor: dict,\n    k: float = 5.0,\n) -> dict:\n    """Decide whether ``signal`` exceeds the noise floor by k sigmas.\n\n    paper.md §8.8. Verdicts:\n        "significant" — |signal − mean| > k·std (or > 1e-10 when\n            is_deterministic=True): a real effect may be claimed.\n        "noise" — |signal − mean| ≤ k·std (and not deterministic):\n            NO effect may be claimed.\n        "no_floor" — deterministic floor and |signal − mean| ≤ 1e-10:\n            nothing happens (signal ~ 0 in a deterministic system; not\n            a bug, but not a "significant effect" either).\n\n    TZ selfcheck §5.6.6 expects "no_floor" for signal = 42.0 + 1e-9,\n    which contradicts the explicit rule above (1e-9 > 1e-10); the rule\n    is followed, so 42.0 + 1e-9 → "significant".\n\n    Args:\n        signal: observed scalar (e.g. κ_U or ε_gap).\n        floor: dict from calibrate_noise_floor.\n        k: number of sigmas; must be > 0 (default 5, the 5-sigma rule).\n\n    Returns:\n        dict with ``verdict``, ``ratio`` (|signal−mean|/std),\n        ``deviation`` (signal−mean), ``threshold`` (k·std),\n        ``is_deterministic``.\n\n    Errors:\n        ValueError: k ≤ 0; NaN/Inf in signal.\n        KeyError: floor dict without the required keys.\n    """\n    if k <= 0:\n        raise ValueError(f"k must be > 0, got {k}")\n    if math.isnan(signal) or math.isinf(signal):\n        raise ValueError(f"signal must be finite, got {signal}")\n    _validate_floor(floor)\n\n    mean = float(floor["mean"])\n    std = float(floor["std"])\n    is_det = bool(floor["is_deterministic"])\n    deviation = float(signal) - mean\n    abs_dev = abs(deviation)\n\n    if is_det:\n        verdict = "significant" if abs_dev > 1e-10 else "no_floor"\n        ratio = float("inf") if abs_dev > 0.0 else 0.0\n    else:\n        if std == 0.0:\n            raise RuntimeError(\n                "floor has std == 0 but is_deterministic=False "\n                "(inconsistent; re-run calibrate_noise_floor)"\n            )\n        verdict = "significant" if abs_dev > k * std else "noise"\n        ratio = abs_dev / std\n\n    return {\n        "verdict": verdict,\n        "ratio": ratio,\n        "deviation": deviation,\n        "threshold": k * std,\n        "is_deterministic": is_det,\n    }\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/nulls_and_stats.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Nulls and statistics (ТЗ E40_math_core §II.9, M8/M9; paper §6 items 22/23).\n\nConsolidation target for the ad-hoc code in ``noise.py`` / ``permutation.py``\nand the per-experiment bootstrap copies. The rule (item 23): any\n``‖OX‖²/‖X‖²``-type claim needs BOTH nulls — the operator null (isotropic X:\nhow much ``O`` eats from random input) and a content-matched null (preserved\nnorms/marginals, randomized directions: separates data structure from\nnorm/aggregation artifacts).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n__all__ = [\n    "operator_null", "mp_upper_edge", "mp_lambda_max_mc", "reduction_control",\n    "paired_bootstrap", "permutation_test", "resolution_floor",\n]\n\n\ndef operator_null(O: np.ndarray, X: np.ndarray, kind: str = "iid",\n                  rng: np.random.Generator | None = None) -> np.ndarray:\n    """Random ``X`` of the same shape as ``X`` for the two-null protocol.\n\n    ``kind="iid"`` — Gaussian, same total Frobenius norm (mechanical operator\n    null: the returned X makes ``‖O X‖²/‖X‖²`` equal the isotropic reference).\n    ``kind="content_matched"`` — per-row (token-axis) norms of ``X`` preserved,\n    directions re-randomized: the dir-matched null of E-39 supplement / E44\n    §5.1.1. Rows = axis ``-2`` (the token axis the operator acts on).\n    """\n    O = np.asarray(O, dtype=np.float64)\n    X = np.asarray(X, dtype=np.float64)\n    rng = rng or np.random.default_rng(0)\n    if X.shape[-2] != O.shape[1]:\n        raise ValueError(f"X token axis {X.shape[-2]} != O dim {O.shape[1]}")\n    if kind == "iid":\n        Z = rng.standard_normal(X.shape)\n        return Z * float(np.sqrt((X ** 2).sum() / (Z ** 2).sum()))\n    if kind == "content_matched":\n        Z = rng.standard_normal(X.shape)\n        Z = Z / np.linalg.norm(Z, axis=-1, keepdims=True)\n        norms = np.linalg.norm(X, axis=-1, keepdims=True)\n        return Z * norms\n    raise ValueError(f"kind must be iid/content_matched, got {kind!r}")\n\n\ndef mp_upper_edge(p: int, n: int) -> float:\n    """Marchenko–Pastur upper edge ``(1 + sqrt(p/n))²`` of ``XᵀX/n``, ``X``\n    ``[n, p]`` — the normalization under which the ТЗ\'s «MP-скобки» reproduce:\n    p=6, n=32 → 2.0535 ≈ 2.05; p=8, n=32 → 2.2500 = 2.25. The second numbers\n    of the early ТЗ pairs are n=192 edges (1.3848 for p=6; p=8 gives 1.4499 —\n    the quoted «1.45» was itself wrong, and the «1.4425» of an earlier draft\n    was retracted too; ТЗ §III.11 keeps 1.4499). патч v2 §2.1 retracts both\n    second numbers as n-unlabelled; finite-n MC means sit below the edge and\n    are the plaintiff (user ruling 2026-11-03)."""\n    if p <= 0 or n <= 0:\n        raise ValueError("p and n must be positive")\n    return float((1.0 + np.sqrt(p / n)) ** 2)\n\n\ndef mp_lambda_max_mc(p: int, n: int, B: int = 400,\n                     rng: np.random.Generator | None = None) -> dict:\n    """Monte-Carlo calibration of ``λ_max`` of ``XᵀX/n``, ``X`` ``[n, p]`` iid\n    N(0,1): mean/median/quantiles vs the MP edge. Test ТЗ §IV.6."""\n    rng = rng or np.random.default_rng(0)\n    lams = np.empty(B)\n    for b in range(B):\n        X = rng.standard_normal((n, p))\n        lams[b] = np.linalg.eigvalsh(X.T @ X / n)[-1]\n    return {"p": p, "n": n, "B": B,\n            "mean": float(lams.mean()), "median": float(np.median(lams)),\n            "q05": float(np.quantile(lams, 0.05)), "q95": float(np.quantile(lams, 0.95)),\n            "mp_edge": mp_upper_edge(p, n)}\n\n\ndef reduction_control(statistic: np.ndarray, trivial_predictor: np.ndarray,\n                      partial_vars: np.ndarray | None = None) -> dict:\n    """Standard pattern for derived metrics: regress the statistic on the\n    trivial predictor (+ optional covariates), return the residual signal and\n    how much the trivial model already explains. Guards «correlates with X»\n    claims where X is mechanical (e.g. energy vs b_mag, E44 §5.1.1)."""\n    y = np.asarray(statistic, dtype=np.float64).ravel()\n    cols = [np.ones_like(y), np.asarray(trivial_predictor, dtype=np.float64).ravel()]\n    if partial_vars is not None:\n        P = np.atleast_2d(np.asarray(partial_vars, dtype=np.float64))\n        if P.shape[0] < P.shape[1]:\n            P = P.T\n        cols += [P[:, j] for j in range(P.shape[1])]\n    A = np.column_stack(cols)\n    beta, *_ = np.linalg.lstsq(A, y, rcond=None)\n    resid = y - A @ beta\n    ss = float(((y - y.mean()) ** 2).sum())\n    # A constant statistic (ss == 0) has nothing to explain: reporting r² = 1\n    # would read as "the trivial model already explains everything" for a\n    # degenerate input (audit4 Q16). Report 0.0 with an explicit flag instead.\n    degenerate = ss <= 0.0\n    r2 = 0.0 if degenerate else 1.0 - float((resid ** 2).sum()) / ss\n    return {"residual": resid, "r2_trivial": float(min(r2, 1.0)),\n            "degenerate_constant_statistic": degenerate,\n            "residual_std": float(resid.std(ddof=A.shape[1]))}\n\n\ndef paired_bootstrap(docs: np.ndarray, stat_fn, B: int = 1000,\n                     seed: int = 0, ci: float = 95.0) -> dict:\n    """Percentile bootstrap over document index (resampling the DOCS, not\n    rows) — the paired CI convention of E-39f/E-39m artifacts."""\n    docs = np.asarray(docs)\n    rng = np.random.default_rng(seed)\n    n = docs.shape[0]\n    vals = np.array([stat_fn(docs[rng.integers(0, n, n)]) for _ in range(B)])\n    a = (100.0 - ci) / 2.0\n    return {"point": float(stat_fn(docs)),\n            "ci_low": float(np.quantile(vals, a / 100.0)),\n            "ci_high": float(np.quantile(vals, 1 - a / 100.0)),\n            "B": B}\n\n\ndef permutation_test(observed: float, null_values: np.ndarray,\n                     alternative: str = "greater") -> float:\n    """One-sided p-value ``(1 + #{null ≥ obs}) / (B + 1)`` (``"less"`` mirrors).\n    ``null_values`` come from ``prism.permutation.generate_null`` or\n    :func:`operator_null` — this function only scores them. A typo in\n    ``alternative`` must not silently mirror the p-value (audit M7).\n    """\n    if alternative not in ("greater", "less"):\n        raise ValueError(f"alternative={alternative!r} not in greater|less "\n                         "(an unlabelled value silently mirrored the verdict)")\n    nv = np.asarray(null_values, dtype=np.float64)\n    B = nv.size\n    if B == 0:\n        raise ValueError("empty null distribution")\n    hits = int((nv >= observed).sum()) if alternative == "greater" else int((nv <= observed).sum())\n    return float((1 + hits) / (B + 1))\n\n\ndef resolution_floor(metric_fn, docs: np.ndarray,\n                     subsamples: tuple[int, ...] = (8, 16, 32),\n                     reps: int = 40, seed: int = 0) -> dict:\n    """Per-test resolution floor: spread of ``metric_fn`` over subsamples of\n    docs at each size (within the test, never a global constant — ТЗ §II.9).\n    Compare an effect against ``floor_full`` BEFORE interpreting absence:\n    effect below the floor ⇒ status ``not_testable``, not «null».\n\n    The full-size floor resamples WITH REPLACEMENT (bootstrap): without\n    replacement, m = n is a permutation and an order-invariant statistic has\n    spread exactly 0, which made the prescribed guard structurally vacuous\n    (audit M5). Sub-full sizes stay without replacement — there the floor is\n    the «which docs you happened to draw» spread.\n    """\n    docs = np.asarray(docs)\n    rng = np.random.default_rng(seed)\n    n = docs.shape[0]\n    sizes = sorted(set(subsamples) | {n})\n    floors = {}\n    for m in sizes:\n        if m > n:\n            continue\n        vals = [float(metric_fn(docs[rng.choice(n, m, replace=(m == n))]))\n                for _ in range(reps)]\n        floors[m] = float(np.std(vals))\n    return {"floor_by_size": floors, "floor_full": floors.get(n),\n            "n_docs": n}\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/operator_checks.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Operator-fraction checks and the two-null reduction control call-signature.\n\nImplements ТЗ E40_math_core §II.7 (M7). Everything here answers one question:\nhow much of ``X`` does an operator ``O`` eat, and is that number information\nabout the DATA or just the norm of the OPERATOR (paper §6 items 19/22/23).\n\nAggregation rule (paper §6 item 23, E6 lesson): for stacks of positions the\ncorpus aggregate sums numerator and denominator over the SAME slice in ONE\nratio. Per-position medians/means are different objects — the identity\n``frac_K = 1 − 2c + s²`` closes exactly for the aggregates and for each\nposition separately, but NOT for a median of one quantity combined with the\naggregate of another (this exact mix produced the phantom ``s² < 0`` of E6).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n__all__ = [\n    "operator_fraction", "isotropic_reference", "apply_P",\n    "c", "s", "O1", "cos_J_PJ", "frac_K",\n]\n\n\ndef _check_OX(O: np.ndarray, X: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    O = np.asarray(O, dtype=np.float64)\n    X = np.asarray(X, dtype=np.float64)\n    if O.ndim != 2 or O.shape[0] != O.shape[1]:\n        raise ValueError(f"O must be square 2-D, got {O.shape}")\n    if X.ndim < 2 or X.shape[-2] != O.shape[1]:\n        raise ValueError(\n            f"X must have its second-to-last axis = O.shape[1]={O.shape[1]}, got {X.shape}")\n    return O, X\n\n\ndef operator_fraction(O: np.ndarray, X: np.ndarray, norm_type: str) -> float:\n    """``‖OX‖²_F/‖X‖²_F`` (``norm_type="energy"``) or ``‖OX‖_F/‖X‖_F`` (``"norm"``).\n\n    ``norm_type`` is identity-relevant (norm vs energy differ by a square —\n    paper §6 item 19); the returned value is meaningless without a ChannelTag.\n    ``norm_type`` has NO default: a default silently labels the number as an\n    energy share, and norm vs energy differ by a square (audit2 recommendation,\n    same error class as M6 — unlabelled values are rejected, unlabelled calls\n    are now impossible).\n    """\n    if norm_type not in ("energy", "norm"):\n        raise ValueError(f"norm_type={norm_type!r} not in energy|norm "\n                         "(norm vs energy differ by a square, paper §6 item 19)")\n    O, X = _check_OX(O, X)\n    OX = np.tensordot(O, X, axes=([1], [-2]))          # [..., m, q]\n    num = float((OX ** 2).sum())\n    den = float((X ** 2).sum())\n    if den <= 0:\n        raise ValueError("operator_fraction: ‖X‖ = 0")\n    return num / den if norm_type == "energy" else float(np.sqrt(num / den))\n\n\ndef isotropic_reference(O: np.ndarray) -> float:\n    """``E‖OX‖²/‖X‖²`` over isotropic ``X`` = ``‖O‖²_F / dim_in``.\n\n    Any claim «O eats f·(reference)» must be quoted against this AND against a\n    content-matched null (paper §6 item 23); the operator alone already eats\n    ``1 − conc`` of a random input (E-39 supplement: conc_iid ≈ 0.115).\n    """\n    O = np.asarray(O, dtype=np.float64)\n    if O.ndim != 2 or O.shape[0] != O.shape[1]:\n        raise ValueError(f"O must be square 2-D, got {O.shape}")\n    return float((O ** 2).sum() / O.shape[1])\n\n\ndef apply_P(J: np.ndarray, p: np.ndarray) -> np.ndarray:\n    """``P(p) J`` along the token axis without materializing P (torch-free).\n\n    Accepts ``J [K, D]`` + ``p [K]``, or stacks ``J [n, K, D]`` with ``p [K]``\n    or per-position ``p [n, K]``.\n    """\n    J = np.asarray(J, dtype=np.float64)\n    p = np.asarray(p, dtype=np.float64)\n    K = J.shape[-2]\n    if p.shape == (K,):\n        pw = p[:, None]                                # broadcast over the D axis\n    elif p.shape == J.shape[:-1]:\n        pw = p[..., None]\n    else:\n        raise ValueError(f"p shape {p.shape} incompatible with J {J.shape}")\n    m = (pw * J).sum(axis=-2, keepdims=True)           # p-weighted row mean\n    return pw * (J - m)                                # (PJ)_i = p_i (J_i − m)\n\n\ndef _split(J: np.ndarray, p: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    PJ = apply_P(J, p)\n    return PJ, p\n\n\ndef _check_p_slice(p: np.ndarray, p_slice: str) -> None:\n    """Label guard (патч v2 §1.6.2): ``p_t`` = per-position, ``p_0`` = fixP.\n    They are DIFFERENT operators ``P(p)``, so the label must be explicit — and\n    ``p_0`` is validated: stacked rows must actually be one fixed p."""\n    if p_slice not in ("p_t", "p_0"):\n        raise ValueError(f"p_slice={p_slice!r} not in p_t|p_0 (stop condition)")\n    if p_slice == "p_0" and p.ndim >= 2:\n        dev = float(np.abs(p - p.reshape(-1, p.shape[-1])[0]).max())\n        if dev > 1e-12:\n            raise ValueError(\n                f"p_slice=\'p_0\' but p varies per position (max dev {dev:.3g}); "\n                "this is per-position p_t — mislabelling mixes operators (E6 class)")\n\n\ndef _agg(num: float | np.ndarray, den: float | np.ndarray, aggregation: str,\n         root: bool = False):\n    """One aggregation over one slice — never mix them across quantities (E6)."""\n    if aggregation == "ratio_of_means":\n        val = float(num.sum() / den.sum())\n        return val ** 0.5 if root else val\n    if aggregation == "energy_weighted":\n        raise ValueError(\n            "aggregation=\'energy_weighted\' was removed (audit4 Q3): the energy-weighted "\n            "mean of per-position ratios IS the ratio of sums — pass \'ratio_of_means\'. "\n            "A genuinely different weighting (‖J_t‖ instead of ‖J_t‖², an external "\n            "covariate) is a different number and needs its own name and tag value.")\n    r = num / den\n    if aggregation == "mean":\n        return float((np.sqrt(r) if root else r).mean())\n    if aggregation == "median":\n        return float(np.median(np.sqrt(r) if root else r))\n    if aggregation == "per_position":   # RAW energy ratios (root is a property of the\n        return r                        # aggregate; frac = 1 − 2c + O1 closes positionally)\n    raise ValueError(\n        f"aggregation={aggregation!r} not in ratio_of_means|mean|median|per_position")\n\n\ndef c(J: np.ndarray, p: np.ndarray, aggregation: str = "ratio_of_means",\n      p_slice: str = "p_t") -> np.ndarray | float:\n    """``c = ⟨J, PJ⟩_F / ‖J‖²_F`` — cross term of ``‖(I−P)J‖²`` (PSD of P\n    licenses ``c ≥ 0`` and NOTHING about alignment: measured cos(J, PJ) is\n    only 0.064–0.108, i.e. PJ is nearly ORTHOGONAL to J, not parallel to it).\n\n    ``p_slice`` labels which ``P(p)`` produced the number (per-position ``p_t``\n    vs fixed ``p_0``); the position slice (all/alive) is whatever you passed —\n    report it with the number (tag, §1.6.5)."""\n    _check_p_slice(p, p_slice)\n    PJ, _ = _split(J, p)\n    num = (PJ * J).sum(axis=(-2, -1))\n    den = (J * J).sum(axis=(-2, -1))\n    return _agg(num, den, aggregation)\n\n\ndef s(J: np.ndarray, p: np.ndarray, aggregation: str = "ratio_of_means",\n      p_slice: str = "p_t") -> np.ndarray | float:\n    """``s = ‖PJ‖_F / ‖J‖_F`` — the NORM ratio (E-33r\'s «98–99 %» object).\n\n    Jensen warning (audit M3): for ``mean``/``median`` this is the aggregate\n    of the PER-POSITION root, so ``s()**2 != O1(..., same aggregation)``\n    (``(mean √r)² ≤ mean r``) and ``frac_K = 1 − 2c + s²`` does NOT close.\n    The identity-closing triples are ``ratio_of_means`` and ``per_position``;\n    ``exp40_1_t4a`` computes the mean cell as ``mean_t(s_t²)`` for exactly\n    this reason."""\n    _check_p_slice(p, p_slice)\n    PJ = apply_P(J, p)\n    num = (PJ * PJ).sum(axis=(-2, -1))\n    den = (J * J).sum(axis=(-2, -1))\n    return _agg(num, den, aggregation, root=True)\n\n\ndef O1(J: np.ndarray, p: np.ndarray, aggregation: str = "ratio_of_means",\n       p_slice: str = "p_t") -> np.ndarray | float:\n    """``O1 = ‖PJ‖²_F / ‖J‖²_F = s²`` — ENERGY kill fraction. Test O1==s² is a\n    tautology guard on the aggregation, keep it asserted (M0 registry)."""\n    _check_p_slice(p, p_slice)\n    PJ = apply_P(J, p)\n    num = (PJ * PJ).sum(axis=(-2, -1))\n    den = (J * J).sum(axis=(-2, -1))\n    return _agg(num, den, aggregation)\n\n\ndef cos_J_PJ(J: np.ndarray, p: np.ndarray, aggregation: str = "ratio_of_means",\n             p_slice: str = "p_t") -> np.ndarray | float:\n    """``cos(J, PJ) = c/s`` — a DIFFERENT number from ``c`` (ТЗ §III.3):\n    measured 0.064–0.108 while c is 0.0005–0.003.\n\n    Restricted to ``ratio_of_means`` (pooled cosine) and ``per_position``:\n    under ``mean``/``median`` this formula is ``mean(c_t)/mean(s_t)`` — a\n    ratio-of-aggregates-of-ratios that is neither the pooled cosine nor the\n    mean cosine (audit M4); compute that hybrid by hand if you need it.\n    """\n    if aggregation not in ("ratio_of_means", "per_position"):\n        raise ValueError(\n            f"cos_J_PJ: aggregation={aggregation!r} gives mean(c)/mean(s), not a "\n            "cosine (audit M4); use ratio_of_means (pooled) or per_position")\n    return c(J, p, aggregation, p_slice) / s(J, p, aggregation, p_slice)\n\n\ndef frac_K(J: np.ndarray, p: np.ndarray, aggregation: str = "ratio_of_means",\n           p_slice: str = "p_t") -> np.ndarray | float:\n    """``frac_K = ‖(I−P)J‖²_F / ‖J‖²_F`` — ENERGY, not norm (E-37 metric).\n\n    Identity ``frac_K = 1 − 2c + s²`` holds per position and for the corpus\n    aggregate computed on one slice; combining it with per-position medians is\n    what produced the phantom E6 (closed 2026-11-03, no longer blocked).\n    """\n    _check_p_slice(p, p_slice)\n    PJ = apply_P(J, p)\n    Kmat = J - PJ\n    num = (Kmat * Kmat).sum(axis=(-2, -1))\n    den = (J * J).sum(axis=(-2, -1))\n    return _agg(num, den, aggregation)\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/p_algebra.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Algebra of the softmax-gauge operator ``P(p) = diag(p) − p pᵀ`` (ТЗ E40_math_core §I.1, §II.3, M3).\n\n``P`` plays three roles at once (paper §2.5): Jacobian of softmax, Fisher–Rao\nmetric in logit coordinates (``JᵀPJ``), and gauge annihilator (``P·1 = 0``).\n\nNot to be confused with ``prism.jacobian.softmax_jacobian(J, p)`` which APPLIES\n``P(p)`` to a Jacobian (torch, never materializes ``P``). This module is the\nnumpy algebra OF the operator itself. Naming follows the physics object, so\nthe ТЗ\'s ``softmax_jacobian(p) -> [K, K]`` lives here as :func:`P_matrix`.\n\nClosed forms (exact, no eigendecomposition):\n- ``tr P = Σp − Σp²``            (= ``1 − Σp²`` for normalized p)\n- ``‖P‖²_F = tr P² = Σp² − 2Σp³ + (Σp²)²``\n- ``P`` is symmetric PSD (``xᵀPx = Var_p(x)``), ``rank = K−1`` for full-support\n  p, and ``P`` is NOT a projector: ``P² = P`` is impossible for interior p\n  (eigenvalues are < 1; at the vertex ``p = e_i`` deg ``P = 0`` IS idempotent —\n  the boundary case, ТЗ §I.1 wording corrected, see docs/E40_math_core_answers.md).\n- eigenvalue bound ``λ_max ≤ p_max`` (from ``P ⪯ diag p``), NOT\n  ``p_max(1−p_max)`` which fails by 2× at the two-point limit (ТЗ §I.1 claim,\n  refuted numerically in the answers file).\n"""\n\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\n\n__all__ = [\n    "P_matrix", "trace_P", "trace_P2", "isotropic_reference",\n    "g_uni", "gauge_share", "secular_residual", "secular_roots",\n]\n\n\ndef _check_p(p: np.ndarray) -> np.ndarray:\n    p = np.asarray(p, dtype=np.float64)\n    if p.ndim != 1 or p.size < 2:\n        raise ValueError(f"p must be a 1-D array of length ≥ 2, got shape {p.shape}")\n    if not np.isfinite(p).all():\n        raise ValueError("p has non-finite entries")\n    if (p < 0).any():\n        raise ValueError("p has negative entries (not a probability vector)")\n    # Допуск нормировки — по точности входа, не фиксированный 1e-8: float32-строка,\n    # нормированная на 1, несёт |Σp−1| до ~K·eps32 (на кэше exp27b замер: медиана\n    # 4.0e-08, p99 1.7e-07, max 2.8e-07 при K=128), поэтому фиксированный 1e-8\n    # отвергал 88 % реальных позиций (docs/audit3_math.md п. 2Б + самопроверка).\n    # Обрезанную строку (P128_RAW, Σp ≈ 0.49) этот допуск по-прежнему ловит.\n    tol = max(1e-8, p.size * np.finfo(np.float32).eps)\n    if abs(p.sum() - 1.0) > tol:\n        raise ValueError(\n            f"p must be normalized (Σp=1); got Σp={p.sum():.12g} (tol {tol:.1e}, "\n            f"= K·eps32 for float32-sourced p). Truncated/raw rows (P128_RAW) must "\n            "be renormalized upstream and tagged p_mode in the ChannelTag (ТЗ §II.1).")\n    return p\n\n\ndef P_matrix(p: np.ndarray) -> np.ndarray:\n    """``P(p) = diag(p) − p pᵀ`` as a dense ``[K, K]`` matrix."""\n    p = _check_p(p)\n    return np.diag(p) - np.outer(p, p)\n\n\ndef trace_P(p: np.ndarray) -> float:\n    """``tr P = Σp − Σp²`` (= ``1 − ‖p‖₂²`` for normalized p)."""\n    p = _check_p(p)\n    return float(p.sum() - np.dot(p, p))\n\n\ndef trace_P2(p: np.ndarray) -> float:\n    """``tr P² = ‖P‖²_F = Σp² − 2Σp³ + (Σp²)²`` (closed form)."""\n    p = _check_p(p)\n    s2, s3 = float(np.dot(p, p)), float(np.dot(p, p * p))\n    return s2 - 2.0 * s3 + s2 * s2\n\n\ndef isotropic_reference(p: np.ndarray) -> float:\n    """``iso = ‖P‖²_F / K`` — what ``‖PJ‖²_F/‖J‖²_F` would be for isotropic J."""\n    p = _check_p(p)\n    return trace_P2(p) / p.size\n\n\ndef g_uni(J: np.ndarray) -> float:\n    """Uniform-direction energy share ``‖Jᵀ1‖² / (K ‖J‖²_F)`` (paper §2.5).\n\n    Corpus-aggregated: accepts ``[K, D]`` or ``[n, K, D]``; sums numerator and\n    denominator over everything, ONE aggregate (never a mean of per-position\n    ratios — aggregation rule, paper §6 item 23). Identical by construction to\n    ``gauge_share`` of the Gramian ``G = Σ_t J_t J_tᵀ`` (identity O2≡O3,\n    max measured discrepancy 7.7e-16). Independent of p — p is not an argument.\n    """\n    J = np.asarray(J, dtype=np.float64)\n    if J.ndim == 2:\n        J = J[None]\n    if J.ndim != 3:\n        raise ValueError(f"J must be [K, D] or [n, K, D], got {J.shape}")\n    K = J.shape[1]\n    col_sums = J.sum(axis=1)                    # [n, D]: Σ_k J_k\n    num = float((col_sums ** 2).sum())          # Σ_t ‖Σ_k J_k‖²\n    den = float((J ** 2).sum())\n    if den <= 0:\n        raise ValueError("g_uni: ‖J‖_F = 0")\n    return num / (K * den)\n\n\ndef gauge_share(G: np.ndarray) -> float:\n    """ESG gauge share ``1ᵀG1 / (K tr G)`` on the Gramian ``G`` (``[K, K]``).\n\n    Equals ``Σ_k (u_kᵀ1)² λ_k / Σ_k λ_k`` = ``spectral.gauge_share(spectrum,\n    eigenvectors)`` on the same ``G`` (identity O2≡O3), but needs no\n    eigendecomposition. Not a duplicate of ``prism.spectral.gauge_share``:\n    different input (Gramian, not spectrum+vectors), numpy not torch.\n    """\n    G = np.asarray(G, dtype=np.float64)\n    if G.ndim != 2 or G.shape[0] != G.shape[1]:\n        raise ValueError(f"G must be square [K, K], got {G.shape}")\n    tr = float(np.trace(G))\n    if tr <= 0:\n        raise ValueError("gauge_share: tr G ≤ 0")\n    K = G.shape[0]\n    ones = np.ones(K)\n    return float(ones @ G @ ones) / (K * tr)\n\n\ndef secular_residual(p: np.ndarray, lam: float) -> float:\n    """``Σ_i p_i²/(p_i − λ) − 1``; zero exactly at the eigenvalues of ``P``\n    strictly between the poles (secular equation, ТЗ §I.1)."""\n    p = _check_p(p)\n    d = p - lam\n    if np.abs(d).min() < 1e-14:\n        raise ValueError("lam coincides with a pole p_i")\n    return float(np.dot(p * p, 1.0 / d) - 1.0)\n\n\ndef secular_roots(p: np.ndarray) -> np.ndarray:\n    """The ``K−1`` nonzero eigenvalues of ``P(p)`` via root search between the\n    poles of the secular equation (structure-preserving, no ``eigvalsh``).\n\n    Assumes distinct ``p_i`` (poles separate the roots: exactly one root in\n    each gap between consecutive sorted ``p_i``; f increases strictly from\n    ``−∞`` at the lower pole to ``+∞`` at the upper one). Falls back loudly if\n    p has duplicates within 1e-12 — the secular form degenerates there\n    (deflated block), and ``np.linalg.eigvalsh(P_matrix(p))`` is the right tool.\n\n    ПОТОЛОК (измерено на реальном срезе softmax, exp27b, K=128, 1580 позиций):\n    на 11.4 % позиций полюса неразрешимы в fp64 и функция отказывается — это\n    честный предел метода бисекции, а не баг; для них — ``eigvalsh``. На\n    остальных 88.6 % корни сходятся с ``eigvalsh(P_matrix(p))`` до 8.7e-12\n    (тезис `docs/audit3_math.md` п. 2Б: падать на реальных данных действительно\n    будет, но по причине разрешения полюсов, а не «одинаковых softmax-значений»;\n    вторая половина тезиса — «заменить на eigvalsh» — не принята, см.\n    `docs/audits_answers.md` §9).\n    Абсолютный порог здесь сознательный: относительный («от p_max») ломает\n    бисекцию, когда зазор оказывается меньше ulp верхнего полюса (проверено —\n    даёт 100 % отказов «корень совпал с полюсом»).\n    """\n    p = np.sort(_check_p(p))[::-1]                    # descending p_1 > p_2 > …\n    K = p.size\n    if len(np.unique(np.round(p, 12))) < K:\n        raise ValueError("secular_roots: duplicate p_i — use eigvalsh instead")\n    roots = np.empty(K - 1)\n    for i in range(K - 1):                             # one root in (p_{i+1}, p_i)\n        lo, hi = p[i + 1], p[i]\n        span = hi - lo\n        if span < 1e-12:\n            raise ValueError("secular_roots: adjacent p_i closer than 1e-12 "\n                             "— use np.linalg.eigvalsh instead")\n        off = min(span * 0.01, max(span * 1e-9, 1e-13))\n        a, b = lo + off, hi - off\n        fa, fb = secular_residual(p, a), secular_residual(p, b)\n        nudge = 0\n        while not (fa < 0.0 < fb):                     # pull endpoints inward\n            nudge += 1\n            if nudge > 40:\n                raise RuntimeError(f"secular_roots: no bracket in ({lo}, {hi})")\n            if fa >= 0:\n                a = 0.5 * (a + lo)\n                fa = secular_residual(p, a)\n            if fb <= 0:\n                b = 0.5 * (b + hi)\n                fb = secular_residual(p, b)\n        for _ in range(200):                           # bisection\n            m = 0.5 * (a + b)\n            fm = secular_residual(p, m)\n            if fm < 0:\n                a, fa = m, fm\n            else:\n                b, fb = m, fm\n            if b - a < 1e-14 * max(1.0, abs(m)):\n                break\n        roots[i] = 0.5 * (a + b)\n    return roots\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/permutation.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Permutation max-statistic (FWER) and cluster-based alternative\n(TZ part 3, §6).\n\nNo forward passes: works on already-computed scalars (from MetricCache)\nand a paired permutation null generator. BH-FDR is deliberately NOT\nimplemented: Westfall–Young max-statistic controls FWER under arbitrary\ncorrelation between layers (L18–L23 are strongly correlated), while\nBH-FDR requires independence/PRDS. This is a choice, not a compromise.\n\npaper.md §8.9, §13 — per-layer profiles and FWER control.\n"""\n\nfrom __future__ import annotations\n\nimport math\nfrom collections.abc import Callable\n\nimport numpy as np\n\n__all__ = [\n    "permutation_max_statistic",\n    "generate_null",\n    "cluster_based_permutation",\n]\n\n\ndef _check_finite(name: str, arr: np.ndarray) -> None:\n    if not np.all(np.isfinite(arr)):\n        bad = int(np.argmax(~np.isfinite(arr)))\n        raise ValueError(\n            f"{name} contains NaN/Inf (first at flat index {bad}); "\n            f"a bug in the null generator or metric is being hidden"\n        )\n\n\ndef _check_common(\n    observed: np.ndarray, null_samples: np.ndarray, alpha: float\n) -> tuple[np.ndarray, np.ndarray]:\n    observed = np.asarray(observed, dtype=float)\n    null_samples = np.asarray(null_samples, dtype=float)\n    if observed.ndim != 1:\n        raise ValueError(\n            f"observed must be a flat [n_layers] profile, got shape {observed.shape}"\n        )\n    if null_samples.ndim != 2:\n        raise ValueError(\n            f"null_samples must be [B, n_layers], got shape {null_samples.shape}"\n        )\n    if null_samples.shape[1] != len(observed):\n        raise ValueError(\n            f"null_samples columns ({null_samples.shape[1]}) != len(observed) "\n            f"({len(observed)})"\n        )\n    if null_samples.shape[0] < 100:\n        raise ValueError(\n            f"B = {null_samples.shape[0]} < 100: too few permutations for FWER"\n        )\n    if not (0.0 < alpha < 1.0):\n        raise ValueError(f"alpha must be in (0, 1), got {alpha}")\n    _check_finite("observed", observed)\n    _check_finite("null_samples", null_samples)\n    # Return the CONVERTED arrays: rebinding parameters alone would not\n    # propagate to the callers, and cluster_based_permutation needs array\n    # semantics (fancy indexing) — a list input must not TypeError there.\n    return observed, null_samples\n\n\ndef permutation_max_statistic(\n    observed: np.ndarray,\n    null_samples: np.ndarray,\n    alpha: float = 0.05,\n) -> dict:\n    """Westfall–Young max-statistic FWER over an [n_layers] profile.\n\n    p = (1 + Σ_b [max_l |null[b, l]| ≥ max_l |observed[l]|]) / (B + 1) —\n    the exact discrete permutation formula (Wright et al., 2015). The\n    max over layers is what absorbs inter-layer correlation; no\n    independence assumption.\n\n    TZ selfcheck §6.6.3 is self-contradictory (asserts "all null maxes\n    below max_obs" AND "k = 100, p = 1.0"); the formula above gives\n    k = 0, p = 1/(B+1) in that case. The formula is followed.\n\n    ``observed`` should be the same statistic that ``metric_fn`` returns\n    in generate_null (e.g. per-layer median(edit − base) across docs).\n\n    Returns:\n        dict with ``p_value``, ``max_observed``, ``null_max_distribution``\n        ([B]), ``significant`` (p < alpha), ``alpha``, ``n_layers``, ``B``.\n\n    Errors:\n        ValueError: shape mismatch, B < 100, alpha outside (0, 1),\n            NaN/Inf in inputs.\n    """\n    observed, null_samples = _check_common(observed, null_samples, alpha)\n\n    max_obs = float(np.max(np.abs(observed)))\n    max_null = np.max(np.abs(null_samples), axis=1)  # [B]\n    if max_obs == 0.0:\n        p_value = 1.0\n    else:\n        k = int(np.sum(max_null >= max_obs))\n        p_value = (1 + k) / (len(max_null) + 1)\n\n    return {\n        "p_value": float(p_value),\n        "max_observed": max_obs,\n        "null_max_distribution": max_null,\n        "significant": p_value < alpha,\n        "alpha": alpha,\n        "n_layers": len(observed),\n        "B": int(len(max_null)),\n    }\n\n\ndef _paired_permutation_indices(n_docs: int, rng: np.random.Generator) -> np.ndarray:\n    """Per-doc coin flip: True keeps (base, edit) as is, False swaps."""\n    return rng.random(n_docs) < 0.5\n\n\ndef generate_null(\n    base_values: np.ndarray,\n    edit_values: np.ndarray,\n    metric_fn: Callable[[np.ndarray, np.ndarray], np.ndarray],\n    n_permutations: int = 1000,\n    seed: int = 0,\n) -> np.ndarray:\n    """Paired permutation null [B, n_layers] (TZ §6.3).\n\n    For each permutation, each document independently swaps its\n    base/edit labels (coin flip) — the standard paired permutation test\n    under H0: base and edit are exchangeable within pairs. ``metric_fn``\n    is called B times with (base_perm, edit_perm) and must return the\n    SAME statistic used for ``observed`` in permutation_max_statistic.\n\n    Args:\n        base_values, edit_values: [n_docs] or [n_docs, n_layers], same shape.\n        metric_fn: (base_perm, edit_perm) -> [n_layers]; pure.\n        n_permutations: B ≥ 100.\n        seed: np.random.default_rng(seed) — deterministic, no global seed.\n\n    Errors:\n        ValueError: shape mismatch, n_docs < 2, B < 100, NaN/Inf,\n            metric_fn output not a 1-D array.\n    """\n    base_values = np.asarray(base_values, dtype=float)\n    edit_values = np.asarray(edit_values, dtype=float)\n    if base_values.shape != edit_values.shape:\n        raise ValueError(\n            f"base_values shape {base_values.shape} != "\n            f"edit_values shape {edit_values.shape}"\n        )\n    n_docs = base_values.shape[0]\n    if n_docs < 2:\n        raise ValueError(f"paired test requires n_docs >= 2, got {n_docs}")\n    if n_permutations < 100:\n        raise ValueError(f"n_permutations must be >= 100, got {n_permutations}")\n    _check_finite("base_values", base_values)\n    _check_finite("edit_values", edit_values)\n\n    rng = np.random.default_rng(seed)\n    rows: list[np.ndarray] = []\n    for b in range(n_permutations):\n        signs = _paired_permutation_indices(n_docs, rng)\n        # signs True: keep (base, edit); False: swap within the pair.\n        # 2-D ([n_docs, n_layers]): broadcast along ROWS — the per-doc coin\n        # flip must swap the whole document row, not per column. (The bare\n        # np.where(signs, ...) broadcasts signs left-padded to (1, n_docs),\n        # which errors when n_docs != n_layers and silently swaps COLUMNS —\n        # breaking the paired structure — when they are equal.)\n        if base_values.ndim == 1:\n            base_perm = np.where(signs, base_values, edit_values)\n            edit_perm = np.where(signs, edit_values, base_values)\n        else:\n            s = signs[:, None]  # [n_docs, 1]\n            base_perm = np.where(s, base_values, edit_values)\n            edit_perm = np.where(s, edit_values, base_values)\n        stat = np.asarray(metric_fn(base_perm, edit_perm), dtype=float)\n        if stat.ndim != 1 or stat.size == 0:\n            raise ValueError(\n                f"metric_fn must return a non-empty 1-D [n_layers] array, "\n                f"got shape {stat.shape} at permutation {b}"\n            )\n        if not np.all(np.isfinite(stat)):\n            raise ValueError(\n                f"metric_fn returned NaN/Inf at permutation {b}"\n            )\n        if rows and stat.shape != rows[0].shape:\n            raise ValueError(\n                f"metric_fn returned inconsistent shapes: {rows[0].shape} "\n                f"vs {stat.shape} at permutation {b}"\n            )\n        rows.append(stat)\n    return np.stack(rows, axis=0)\n\n\ndef _find_clusters(values: np.ndarray, threshold: float) -> list[list[int]]:\n    """Contiguous runs of indices with |values[l]| > threshold."""\n    mask = np.abs(values) > threshold\n    clusters: list[list[int]] = []\n    cur: list[int] = []\n    for l in range(len(values)):\n        if mask[l]:\n            cur.append(l)\n        elif cur:\n            clusters.append(cur)\n            cur = []\n    if cur:\n        clusters.append(cur)\n    return clusters\n\n\ndef cluster_based_permutation(\n    observed: np.ndarray,\n    null_samples: np.ndarray,\n    threshold: float = 2.0,\n    alpha: float = 0.05,\n) -> dict:\n    """Cluster-based permutation FWER (Maris & Oostenveld, 2007).\n\n    Groups adjacent super-threshold layers and tests the cluster sum\n    Σ |values[l]| against the max cluster sum over the null. The caller\n    must pass a NORMALIZED statistic (e.g. paired t or Cohen\'s d per\n    layer); this function uses |values| as the statistic.\n\n    Secondary to permutation_max_statistic: use it when the max-statistic\n    p is slightly above alpha but 2–3 adjacent layers clearly show the\n    effect (grouping raises power).\n\n    Returns:\n        dict with ``clusters`` (list of {"layers", "statistic", "p_value",\n        "significant"}), ``null_max_cluster_distribution`` ([B]),\n        ``alpha``, ``n_layers``, ``B``.\n\n    Errors:\n        ValueError: threshold ≤ 0, alpha outside (0, 1), shape issues,\n            B < 100, NaN/Inf.\n    """\n    if threshold <= 0:\n        raise ValueError(f"threshold must be > 0, got {threshold}")\n    observed, null_samples = _check_common(observed, null_samples, alpha)\n\n    obs_clusters = _find_clusters(observed, threshold)\n\n    # Null: for each b, threshold |null[b]|, sum |.| per cluster, take max\n    # over clusters (0.0 if none).\n    null_max_cluster = np.zeros(len(null_samples))\n    for b in range(len(null_samples)):\n        clusters = _find_clusters(null_samples[b], threshold)\n        if clusters:\n            null_max_cluster[b] = max(\n                float(np.sum(np.abs(null_samples[b][c]))) for c in clusters\n            )\n\n    clusters_out = []\n    for c in obs_clusters:\n        stat = float(np.sum(np.abs(observed[c])))\n        k = int(np.sum(null_max_cluster >= stat))\n        p = (1 + k) / (len(null_max_cluster) + 1)\n        clusters_out.append({\n            "layers": c,\n            "statistic": stat,\n            "p_value": float(p),\n            "significant": p < alpha,\n        })\n\n    return {\n        "clusters": clusters_out,\n        "null_max_cluster_distribution": null_max_cluster,\n        "alpha": alpha,\n        "n_layers": len(observed),\n        "B": int(len(null_max_cluster)),\n    }\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/pipeline.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Experiment specification + thin driver (TZ part 4).\n\n``ExperimentSpec`` freezes every parameter of a run; ``run_spec``\norchestrates validation, environment capture, seeds, output paths,\nmeta.json/results.json with atomic writes, exception diagnostics and\nidempotent reuse. ``run_spec`` does NOT load a model (that is\n``exp_fn``\'s job) and does NOT import any experiment-specific module\n(spectral/holonomy/noise/permutation/topology) — if it did, that would\nbe an architectural error.\n\npaper.md §19.A — spec structure.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nimport uuid\nfrom collections.abc import Callable, Mapping, Sequence\nfrom dataclasses import dataclass, field\nfrom datetime import datetime, timezone\nfrom types import MappingProxyType\nfrom typing import Any\n\nimport numpy as np\nimport torch\n\nfrom .cache import deterministic_mode\nfrom .interventions import EditSpec\n\n__all__ = [\n    "ExperimentSpec",\n    "OutputArtifacts",\n    "run_spec",\n    "run_many",\n    "validate_spec",\n    "load_artifacts",\n    "default_output_subdir",\n]\n\n_VALID_EXPERIMENTS = ("T0", "eps", "T2", "T1", "delta")\n\n\ndef _json_safe(obj: Any) -> Any:\n    """Recursive JSON conversion (TZ §3.3).\n\n    np.ndarray/torch.Tensor -> lists; NaN/Inf -> "nan"/"inf"/"-inf";\n    np scalars -> Python scalars; dict keys -> str; anything else ->\n    "<unserializable: Type>" (diagnostics, not strict validation).\n    """\n    if obj is None or isinstance(obj, (bool, int, str)):\n        return obj\n    if isinstance(obj, float):\n        if np.isnan(obj):\n            return "nan"\n        if np.isposinf(obj):\n            return "inf"\n        if np.isneginf(obj):\n            return "-inf"\n        return obj\n    if isinstance(obj, np.ndarray):\n        return _json_safe(obj.tolist())\n    if isinstance(obj, torch.Tensor):\n        return _json_safe(obj.detach().cpu().tolist())\n    if isinstance(obj, np.generic):\n        return _json_safe(obj.item())\n    if isinstance(obj, Mapping):\n        return {str(k): _json_safe(v) for k, v in obj.items()}\n    if isinstance(obj, set):  # deterministic order (to_key stability)\n        return [_json_safe(v) for v in sorted(obj, key=repr)]\n    if isinstance(obj, (list, tuple)):\n        return [_json_safe(v) for v in obj]\n    return f"<unserializable: {type(obj).__name__}>"\n\n\ndef _full_hash(spec: "ExperimentSpec") -> str:\n    payload = json.dumps(spec.to_meta_dict(), sort_keys=True, default=str)\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()\n\n\n@dataclass(frozen=True)\nclass ExperimentSpec:\n    """Frozen run specification (TZ §2).\n\n    ``config`` is a Mapping (typically MappingProxyType) with\n    JSON-serializable values; ``strict=True`` in config enables\n    ``deterministic_mode(strict=True)`` (do NOT default to strict —\n    some GDN kernels have no deterministic implementation).\n    """\n\n    experiment: str\n    edit: EditSpec\n    layers: tuple[int, ...]\n    token_set: tuple[int, ...]\n    n_docs: int\n    S: int\n    r: int = 16\n    n_theta: int = 16\n    n_steps: int = 20\n    epsilon_ratio: float = 0.1\n    B_perm: int = 1000\n    B_boot: int = 1000\n    k_sigma: float = 5.0\n    alpha: float = 0.05\n    seed: int = 0\n    cache_root: str = "results/.cache"\n    output_root: str = "results"\n    model_name: str = "Qwen/Qwen3.5-0.8B"\n    config: Mapping[str, Any] = field(\n        default_factory=lambda: MappingProxyType({})\n    )\n\n    def to_meta_dict(self) -> dict:\n        """JSON-serializable dict, EXCLUDING cache_root/output_root\n        (paths do not affect results)."""\n        d = {\n            "experiment": self.experiment,\n            "edit": {\n                "name": self.edit.name,\n                "norm_frobenius": _json_safe(self.edit.norm_frobenius),\n                "projection_onto_top_r": _json_safe(self.edit.projection_onto_top_r),\n            },\n            "layers": list(self.layers),\n            "token_set": list(self.token_set),\n            "n_docs": self.n_docs,\n            "S": self.S,\n            "r": self.r,\n            "n_theta": self.n_theta,\n            "n_steps": self.n_steps,\n            "epsilon_ratio": self.epsilon_ratio,\n            "B_perm": self.B_perm,\n            "B_boot": self.B_boot,\n            "k_sigma": self.k_sigma,\n            "alpha": self.alpha,\n            "seed": self.seed,\n            "model_name": self.model_name,\n            "config": {\n                str(k): _json_safe(v) for k, v in sorted(self.config.items())\n            },\n        }\n        return d\n\n    def to_key(self) -> str:\n        """Stable 12-hex identifier (SHA-256 prefix) of the spec.\n\n        Path-insensitive (cache_root/output_root excluded); sensitive to\n        every result-affecting field including config and the edit\n        identity (name/norm/token_set).\n        """\n        return _full_hash(self)[:12]\n\n\n@dataclass(frozen=True)\nclass OutputArtifacts:\n    """Handle to a saved (or reused) run: results, meta, path, key."""\n\n    spec: ExperimentSpec\n    results: dict\n    meta: dict\n    path: str\n    spec_key: str\n    reused: bool\n\n\ndef validate_spec(spec: ExperimentSpec) -> None:\n    """TZ §2.1 field checks.\n\n    Note: ``pipeline`` does NOT know ``n_layers`` (the model is loaded\n    only inside ``exp_fn``); it checks ``l >= 0`` and ``len(layers) >= 1``.\n    Full ``l < n_layers`` validation is the exp_fn\'s responsibility.\n\n    Raises:\n        ValueError: any field outside its contract.\n    """\n    if spec.experiment not in _VALID_EXPERIMENTS:\n        raise ValueError(\n            f"experiment must be one of {_VALID_EXPERIMENTS}, "\n            f"got {spec.experiment!r}"\n        )\n    if tuple(spec.edit.token_set) != tuple(spec.token_set):\n        raise ValueError(\n            f"edit.token_set {tuple(spec.edit.token_set)} != "\n            f"spec.token_set {tuple(spec.token_set)}"\n        )\n    if spec.edit.delta_W is not None and \\\n            spec.edit.delta_W.shape[0] != len(spec.token_set):\n        raise ValueError(\n            f"edit.delta_W rows {spec.edit.delta_W.shape[0]} != "\n            f"len(token_set) {len(spec.token_set)}"\n        )\n    if len(spec.token_set) < 2:\n        raise ValueError(f"len(token_set) must be >= 2, got {len(spec.token_set)}")\n    if len(spec.layers) < 1:\n        raise ValueError("layers must be non-empty")\n    for l in spec.layers:\n        if not (isinstance(l, int) and l >= 0):\n            raise ValueError(f"layer must be a non-negative int, got {l!r}")\n    if spec.n_docs < 1:\n        raise ValueError(f"n_docs must be >= 1, got {spec.n_docs}")\n    if spec.S < 2:\n        raise ValueError(f"S must be >= 2, got {spec.S}")\n    if spec.r < 1:\n        raise ValueError(f"r must be >= 1, got {spec.r}")\n    if spec.r > len(spec.token_set):\n        raise ValueError(f"r = {spec.r} > len(token_set) = {len(spec.token_set)}")\n    if spec.n_theta < 4:\n        raise ValueError(f"n_theta must be >= 4, got {spec.n_theta}")\n    if spec.n_steps < 2:\n        raise ValueError(f"n_steps must be >= 2, got {spec.n_steps}")\n    if spec.epsilon_ratio <= 0:\n        raise ValueError(f"epsilon_ratio must be > 0, got {spec.epsilon_ratio}")\n    if spec.B_perm < 100:\n        raise ValueError(f"B_perm must be >= 100, got {spec.B_perm}")\n    if spec.B_boot < 100:\n        raise ValueError(f"B_boot must be >= 100, got {spec.B_boot}")\n    if spec.k_sigma <= 0:\n        raise ValueError(f"k_sigma must be > 0, got {spec.k_sigma}")\n    if not (0.0 < spec.alpha < 1.0):\n        raise ValueError(f"alpha must be in (0, 1), got {spec.alpha}")\n    if spec.seed < 0:\n        raise ValueError(f"seed must be >= 0, got {spec.seed}")\n    if not spec.cache_root:\n        raise ValueError("cache_root must be non-empty")\n    if not spec.output_root:\n        raise ValueError("output_root must be non-empty")\n    if not spec.model_name:\n        raise ValueError("model_name must be non-empty")\n\n\ndef default_output_subdir(spec: ExperimentSpec) -> str:\n    """``{edit.name}/L{min:02d}-{max:02d}_r{r}_seed{seed}`` (TZ §4.1)."""\n    return (\n        f"{spec.edit.name}/"\n        f"L{min(spec.layers):02d}-{max(spec.layers):02d}"\n        f"_r{spec.r}_seed{spec.seed}"\n    )\n\n\ndef _expected_output_path(spec: ExperimentSpec, output_subdir: str | None) -> str:\n    return os.path.join(\n        spec.output_root, spec.experiment,\n        output_subdir or default_output_subdir(spec),\n    )\n\n\ndef _ensure_output_dir(path: str) -> None:\n    try:\n        os.makedirs(path, exist_ok=True)\n    except PermissionError as e:\n        raise RuntimeError(\n            f"cannot create output dir {path!r} (read-only / Kaggle FUSE?): {e}"\n        ) from e\n\n\ndef _write_json_atomic(path: str, obj: Any) -> None:\n    d = os.path.dirname(path) or "."\n    os.makedirs(d, exist_ok=True)\n    tmp = f"{path}.tmp.{os.getpid()}.{uuid.uuid4().hex[:8]}"\n    with open(tmp, "w", encoding="utf-8") as f:\n        json.dump(_json_safe(obj), f, ensure_ascii=False, indent=2)\n    os.replace(tmp, path)\n\n\ndef _read_json(path: str) -> Any:\n    try:\n        with open(path, encoding="utf-8") as f:\n            return json.load(f)\n    except (json.JSONDecodeError, OSError) as e:\n        raise RuntimeError(f"cannot read/parse JSON at {path!r}: {e}") from e\n\n\ndef _now_iso() -> str:\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef _detect_env() -> dict:\n    env = {\n        "python": sys.version.split()[0],\n        "torch": torch.__version__,\n        "numpy": np.__version__,\n        "transformers": "unknown",\n        "device": "cpu",\n        "git_commit": "unknown",\n    }\n    try:\n        import transformers\n        env["transformers"] = transformers.__version__\n    except ImportError:\n        pass\n    if torch.cuda.is_available():\n        env["device"] = f"cuda:{torch.cuda.current_device()}"\n    elif getattr(torch.backends, "mps", None) is not None and \\\n            torch.backends.mps.is_available():\n        env["device"] = "mps"\n    try:\n        out = subprocess.run(\n            ["git", "rev-parse", "HEAD"], capture_output=True, text=True,\n            timeout=5, check=True,\n        )\n        env["git_commit"] = out.stdout.strip()\n    except Exception:\n        pass\n    return env\n\n\ndef _reconstruct_edit(meta_spec: dict) -> EditSpec:\n    """EditSpec from meta metadata; delta_W is ALWAYS None here (it is\n    not stored — size). Re-applying the edit requires the caller\'s own\n    EditSpec (TZ §7)."""\n    e = meta_spec.get("edit", {})\n    token_set = tuple(meta_spec.get("token_set", ()))\n    return EditSpec(\n        name=str(e.get("name", "unknown")),\n        delta_W=None,\n        token_set=token_set,\n        target_layer="lm_head",\n        norm_frobenius=float(e.get("norm_frobenius", 0.0)),\n        projection_onto_top_r=float(e.get("projection_onto_top_r", 0.0)),\n        seed=None,\n        meta={"reconstructed": True,\n              "note": "delta_W is not stored in meta (size); "\n                      "re-apply with the caller\'s own EditSpec"},\n    )\n\n\ndef _load_or_reuse(meta_path: str, spec_key: str) -> OutputArtifacts | None:\n    """Reuse a complete run whose spec_key matches; None otherwise."""\n    if not os.path.exists(meta_path):\n        return None\n    try:\n        meta = _read_json(meta_path)\n    except RuntimeError:\n        return None  # corrupt meta: re-run\n    if meta.get("spec_key") != spec_key or not meta.get("complete"):\n        return None\n    out_dir = os.path.dirname(meta_path)\n    results_path = os.path.join(out_dir, "results.json")\n    results = _read_json(results_path) if os.path.exists(results_path) else {}\n    if not os.path.exists(results_path):\n        meta["results_available"] = False\n    spec = _spec_from_meta(meta)\n    return OutputArtifacts(\n        spec=spec, results=results, meta=meta, path=meta_path,\n        spec_key=spec_key, reused=True,\n    )\n\n\ndef _spec_from_meta(meta: dict) -> ExperimentSpec:\n    ms = meta.get("spec", {})\n    edit = _reconstruct_edit(ms)\n    return ExperimentSpec(\n        experiment=ms.get("experiment", ""),\n        edit=edit,\n        layers=tuple(ms.get("layers", ())),\n        token_set=tuple(ms.get("token_set", ())),\n        n_docs=int(ms.get("n_docs", 0)),\n        S=int(ms.get("S", 0)),\n        r=int(ms.get("r", 16)),\n        n_theta=int(ms.get("n_theta", 16)),\n        n_steps=int(ms.get("n_steps", 20)),\n        epsilon_ratio=float(ms.get("epsilon_ratio", 0.1)),\n        B_perm=int(ms.get("B_perm", 1000)),\n        B_boot=int(ms.get("B_boot", 1000)),\n        k_sigma=float(ms.get("k_sigma", 5.0)),\n        alpha=float(ms.get("alpha", 0.05)),\n        seed=int(ms.get("seed", 0)),\n        model_name=str(ms.get("model_name", "")),\n        config=MappingProxyType({k: v for k, v in ms.get("config", {}).items()}),\n    )\n\n\ndef run_spec(\n    spec: ExperimentSpec,\n    exp_fn: Callable[[ExperimentSpec], dict],\n    output_subdir: str | None = None,\n    skip_if_exists: bool = True,\n    dry_run: bool = False,\n    verbose: bool = True,\n) -> OutputArtifacts:\n    """Validate, seed, run exp_fn, save results.json + meta.json (TZ §3).\n\n    meta is written BEFORE exp_fn (complete=False) so a crash/Ctrl-C\n    leaves on-disk diagnostics; written again after success. Atomic\n    (tmp + os.replace). exp_fn must return a JSON-convertible dict and\n    must NOT write to the output dir itself.\n\n    Seeds: deterministic_mode(seed, strict=config.get("strict", False)) —\n    strict is NEVER the default (GDN compatibility).\n\n    Args:\n        spec: the frozen spec.\n        exp_fn: Callable[[ExperimentSpec], dict] — all experiment logic.\n        output_subdir: override default_output_subdir(spec).\n        skip_if_exists: reuse a complete meta with matching spec_key.\n        dry_run: validate + print plan, do not call exp_fn.\n        verbose: progress prints.\n\n    Returns:\n        OutputArtifacts (reused=True when loaded from disk).\n\n    Raises:\n        ValueError: spec invalid.\n        TypeError: exp_fn returned a non-dict (meta marked incomplete).\n        Any exception from exp_fn: re-raised AFTER meta diagnostics.\n    """\n    validate_spec(spec)\n    spec_key = spec.to_key()\n    output_path = _expected_output_path(spec, output_subdir)\n    meta_path = os.path.join(output_path, "meta.json")\n\n    if dry_run:\n        if verbose:\n            print(f"[run_spec] DRY RUN experiment={spec.experiment} "\n                  f"key={spec_key} path={output_path}")\n            print(json.dumps(spec.to_meta_dict(), indent=2, ensure_ascii=False,\n                             default=str))\n        return OutputArtifacts(spec=spec, results={}, meta={}, path="",\n                               spec_key=spec_key, reused=False)\n\n    _ensure_output_dir(output_path)\n    if skip_if_exists:\n        existing = _load_or_reuse(meta_path, spec_key)\n        if existing is not None and verbose:\n            print(f"[run_spec] reused experiment={spec.experiment} "\n                  f"key={spec_key} path={meta_path}")\n        if existing is not None:\n            return existing\n\n    meta = {\n        "spec": spec.to_meta_dict(),\n        "spec_key": spec_key,\n        "spec_key_full": _full_hash(spec),\n        "experiment": spec.experiment,\n        "edit_name": spec.edit.name,\n        "env": _detect_env() | {"timestamp_start": _now_iso()},\n        "complete": False,\n        "error": None,\n        "reused": False,\n    }\n    # Write BEFORE exp_fn so a crash/Ctrl-C leaves on-disk diagnostics\n    # (docstring contract; SIGKILL/OOM still leave nothing).\n    _write_json_atomic(meta_path, meta)\n    if verbose:\n        print(f"[run_spec] experiment={spec.experiment} key={spec_key} "\n              f"path={output_path}")\n\n    t0 = time.perf_counter()\n    try:\n        with deterministic_mode(seed=spec.seed,\n                                strict=bool(spec.config.get("strict", False))):\n            results = exp_fn(spec)\n    except BaseException as e:  # KeyboardInterrupt included\n        meta["error"] = f"{type(e).__name__}: {e}"\n        meta["env"]["timestamp_end"] = _now_iso()\n        meta["env"]["duration_sec"] = time.perf_counter() - t0\n        try:\n            _write_json_atomic(meta_path, meta)\n        except OSError:\n            pass  # diagnostics best-effort; re-raise the original\n        raise\n\n    if not isinstance(results, dict):\n        meta["error"] = (\n            f"expected dict from exp_fn, got {type(results).__name__}"\n        )\n        meta["env"]["timestamp_end"] = _now_iso()\n        meta["env"]["duration_sec"] = time.perf_counter() - t0\n        _write_json_atomic(meta_path, meta)\n        raise TypeError(meta["error"])\n\n    meta["complete"] = True\n    meta["env"]["timestamp_end"] = _now_iso()\n    meta["env"]["duration_sec"] = time.perf_counter() - t0\n    _write_json_atomic(os.path.join(output_path, "results.json"), results)\n    _write_json_atomic(meta_path, meta)\n    if verbose:\n        print(f"[run_spec] done in {meta[\'env\'][\'duration_sec\']:.1f}s")\n\n    return OutputArtifacts(spec=spec, results=results, meta=meta,\n                           path=meta_path, spec_key=spec_key, reused=False)\n\n\ndef run_many(\n    specs: Sequence[ExperimentSpec],\n    exp_fn: Callable[[ExperimentSpec], dict],\n    output_subdir_fn: Callable[[ExperimentSpec], str] | None = None,\n    skip_if_exists: bool = True,\n    verbose: bool = True,\n) -> list[OutputArtifacts]:\n    """Sequential batch of run_spec (TZ §3.5).\n\n    Convenience batch wrapper — it adds NO new logic: each spec goes\n    through the identical run_spec path (validate, cache-key, meta,\n    seeds). The only added behavior is that an exception in one spec\n    ABORTS the remaining batch (not swallowed); wrap in try/except for\n    partial batches.\n    """\n    out = []\n    for spec in specs:\n        out.append(run_spec(\n            spec, exp_fn,\n            output_subdir=output_subdir_fn(spec) if output_subdir_fn else None,\n            skip_if_exists=skip_if_exists,\n            verbose=verbose,\n        ))\n    return out\n\n\ndef load_artifacts(path: str) -> OutputArtifacts:\n    """Inverse of run_spec\'s save (TZ §3.4).\n\n    ``path`` — meta.json or the directory containing it. The spec is\n    reconstructed from meta (delta_W is None — it is not stored). A\n    meta with complete=False raises RuntimeError (incomplete run).\n    results.json missing → results={} and meta["results_available"]=False.\n\n    Raises:\n        RuntimeError: broken JSON, missing meta, incomplete run.\n    """\n    meta_path = (os.path.join(path, "meta.json")\n                 if os.path.isdir(path) else path)\n    if not os.path.exists(meta_path):\n        raise RuntimeError(f"no meta.json at {meta_path!r}")\n    meta = _read_json(meta_path)\n    if not meta.get("complete"):\n        raise RuntimeError(\n            f"meta at {meta_path!r} is incomplete "\n            f"(error: {meta.get(\'error\')!r}); re-run the experiment"\n        )\n    results_path = os.path.join(os.path.dirname(meta_path), "results.json")\n    results = _read_json(results_path) if os.path.exists(results_path) else {}\n    if not os.path.exists(results_path):\n        meta["results_available"] = False\n    spec = _spec_from_meta(meta)\n    validate_spec(spec)  # hand-edited/stale meta → catch, don\'t return bad spec\n    return OutputArtifacts(\n        spec=spec, results=results, meta=meta, path=meta_path,\n        spec_key=str(meta.get("spec_key", spec.to_key())), reused=True,\n    )\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/profiles.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Recipes on top of the kernel — NOT core (ТЗ E40_math_core §II.11).\n\nPromotion rule: a recipe enters the core only when a second consumer appears\n(after E-41). Do not call ``R_can`` an order parameter before it passes\ncriteria (a′)∧(a)∧(b)∧(c) (ТЗ §V).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .operator_checks import apply_P\n\n__all__ = ["R_can", "R_raw", "G_l", "r_cross"]\n\n\ndef _lam_share(G: np.ndarray) -> float:\n    """λ_max(G)/tr(G) for a symmetric PSD ``G``."""\n    G = np.asarray(G, dtype=np.float64)\n    if G.ndim != 2 or G.shape[0] != G.shape[1]:\n        raise ValueError(f"G must be square, got {G.shape}")\n    tr = float(np.trace(G))\n    if tr <= 0:\n        raise ValueError("λ-share: tr G ≤ 0")\n    return float(np.linalg.eigvalsh(G)[-1] / tr)\n\n\ndef R_raw(J: np.ndarray) -> float:\n    """``λ_max(JJᵀ)/tr(JJᵀ)`` — top-mode share of the RAW Jacobian."""\n    J = np.asarray(J, dtype=np.float64)\n    if J.ndim != 2:\n        raise ValueError(f"J must be [K, D], got {J.shape}")\n    return _lam_share(J @ J.T)\n\n\ndef R_can(J: np.ndarray, p: np.ndarray) -> float:\n    """``λ_max(E_K)/tr(E_K)`` with ``E_K = J^(p)(J^(p))ᵀ`` on the canonical\n    slice ``J^(p) = P(p) J`` (ТЗ E40_math_core §I «Определение», paper §2.11.1,\n    e40_todo §8.1). The slice must be named in the ChannelTag — R-values from\n    different slices are not comparable (slice-locality, ТЗ §I.2).\n\n    NOT ``(I − P)J``: that complement is the E-37 ``K`` object (frac_K) and is\n    ≈ J in norm (P kills 98–99 % of it), which made an ``R_can`` built on it\n    indistinguishable from ``R_raw`` (audit_core_math_v1 M1, fixed 2026-11-03)."""\n    J = np.asarray(J, dtype=np.float64)\n    Kmat = apply_P(J, p)\n    return _lam_share(Kmat @ Kmat.T)\n\n\ndef G_l(delta_raw: float, delta_fixP: float) -> float:\n    """``G(l) = 1 − Δ_fixP/Δ_raw`` — gauge-reduction of a transport distance.\n    Takes the two ALREADY-computed distances (aggregation of Δ is the caller\'s\n    contract; state it in the tag)."""\n    if delta_raw <= 0:\n        raise ValueError("G_l: Δ_raw must be > 0")\n    return 1.0 - float(delta_fixP) / float(delta_raw)\n\n\ndef r_cross(curve1: np.ndarray, curve2: np.ndarray) -> float | None:\n    """First crossing index of two curves on a common grid (linear\n    interpolation); ``None`` if they never cross."""\n    c1 = np.asarray(curve1, dtype=np.float64)\n    c2 = np.asarray(curve2, dtype=np.float64)\n    if c1.shape != c2.shape or c1.ndim != 1:\n        raise ValueError("curves must be equal-length 1-D")\n    d = c1 - c2\n    sign = np.sign(d)\n    for i in range(1, d.size):\n        if sign[i] == 0:\n            return float(i)\n        if sign[i] != sign[i - 1] and sign[i - 1] != 0:\n            t = d[i - 1] / (d[i - 1] - d[i])\n            return float(i - 1 + t)\n    return None\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/spectral.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Spectral metrics over already-computed Jacobians / Gramians.\n\nPure functions: no model, no forward passes, no state. All metrics take a\nspectrum tensor (``[K]``) or a matrix and return scalars/tensors, so the same\n``eigengap`` works for ``Δ_U(t, t\')``, ESG Gramians, and arbitrary\nsymmetric matrices (paper §6, §8.5, §8.9).\n\nRelationship to ``esg.py``:\n- ``esg._spectral_decomposition(G, n_components)`` (eigh, K×K) returns\n  top-n eigenvalues + eigenvectors (ROWS = eigenvectors, esg convention).\n  Not duplicated here.\n- ``spectral.spectrum_from_gramian(G)`` returns the FULL eigenvalue list\n  only (no vectors) — ``eigvalsh``, ~2× faster than ``eigh``. Use it when\n  only the spectrum is needed (``entropy_effective_rank`` etc.).\n- ``spectral.top_r_svd(J, r)`` (SVD, K×D) returns ``(U_r, S_r, V_r)`` for\n  holonomy / top-r projections. SVD on K×D is a different operation from\n  eigh on the K×K Gramian.\n\nSign convention: singular/eigenvector signs are LAPACK-version-dependent.\nConsumers (holonomy, ε) must use sign-invariant quantities (``|cos|``,\nProcrustes residuals).\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport warnings\n\nimport torch\n\n__all__ = [\n    "top_r_svd",\n    "spectrum_from_gramian",\n    "eigengap",\n    "eigengap_min",\n    "entropy_effective_rank",\n    "stable_rank",\n    "participation_ratio",\n    "snr_top1_vs_rest",\n    "snr_e14",\n    "snr_e14_parts",\n    "gauge_share",\n    "gauge_share_top1",\n]\n\n_LOW_PREC = (torch.float16, torch.bfloat16)\n\n\ndef _check_1d_finite(spectrum: torch.Tensor, name: str = "spectrum") -> None:\n    """1-D tensor, no NaN/Inf (index of the first offender in the message)."""\n    if spectrum.dim() != 1:\n        raise ValueError(f"{name}: expected 1-D tensor, got {spectrum.dim()}-D")\n    if spectrum.numel() == 0:\n        raise ValueError(f"{name}: empty tensor")\n    bad = torch.nonzero(~torch.isfinite(spectrum), as_tuple=False).flatten()\n    if bad.numel() > 0:\n        raise ValueError(f"{name}: non-finite value at index {int(bad[0])}")\n\n\n# ---------------------------------------------------------------------------\n# SVD primitives\n# ---------------------------------------------------------------------------\n\n\ndef top_r_svd(\n    J: torch.Tensor, r: int\n) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n    """Truncated SVD: ``J ≈ U_r @ diag(S_r) @ V_r``.\n\n    Args:\n        J: [K, D] Jacobian (float32/float64; fp16/bf16 is cast to float32\n           for SVD numerical stability).\n        r: 1 ≤ r ≤ min(K, D).\n\n    Returns:\n        (U_r [K, r], S_r [r] descending, V_r [r, D]) with the same dtype\n        (post-cast) and device as the input.\n\n    Sign caveat: signs of U_r / V_r depend on the LAPACK version — use\n    |cos| / Procrustes residuals, never raw sign comparisons.\n    """\n    if J.dim() != 2:\n        raise ValueError(f"top_r_svd: expected 2-D J, got {J.dim()}-D")\n    if not torch.isfinite(J).all():\n        bad = int((~torch.isfinite(J)).nonzero(as_tuple=False).flatten()[0])\n        raise ValueError(\n            f"top_r_svd: J contains NaN/Inf (first at flat index {bad}); "\n            "refuse to propagate a corrupted cache silently"\n        )\n    K, D = J.shape\n    if not (1 <= r <= min(K, D)):\n        raise ValueError(\n            f"top_r_svd: r={r} out of range [1, min(K, D)] = [1, {min(K, D)}]"\n        )\n    if J.dtype in _LOW_PREC:\n        J = J.float()\n    U, S, V = torch.linalg.svd(J, full_matrices=False)\n    return U[:, :r].contiguous(), S[:r].contiguous(), V[:r, :].contiguous()\n\n\ndef spectrum_from_gramian(G: torch.Tensor, ascending: bool = False) -> torch.Tensor:\n    """All eigenvalues of a symmetric Gramian, sorted.\n\n    ``eigvalsh`` (no eigenvectors) computed in float64 for stability on\n    ill-conditioned ``Δ_U``; the result is cast back to the input dtype.\n\n    Args:\n        G: [K, K] symmetric (PSD NOT required: ``Δ_U(t, t\')`` for\n           ``t ≠ t\'`` is symmetric but can have negative eigenvalues).\n        ascending: False (default) = descending λ₁ ≥ λ₂ ≥ … (paper §6\n           convention); True = ascending (numerical checks).\n\n    Raises:\n        ValueError: non-2-D, non-square, or asymmetric (``atol=1e-6``)\n           input — no silent symmetrization, that would hide Δ_U bugs.\n    """\n    if G.dim() != 2 or G.shape[0] != G.shape[1]:\n        raise ValueError(f"spectrum_from_gramian: expected square 2-D, got {tuple(G.shape)}")\n    if not torch.allclose(G, G.T, atol=1e-6):\n        raise ValueError("spectrum_from_gramian: G is not symmetric (no silent symmetrization)")\n    lam = torch.linalg.eigvalsh(G.double())\n    lam = torch.sort(lam, descending=not ascending).values\n    return lam.to(G.dtype)\n\n\n# ---------------------------------------------------------------------------\n# Scalar spectral metrics (take a spectrum, not a matrix)\n# ---------------------------------------------------------------------------\n\n\ndef eigengap(spectrum: torch.Tensor) -> float:\n    """λ₁ − λ₂ (two largest eigenvalues; paper §6).\n\n    Input order does not matter (sorted internally). Negative eigenvalues\n    are NOT dropped: for a proper eigengap ``Δ_U`` should be PSD, but the\n    function does not enforce it.\n    """\n    _check_1d_finite(spectrum)\n    if spectrum.numel() < 2:\n        raise ValueError(f"eigengap: need K >= 2, got {spectrum.numel()}")\n    s = torch.sort(spectrum, descending=True).values\n    return float(s[0] - s[1])\n\n\ndef eigengap_min(\n    spectrum: torch.Tensor,      # [k] eigenvalues (NOT singular values), descending not required\n    eps_frac: float = 0.01,      # threshold fraction of λ_1 = max(spectrum)\n) -> tuple[float, bool]:\n    """Pre-flight [M-4]: min consecutive eigenvalue gap inside top-k.\n\n    Returns (gap_min, well_separated), well_separated = gap_min >= eps_frac\n    * spectrum[0] (after sorting descending; so spectrum[0] is λ_1 = max).\n\n    SEMANTICS (docs/E38_fixes.md §1.1): ``well_separated`` is a\n    mathematically neutral predicate — the spectrum is well separated\n    (no near-degenerate pair). It is NOT the [M-4] verdict: per [M-4]\n    ``min_t gap_min < ε`` (i.e. NOT well_separated) is when the non-Abelian\n    formalism is applicable. Callers decide the mapping; E-38c reports\n    ``fraction_well_separated`` and drives verdicts from\n    ``min_t_gap_min_over_threshold`` directly.\n\n    NOT min(eigengap(...)): eigengap returns only the FIRST gap λ₁−λ₂;\n    this returns the MINIMUM over ALL consecutive gaps λ_i − λ_{i+1}.\n    Same input convention as eigengap/stable_rank/participation_ratio\n    (eigenvalues, i.e. σ²), so it can follow spectrum_from_gramian\n    directly without a manual sqrt(). Do NOT pass singular values here.\n\n    The ``>=`` (not ``>``) is deliberate: an exact equality\n    ``gap_min == threshold`` is classified as well_separated (boundary =\n    "acceptable, not degenerate", per [M-4]).\n\n    Args:\n        spectrum: [k] eigenvalues (≥ 0), any order, k ≥ 2.\n        eps_frac: threshold as a fraction of λ_1; must lie in (0, 1).\n\n    Returns:\n        (gap_min, well_separated) as Python (float, bool).\n\n    Warnings:\n        RuntimeWarning: spectrum all zero → returns (0.0, False).\n\n    Errors:\n        ValueError: k < 2, NaN/Inf (index reported), negative eigenvalue\n            (index reported), eps_frac outside (0, 1).\n    """\n    _check_1d_finite(spectrum, "spectrum")\n    if spectrum.numel() < 2:\n        raise ValueError(f"eigengap_min: need k >= 2, got {spectrum.numel()}")\n    neg = torch.nonzero(spectrum < 0, as_tuple=False).flatten()\n    if neg.numel() > 0:\n        raise ValueError(\n            f"eigengap_min: negative eigenvalue at index {int(neg[0])} "\n            f"(spectrum must be PSD eigenvalues)"\n        )\n    if not (0.0 < eps_frac < 1.0):\n        raise ValueError(f"eigengap_min: eps_frac must be in (0, 1), got {eps_frac}")\n    s = torch.sort(spectrum, descending=True).values\n    if s[0] == 0:\n        warnings.warn("eigengap_min: all eigenvalues are zero", RuntimeWarning)\n        return 0.0, False\n    # Gaps computed in float64: in float32 (eps ≈ 1.2e-7) a gap near\n    # eps_frac * λ_1 with eps_frac ≤ 0.01 would sit at the precision floor\n    # (absolute error ~1e-7·λ_1). Same policy as spectrum_from_gramian,\n    # which already casts to double internally (API symmetry).\n    gaps = (s[:-1] - s[1:]).double()\n    gap_min = float(gaps.min())\n    threshold = float(eps_frac * s[0])\n    return gap_min, gap_min >= threshold\n\n\ndef entropy_effective_rank(spectrum: torch.Tensor) -> float:\n    """r_eff = exp(−Σ p_k log p_k), p_k = λ_k / Σλ (paper §6, §15; E-14, E-27.6).\n\n    Non-positive eigenvalues are DROPPED (p_k = 0); if all λ ≤ 0 the\n    function returns 0.0 with a RuntimeWarning. For ``Δ_U(t, t\')`` with\n    t ≠ t\' (mixed spectrum) r_eff is computed on the positive part only —\n    semantically correct (negative "energy" has no meaning) but the value\n    can be smaller than naively expected.\n    """\n    _check_1d_finite(spectrum)\n    pos = spectrum[spectrum > 0]\n    if pos.numel() == 0:\n        warnings.warn(\n            "entropy_effective_rank: all eigenvalues <= 0, returning 0.0",\n            RuntimeWarning,\n        )\n        return 0.0\n    p = pos / pos.sum()\n    entropy = -(p * torch.log(p)).sum()\n    return float(torch.exp(entropy))\n\n\ndef stable_rank(spectrum: torch.Tensor) -> float:\n    """Σλ / λ₁ (paper §6).\n\n    "How far the spectrum is from being concentrated in one mode."\n    NOT a synonym for participation_ratio: for λ = (4,1,1,1,1),\n    stable_rank = 8/4 = 2.0 while PR = 64/20 = 3.2 (TZ said 64/18 ≈ 3.555 — typo, Σλ² = 20).\n\n    Negative eigenvalues are not dropped (ratio definition). λ₁ ≤ 0\n    (all zero, or a mixed-sign spectrum where the largest is negative)\n    → 0.0 with a RuntimeWarning.\n    """\n    _check_1d_finite(spectrum)\n    s = torch.sort(spectrum, descending=True).values\n    lam1 = s[0]\n    if lam1 <= 0:\n        warnings.warn(\n            f"stable_rank: λ₁ = {float(lam1):.3e} <= 0, returning 0.0", RuntimeWarning\n        )\n        return 0.0\n    return float(s.sum() / lam1)\n\n\ndef participation_ratio(spectrum: torch.Tensor) -> float:\n    """PR = (Σλ)² / Σλ² ∈ [1, K] (paper §6, §17; E-14).\n\n    Defined on all eigenvalues (negatives not dropped). Σλ² = 0 → 0.0.\n    Originally ``_participation_ratio`` in experiments/exp14_snr_pr.py.\n    """\n    _check_1d_finite(spectrum)\n    s2 = spectrum.pow(2).sum()\n    if s2 == 0:\n        return 0.0\n    return float(spectrum.sum().pow(2) / s2)\n\n\ndef snr_top1_vs_rest(spectrum: torch.Tensor) -> float:\n    """λ₁ / mean(λ₂..λ_K), λ₁ = largest eigenvalue.\n\n    DEFINITION UNDER REVIEW (TZ docs/new_code_1.md §1.3): the paper §6\n    mentions a "leave-one-out SNR" without an exact formula, and the\n    original E-14 SNR is sample-based, not spectral (see :func:`snr_e14`).\n    This spectral proxy is provided for E-ε; if the original E-14 formula\n    turns out to be the one used downstream, prefer ``snr_e14`` and record\n    which one is used in ``meta.json``.\n\n    mean(λ_{-1}) = 0 → ``inf``. K = 1 → ValueError. Negatives are not\n    dropped. No positive λ → ValueError.\n    """\n    _check_1d_finite(spectrum)\n    if spectrum.numel() < 2:\n        raise ValueError(f"snr_top1_vs_rest: need K >= 2, got {spectrum.numel()}")\n    s = torch.sort(spectrum, descending=True).values\n    lam1 = s[0]\n    if lam1 <= 0:\n        raise ValueError("snr_top1_vs_rest: no positive eigenvalue (λ₁ <= 0)")\n    rest = s[1:].mean()\n    if rest == 0:\n        return float("inf")\n    return float(lam1 / rest)\n\n\ndef snr_e14_parts(\n    jacobians: list[torch.Tensor],\n) -> tuple[float, float, float]:\n    """Original E-14 sample-based SNR with intermediates\n    (experiments/exp14_snr_pr.py::``_snr_sample`` — MOVED here, not copied).\n\n    Returns ``(snr, mean_sq, var)`` where\n    ``mean_sq = mean‖J‖²_F``, ``var = mean‖J‖²_F − ‖mean J‖²_F`` (per the\n    original E-14 definition), and ``snr = ‖mean J‖²_F / max(1e-12, var)``.\n    Measures cross-position CONSISTENCY of J, not a spectral property — do\n    not confuse with :func:`snr_top1_vs_rest`.\n\n    VALIDATED against results/exp14_snr_pr.json (committed): for all 24\n    layers the formula ``(mean_sq − var_sq)/var_sq`` reproduces the stored\n    ``SNR`` bit-for-bit (L0 = 0.1415, L21 = 3.6509, L23 = 199.3080 — the\n    paper\'s numbers). See tests/test_spectral.py::test_snr_e14_validated_against_exp14_json.\n    """\n    if len(jacobians) < 2:\n        raise ValueError(f"snr_e14_parts: need >= 2 Jacobians, got {len(jacobians)}")\n    J = torch.stack([j.float() for j in jacobians])\n    M = J.mean(dim=0)\n    signal = M.pow(2).sum().item()\n    mean_sq = J.pow(2).sum(dim=(1, 2)).mean().item()\n    var = mean_sq - signal\n    return float(signal / max(1e-12, var)), float(mean_sq), float(var)\n\n\ndef snr_e14(jacobians: list[torch.Tensor]) -> float:\n    """Original E-14 sample-based SNR, scalar form (``snr_e14_parts()[0]``)."""\n    return snr_e14_parts(jacobians)[0]\n\n\ndef gauge_share(\n    spectrum: torch.Tensor,\n    eigenvectors: torch.Tensor,\n    uniform: torch.Tensor | None = None,\n) -> float:\n    """Σ_k (u_kᵀ uniform)² λ_k / Σ_k λ_k (paper §6, "gauge_share ESG-version").\n\n    Fraction of ``JJ^T`` energy lying on the uniform direction in\n    token-space.\n\n    Args:\n        spectrum: [K] eigenvalues of the Gramian (full spectrum, matching\n            ``eigenvectors``).\n        eigenvectors: [K, K] with COLUMNS = u_k, the k-th column\n            corresponding to ``spectrum[k]``. NOTE: ``esg`` returns\n            eigenvectors with ROWS = eigenvectors — pass ``eig.T``.\n        uniform: [K] direction; L2-normalized internally. Default\n            ``1/√K · ones(K)``.\n\n    The spectrum/eigenvectors pair must cover ALL modes (not top-r) — the\n    ``eigenvectors.shape == (K, K)`` check (K = len(spectrum)) enforces this:\n    a top-r spectrum paired with a full K×K matrix fails the shape check.\n    For top-r use :func:`gauge_share_top1` (or pass the full trace\n    explicitly).\n\n    SAME functional as ``g_uni`` when both are computed on the same ``J``:\n    with ``G = JJᵀ`` this is ``1ᵀG1/(K·tr G) = ‖Jᵀ1‖²/(K‖J‖²_F)``, which is the\n    definition ``p_algebra.g_uni`` implements — measured equal to 7.7e-16\n    (E-39 supplement, O2≡O3). They differ only when applied to DIFFERENT ``J``\n    (raw / ``P(p)J`` / centered) — e.g. ``g_uni(J^gf) ≡ 0`` because\n    ``P(p)·1 = 0``, so a "gauge share" measured after ``P(p)`` is empty by\n    construction (paper §6 item 20). Former wording «Distinct from g_uni»\n    predated E-39 and contradicted it (audit4 Q2).\n\n    Σλ = 0 → 0.0 with a RuntimeWarning.\n    """\n    _check_1d_finite(spectrum)\n    K = spectrum.numel()\n    if eigenvectors.shape != (K, K):\n        raise ValueError(\n            f"gauge_share: eigenvectors {tuple(eigenvectors.shape)} "\n            f"!= (K, K) = ({K}, {K})"\n        )\n    if uniform is None:\n        uniform = torch.full((K,), 1.0 / math.sqrt(K), device=eigenvectors.device, dtype=eigenvectors.dtype)\n    uniform = uniform / uniform.norm()\n    coeffs = (eigenvectors.T @ uniform) ** 2  # [K]: (u_k · uniform)^2\n    total = spectrum.sum()\n    if total == 0:\n        warnings.warn("gauge_share: Σλ = 0, returning 0.0", RuntimeWarning)\n        return 0.0\n    return float((coeffs * spectrum).sum() / total)\n\n\ndef gauge_share_top1(\n    spectrum: torch.Tensor, eigenvectors: torch.Tensor\n) -> float:\n    """λ₁ · (u₁ᵀ uniform)² / Σλ — top-1 proxy (paper §6; E-10a ``gauge_share_0``).\n\n    The value reported in E-10a as ``gauge_share_0`` IS this function\n    (top mode only, uniform = 1/√K). ``spectrum[0]`` must be the largest\n    eigenvalue (descending convention) and ``eigenvectors[:, 0]`` the\n    corresponding eigenvector (columns convention; pass ``eig.T`` from esg).\n\n    Σλ = 0 → 0.0 with a RuntimeWarning.\n    """\n    _check_1d_finite(spectrum)\n    K = spectrum.numel()\n    if eigenvectors.shape != (K, K):\n        raise ValueError(\n            f"gauge_share_top1: eigenvectors {tuple(eigenvectors.shape)} "\n            f"!= (K, K) = ({K}, {K})"\n        )\n    uniform = torch.full((K,), 1.0 / math.sqrt(K), device=eigenvectors.device, dtype=eigenvectors.dtype)\n    c1 = (eigenvectors[:, 0] @ uniform) ** 2\n    total = spectrum.sum()\n    if total == 0:\n        warnings.warn("gauge_share_top1: Σλ = 0, returning 0.0", RuntimeWarning)\n        return 0.0\n    return float(spectrum[0] * c1 / total)\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/svd_bundle.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""SVD bundle: one place to cut a Jacobian into ``U Σ Vᵀ`` + metadata (ТЗ E40_math_core §II.4, M4).\n\nConsolidates three private copies of principal angles that existed in\n``experiments/`` (exp28 ``principal_angles_max``, exp31 ``principal_angle_max``,\nexp39a ``_principal_angles_cos``) and attaches the ChannelTag the ТЗ demands.\n``slice`` is renamed ``slice_name`` (``slice`` is a python builtin and shadows\nit inside the function — deliberate deviation, ТЗ §II.4).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\nfrom .transport import _check_frame\n\n__all__ = ["svd_bundle", "principal_angles", "effective_rank"]\n\n\ndef effective_rank(Sigma: np.ndarray, threshold: float = 1e-2) -> int:\n    """Number of singular values with ``σ_i ≥ threshold · σ_1`` (thresholded PR,\n    the ``r_eff^U(τ)`` convention of E-39a — NOT the entropic r_eff of the ESG\n    spectrum; always subscript in artifacts, E-40.1 audit A4)."""\n    Sigma = np.asarray(Sigma, dtype=np.float64)\n    if Sigma.ndim != 1 or Sigma.size == 0:\n        raise ValueError(f"Sigma must be 1-D, got {Sigma.shape}")\n    if not (0 < threshold <= 1):\n        raise ValueError(f"threshold must be in (0, 1], got {threshold}")\n    if Sigma[0] <= 0:\n        # A zero spectrum has NO directions above threshold. With σ₁ = 0 the\n        # comparison `σ >= threshold·σ₁` is `0 >= 0` and would return the full\n        # dimension — a rank-0 Jacobian reported as full rank (audit4 Q1).\n        return 0\n    return int((Sigma >= threshold * Sigma[0]).sum())\n\n\ndef principal_angles(V1: np.ndarray, V2: np.ndarray, r: int | None = None) -> np.ndarray:\n    """Cosines of principal angles between column spaces of ``V1``, ``V2``\n    (``[d, r]`` frames): ``σ(eigvals of V1ᵀV2)``, descending. Frames must have\n    orthonormal columns (``_check_frame``) — on non-orthonormal inputs the SVD\n    values are not angle cosines and clipping to [0,1] would pass the wrong\n    object off as a valid one (audit M10). Supersedes the per-experiment\n    copies (same math)."""\n    V1 = _check_frame(V1, "V1")\n    V2 = _check_frame(V2, "V2")\n    if V1.shape[0] != V2.shape[0]:\n        raise ValueError(f"frames must be [d, r] on a common d, got {V1.shape}, {V2.shape}")\n    if r is not None:\n        V1, V2 = V1[:, :r], V2[:, :r]\n    s = np.linalg.svd(V1.T @ V2, compute_uv=False)\n    return np.clip(s, 0.0, 1.0)\n\n\ndef svd_bundle(J: np.ndarray, rank: int, side: str = "right",\n               slice_name: str = "raw", tag=None) -> dict:\n    """Top-``rank`` SVD bundle of a single ``J [K, D]`` with bookkeeping.\n\n    Returns dict with ``U [K, r]``, ``Sigma [r]``, ``V [D, r]`` (columns are\n    singular vectors, ``J ≈ U diag(Σ) Vᵀ``), ``energy`` — captured Frobenius\n    energy fraction, ``rank_effective`` — thresholded effective rank of the\n    FULL spectrum, ``stable`` — False when the top-r block has a σ gap below\n    ``1e-2 σ₁`` anywhere (rotation ambiguity, E40_ans §5.2). CONVENTION WARNING\n    (audit M9): this gap is in σ units (the E-38c ``gap_min`` convention);\n    ``kinetic.rotation_unique`` uses the Davis–Kahan σ² form\n    (``min|σᵢ²−σⱼ²| > eps·σ₁²``) — on a spectrum like [1.0, 0.991] the two\n    disagree by construction. Quote the convention with the flag.\n    ``tag`` — the\n    ChannelTag passed in (ТЗ: a number without a passport is invalid).\n\n    ``side`` is metadata for the consumer (which axis of the transport the\n    bundle addresses); it does not change the SVD.\n    """\n    J = np.asarray(J, dtype=np.float64)\n    if J.ndim != 2:\n        raise ValueError(f"J must be [K, D], got {J.shape}")\n    if side not in ("left", "right", "mixed"):\n        raise ValueError(f"side must be left/right/mixed, got {side!r}")\n    K, D = J.shape\n    if not (0 < rank <= min(K, D)):\n        raise ValueError(f"rank must be in (0, min(K,D)={min(K, D)}], got {rank}")\n    U, S, Vt = np.linalg.svd(J, full_matrices=False)\n    Vr = Vt[:rank].T\n    gaps = S[:rank - 1] / S[0] - S[1:rank] / S[0] if rank > 1 else np.array([1.0])\n    return {\n        "U": U[:, :rank],\n        "Sigma": S[:rank],\n        "V": Vr,\n        "Sigma_full": S,\n        "energy": float((S[:rank] ** 2).sum() / (S ** 2).sum()),\n        "rank_effective": effective_rank(S),\n        "stable": bool(np.abs(gaps).min() > 1e-2),\n        "side": side,\n        "slice": slice_name,\n        "tag": tag,\n    }\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/tag.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""ChannelTag: mandatory passport for every E-40.x result (ТЗ E40_math_core §II.1, M1).\n\nEvery artifact number must say HOW it was produced before it says HOW MUCH.\nThree fields are ``identity-relevant``: two results may only be compared when\nthey agree on them.\n\n- ``p_mode`` — absolute values of ``tr P``, ``iso``, ``O1`` change when p\n  changes (renormalized P128 vs raw P128_RAW moved L6 O1/iso 0.511 → 0.322);\n  ``g_uni`` does not depend on p at all (``p_algebra.g_uni``). Unknown mode is\n  a stop condition, not a legal tag value (``other`` removed, патч v2 §1.6).\n- ``p_slice`` — ``p_t`` (per-position) vs ``p_0`` (fixP) are DIFFERENT operators\n  ``P(p)``; triple numbers without ``p_slice`` are not comparable. ``g_uni``\n  needs no ``p_slice`` (p-free in both branches: raw has no p; gf ≡ 0).\n- ``norm_type`` — norm vs energy differ by a square (E-33r\'s «98–99 %» was a\n  norm labelled as energy; paper §6 item 19).\n- ``slice`` — identities hold inside a slice only; ``g_uni(J^gf) ≡ 0`` is true\n  exclusively on the gf slice (ТЗ §I.2 slice-locality rule).\n- ``aggregation`` — median of a ratio ≠ ratio of sums; the identity\n  ``frac_K = 1 − 2c + s²`` closes only under ``ratio_of_means`` (E6, ТЗ §I.5).\n  Default ``ratio_of_means``.\n\nUsage::\n\n    from prism.tag import ChannelTag, make_tag, require_triple_tag\n    t = make_tag(protocol="natural", slice="raw", window="full", side="right",\n                 p_mode="truncated_U128", p_slice="p_t", layer=23, rank=16,\n                 norm_type="energy", source="cache", status="measured")\n    # aggregation defaults to "ratio_of_means"; any real/null ratio must set it.\n    require_triple_tag(t)   # ValueError if a triple number would be reported\n                            # without slice/aggregation/p_slice (патч v2 §1.6.5)\n\nAn artifact without a ``ChannelTag`` is invalid (ТЗ §II.1 rules).\n"""\n\nfrom __future__ import annotations\n\nfrom collections import namedtuple\n\n__all__ = ["ChannelTag", "make_tag", "IDENTITY_FIELDS", "TRIPLE_EXTRA_FIELDS",\n           "require_triple_tag", "TAG_DOMAINS"]\n\n#: Allowed values per field (fail-loud membership, trust-boundary style).\nTAG_DOMAINS: dict[str, tuple[str, ...]] = {\n    "protocol": ("natural", "deltaW"),\n    "slice": ("raw", "gf", "fixP"),\n    "window": ("top", "tail", "full"),\n    "side": ("left", "right", "mixed"),\n    "p_mode": ("truncated_U128", "full_softmax"),\n    "p_slice": ("p_t", "p_0"),\n    # The vocabulary the aggregator actually implements — the two lists must agree\n    # exactly. `energy_weighted` used to be listed here and was an ALIAS of\n    # `ratio_of_means` (energy-weighted mean of per-position ratios = ratio of sums),\n    # i.e. a fourth name for a third thing: removed, and `per_position`, which really\n    # is a different number and was implemented but untaggable, is in (audit4 Q3).\n    "aggregation": ("ratio_of_means", "mean", "median", "per_position"),\n    "norm_type": ("norm", "energy", "angle"),\n    "source": ("cache", "recomputed", "audit-only"),\n    "status": ("measured", "inferred", "not_testable"),\n}\n\n#: Fields two results must agree on before numbers are comparable (ТЗ §II.1).\nIDENTITY_FIELDS = ("p_mode", "norm_type", "slice", "aggregation")\n#: Extra fields mandatory when reporting a triple number (c, s, cos, frac_K);\n#: g_uni does NOT need them (p-free), патч v2 §1.6 rules 4–5.\nTRIPLE_EXTRA_FIELDS = ("p_slice",)\n\nChannelTag = namedtuple(\n    "ChannelTag",\n    ["protocol", "slice", "window", "side", "p_mode", "p_slice", "aggregation",\n     "layer", "rank", "norm_type", "source", "status"],\n)\n\n\ndef require_triple_tag(tag: "ChannelTag") -> "ChannelTag":\n    """Fail loud if ``tag`` may not back a triple/real-null number: slice,\n    aggregation and p_slice must all be set (патч v2 §1.6.5)."""\n    missing = [f for f in ("slice", "aggregation", "p_slice")\n               if getattr(tag, f) is None]\n    if missing:\n        raise ValueError(\n            f"ChannelTag: triple/real-null reporting requires {missing} "\n            f"(p_t vs p_0 are different operators P(p); median ≠ ratio; "\n            f"identities are slice-local)")\n    return tag\n\n\ndef make_tag(**kwargs) -> ChannelTag:\n    """Build a ChannelTag, validating every enumerated field and requiring the\n    identity-relevant ones. ``aggregation`` defaults to ``ratio_of_means``\n    (ТЗ §II.1: mandatory for any ratio, this is the identity-closing default).\n    ``layer``/``rank`` must be ints or None."""\n    kwargs.setdefault("aggregation", "ratio_of_means")\n    missing = [f for f in IDENTITY_FIELDS if f not in kwargs or kwargs[f] is None]\n    if missing:\n        raise ValueError(\n            f"ChannelTag: identity-relevant fields missing: {missing} "\n            f"(they are what makes two numbers comparable, ТЗ §II.1)"\n        )\n    for field, allowed in TAG_DOMAINS.items():\n        val = kwargs.get(field)\n        if val is None:\n            continue\n        if val not in allowed:\n            raise ValueError(\n                f"ChannelTag.{field}={val!r} not in {allowed}")\n    for field in ("layer", "rank"):\n        val = kwargs.get(field)\n        if val is not None and not isinstance(val, (int,)):\n            raise ValueError(f"ChannelTag.{field}={val!r} must be int or None")\n    return ChannelTag(**{f: kwargs.get(f) for f in ChannelTag._fields})\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/topology.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Zigzag persistent homology across point clouds + generic projections\n(TZ part 3, §7).\n\nOptional backends: giotto-tda and/or dionysus. This module imports\nWITHOUT them; only ``zigzag_persistence`` requires a backend (lazy\nImportError with pip instructions). The projections are pure numpy.\n\nZigzag vs ordinary PH: ordinary persistence requires a NESTED filtration\n(K_0 ⊂ K_1 ⊂ ...); zigzag allows BACKWARD steps (K_i → K_{i+1} → K_i),\nso features can be tracked as they are born and die when moving between\nlayers (the clouds are not nested).\n\npaper.md §8.10 — topology across layers.\n\nNOT covered here (TZ §7.8): compare_tail_vs_pca (experiments/exp_T1.py),\ntotal_persistence (caller), reducing differing d_l to a common d\n(caller: use project_to_pca_subspace before the zigzag call — the\nfunction does NOT project implicitly), loading tail_basis, plotting.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n__all__ = [\n    "zigzag_persistence",\n    "project_to_subspace",\n    "project_to_pca_subspace",\n]\n\n\ndef _import_giotto():\n    try:\n        from gtda.diagrams import ZigzagPersistence\n    except ImportError as e:\n        raise ImportError(\n            "giotto-tda is required for backend=\'giotto\'. "\n            "Install: pip install giotto-tda"\n        ) from e\n    return ZigzagPersistence\n\n\ndef _import_dionysus():\n    try:\n        import dionysus\n    except ImportError as e:\n        raise ImportError(\n            "dionysus is required for backend=\'dionysus\'. "\n            "Install: pip install dionysus"\n        ) from e\n    return dionysus\n\n\ndef _validate_point_clouds(point_clouds: list[np.ndarray]) -> int:\n    """Checks: len ≥ 2, each 2-D, same n_points, d ≥ 1, n_points ≥ 5,\n    finite. Returns n_points."""\n    if len(point_clouds) < 2:\n        raise ValueError(\n            f"need >= 2 point clouds for zigzag, got {len(point_clouds)}"\n        )\n    n_points = None\n    for i, x in enumerate(point_clouds):\n        x = np.asarray(x, dtype=float)\n        if x.ndim != 2:\n            raise ValueError(f"point cloud {i} is {x.ndim}-D, need 2-D [n, d]")\n        n, d = x.shape\n        if d < 1:\n            raise ValueError(f"point cloud {i} has d = 0")\n        if n_points is None:\n            n_points = n\n        elif n != n_points:\n            raise ValueError(\n                f"point cloud {i} has n = {n} != {n_points} "\n                f"(the same positions/docs across all clouds)"\n            )\n        if not np.all(np.isfinite(x)):\n            raise ValueError(f"point cloud {i} contains NaN/Inf")\n    if n_points is None or n_points < 5:\n        raise ValueError(\n            f"n_points = {n_points} < 5: PH on < 5 points is trivial"\n        )\n    return n_points\n\n\ndef zigzag_persistence(\n    point_clouds: list[np.ndarray],\n    maxdim: int = 1,\n    metric: str = "euclidean",\n    backend: str = "giotto",\n) -> dict:\n    """Zigzag PH over a sequence of point clouds (layers).\n\n    Maps between clouds are implicitly identity — the caller must reduce\n    all clouds to a COMMON dimension first (e.g. project_to_pca_subspace\n    to d=15, or tail modes with the same r_tail). Differing d_l is NOT\n    handled here.\n\n    Args:\n        point_clouds: list of [n_points, d], same n_points (≥ 5), len ≥ 2.\n        maxdim: 0 or 1 (H2 at N ~ 128 is noise).\n        metric: "euclidean" | "cosine" | "precomputed" (distance matrices).\n        backend: "giotto" (default) | "dionysus"; lazy import.\n\n    Returns:\n        dict with ``dgms`` (list of [n_features, 2] birth/death arrays,\n        np.inf for essential features), ``n_layers``, ``maxdim``,\n        ``backend``, ``metric``, ``n_points``. Feature order may differ\n        between backends/versions — do not rely on it.\n\n    Errors:\n        ValueError: maxdim > 1, bad metric, cloud validation failures.\n        ImportError: backend not installed (with pip instruction).\n    """\n    if maxdim not in (0, 1):\n        raise ValueError(f"maxdim must be 0 or 1, got {maxdim}")\n    if metric not in ("euclidean", "cosine", "precomputed"):\n        raise ValueError(f"metric must be euclidean/cosine/precomputed, got {metric!r}")\n    n_points = _validate_point_clouds(point_clouds)\n\n    if backend == "giotto":\n        ZigzagPersistence = _import_giotto()\n        # gtda: list of point clouds; for "precomputed" the caller passes\n        # distance matrices (accepted as-is).\n        zz = ZigzagPersistence(metric=metric if metric != "precomputed" else "precomputed")\n        # gtda ZigzagPersistence.fit_transform returns [n_features, 2]\n        dgm = zz.fit_transform(point_clouds)\n        dgms = [np.asarray(dgm, dtype=float)]\n        if maxdim == 1:\n            # gtda returns H0+H1 combined when maxdim allows; split by\n            # persistence convention: H0 features are those born at 0.\n            # HEURISTIC split (docs/changed_review.md, 2026-10): for zigzag\n            # the first column may be a time index rather than a\n            # filtration radius, so some H0 features can be born != 0 (and\n            # some H1 at 0). The gtda API gives no explicit H0/H1\n            # separation; treat dgms[0]/dgms[1] as "born-at-0" /\n            # "born-at-other", NOT as guaranteed H0/H1.\n            h0 = dgm[np.asarray(dgm, dtype=float)[:, 0] == 0.0]\n            h1 = dgm[np.asarray(dgm, dtype=float)[:, 0] != 0.0]\n            dgms = [np.asarray(h0, dtype=float), np.asarray(h1, dtype=float)]\n    elif backend == "dionysus":\n        dionysus = _import_dionysus()\n        # dionysus API differs across versions; use the zigzag helper when\n        # available. This path is exercised only if dionysus is installed.\n        if not hasattr(dionysus, "zigzag_persistence"):\n            raise ImportError(\n                "dionysus.zigzag_persistence not available in this version; "\n                "use backend=\'giotto\'"\n            )\n        res = dionysus.zigzag_persistence(point_clouds, maxdim=maxdim)\n        dgms = [np.asarray(r, dtype=float) for r in res]\n    else:\n        raise ValueError(f"unknown backend {backend!r} (giotto | dionysus)")\n\n    return {\n        "dgms": dgms,\n        "n_layers": len(point_clouds),\n        "maxdim": maxdim,\n        "backend": backend,\n        "metric": metric,\n        "n_points": n_points,\n    }\n\n\ndef project_to_subspace(h: np.ndarray, basis: np.ndarray) -> np.ndarray:\n    """Project h onto the subspace spanned by ``basis``.\n\n    ``basis`` is [r, D] (rows = basis vectors, default) or [D, r]\n    (columns). Disambiguation: if basis.shape[1] == h.shape[-1] → rows;\n    if basis.shape[0] == h.shape[-1] → cols; if BOTH match (square) →\n    ValueError. Orthonormality is NOT checked (caller\'s responsibility;\n    V_r from top_r_svd is already orthonormal).\n\n    Returns [N, r] (or [r] for 1-D h), dtype float32.\n\n    Errors:\n        ValueError: incompatible shapes, ambiguous square basis,\n            basis.dim() != 2.\n    """\n    h = np.asarray(h, dtype=np.float32)\n    basis = np.asarray(basis, dtype=np.float32)\n    if basis.ndim != 2:\n        raise ValueError(f"basis must be 2-D, got {basis.ndim}-D")\n    d = h.shape[-1]\n    rows_ok = basis.shape[1] == d\n    cols_ok = basis.shape[0] == d\n    if rows_ok and cols_ok:  # square basis: compatible both ways\n        raise ValueError(\n            f"basis shape {basis.shape} is square and compatible both ways; "\n            f"pass an explicit [r, D] or [D, r] basis"\n        )\n    if rows_ok:\n        return h @ basis.T\n    if cols_ok:\n        return h @ basis\n    raise ValueError(\n        f"incompatible shapes: h last dim = {d}, basis = {basis.shape} "\n        f"(need basis [r, {d}] or [{d}, r])"\n    )\n\n\ndef project_to_pca_subspace(\n    h_batch: np.ndarray,\n    n_components: int = 15,\n) -> tuple[np.ndarray, np.ndarray]:\n    """PCA to n_components (centered; numpy SVD, float32).\n\n    Returns (h_proj, components): h_proj [N, n_components] (centered —\n    add the mean back manually for uncentered projections), components\n    [n_components, D] (rows, orthonormal). Signs of components may differ\n    between numpy versions — do not rely on them.\n\n    Deliberately NOT built on ``spectral.top_r_svd`` (consolidate.md §2.2):\n    ``top_r_svd`` is torch-only, and this module is numpy-only by design\n    (projections must work on plain ndarrays without torch). Calling it\n    would force a hard torch import into a numpy module — a regression.\n    The math is identical (center + truncated SVD + float32 + the same\n    sign-caveat), so the SVD call appearing twice is nominal, not a\n    semantic duplicate.\n\n    Errors:\n        ValueError: N < 2, n_components > min(N, D).\n    """\n    h = np.asarray(h_batch, dtype=np.float64)\n    if h.ndim != 2:\n        raise ValueError(f"h_batch must be 2-D [N, D], got {h.ndim}-D")\n    n, d = h.shape\n    if n < 2:\n        raise ValueError(f"N = {n} < 2: PCA undefined")\n    if n_components > min(n, d):\n        raise ValueError(\n            f"n_components = {n_components} > min(N, D) = {min(n, d)}"\n        )\n    centered = h - h.mean(axis=0, keepdims=True)\n    _, _, vt = np.linalg.svd(centered, full_matrices=False)\n    components = vt[:n_components].astype(np.float32)\n    h_proj = (centered @ components.T).astype(np.float32)\n    return h_proj, components\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'prism/transport.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""Procrustes transport between orthonormal frames (ТЗ E40_math_core §II.5, M5).\n\nnumpy-side companion to ``prism.holonomy`` (which is torch and does the\nlayer-loop geometry); this module provides the PRECONDITIONS the ТЗ makes\nmandatory before any ``log Q``:\n\n``align_signs_and_columns`` before polar: ``Q_U = polar(U_tᵀ U_{t+1})`` is\nmeaningful only when the columns are sign-consistent and matched. A sign flip\nleaves ``−1`` on the diagonal → principal ``log`` complex; a permutation at\n``σ_j ≈ σ_{j+1}`` makes ``log`` branch-ambiguous (ТЗ §I.8).\n\nFlags (all reported into the artifact, ТЗ §I.8 pt 4):\n- ``sign_flips``   — columns flipped to make ``u_j(t)ᵀu_j(t+1) ≥ 0``;\n- ``order_cross``  — Hungarian matching ≠ rank order (Σ reorders between t\'s);\n- ``unstable_transport`` — spectrum of the polar factor close to ``−1`` or\n  ``‖log Q‖_F`` beyond ``angle_max``.\n"""\n\nfrom __future__ import annotations\n\nimport math\n\nimport numpy as np\nfrom scipy.linalg import logm\nfrom scipy.optimize import linear_sum_assignment\n\n__all__ = [\n    "polar_factor", "align_signs_and_columns", "procrustes_transport",\n    "transport_log", "holonomy", "holonomy_angles", "kappa",\n]\n\n\ndef _check_frame(U: np.ndarray, name: str) -> np.ndarray:\n    U = np.asarray(U, dtype=np.float64)\n    if U.ndim != 2:\n        raise ValueError(f"{name} must be [d, r] with orthonormal columns, got {U.shape}")\n    r = U.shape[1]\n    gram_err = float(np.abs(U.T @ U - np.eye(r)).max())\n    if gram_err > 1e-6:\n        raise ValueError(f"{name}: columns not orthonormal (max |UᵀU−I| = {gram_err:.2e})")\n    return U\n\n\ndef polar_factor(M: np.ndarray) -> np.ndarray:\n    """Nearest orthogonal matrix ``Q = U Vᵀ`` from ``M = U S Vᵀ`` (∈ O(r))."""\n    M = np.asarray(M, dtype=np.float64)\n    if M.ndim != 2 or M.shape[0] != M.shape[1]:\n        raise ValueError(f"polar_factor wants a square matrix, got {M.shape}")\n    U, _, Vt = np.linalg.svd(M)\n    return U @ Vt\n\n\ndef align_signs_and_columns(U0: np.ndarray, U1: np.ndarray,\n                            tol: float = 1e-12) -> tuple[np.ndarray, np.ndarray, dict]:\n    """Match columns of ``U1`` to ``U0`` (Hungarian on ``|U0ᵀU1|``), then fix\n    signs so the matched diagonal overlaps are ``≥ 0``.\n\n    Returns ``(U0, U1_aligned, flags)`` with ``flags["perm"]`` — the column\n    matching (``U1a[:, j]`` came from ``U1[:, perm[j]]``), so consumers can\n    carry per-column quantities (σ!) along with the frame (M2 fix). \n    Matching by ``|U0ᵀU1|`` — NOT by rank ``σ`` — is the ТЗ requirement;\n    disagreement with σ-order is reported as ``order_cross`` (a fact about\n    Σ reordering, not an error).\n    """\n    U0 = _check_frame(U0, "U0")\n    U1 = _check_frame(U1, "U1")\n    if U0.shape != U1.shape:\n        raise ValueError(f"frame shape mismatch {U0.shape} vs {U1.shape}")\n    S = np.abs(U0.T @ U1)\n    _, perm = linear_sum_assignment(-S)\n    flags = {\n        "order_cross": bool(np.any(perm != np.arange(perm.size))),\n        "sign_flips": [],\n        "perm": [int(k) for k in perm],\n    }\n    U1a = U1[:, perm]\n    diag = np.einsum("ij,ij->j", U0, U1a)             # u_j(t)ᵀ u_j(t+1)\n    if (np.abs(diag) < tol).any():\n        raise ValueError(\n            "align_signs_and_columns: a matched pair is ~orthogonal "\n            f"(min |diag| = {np.abs(diag).min():.2e}) — frames too far apart, "\n            "column identity is lost; refuse to guess signs.")\n    signs = np.sign(diag)\n    flags["sign_flips"] = [int(j) for j in np.where(signs < 0)[0]]\n    U1a = U1a * signs\n    return U0, U1a, flags\n\n\ndef procrustes_transport(V0: np.ndarray, V1: np.ndarray) -> np.ndarray:\n    """``Q = polar(V0ᵀ V1)`` — the optimal rotation ``O(r)`` between frames."""\n    V0 = _check_frame(V0, "V0")\n    V1 = _check_frame(V1, "V1")\n    return polar_factor(V0.T @ V1)\n\n\ndef transport_log(Q: np.ndarray, angle_max: float = math.pi - 1e-3) -> np.ndarray:\n    """Principal matrix logarithm of an orthogonal ``Q``; refuses (ValueError)\n    when the spectrum touches ``−1`` (log undefined on the principal branch)."""\n    Q = np.asarray(Q, dtype=np.float64)\n    ang = np.abs(np.angle(np.linalg.eigvals(Q)))\n    if ang.max() > angle_max:\n        raise ValueError(\n            f"transport_log: eigenangle {ang.max():.6f} rad at/over the branch "\n            "cut (−1): log Q undefined on the principal branch (unstable_transport)")\n    L = logm(Q)\n    if float(np.abs(L.imag).max()) > 1e-8:\n        raise ValueError(f"transport_log: complex log (max |Im| = {np.abs(L.imag).max():.2e})")\n    return L.real\n\n\ndef holonomy(V0: np.ndarray, V1: np.ndarray, V2: np.ndarray) -> np.ndarray:\n    """Loop holonomy ``R01 · R12 · R02⁻¹`` for three consecutive frames\n    (each ``Rij = polar(Viᵀ Vj)``, columns aligned first)."""\n    V0a, V1a, _ = align_signs_and_columns(V0, V1)\n    _, V2a, _ = align_signs_and_columns(V0a, V2)      # align all to V0 gauge\n    R01 = procrustes_transport(V0a, V1a)\n    R12 = procrustes_transport(V1a, V2a)\n    R02 = procrustes_transport(V0a, V2a)\n    return R01 @ R12 @ R02.T                          # R02⁻¹ = R02ᵀ (orthogonal)\n\n\ndef holonomy_angles(H: np.ndarray) -> np.ndarray:\n    """Rotation angles ``θ_j = arg eig(H)`` in ``[0, π]`` (± pairs)."""\n    H = np.asarray(H, dtype=np.float64)\n    return np.sort(np.abs(np.angle(np.linalg.eigvals(H))))\n\n\ndef kappa(H: np.ndarray) -> float:\n    """``κ(H) = ‖log H‖_F`` — invariant under orthogonal similarity\n    ``H → Oᵀ H O`` (the M0 registry asserts exactly this)."""\n    return float(np.linalg.norm(transport_log(H), "fro"))\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'experiments/exp47b_gen_kinematics.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('#!/usr/bin/env python\n"""E-47b (P1): кинематика листа L23 при СВОБОДНОЙ ГЕНЕРАЦИИ (Kaggle T4 / локально).\n\nПостановка/критерии (K1.1–K1.4, пререгистрированы) — docs/leaf_dynamics.md §5.2,§6;\nчисла P0 и прогноз (насыщение s=0.383·tanh(a/0.715), выход с полки на мелком шаге) — §8.1.\n\nПротокол:\n  * 4 домена x n_docs документов; промпт = первые prompt_len токенов калибровочного пула\n    (те же датасеты, что строили кэш exp27b; точное совпадение документов НЕ гарантируется\n    потоком датасета — пишем sha256 промптов в артефакт);\n  * greedy-генерация max_new шагов с DynamicCache; на каждый шаг берём h = hidden_states[n_layers]\n    (выход L23 = вход финальной RMSNorm; НЕ last_hidden_state — тот уже post-norm);\n  * кинематика из exp47_common: a_t, s_t (главные углы V16), p_t, g_t (доля движения u,\n    видимая листу: ||Pi_{span(Va∪Vb)} du|| / ||du||), Omega_t; тривиальные фичи-конкуренты:\n    энтропия logits, top-1 prob, флаг «токен=предыдущий»;\n  * ПЕТЛИ: find_loops по сгенерированным токенам (onset = начало 3-го повтора);\n  * СОХРАНЯЕМ СЫРЬЁ (npz): фичи per step, V16 fp16 [T,16,D], u fp16 — весь P2/P3-анализ\n    пересобирается офлайн без перепрогона.\n\nБазис U128 — КОНСТАНТА из кэша exp27b (источник истины всех опубликованных чисел;\nисточник: exp27b_jcache_L23.pt, pool_fp wiki 0cc16ec344e2 / math e71cc282c85d / code\n209e83b1898b / chat 2afac288344a), с двумя самопроверками на лету:\n  (i)  маршрут: H_head @ W_lm^T vs model(...).logits < 1e-3 (как exp45b) — падает громко;\n  (ii) частотный базис, пересобранный с пула этим же датасетом — пересечение в % (дескриптив;\n       поток датасета мог сдвинуться — на выводы не влияет, U128 фиксирован константой).\n\nKaggle: сигнатура main(model=..., device=..., results_dir=...) — драйвер билда подставляет\nлокальный путь модели/устройство/каталог. Локально: аргументы CLI.\n\nЗапуск (локально, CPU медленно): ../.venv/bin/python exp47b_gen_kinematics.py --n-docs 2\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport torch\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\n_ROOT = os.path.dirname(_HERE)          # Kaggle: бандл примонтирован как <root>/{prism,experiments}\nif _ROOT not in sys.path:\n    sys.path.insert(0, _ROOT)\n\nimport exp47_common as C   # noqa: E402\nfrom prism.calibration import build_global_token_set, load_calibration_data  # noqa: E402\n\nRNG_SEED = 47\n# U128 — частотный базис exp27b (константа-источник; см. докстринг)\nU128 = [0, 3, 4, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25,\n        26, 30, 32, 82, 87, 198, 220, 264, 271, 279, 303, 310, 314, 318, 321, 328, 348,\n        351, 353, 357, 364, 369, 383, 393, 411, 413, 421, 423, 424, 430, 440, 449, 458,\n        466, 471, 494, 498, 506, 513, 524, 539, 579, 599, 615, 628, 635, 641, 653, 660,\n        668, 694, 695, 716, 725, 735, 748, 777, 791, 799, 821, 864, 948, 984, 1000, 1040,\n        1066, 1076, 1088, 1092, 1116, 1132, 1172, 1220, 1238, 1317, 1330, 1332, 1345,\n        1358, 1381, 1472, 1518, 1521, 1532, 1602, 1608, 1680, 1754, 1831, 1973, 2107,\n        2228, 2394, 2397, 2424, 2574, 2582, 2785, 2957, 2972, 3018, 3067, 4370, 4558,\n        56014, 248044, 248046]\nPOOL_FP_EXPECT = {"wiki": "0cc16ec344e2", "math": "e71cc282c85d",\n                  "code": "209e83b1898b", "chat": "2afac288344a"}\n\n\ndef _sha(x) -> str:\n    return hashlib.sha256(str(x).encode()).hexdigest()[:12]\n\n\ndef _step(V_prev, u_prev, u_cur, A, eps, h_cur):\n    """Фичи перехода: V/u текущего шага + (a,s,p,g); при V_prev=None (первый шаг)\n    кинематика не определена — nan, но V/u считаем."""\n    Vb, _ = C.V16_from_J(C.J_analytic(A, h_cur, eps)[0])\n    if V_prev is None:\n        return float("nan"), float("nan"), float("nan"), float("nan"), Vb\n    Va = V_prev\n    s, p, _P, _Th = C.frame_kinematics(Va, Vb)\n    a = float(torch.acos(((u_prev * u_cur).sum()).clamp(max=1 - 1e-12)))\n    du = u_cur - (u_prev @ u_cur) * u_prev\n    ndu = float(du.norm())\n    g = float("nan")\n    if ndu > 1e-9:\n        q = torch.linalg.qr(torch.cat([Va, Vb], dim=1))[0]\n        g = float((q.T @ (du / ndu)).norm())\n    return a, s, p, g, Vb\n\n\ndef main(model="/data/Qwen/Qwen3.5-0.8B", n_docs=8, max_new=384, prompt_len=64,\n         device="cpu", results_dir="results", do_verify_basis=True,\n         rtn_bits=None, rtn_group=128, dtype="float32", load_in_8bit=False):\n    torch.set_num_threads(4)\n    torch.manual_seed(RNG_SEED)\n    dev = torch.device(device)\n    from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache  # type: ignore\n\n    t0 = time.time()\n    kw = {}\n    if load_in_8bit:      # честный PTQ (bnb int8): trunk квант, embed/head точные\n        from transformers import BitsAndBytesConfig\n        kw = dict(quantization_config=BitsAndBytesConfig(load_in_8bit=True),\n                  device_map="cuda:0")\n    m = AutoModelForCausalLM.from_pretrained(model, dtype=getattr(torch, dtype),\n                                             attn_implementation="eager", **kw)\n    m.eval()\n    if not load_in_8bit:\n        m.to(dev)\n    if rtn_bits:      # full-model RTN (тест A/B честный: меняется и динамика u).\n        lev = 2 ** (rtn_bits - 1) - 1\n        nq = 0\n        for mod in m.modules():\n            if isinstance(mod, torch.nn.Linear):\n                w = mod.weight.data\n                wf = w.reshape(w.shape[0], -1, rtn_group).double()\n                sc = wf.abs().amax(-1, keepdim=True).clamp(min=1e-12) / lev\n                mod.weight.data = ((wf / sc).round().clamp(-lev, lev) * sc).reshape_as(w).to(w.dtype)\n                nq += 1\n        print(f"[rtn] int{rtn_bits} per-group({rtn_group}) деQuantized Linear-слоёв: {nq}", flush=True)\n    tok = AutoTokenizer.from_pretrained(model)\n    n_layers = int(m.config.num_hidden_layers)\n    norm = m.model.norm\n    eps = float(getattr(norm, "variance_epsilon",\n                        getattr(norm, "eps", m.config.rms_norm_eps)))\n    # e52k падал здесь на GPU: `pre`/`raw` ниже снимаются на CPU, а `wn` оставался на\n    # cuda:0, и проверка g-режима (голый w против 1+w) давала cuda/cpu mismatch. wn дальше\n    # употребляется только через .cpu() (C.head_M, feats), поэтому CPU-размещение безопасно.\n    wn = norm.weight.detach().float().cpu().clone()   # clone: у quantized-весов .cpu() может оставаться view\n    # эмпирика семантики RMSNorm: gemma-style g=1+w против голого g=w (Qwen2/3)\n    if os.environ.get("PRISM_SKIP_NORM_TEST") == "1":\n        # семантика g уже проверен локально (E-47b): qwen3_5/gemma-style -> g=1+w\n        g_mode = "1p_w" if "Qwen3_5" in type(norm).__name__ else "w"\n        print(f"[norm] тест пропущен (PRISM_SKIP_NORM_TEST=1), g_mode={g_mode}", flush=True)\n        wn = norm.weight.detach().float().cpu().clone()\n        g_n = (wn if g_mode == "w" else 1.0 + wn)\n        eps = float(getattr(norm, "variance_epsilon", getattr(norm, "eps", 1e-6)))\n    else:\n     with torch.no_grad():\n        prb = torch.randint(1, 1000, (1, 8), device=dev)\n        pre_c = []\n        hk = norm.register_forward_hook(lambda _m, inp, _o: pre_c.append(inp[0].detach()))\n        raw = m.model(input_ids=prb).last_hidden_state[0, -1].detach().float().cpu()\n        hk.remove()\n        pre = pre_c[0][0, -1].detach().float().cpu()   # raw уже cpu — device-консистентность (e52k cuda fix)\n        r = (pre.pow(2).mean(-1, keepdim=True) + eps).rsqrt()\n        e_w = float((wn * r * pre - raw).abs().max())\n        e_1w = float(((1.0 + wn) * r * pre - raw).abs().max())\n        g_mode = "w" if e_w <= e_1w else "1p_w"\n        assert min(e_w, e_1w) < 1e-3, f"norm не закрыт ни одной g: w={e_w:.1e} 1+w={e_1w:.1e}"\n        print(f"[norm] g-режим: {g_mode} (err w={e_w:.1e}, 1+w={e_1w:.1e})", flush=True)\n    g_n = (wn if g_mode == "w" else 1.0 + wn)\n    W = m.lm_head.weight.detach()\n    U = torch.tensor(U128, device=dev)\n    A = C.head_M(W[U].cpu().float(), g_n.cpu()).to(dev)      # [128,D] fp64 на устройстве\n    tied = m.lm_head.weight.data_ptr() == m.get_input_embeddings().weight.data_ptr()\n\n    # маршрут: H_head @ W_lm[U]^T == logits[U]\n    with torch.no_grad():\n        probe = torch.randint(1, 1000, (1, 8), device=dev)\n        hs = m.model(input_ids=probe).last_hidden_state[0, -1].float()\n        lg = m(input_ids=probe).logits[0, -1, U].float()\n        rt = float((hs @ W[U].T.float().cpu().to(dev) - lg).abs().max())\n        rel = rt / max(float(lg.abs().max()), 1.0)\n    print(f"[route] max|H@W_U^T - logits_U| = {rt:.2e} (rel {rel:.1e}); tied={tied}", flush=True)\n    # fp32: абсолютный порог; bf16/int8: масштаб шкурного округления ~ rel·scale\n    # (на 0.8B bf16 абс. 3e-2 при scale~30 это ЧЕСТНОЕ bf16-округление, e52k v5)\n    assert rt < 1e-3 or rel < 2e-2, f"маршрут логитов не сошёлся: abs {rt:.3e} rel {rel:.1e}"\n\n    docs_out, feats_npz = [], {}\n    basis_overlap = None\n    for dom in ("wiki", "math", "code", "chat"):\n        pool = load_calibration_data(model, max_length=256, min_length=64,\n                                     n_samples=n_docs, domain=dom)\n        assert len(pool) >= n_docs\n        if do_verify_basis and dom == "wiki":       # дорого: один домен\n            fresh = build_global_token_set(m, pool[:n_docs], K=len(U128))\n            basis_overlap = len(set(fresh) & set(U128)) / len(U128)\n            print(f"[basis] overlap fresh-vs-cache U128: {basis_overlap:.2%}", flush=True)\n        for si in range(n_docs):\n            ids = pool[si]["input_ids"][:, :prompt_len].to(dev)\n            prompt_sha = _sha(pool[si]["input_ids"][0, :prompt_len].tolist())\n            # префиксный проход: h_23 промпта (последняя позиция) + cache\n            with torch.no_grad():\n                cache = DynamicCache(config=m.config)\n                out = m(input_ids=ids, past_key_values=cache, use_cache=True,\n                        output_hidden_states=True)\n            h = out.hidden_states[n_layers][0, -1].double().to(dev)   # h_23 (pre-final-norm)\n            logits = out.logits[0, -1]\n            feats = {"a": [], "s": [], "p": [], "g": [], "ent": [], "top1": [],\n                     "eq_prev": [], "hnorm": []}\n            u = h / h.norm()\n            V_prev = u_prev = None\n            # Кадр при промптной позиции — отдельно от V_prev (NaN первого шага не трогаем).\n            V_at_cur = C.V16_from_J(C.J_analytic(A, h, eps)[0])[0]\n            tokens = ids[0].tolist()\n            V_stack, u_stack = [], []\n            for step in range(max_new):\n                p = torch.softmax(logits.float(), dim=-1)\n                ent = float(-(p.clamp_min(1e-12) * p.clamp_min(1e-12).log()).sum())\n                nxt = int(torch.argmax(logits))\n                feats["ent"].append(ent)\n                feats["top1"].append(float(p.max()))\n                feats["eq_prev"].append(int(nxt == tokens[-1]))\n                feats["hnorm"].append(float(h.norm()))\n                u_stack.append(u.float().cpu().numpy())\n                tokens.append(nxt)\n                with torch.no_grad():\n                    out = m(input_ids=torch.tensor([[nxt]], device=dev),\n                            past_key_values=cache, use_cache=True,\n                            output_hidden_states=True)\n                h_new = out.hidden_states[n_layers][0, -1].double().to(dev)\n                logits = out.logits[0, -1]\n                u_new = h_new / h_new.norm()\n                a, s, pl, gg, V_cur = _step(V_prev, u, u_new, A, eps, h_new)\n                feats["a"].append(a); feats["s"].append(s)\n                feats["p"].append(pl); feats["g"].append(gg)\n                V_prev, u_prev = V_cur, u\n                u = u_new\n                # Аудит E46+ (A6): писался V_cur = кадр СЛЕДУЮЩЕЙ позиции, то есть\n                # сохранённые u[i] и V[i] были сдвинуты на шаг. Теперь оба на позиции i.\n                V_stack.append(V_at_cur.float().cpu().numpy())\n                V_at_cur = V_cur\n                h = h_new\n            gen_tokens = tokens[prompt_len:]\n            loops = C.find_loops(gen_tokens)\n            aa = np.array(feats["a"], dtype=float)\n            ss = np.array(feats["s"], dtype=float)\n            pp = np.array(feats["p"], dtype=float)\n            # посегментная сводка (in-loop по ДЕТЕРМИНИРОВАННОМУ детектору; вне — всё остальное)\n            in_mask = np.zeros(max_new, dtype=bool)\n            for lp in loops:\n                on0 = min(lp[0], max_new - 1)\n                in_mask[on0: min(lp[2], max_new)] = True\n            acf = [C.autocorr_peak(torch.from_numpy(\n                       ss[min(lp[0], len(ss)-1):min(lp[2], len(ss))] - np.nanmean(ss)), lp[3])\n                   for lp in loops]\n            r_seg = float("nan")\n            if in_mask.any() and (~in_mask).sum() > 20:\n                x, y = ss[in_mask], aa[in_mask]\n                ok = ~(np.isnan(x) | np.isnan(y))\n                if ok.sum() > 5:\n                    x = x - x.mean(); y = y - y.mean()\n                    d = math.sqrt(float((x * x).sum() * (y * y).sum()))\n                    r_seg = float((x * y).sum() / d) if d > 0 else float("nan")\n            docs_out.append({\n                "dom": dom, "si": si, "prompt_sha": prompt_sha,\n                "n_loops": len(loops),\n                "loops": [list(lp) for lp in loops],\n                "a_med": float(np.nanmedian(aa)), "s_med": float(np.nanmedian(ss)),\n                "p_med": float(np.nanmedian(pp)),\n                "g_med": float(np.nanmedian(feats["g"])),\n                "excess_a": C.band_power(torch.from_numpy(np.nan_to_num(aa))),\n                "excess_s": C.band_power(torch.from_numpy(np.nan_to_num(ss))),\n                "in_loop": {"a": float(np.nanmedian(aa[in_mask])) if in_mask.any() else None,\n                            "s": float(np.nanmedian(ss[in_mask])) if in_mask.any() else None,\n                            "p": float(np.nanmedian(pp[in_mask])) if in_mask.any() else None,\n                            "acf_peak_by_loop": [None if x != x else float(x) for x in acf],\n                            "r_s_a_in": r_seg if r_seg == r_seg else None},\n                "trivial": {"ent_med": float(np.median(feats["ent"])),\n                            "top1_med": float(np.median(feats["top1"])),\n                            "eq_prev_rate": float(np.mean(feats["eq_prev"]))},\n                "gen_sha": _sha(gen_tokens),\n            })\n            feats_npz[f"{dom}_{si}"] = {k: np.array(v, dtype=np.float32)\n                                        for k, v in feats.items()}\n            feats_npz[f"{dom}_{si}_V"] = np.stack(V_stack).astype(np.float16)\n            feats_npz[f"{dom}_{si}_u"] = np.stack(u_stack).astype(np.float16)\n            d0 = docs_out[-1]\n            print(f"{dom} s{si}: loops={d0[\'n_loops\']} a={d0[\'a_med\']:.3f} s={d0[\'s_med\']:.3f} "\n                  f"p={d0[\'p_med\']:.3f} g={d0[\'g_med\']:.3f} ent={d0[\'trivial\'][\'ent_med\']:.2f} "\n                  f"[{time.time()-t0:.0f}s]", flush=True)\n        del pool\n\n    os.makedirs(results_dir, exist_ok=True)\n    out = {"params": {"protocol": "docs/leaf_dynamics.md §5.2 (P1)", "model": str(model),\n                      "n_docs": n_docs, "max_new": max_new, "prompt_len": prompt_len,\n                      "decoding": "greedy", "seed": RNG_SEED,\n                      "U128_source": "exp27b cache constant (pool_fp см. в докстринге)",\n                      "basis_overlap_fresh": basis_overlap,\n                      "route_check": rt, "tied": bool(tied)},\n           "docs": docs_out}\n    with open(os.path.join(results_dir, "exp47b_gen_kinematics.json"), "w") as f:\n        json.dump(out, f, ensure_ascii=False, allow_nan=False)\n    feats_npz["g_norm"] = g_n.cpu().numpy().astype(np.float32)   # источник истины для e52a\n    feats_npz["eps"] = np.float64(eps)\n    np.savez_compressed(os.path.join(results_dir, "exp47b_gen_kinematics.npz"), **feats_npz)\n    print(f"saved. total {time.time()-t0:.0f}s", flush=True)\n    return out\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", default="/data/Qwen/Qwen3.5-0.8B")\n    ap.add_argument("--n-docs", type=int, default=8)\n    ap.add_argument("--max-new", type=int, default=384)\n    ap.add_argument("--prompt-len", type=int, default=64)\n    ap.add_argument("--device", default="cpu")\n    ap.add_argument("--rtn-bits", type=int, default=None,\n                    help="full-model RTN: симметрично per-group квантовать ВСЕ Linear (dequant in-place)")\n    ap.add_argument("--rtn-group", type=int, default=128)\n    ap.add_argument("--results-dir", default=os.path.join(_HERE, "..", "results"))\n    a = ap.parse_args()\n    main(model=a.model, n_docs=a.n_docs, max_new=a.max_new, prompt_len=a.prompt_len,\n         device=a.device, results_dir=a.results_dir,\n         rtn_bits=a.rtn_bits, rtn_group=a.rtn_group)\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'experiments/exp52a_perturb_axis.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('#!/usr/bin/env python\n"""E-52a PREFLIGHT: ось отклонения τ = τ* + ε (E52_review, тесты A–C).\n\nВопрос не «болеет ли», а «куда и на сколько отклоняется τ под батареей\nвозмущений». Нули/метрика — E-49b (W4, paired по документам, τ-медиана).\n\nВозмущения (дозные семейства, всё детерминировано по сидам):\n  FLAT  : готовые строки головы E-45 (flat_l0.01 < flat_l0.1 < flat_noleash,\n          доза = глубина уплощения; qwen-only — кэш exp45_flat_rows.pt);\n  RTN   : симметричная per-row квантизация A, int8 → int4 → int2 (доза = биты);\n  NOISE : мультипликативный шум строк A, σ = 0.005/0.02/0.08 (реад-аут);\n  U-NS  : шум скрытых u перед сглаживанием, σ = 0.01/0.02/0.04 (трэнк-прокси).\nТесты: A — знак Δτ (одно семейство ⇒ знак однообразен?); B — монотонность\nпо дозе (Spearman); C — перенос на вторую модель (сравнение потом по JSON).\n\nК администратору чисел: τ считается top-16 РЕБРАМИ J через спектр JJᵀ [128,128]\n(ранг ≤128 ⇒ тождественно [D,D]-варианту exp49b2, в ~50 раз быстрее); T=96,\nW=4 — самосогласовано внутри прогона, с baseline E-49b (T=384) абсолютные\nуровни не сравниваем — только paired-Δ.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport sys\n\nimport numpy as np\nimport torch\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\nimport exp47_common as C  # noqa: E402\nfrom exp49_leaf_info import load_model, smoothed_us, tau_triplets, boot_ci  # noqa: E402\n\nTMAX = 96\nW = 4\n\n\ndef _seed(s):\n    return int.from_bytes(hashlib.sha256(s.encode()).digest()[:8], "little") % (1 << 62)\n\n\ndef rtn(A, bits):\n    """Симметричная per-row RTN: scale = row-absmax/(2^(b-1)-1)."""\n    lev = 2 ** (bits - 1) - 1\n    sc = A.abs().amax(dim=1, keepdim=True).clamp(min=1e-12) / lev\n    return (torch.clamp((A / sc).round(), -lev, lev) * sc)\n\n\ndef frames16(A, us, eps):\n    """Top-16 рам листа через спектр JJᵀ [K,K] (ранг J ≤ K=128 < D)."""\n    out = []\n    for x in us:\n        J, _, _ = C.J_analytic(A, x, eps)\n        Gs = J @ J.T\n        Gs = 0.5 * (Gs + Gs.T)\n        _, Ue = torch.linalg.eigh(Gs)\n        V = J.T @ Ue[:, -16:]\n        out.append(torch.linalg.qr(V / V.norm(dim=0, keepdim=True))[0])\n    return out\n\n\ndef tau_of(A, us, eps):\n    t = tau_triplets(frames16(A, us, eps))\n    return float(np.median(t)) if t else None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--model", choices=("qwen", "lfm3"), default="qwen")\n    ap.add_argument("--n-docs", type=int, default=32)\n    ap.add_argument("--families", default="flat,rtn,noise,u_ns")\n    ap.add_argument("--npz", default=None); ap.add_argument("--docs-json", default=None)\n    ap.add_argument("--model-dir", default=None,\n                    help="пересобрать A из этого чекпойнта (тест D, без full-load: safetensors slice)")\n    ap.add_argument("--out", default=None)\n    args = ap.parse_args()\n    root = os.path.join(_HERE, "..")\n    fams = set(args.families.split(","))\n\n    ns = argparse.Namespace(model=args.model)\n    A0, U, eps, npz, jp = load_model(ns)\n    g = load_model.last_g\n    if args.npz:\n        npz = args.npz\n    docs = json.load(open(args.docs_json or os.path.join(root, jp)))["docs"][: args.n_docs]\n    z = np.load(npz, allow_pickle=True)\n    if args.model_dir:      # тест D: A/g/eps от модели самой трассы\n        import glob\n        from safetensors import safe_open\n        mp = args.model_dir\n        files = sorted(glob.glob(os.path.join(mp, "*.safetensors")))\n        HCAND = ("lm_head.weight", "model.language_model.lm_head.weight",\n                 "model.embed_tokens.weight", "model.language_model.embed_tokens.weight")\n        NCAND = ("model.norm.weight", "model.language_model.norm.weight")\n        head = normw = None\n        for f in files:\n            with safe_open(f, framework="pt") as sf:\n                ks = set(sf.keys())\n                if head is None:\n                    head = next((sf.get_tensor(c) for c in HCAND if c in ks), None)\n                if normw is None:\n                    normw = next((sf.get_tensor(c) for c in NCAND if c in ks), None)\n            if head is not None and normw is not None:\n                break\n        assert head is not None and normw is not None, f"head/norm не найдены в {mp}"\n        wn = normw.float().double()\n        if "g_norm" in z.files:          # свежая трасса: единый источник истины\n            g = torch.from_numpy(z["g_norm"].astype(np.float64))\n            eps = float(z["eps"])\n            print("[model-dir] g/eps из npz", flush=True)\n        else:                            # старая трасса: статистика w (w~0 ⇒ 1+w; w~1 ⇒ w)\n            g = wn if float(wn.abs().mean()) > 0.5 else 1.0 + wn\n            cfg = json.load(open(os.path.join(mp, "config.json")))\n            eps = float(cfg.get("rms_norm_eps")\n                        or cfg.get("text_config", {}).get("rms_norm_eps") or 1e-6)\n            mode = "w" if float(wn.abs().mean()) > 0.5 else "1+w"\n            print(f"[model-dir] fallback: g={mode}, eps={eps}, |w|mean={float(wn.abs().mean()):.3f}", flush=True)\n        A0 = C.head_M(head.float()[U], g)\n\n    base_us = {}\n    for d in docs:\n        k = f"{d[\'dom\']}_{d[\'si\']}"\n        u = torch.from_numpy(z[k + "_u"].astype(np.float64))[:TMAX]\n        hn = np.asarray(z[k].item()["hnorm"], dtype=float)[:TMAX]\n        base_us[k] = (u, hn)\n        # sanity: рамочный маршрут 128-трюка против [D,D] — один раз\n    if True:\n        k0 = next(iter(base_us)); u0, hn0 = base_us[k0]\n        us0 = smoothed_us(u0, hn0, W)\n        J, _, _ = C.J_analytic(A0, us0[5], eps)\n        V_small = J.T @ torch.linalg.eigh(J @ J.T)[1][:, -16:]\n        V_big = (J.T @ J) @ torch.linalg.eigh((J.T @ J).double())[1][:, -16:]  # [D,D]\n        ang = torch.acos(torch.linalg.svdvals(\n            torch.linalg.qr(V_small)[0].T @ torch.linalg.qr(V_big)[0]).clamp(-1 + 1e-12, 1 - 1e-12))\n        assert float(ang.max()) < 1e-4, f"128-трюк расходится с [D,D]: {float(ang.max())}"  # 1.4e-6实测\n\n    variants = [("base", A0, None)]\n    if "flat" in fams and args.model == "qwen":\n        fl = torch.load(os.path.join(root, "results", "exp45_flat_rows.pt"),\n                        map_location="cpu", weights_only=False)\n        for nm in ("flat_l0.01", "flat_l0.1", "flat_noleash"):\n            variants.append((nm, C.head_M(fl[nm].float(), g), None))\n    if "rtn" in fams:\n        for b in (8, 4, 2):\n            variants.append((f"rtn{b}", rtn(A0, b), None))\n    if "noise" in fams:\n        for s in (0.005, 0.02, 0.08):\n            xi = torch.randn(A0.shape, dtype=torch.float64,\n                             generator=torch.Generator().manual_seed(_seed(f"An{s}")))\n            variants.append((f"noise{s}", A0 * (1.0 + s * xi), None))\n    u_variants = ([("u_ns0.01", 0.01), ("u_ns0.02", 0.02), ("u_ns0.04", 0.04)]\n                  if "u_ns" in fams else [])\n\n    out = {"model": args.model, "W": W, "T": TMAX, "n_docs": len(docs),\n           "note": "paired Δτ (variant−base) per doc; 128-трюк верифицирован против [D,D]"}\n    taus = {}\n    for nm, A, _ in variants:\n        vals = {}\n        for k, (u, hn) in base_us.items():\n            vals[k] = tau_of(A, smoothed_us(u, hn, W), eps)\n        taus[nm] = vals\n        print(nm, "done", flush=True)\n    for nm, s in u_variants:\n        xi = torch.randn((1, A0.shape[1]), dtype=torch.float64,\n                         generator=torch.Generator().manual_seed(_seed(f"u{nm}")))\n        vals = {}\n        for k, (u, hn) in base_us.items():\n            un = u + s * u.norm(dim=1, keepdim=True) * xi.to(u.dtype)\n            vals[k] = tau_of(A0, smoothed_us(un, hn, W), eps)\n        taus[nm] = vals\n        print(nm, "done", flush=True)\n\n    dom = {k: k.split("_")[0] for k in base_us}\n    summ = {}\n    for nm in taus:\n        if nm == "base":\n            summ[nm] = {"tau": float(np.median(list(taus["base"].values())))}\n            continue\n        dd = [taus[nm][k] - taus["base"][k] for k in base_us\n              if taus[nm][k] is not None and taus["base"][k] is not None]\n        ddom = {}\n        for d0 in set(dom.values()):\n            sub = [taus[nm][k] - taus["base"][k] for k in base_us\n                   if dom[k] == d0 and taus[nm][k] is not None]\n            ddom[d0] = float(np.median(sub)) if sub else None\n        summ[nm] = {"d_tau": float(np.median(dd)), "ci": boot_ci(dd),\n                    "dom_sign_pos": sum(1 for v in ddom.values() if (v or 0) > 0),\n                    "n_dom": len(ddom), "per_dom": ddom,\n                    "tau": float(np.median(list(taus[nm].values())))}\n        print(nm, summ[nm]["d_tau"], summ[nm]["ci"], flush=True)\n    out["per_doc"] = taus\n    out["summary"] = summ\n    path = args.out or os.path.join(root, "results", f"exp52a_perturb_{args.model}.json")\n    json.dump(out, open(path, "w"), indent=1, allow_nan=False)\n    print("WROTE", path)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'experiments/exp47_common.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('"""E-47 common: численная кинематика листа L23 (общое ядро для exp47a/b/c/d).\n\nМатематика (полный вывод и верификация — docs/leaf_dynamics.md §2,§4):\n\n    J(h) = r(h) * M (I - rho * u u^T),   u = h/||h||,  rho = ||h||^2/(||h||^2 + D*eps),\n    M    = W_head[U128] * diag(1 + w_norm),          # Qwen RMSNorm: (1+w), НЕ w\n    r(h) = rsqrt(mean(h^2) + eps)                     # скаляр -> на V не влияет\n\nВся зависимость V_16(h) от позиции — ЧЕРЕЗ ОДНУ u (ранг-1 наклон). Отсюда:\n  * V инвариантна к масштабу h;\n  * delta J от поворота u на du ⟂ u: -rho*M(du u^T + u du^T) — ранг 2;\n  * скорость листа s_t пропорциональна скорости потока a_t = angle(u_t,u_{t+1})\n    с резольвентным множителем 1/(sigma_i^2 - sigma_j^2) (док. gap16 мал ⇒ рама жёсткая).\n\nЧисленная гигиена (урок верификации 2026-11-05): собственные векторы Gram берём от\nJJ^T размера [K,K] (K=128), а не J^T J [D,D]: второе удваивает обусловленность на блоке D\nи в fp32 даёт ложный пол ~1e-3 рад на масштабе s_t. Путь: eigh(JJ^T) -> U, sigma^2;\nправые векторы V = J^T U[:,top] / sigma. Пол метода в fp64: 1.4e-6 рад (проверено).\n\nВсе функции чистые, torch-only (scipy.linalg.schur только для углов Шюра).\n"""\nfrom __future__ import annotations\n\nimport math\n\nimport torch\n\n\n# ---------------------------------------------------------------- Jacobian / V\n\ndef head_M(head_W: torch.Tensor, g_norm: torch.Tensor) -> torch.Tensor:\n    """M = W_head[U128] diag(g), g = 1+w_norm. fp64 — для кинематики малых углов."""\n    return (head_W.double() * g_norm.double().unsqueeze(0)).contiguous()\n\n\ndef J_analytic(A: torch.Tensor, h: torch.Tensor, eps: float) -> tuple[torch.Tensor, float, torch.Tensor]:\n    """J(h) без скаляра r (на V не влияет): J = A (I - rho u u^T) = A - rho (A u) u^T.\n\n    Возвращает (J [K,D], rho, u [D]). A = M (fp64). Одна внешняя произведение — O(K D).\n    """\n    hh = float(h @ h)\n    D = h.numel()\n    rho = hh / (hh + D * eps)\n    u = h / math.sqrt(hh)\n    J = A - rho * torch.outer(A @ u, u)\n    return J, rho, u\n\n\ndef V16_from_J(J: torch.Tensor, r: int = 16):\n    """Топ-r правых сингулярных подпространств через Gram JJ^T [K,K] (fp64).\n\n    V = J^T U_k / sigma_k (U_k — собственные векторы JJ^T с бОльшими lmbda=sigma^2).\n    Возвращает (V [D,r], sigma [K] — все, по порядку убывания).\n    """\n    G = J @ J.T\n    G = 0.5 * (G + G.T)\n    lam, U = torch.linalg.eigh(G)          # по возрастанию\n    idx = torch.argsort(lam, descending=True)\n    lam, U = lam[idx], U[:, idx]\n    sig = lam.clamp_min(1e-30).sqrt()\n    V = (J.T @ U[:, :r]) / sig[:r]\n    return V.contiguous(), sig\n\n\ndef procrustes_R(Va: torch.Tensor, Vb: torch.Tensor) -> torch.Tensor:\n    """Ортогональный транспорт рамы Va->Vb: R = polar(Va^T Vb) = U W^T (SVD 16x16).\n\n    R определён корректно, когда столбцы — УПОРЯДОЧЕННЫЕ базисы (наши V из eigh — да).\n    Знаки столбцов SVD-базисов тут не мешают: Va->Va даёт точную I только при совпадении\n    знаков; поэтому дальше используем только инварианты R (углы Шюра), которые от знака\n    базиса НЕ зависят (подправка знаком вписана в сам полярный множитель).\n    """\n    U, _, Wt = torch.linalg.svd(Va.T @ Vb, full_matrices=False)\n    return U @ Wt\n\n\n# ---------------------------------------------------------------- инварианты R\n\ndef schur_angles(R: torch.Tensor):\n    """Главная часть log(R) для R in SO(16): углы 2x2-блоков вещественного Шюра.\n\n    Вещественный Шур R = Q T Q^T: диагональные 1x1 блоки = ±1 (угол 0 или pi),\n    2x2 блоки [[a,b],[c,d]]bc<0: собственные a ± i sqrt(|bc|) => theta = atan2(|b|,a)\n    (берём |theta| в [0,pi)). Возвращает (theta sorted desc [m], Q — векторы Шюра,\n    bsize — размеры блоков): Q[:, k] — ИНВАРИАНТНОЕ подпространство блока k\n    (для 2-блока — плоскость вращения с наибольшим |theta|, если блоки отсортированы).\n    """\n    import numpy as np\n    from scipy.linalg import schur\n\n    T, Q = schur(R.detach().cpu().numpy(), output="real")\n    n = T.shape[0]\n    thetas: list[float] = []\n    bsize: list[int] = []\n    i = 0\n    while i < n:\n        if i + 1 < n and abs(T[i + 1, i]) > 1e-10:\n            # собственный блок Шюра: [[a,b],[c,d]], bc<0, собств. = (a+d)/2 ± i*sqrt|bc|\n            re = 0.5 * (float(T[i, i]) + float(T[i + 1, i + 1]))\n            im = math.sqrt(abs(float(T[i, i + 1]) * float(T[i + 1, i])))\n            th = math.atan2(im, re) % math.pi\n            thetas.append(th)\n            bsize.append(2)\n            i += 2\n        else:\n            th = 0.0 if float(T[i, i]) > 0 else math.pi\n            thetas.append(th)\n            bsize.append(1)\n            i += 1\n    order = sorted(range(len(thetas)), key=lambda k: -thetas[k])\n    # переставляем векторы Шюра блоками в порядке убывания угла\n    cols: list[int] = []\n    start: list[int] = []\n    s = 0\n    for bs in bsize:\n        start.append(s)\n        s += bs\n    for k in order:\n        cols.extend(range(start[k], start[k] + bsize[k]))\n    Q2 = torch.from_numpy(Q[:, cols])\n    return torch.tensor([thetas[k] for k in order], dtype=torch.float64), Q2, [bsize[k] for k in order]\n\n\ndef frame_kinematics(Va: torch.Tensor, Vb: torch.Tensor):\n    """Один шаг кинематики ЛИСТА (подпространства) — базис-инвариантные величины.\n\n    ⚠ УРОК (найден первым прогоном P0 2026-11-05): углы Шюра одного шага Procrustes\n    R = polar(Va^T Vb) НЕ инвариантны к внутренним базисам рам: Va->Va@Q1, Vb->Vb@Q2\n    дают R -> Q1^T R Q2, собственные углы при этом меняются. Внутренний базис eigh/SVD\n    при плотном кластере сингулярных чисел выбирается произвольно — одношаговый R мерил\n    «рассогласование базисов» (s~π, e~π/2 на ровных данных). Инвариантен ТОЛЬКО цикличный\n    произведение (голономия κ — потому E-28 и корректен). Для одного шага — ГЛАВНЫЕ УГЛЫ\n    подпространств (Грассманиан):\n        Va^T Vb = X diag(cos Θ) Y^T ;  s = ||Θ||_2,  p = Θ_max/s,\n        собственная 2-плоскость в D-координатах: span{Va X_1, Vb Y_1} (инвариантна).\n    В меловом limite Θ = углы вращения, так что семантика «скорость/планарность» сохранена.\n    Возвращает (s, p, P [D,2], Θ [r] desc).\n    """\n    M = Va.T @ Vb\n    X, c, Yh = torch.linalg.svd(M, full_matrices=False)\n    Y = Yh.T\n    Th = torch.acos(c.clamp(-1.0 + 1e-12, 1.0 - 1e-12))\n    # svd отдаёт cos Θ по убыванию ⇒ Θ по ВОЗРАСТАНИЮ; доминирующий — последний\n    order = torch.argsort(Th, descending=True)\n    Th = Th[order]\n    i0 = int(order[0])\n    s = float(torch.sqrt((Th * Th).sum()))\n    p = float(Th[0] / s) if s > 1e-12 else float("nan")\n    P = torch.column_stack([Va @ X[:, i0], Vb @ Y[:, i0]]).double().contiguous()\n    return s, p, P, Th\n\n\ndef align_to_u_plane(P: torch.Tensor, u0: torch.Tensor, u1: torch.Tensor) -> float:\n    """Худший главный угол (рад) между 2-плоскостью P и span{u0,u1}; nan если span вырожден.\n\n    Для 2-плоскости в D>4 корректнее брать ПЛОЩАДНУЮ согласованность: здесь худший\n    главный угол — если P и span расходятся хотя бы в одном направлении, угол велик.\n    cos главного угла = сингулярные числа P^T Q_U; acos(min) — угол самого «непокрытого»\n    направления: 0 = плоскости совпадают.\n    """\n    B = torch.column_stack([u0, u1])\n    svB = torch.linalg.svdvals(B)\n    if float(svB.min()) < 1e-8 * float(svB.max()):\n        return float("nan")            # u0 || u1: шага нет, выравнивание не определено\n    Qb = torch.linalg.qr(B)[0]\n    svP = torch.linalg.svdvals(P)\n    if float(svP.min()) < 1e-6 * float(svP.max()):\n        return float("nan")            # Θ мал: «плоскость» выродилась в прямую, QR бы соврал\n    Qp = torch.linalg.qr(P)[0]         # P — два ПРИНЦИПАЛЬНЫХ вектора (угол между ними = Θ),\n                                       # не ортонормированы; без QR svd(P^T Q) врал (>1)\n    sv = torch.linalg.svdvals(Qp.T @ Qb)\n    return math.acos(float(sv.min().clamp(max=1.0 - 1e-12)))\n\n\ndef Q_orth(X: torch.Tensor) -> torch.Tensor:  # noqa: N802 (имя из формул)\n    return torch.linalg.qr(X)[0]\n\n\n# ---------------------------------------------------------------- скалярные ряды\n\ndef u_series(hs: torch.Tensor) -> torch.Tensor:\n    """По [T,D] -> единичные u_t (fp64)."""\n    H = hs.double()\n    return H / H.norm(dim=-1, keepdim=True)\n\n\ndef angle_series(u: torch.Tensor) -> torch.Tensor:\n    """a_t = angle(u_t, u_{t+1}) [T-1]."""\n    c = (u[:-1] * u[1:]).sum(-1).clamp(max=1.0 - 1e-12)\n    return torch.acos(c)\n\n\ndef omega_geodesic(u: torch.Tensor) -> torch.Tensor:\n    """Геодезическая кривизна [M-3]: |A01 + A12 - A02| по тройкам (T-2)."""\n    def A(x, y):\n        return torch.acos((x * y).sum(-1).clamp(max=1.0 - 1e-12))\n    return (A(u[:-2], u[1:-1]) + A(u[1:-1], u[2:]) - A(u[:-2], u[2:])).abs()\n\n\ndef band_power(x: torch.Tensor, lo: int = 4, hi: int = 32) -> float:\n    """EXCESS спектральной плотности в полосе периодов [lo,hi] против вне-полосы.\n\n    =1 белый шум; ≫1 — периодическая компонента на этих периодах (детектор орбиты).\n    Доля-в-полосе не годится: полоса покрывает почти весь спектр короткого ряда, и даже\n    чистая синусоида даёт ≈1. Поэтому делим среднюю плотность В полосе на среднюю ВНЕ\n    (пол с 1e-6 от общей плотности — защита от деления на ноль у чистой синусоиды).\n    Полоса по умолчанию 4..32: ниже 4 — токсично (токенизированные повторы букв),\n    выше 32 — ловим тренды вместо орбит."""\n    x = x - x.mean()\n    if x.numel() < 16 or float(x.pow(2).sum()) < 1e-20:\n        return float("nan")\n    f = torch.fft.rfft(x.double())\n    pw = (f.real ** 2 + f.imag ** 2)[1:]        # bin k <-> период N/k, DC выброшен\n    n = x.numel()\n    k_lo, k_hi = max(1, n // hi), min(pw.numel() - 1, n // lo)\n    mask = torch.zeros(pw.numel(), dtype=torch.bool)\n    mask[k_lo:k_hi + 1] = True\n    d_in = float(pw[mask].mean())\n    d_out = float(pw[~mask].mean())\n    d_tot = float(pw.mean())\n    return d_in / max(d_out, 1e-6 * d_tot)\n\n\ndef autocorr_peak(x: torch.Tensor, p: int, base_slice=slice(5, None)) -> float:\n    """ACF(x)[p] относительно медианы |ACF| хвоста — пик/база (детектор периода p).\n\n    ACF нормирована на полную энергию (смещённая оценка — ряд короткий, monotone затухание\n    вычитается базой медианы хвоста)."""\n    xf = (x - x.mean()).double()\n    if xf.numel() < 3 * p:\n        return float("nan")\n    n = xf.numel()\n    e0 = float((xf * xf).sum())\n    if e0 < 1e-20:\n        return float("nan")\n    acf = torch.tensor([float((xf[: n - k] * xf[k:]).sum() / e0)\n                        for k in range(1, min(n - 1, 3 * p + 1))])\n    base = acf[base_slice].abs().median()\n    return float(acf[p - 1] / base) if base == base and float(base) > 1e-9 else float("nan")\n\n\n# ---------------------------------------------------------------- детектор петель\n\ndef find_loops(tokens: list[int], max_p: int = 32, min_reps: int = 3):\n    """Детерминированный детектор повторов (пререгистрация docs/leaf_dynamics.md §5.2).\n\n    Эпизод: позиция j и ФУНДАМЕНТАЛЬНЫЙ период p (минимальный из подходящих), при котором\n    блок tokens[j:j+p] повторяется k>=min_reps раз подряд вперёд ( maximal\'ный прогон).\n    onset = j + (min_reps-1)*p — начало ТРЕТЬЕГО повтора («петля объявила себя»). Эпизоды\n    с одинаковым onset склеиваются; в выход кладётся максимальный прогон по каждому onset.\n    Возвращает [(onset, start=j, end, p, k)].\n    """\n    n = len(tokens)\n    raw: dict[int, tuple[int, int, int, int]] = {}\n    for j in range(n - 2):\n        for p in range(2, max_p + 1):\n            block = tokens[j: j + p]\n            if len(block) < p:\n                break\n            k = 1\n            e = j + p\n            while e + p <= n and tokens[e: e + p] == block:\n                k += 1\n                e += p\n            if k >= min_reps and p * (k - 1) >= 6:   # подавление тривиальных «aa», «aab aab»\n                cand = (j, e, p, k)\n                prev = raw.get(j)\n                if prev is None or p * k > prev[2] * prev[3]:\n                    raw[j] = cand\n    out = sorted((j + (min_reps - 1) * cand[2], j, cand[1], cand[2], cand[3])\n                 for j, cand in raw.items())\n    # склейка: эпизод, начинающийся внутри более длинного с тем же периодом, отбрасывается\n    merged: list[tuple[int, int, int, int, int]] = []\n    for ep in sorted(out, key=lambda x: (x[1], -x[4] * x[3])):\n        if merged and merged[-1][1] <= ep[1] < merged[-1][2] and merged[-1][3] == ep[3]:\n            continue\n        merged.append(ep)\n    return merged\n', encoding='utf-8')
p = pathlib.Path(ROOT, 'experiments/exp49_leaf_info.py'); p.parent.mkdir(parents=True, exist_ok=True)
p.write_text('#!/usr/bin/env python\n"""E-49: НЕСЁТ ЛИ ЛИСТ ИНФОРМАЦИЮ О ТРАЕКТОРИИ — размороженный протокол с контролями.\n\nКонтекст: тождество J и «облако» V-рам воспроизводятся на двух архитектурах (E-48), но\nпопарные эвклидовы метрики r=16 оказались null-эквивалентны (§10). Проверяем три гипотезы\nо том, что МЫ МЕРИЛИ НЕ ТАК, каждая — несколькими методами:\n\n  H1 «не тот top»     — r ∈ {1,2,4} (верхние моды, σ1/σ2≈2.3) vs 16 (вырожденный хвост).\n  H2 «не та гранулярн.»— сглаживание h по окну W ∈ {1,4,16} открывает медленный канал.\n  H3 «не та метрика»  — углы в pullback-метрике ридаута G = AᵀA (M-метрика, G-ортонорм.\n                        базисы через L=chol(G)).\n\nМетоды на каждом (r, W, метрика):\n  M1 лаг-кривая s(lag), lag ∈ {1,2,4,8,16,32} (full-length серии, без прореживания);\n  M2 когерентность τ = [s(t,t+1)+s(t+1,t+2)−s(t,t+2)]/s(t,t+1)  (0 = связное движение,\n     1 = белое; главный дискриминант, работает даже когда лаг-кривая насыщена);\n  M3 голономия окна L=8 — НЕ ЗАПУСКАЛАСЬ в production-сетке (функция holonomy()\n     есть, не вызывалась; исправлено по ревью E49 §1.2 — заявка снята);\n  M4 v1-угол со знакомым транспортом (r=1 чувствителен к знаку — фиксируем трассировкой).\n\nКОНТРОЛИ (падение этапа, если не различились):\n  C-A white u (независимые направления через тот же конвейер): τ≈1, лаг flat;\n  C-B coherent u (шаги δ=0.02 по геодезическим, медленно поворачивающаяся плоскость):\n      τ<0.5, лаг-кривая растёт.\n\nЗапуск: --stage 1 (только контроли, быстро) → --stage 2 (данные) → обе модели.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nimport sys\n\nimport numpy as np\nimport torch\n\n_HERE = os.path.dirname(os.path.abspath(__file__))\nif _HERE not in sys.path:\n    sys.path.insert(0, _HERE)\nimport exp47_common as C  # noqa: E402\n\nEPS_LFM, EPS_QWEN = 1e-5, 1e-6\nRNG = np.random.default_rng(47)\nLAGS = (1, 2, 4, 8, 16, 32)\n\n\ndef med(x):\n    x = np.asarray(x, float)\n    x = x[np.isfinite(x)]\n    return round(float(np.median(x)), 4) if len(x) else None\n\n\ndef boot_ci(v, B=2000):\n    v = np.asarray(v, float); v = v[np.isfinite(v)]\n    if len(v) < 3:\n        return [None, None]\n    # seed фиксирован НА ВЫЗОВ (ревью §2.6): CI не зависит от порядка вызовов\n    idx = np.random.default_rng(47).integers(0, len(v), (B, len(v)))\n    m = np.median(v[idx], axis=1)\n    return [round(float(np.quantile(m, 0.025)), 4), round(float(np.quantile(m, 0.975)), 4)]\n\n\n# ------------------------------------------------------------------ каркас метрик\n\ndef frame_in_metric(J, r, L):\n    """Базис top-r инвариантного подпространства JᵀJ, G-ортонормированный.\n    Фикс (ревью E49 §1.1): torch.linalg.cholesky даёт G = L·Lᵀ (НЕ LᵀL), значит\n    ‖x‖_G = ‖Lᵀx‖₂ и G-ортонорм — qr(LᵀV), а не qr(LV). Возвращаемый базис —\n    Lᵀ·span(V) (НЕ span(V)); углы между подпространствами в G-метрике равны\n    евклидовым углам между Lᵀ-образами — это то, зачем встроивание нужно.\n    ПРЕЖНИЕ M-ячейки сетки\n    exp49_leaf_info_*.json считали метрику LᵀL ≠ G — НЕИНТЕРПРЕТИРУЕМЫ (евклидовы\n    H2-выводы exp49b2 не затронуты: там L=None)."""\n    V = C.V16_from_J(J, r=r)[0]                       # [D,r] ортонорм (евклид)\n    if L is None:\n        return V\n    Wm = L.T @ V\n    return torch.linalg.qr(Wm)[0]\n\n\ndef tau_triplets(Vs):\n    a = [C.frame_kinematics(Vs[t], Vs[t + 1])[0] for t in range(len(Vs) - 2)]\n    b = [C.frame_kinematics(Vs[t + 1], Vs[t + 2])[0] for t in range(len(Vs) - 2)]\n    c = [C.frame_kinematics(Vs[t], Vs[t + 2])[0] for t in range(len(Vs) - 2)]\n    return [(x + y - z) / x for x, y, z in zip(a, b, c) if x > 1e-6]\n\n\ndef v1_signed_series(us, A, eps):\n    out, prev = [], None\n    for u in us:\n        J, _, _ = C.J_analytic(A, u, eps)\n        G = J @ J.T; G = 0.5 * (G + G.T)\n        lam, Ue = torch.linalg.eigh(G)\n        v1 = J.T @ Ue[:, -1]\n        v1 = v1 / v1.norm()\n        if prev is not None and float(v1 @ prev) < 0:\n            v1 = -v1\n        out.append(v1); prev = v1\n    return out\n\n\ndef v1_lag(v1s):\n    return {L: [float(torch.acos((v1s[t] @ v1s[t + L]).abs().clamp(max=1 - 1e-12)))\n                for t in range(len(v1s) - L)] for L in LAGS}\n\n\ndef holonomy(Vs, L=8):\n    out = []\n    r = Vs[0].shape[1]\n    for t in range(len(Vs) - L):\n        P = torch.eye(r, dtype=torch.float64)\n        for k in range(L):\n            P = P @ C.procrustes_R(Vs[t + k], Vs[t + k + 1])\n        P = C.procrustes_R(Vs[t + L], Vs[t]) @ P\n        th, _, _ = C.schur_angles(P)\n        out.append(float(torch.sqrt((th * th).sum())))\n    return out\n\n\ndef doc_stats(Vs, v1s=None):\n    d = {"s": {str(L): [C.frame_kinematics(Vs[t], Vs[t + L])[0] for t in range(len(Vs) - L)]\n               for L in LAGS},\n         "tau": tau_triplets(Vs)}\n    if v1s is not None:\n        d["v1"] = {str(L): v for L, v in v1_lag(v1s).items()}\n    return d\n\n\ndef agg(docss):\n    out = {"s_lag": {str(L): med([x for d in docss for x in d["s"][str(L)]]) for L in LAGS},\n           "tau": med([x for d in docss for x in d["tau"]])}\n    if docss and "v1" in docss[0]:\n        out["v1_lag"] = {str(L): med([x for d in docss for x in d["v1"][str(L)]]) for L in LAGS}\n    # bootstrap по документам на τ и на s(1)\n    taus = [med(d["tau"]) for d in docss]\n    out["tau_doc_boot95"] = boot_ci(taus)\n    return out\n\n\n# ------------------------------------------------------------------ контроли\n\ndef nz(x):\n    return x / x.norm()\n\n\ndef control_white(T, D):\n    g = torch.Generator().manual_seed(7)\n    return [nz(torch.randn(D, dtype=torch.float64, generator=g))\n            for _ in range(T)]\n\n\ndef control_coherent(T, D, delta=0.02):\n    """ГЕОДЕЗИЧЕСКИЙ контроль (фикс ревью E49-4 §2.1 — та же ортогонализация, что в\n    synth_coh exp49b2; прежняя версия без w⟂u давала невалидный CB)."""\n    g = torch.Generator().manual_seed(11)\n    u = nz(torch.randn(D, dtype=torch.float64, generator=g))\n    w = nz(torch.randn(D, dtype=torch.float64, generator=g))\n    w = nz(w - u * (u @ w))\n    out = []\n    for t in range(T):\n        out.append(u.clone())\n        u2 = math.cos(delta) * u + math.sin(delta) * w\n        u = nz(u2)\n        z = nz(torch.randn(D, dtype=torch.float64, generator=g))\n        w = nz(w + 0.05 * (z - (z @ u) * u - (z @ w) * w))\n        w = nz(w - u * (u @ w))\n    return out\n\n\ndef run_stage1(A, eps, D, T=96):\n    res = {}\n    for name, us in [("C-A_white", control_white(T, D)), ("C-B_coherent", control_coherent(T, D))]:\n        Vs = [frame_in_metric(C.J_analytic(A, u, eps)[0], 16, None) for u in us]\n        v1s = v1_signed_series(us, A, eps)\n        d = doc_stats(Vs, v1s)\n        res[name] = {"s_lag": {k: med(v) for k, v in d["s"].items()},\n                     "tau": med(d["tau"]),\n                     "v1_lag": {k: med(v) for k, v in d["v1"].items()}}\n    ok = (res["C-A_white"]["tau"] > 0.7 and res["C-B_coherent"]["tau"] < 0.5\n          and res["C-B_coherent"]["s_lag"]["16"] and res["C-B_coherent"]["s_lag"]["1"]\n          and res["C-B_coherent"]["s_lag"]["16"] > res["C-B_coherent"]["s_lag"]["1"])\n    res["SELFTEST"] = "ok" if ok else "ВАЛИТСЯ: конвейер не различает белое/связное"\n    return res\n\n\n# ------------------------------------------------------------------ данные\n\ndef smoothed_us(u, nrm, W):\n    """Сглаживание шагов (причинное окно). ИЗВЕСТНОЕ ОГРАНИЧЕНИЕ (ревью E49 §2.7):\n    на первых W−1 позициях окно короче — краевой эффект ~(W−1)/T (~4% при W=16 на\n    T=384); в исторических прогонах оставлено как есть; новые скрипты (e52) должны\n    отбрасывать первые W−1 или брать центрированное окно."""\n    h = u * torch.tensor(nrm, dtype=torch.float64).unsqueeze(1)\n    if W > 1:\n        hs = torch.stack([h[max(0, t - W + 1):t + 1].mean(0) for t in range(len(h))])\n    else:\n        hs = h\n    return [hs[t] / hs[t].norm() for t in range(len(hs))]\n\n\ndef load_model(args):\n    from transformers import AutoConfig, AutoModelForCausalLM  # type: ignore\n    if args.model == "qwen":\n        mp = os.environ.get("PRISM_QWEN_DIR", "/data/Qwen/Qwen3.5-0.8B")\n        ck = torch.load(os.environ.get("PRISM_JCACHE",\n                        "/data/prism_kernels/exp27b/results/exp27b_jcache_L23.pt"),\n                        map_location="cpu", weights_only=False)\n        U = ck["U128"]\n        U = U.tolist() if torch.is_tensor(U) else list(U)\n        cfg = AutoConfig.from_pretrained(mp)\n        m = AutoModelForCausalLM.from_pretrained(mp, dtype=torch.float32).eval()\n        g = 1.0 + m.model.norm.weight.detach().float()\n        npz, jp, eps = (os.environ.get("PRISM_NPZ",\n                        "/data/prism_kernels/exp47b/exp47b_gen_kinematics.npz"),\n                        os.environ.get("PRISM_DOCS_JSON",\n                                       "results/exp47b_gen_kinematics.json"), EPS_QWEN)\n    else:\n        from transformers.cache_utils import LinearAttentionLayer  # type: ignore\n        mp = os.environ.get("PRISM_LFM3_DIR", "/data/LiquidAI/LFM3-1.7B")\n        cfg = AutoConfig.from_pretrained(mp, trust_remote_code=True)\n        cfg.layer_types = list(json.load(open(os.path.join(mp, "config.json")))["layer_types"])\n        _o = LinearAttentionLayer.update_conv_state\n        LinearAttentionLayer.update_conv_state = (\n            lambda self, cs, si=0, **kw: _o(self, cs.squeeze(1) if cs.ndim == 4 else cs, si, **kw))\n        m = AutoModelForCausalLM.from_pretrained(mp, config=cfg, dtype=torch.float32,\n                                                 trust_remote_code=True).eval()\n        g = m.model.embedding_norm.weight.detach().float()\n        U = list(range(0, 384, 3))[:128]\n        npz, jp, eps = ("/data/prism_kernels/exp48b/exp48b_lfm3_gen_kinematics.npz",\n                        "results/exp48b_lfm3_gen_kinematics.json", EPS_LFM)\n    W_head = m.get_input_embeddings().weight.detach()\n    # tied-проверка универсальная (ревью E49 §1.4/1.6): у LFM3-1.7B\n    # tie_word_embeddings=True и lm_head в чекпоинте ОТСУТСТВУЕТ (проверено\n    # 2026-11-06 по config/safetensors) — head ≡ embeddings, A был верным;\n    # для untied-моделей берём output head.\n    head = getattr(m, "lm_head", None)\n    if head is not None and head.weight.data_ptr() != W_head.data_ptr():\n        W_head = head.weight.detach()\n    A = C.head_M(W_head.float()[U], g)\n    load_model.last_g = g          # для e52a: пересборка A из изменённых строк head\n    return A, U, eps, npz, jp\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--stage", type=int, default=2)\n    ap.add_argument("--model", choices=("qwen", "lfm3"), default="qwen")\n    args = ap.parse_args()\n    root = os.path.join(_HERE, "..")\n    A, U, eps, npz, jp = load_model(args)\n    D = int(A.shape[1])\n    out = {"model": args.model, "U_head": U[:6], "eps": eps}\n\n    if args.stage == 1:\n        out["controls"] = run_stage1(A, eps, D)\n        print("CONTROLS:", json.dumps(out["controls"], indent=1), flush=True)\n        if out["controls"]["SELFTEST"] != "ok":\n            sys.exit("selftest failed")\n\n    if args.stage >= 2:\n        G = A.T @ A\n        Lch = torch.linalg.cholesky(G + 1e-10 * torch.eye(D, dtype=torch.float64) * float(G.diag().mean()))\n        docs = json.load(open(os.path.join(root, jp)))["docs"]\n        z = np.load(npz, allow_pickle=True)\n        ud = {}\n        for doc in docs:\n            f = z[f"{doc[\'dom\']}_{doc[\'si\']}"].item()\n            ud[f"{doc[\'dom\']}_{doc[\'si\']}"] = (\n                torch.from_numpy(z[f"{doc[\'dom\']}_{doc[\'si\']}_u"].astype(np.float64)),\n                np.asarray(f["hnorm"], dtype=float))\n        rep = {}\n        # стриминг: один (W, doc) в памяти; для него eig один раз, все (r,metric) из него\n        comb = {}\n        for W in (1, 4, 16):\n            for key, (u, nrm) in ud.items():\n                us = smoothed_us(u, nrm, W)\n                eig = []\n                for x in us:\n                    J, _, _ = C.J_analytic(A, x, eps)\n                    Gm = J @ J.T; Gm = 0.5 * (Gm + Gm.T)\n                    lam, Ue = torch.linalg.eigh(Gm)\n                    eig.append((J, Ue))\n                for r in (1, 2, 4, 16):\n                    for metric in ("eucl", "M"):\n                        Vs = []\n                        for (J, Ue) in eig:\n                            V = J.T @ Ue[:, -r:]\n                            V = V / V.norm(dim=0, keepdim=True)\n                            V = torch.linalg.qr(V)[0]\n                            if metric == "M":\n                                # ревью E49-2 §2.1: cholesky даёт G=L·Lᵀ ⇒ G-ортонорм\n                                # = qr(LᵀV); инлайн-ветка ранее крутила метрику LᵀL\n                                V = torch.linalg.qr(Lch.T @ V)[0]\n                            Vs.append(V)\n                        comb.setdefault(f"W{W}_r{r}_{metric}", []).append(doc_stats(Vs, None))\n                del eig\n            print(f"W{W} done", flush=True)\n        for k, ds in comb.items():\n            rep[k] = agg(ds)\n            print(f"{k}: tau={rep[k][\'tau\']} s1={rep[k][\'s_lag\'][\'1\']} "\n                  f"s32={rep[k][\'s_lag\'][\'32\']}", flush=True)\n        # v1-метрики отдельно (W=1): угол между v1 со знакомым транспортом\n        v1ds = {}\n        for key, (u, nrm) in ud.items():\n            us = smoothed_us(u, nrm, 1)\n            v1ds[key] = v1_signed_series(us, A, eps)\n        rep["v1"] = {str(L): med([float(torch.acos((v1ds[k][t] @ v1ds[k][t + L]).abs().clamp(max=1 - 1e-12)))\n                                 for k in v1ds for t in range(len(v1ds[k]) - L)]) for L in LAGS}\n        out["grid"] = rep\n\n    path = os.path.join(root, "results", f"exp49_leaf_info_{args.model}.json")\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    json.dump(out, open(path, "w"), indent=1)\n    print("WROTE", path)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('[bundle] wrote 34 files under', ROOT)


In [ ]:
# ---------- driver ----------
# ---------- driver ----------
# E-52k driver (DRIVER MODE, инлайн-бандл, без датасетов).
# Кривая τ(size) на GPU bf16: 0.8B → 2B → 4B (+повтор 0.8B как якорь протокола)
# и честный PTQ с HF-весов: bnb-int8 на 4B и 9B (7B в семье Qwen3.5 нет — 9B
# и есть «третий-крупный»; bf16-9B в T4 не влезает, int8 влезает).
import os, sys, time, glob, json, shutil, traceback, gc, subprocess
t0 = time.time()
os.environ.setdefault("OMP_NUM_THREADS", "3")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ["PRISM_SKIP_NORM_TEST"] = "1"  # семантика g проверена локально (E-47b)
os.environ["HF_HOME"] = "/kaggle/working/.hf"          # staging под local_dir — в working, не в home
os.environ["HF_HUB_CACHE"] = "/kaggle/working/.hf/hub"
CODE = "/kaggle/working/code"
sys.path.insert(0, CODE)
sys.path.insert(0, os.path.join(CODE, "experiments"))
try:
    subprocess.run(["pip", "install", "-q", "-U", "transformers", "bitsandbytes"], check=False)
    from huggingface_hub import snapshot_download
    import importlib.util, torch
    def complete(p):
        import re
        ss = [f for f in os.listdir(p) if f.endswith(".safetensors")]
        if not ss:
            return False
        mt = [re.search(r"-(\d+)-of-(\d+)\.safetensors$", f) for f in ss]
        if any(x is None for x in mt):
            return True
        tot = {int(x.group(2)) for x in mt}
        have = {int(x.group(1)) for x in mt}
        return len(tot) == 1 and have == set(range(1, tot.pop() + 1))
    def load(mod, fn):
        spec = importlib.util.spec_from_file_location(mod, os.path.join(CODE, "experiments", fn))
        X = importlib.util.module_from_spec(spec)
        sys.modules[mod] = X
        spec.loader.exec_module(X)
        return X
    E47 = load("e47", "exp47b_gen_kinematics.py")
    # PRISM-каналы: jcache синтезируем из литерала U128 (бинарник не нужен)
    d08 = "/kaggle/working/m_Qwen3.5-0.8B"
    if not glob.glob(d08 + "/*.safetensors"):
        prev = [p for p in glob.glob("/kaggle/input/**/m_Qwen3.5-0.8B", recursive=True) if complete(p)]
        if prev:
            d08_src = prev[0]
        else:
            d08_src = snapshot_download("Qwen/Qwen3.5-0.8B", local_dir=d08)
    os.environ["PRISM_QWEN_DIR"] = d08 if glob.glob(d08 + "/*.safetensors") else d08_src
    jc = "/kaggle/working/U128_synth.pt"
    torch.save({"U128": list(E47.U128)}, jc)
    os.environ["PRISM_JCACHE"] = jc
    print("[env] QWEN_DIR:", os.environ["PRISM_QWEN_DIR"], flush=True)
    E52 = load("e52", "exp52a_perturb_axis.py")   # после env (load_model читает при вызове)
    JOBS = [  # (tag, hf_repo, dtype, bnb-int8, последний заход за этой моделью)
        ("q08_bf16", "Qwen/Qwen3.5-0.8B", "bfloat16", False, True),
        ("q2_bf16",  "Qwen/Qwen3.5-2B",   "bfloat16", False, True),
        ("q4_bf16",  "Qwen/Qwen3.5-4B",   "bfloat16", False, False),
        ("q4_int8",  "Qwen/Qwen3.5-4B",   "bfloat16", True,  True),
        ("q9_int8",  "Qwen/Qwen3.5-9B",   "bfloat16", True,  True),
    ]
    for tag, repo, dt, q8, last in JOBS:
        try:
            name = repo.split("/")[-1]
            md = f"/kaggle/working/m_{name}"
            if not glob.glob(md + "/*.safetensors"):
                prev = [p for p in glob.glob(f"/kaggle/input/**/m_{name}", recursive=True) if complete(p)]
                md = prev[0] if prev else snapshot_download(repo, local_dir=md)
            rd = f"/kaggle/working/r52k_{tag}"; os.makedirs(rd, exist_ok=True)
            print(f"\n##### {tag} ({repo}, {dt}{', bnb-int8' if q8 else ''})", flush=True)
            E47.main(model=md, n_docs=4, max_new=160, device="cuda:0", results_dir=rd,
                     do_verify_basis=False, dtype=dt, load_in_8bit=q8)
            sys.argv = ["e52a", "--model", "qwen", "--n-docs", "16", "--model-dir", md,
                        "--families", "rtn",
                        "--npz", rd + "/exp47b_gen_kinematics.npz",
                        "--docs-json", rd + "/exp47b_gen_kinematics.json",
                        "--out", f"/kaggle/working/exp52k_tau_{tag}.json"]
            E52.main()
            if last and name != "Qwen3.5-0.8B" and md.startswith("/kaggle/working"):
                shutil.rmtree(md, ignore_errors=True)   # освобождаем диск под следующий размер
            shutil.rmtree("/kaggle/working/.hf", ignore_errors=True)  # staging-дубли снапшотов
            free = shutil.disk_usage("/kaggle/working").free / 2**30
            print(f"[disk] free {free:.1f} GiB", flush=True)
            gc.collect(); torch.cuda.empty_cache()
        except Exception:
            print(f"##### {tag} FAILED", flush=True)
            traceback.print_exc()
    print("\n===== SUMMARIES =====", flush=True)
    for f in sorted(glob.glob("/kaggle/working/exp52k_tau_*.json")):
        s = json.load(open(f))["summary"]
        print(f.split("exp52k_tau_")[1], "base:", s["base"]["tau"], flush=True)
except Exception:
    traceback.print_exc()
print("DONE %.1fs" % (time.time() - t0))
